# 05 — Historical experiment source and replay

This notebook completes the source record for the report's rejected hypotheses, alongside the four core notebooks. It includes the full migrated source in the appendix, the exact archive/member/original hashes, and callable replay entries. Default execution reads retained results and compiles/imports the code. It **does not train**.

Original experiment settings and model-selection logic are preserved. Filesystem paths and entry points are adapted to the current layout. Optional replay writes only to a new explicit output directory and never promotes or overwrites the selected submission. No historical experiment was rerun during this source restoration; matching historical numerical results after an optional refit is not claimed.


In [2]:
from pathlib import Path
import sys, json
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p/'models/selection.json').is_file() and (p/'tools/src').is_dir())
for directory in (ROOT/'tools/src', ROOT/'tools/src/historical'):
    if str(directory) not in sys.path:
        sys.path.insert(0, str(directory))
import pandas as pd
from IPython.display import display
import replay
RUN_TRAINING = False
RUN_HISTORICAL_TRAINING = False
print('Historical source review; training:', RUN_HISTORICAL_TRAINING)


## Verify source availability and dependencies

The check below compiles and imports all restored runners, checks their migration hashes, checks the small recovered input files against their original hashes, and confirms the saved reference files are present. It does not fit an estimator. `local_selection` is the already included nested local-model study; the other ten entries are restored historical runners.


In [3]:
verification = replay.check(ROOT)
display(verification)
provenance = json.loads((ROOT/'tools/src/historical/source_provenance.json').read_text())
display(pd.DataFrame([{'module': name, 'archive': rec['nested_archive'],
                       'member': rec['member'], 'original_sha256': rec['original_sha256'],
                       'ported_sha256': rec['ported_sha256']}
                      for name, rec in provenance['source_files'].items()]))


{'status': 'PASS',
 'ported_runners': 10,
 'replay_studies': 11,
 'training_performed': False,
 'check_scope': 'compile/import, source hashes, required saved inputs; no experimental refits'}

,module,archive,member,original_sha256,ported_sha256
0,midterm_graph_refinement,archive/midterm_before_cleanup_2026-11-02.zip,midterm/midterm_graph_refinement.py,e91a5c403fb9d8dc6d9fec3a98949044135e9b49a99597...,0cf800ef7722898f4ae6a5b1e1654ce3f0e0cabb4820f5...
1,midterm_rank_part1,archive/midterm_before_cleanup_2026-11-02.zip,midterm/midterm_rank_part1.py,151f2c2cba72a8a834191e92764caf663bb988eb2fa4f3...,972b59415afd73c0d0caf5ecd3851367fff3becd628793...
2,midterm_part1_round2,archive/midterm_before_cleanup_2026-11-02.zip,midterm/midterm_part1_round2.py,3cf21506640e06895a2442c8732f460d08d75fb3482b70...,c8550d6744e7cbdb02f287dc8443b08888413f092d3afb...
3,midterm_part1_label_shift,archive/midterm_before_cleanup_2026-11-02.zip,midterm/midterm_part1_label_shift.py,e1c8a5d7d787e225ddd73a9e002dcc52cbf879d7bdd027...,ce8df2a12be4349d812047955423ba071ad9b3a8555f32...
4,midterm_part1_other_views,archive/midterm_before_cleanup_2026-11-02.zip,midterm/midterm_part1_other_views.py,11eaa037a272c2f88e0864420ba9e4d4d97916281eed4b...,c7dd870cdcec879a1f86311c5b51e499fd4b7e076d2918...
5,midterm_part1_mixture_followup,archive/midterm_before_cleanup_2026-11-02.zip,midterm/midterm_part1_mixture_followup.py,7dd18ac24c2bb1e3a60ada09db9d93cc21fa55665496e8...,d4d6ce1092b788292d287dd157d31b419b74b5079f36b5...
6,midterm_part1_val_refit_weight,archive/midterm_before_cleanup_2026-11-02.zip,midterm/midterm_part1_val_refit_weight.py,c794e8982181d2b529fb2146a85788e8b6225bf26e1f40...,0f83e26ec643f2ac3b86ee075cec8db657fabfd8e609ba...
7,midterm_part1_parity,archive/midterm_before_report_cleanup_2026-11-...,midterm/midterm_part1_parity.py,eaec41010ec6834b831e2457e029f84b0e1ee6548be092...,a6c7eb905172acebcba8aa781979ab43d72ebb97e1191d...
8,midterm_part1_local_followup,archive/midterm_before_report_cleanup_2026-11-...,midterm/midterm_part1_local_followup.py,03652f39d3f9bbaee479028f1a96438197dc4d0c0737ab...,b4580da93ade55cf8fd96f126a96638ac9122b847d4506...
9,midterm_part1_ridge_boundary,archive/midterm_before_report_cleanup_2026-11-...,midterm/midterm_part1_ridge_boundary.py,f8d8e3b45528e71906613b84d18a2bc3d4cf102a66f6dd...,6e167a7196f9086db7ade215b3806d524fed7601a87947...


## Recorded experimental results

These are historical observations with different fitting and validation protocols; the rows are not a common benchmark. Table 6's graph/soft labels, primary-secondary/whitening, label shift, Other/text, local follow-up, and ridge-boundary studies have replayable sources below. The parity, small-mixture, and validation-weight studies preserve additional diagnostic history.


In [4]:
summaries = replay.recorded_summaries(ROOT)
display(pd.DataFrame([{'study': name, 'source_module': module,
                       'recorded_summary': summary, 'source_callable': function}
                      for name, (module, function, summary) in replay.STUDIES.items()]))
for name, summary in summaries.items():
    print('\nRECORDED:', name)
    display(summary)


,study,source_module,recorded_summary,source_callable
0,graph,midterm_graph_refinement,models/part1/exploration/graph/summary.json,run_geometry
1,selective_pseudo,midterm_rank_part1,models/part1/exploration/selective_pseudo/summ...,run
2,structure_geometry,midterm_part1_round2,models/part1/exploration/structure_geometry/su...,run
3,label_shift,midterm_part1_label_shift,models/part1/exploration/label_shift/backtest_...,run
4,other_views,midterm_part1_other_views,models/part1/exploration/other_views/decision....,run
5,mixture_followup,midterm_part1_mixture_followup,models/part1/exploration/structure_geometry/mi...,run
6,val_refit_weight,midterm_part1_val_refit_weight,models/part1/exploration/structure_geometry/va...,run
7,parity,midterm_part1_parity,models/part1/02_kmeans_lr/evidence/baseline_pa...,run
8,local_followup,midterm_part1_local_followup,models/part1/exploration/local_followup/summar...,run
9,ridge_boundary,midterm_part1_ridge_boundary,models/part1/exploration/ridge_boundary/summar...,run


{'graph_cv_winner': 'p1geo_graph_lr_k15_v1',
 'knn_cv_winner': 'p1geo_knn_k5_v1',
 'graph_validation_f1': 0.8931623931623932,
 'current_validation_f1': 0.8782051282051282,
 'improved_official_validation': True,
 'selected_submission': 'p1geo_graph_lr_k15_v1',
 'test_used': False}

{'selective_pseudo': {'winner': 2,
  'setting': {'positive_cutoff': 0.5,
   'negative_cutoff': 0.2,
   'pseudo_weight': 0.03},
  'oof_control_f1': 0.8895966029723991,
  'oof_candidate_f1': 0.8337579617834393,
  'validation_control_f1': 0.8782051282051282,
  'validation_candidate_f1': 0.8021367521367521,
  'passes_local_gate': False,
  'final_replaced': False,
  'kaggle_uploaded': False},
 'thresholds': {'selected_thresholds': {'Safety': 0.25,
   'Funding': 0.35,
   'Efficacy': 0.3,
   'Recruitment': 0.35,
   'Other': 0.3},
  'oof_control_f1': 0.8895966029723991,
  'oof_candidate_f1': 0.8874734607218684,
  'validation_control_f1': 0.8782051282051282,
  'validation_candidate_f1': 0.856837606837607,
  'passes_local_gate': False,
  'final_replaced': False,
  'kaggle_uploaded': False}}

{'selected': 'control',
 'oof_f1': 0.8895966029723991,
 'time_f1': 0.8416666666666666,
 'pre_refit_validation_f1': 0.8782051282051282,
 'frozen_validation_f1': 0.8782051282051282,
 'refit_gold_count': 235,
 'test_predicted_multi': 66,
 'candidate_path': 'refinement/part1_structure_geometry_2026_11_02/part1_refit_235_candidate.csv',
 'candidate_sha256': '94b4a22eb2aa5a1b6c956184f55f223803b06ccaeb426911eb5621b6cae981f1',
 'original_final_replaced': False,
 'kaggle_uploaded': False,
 'warning': 'Validation labels entered final refit; validation score is no longer an independent estimate for that fitted model.'}

{'baseline_f1': 0.8392857142857143,
 'train_only_early_baseline_f1': 0.869047619047619,
 'best': {'alpha': 1.0,
  'sample_f1': 0.8452380952380951,
  'gain_vs_alpha0': 0.00595238095238082,
  'recruitment_f1': 0.9767441860465116,
  'changed_decision_rows': 1,
  'positive_row_deltas': 1,
  'negative_row_deltas': 0,
  'predicted_multi': 20},
 'recruitment_source_prior': 0.7329545454545454,
 'recruitment_em_target_prior': 0.5479472360959892,
 'recruitment_bbse_target_prior': 0.6071672362169077,
 'em_bbse_difference': 0.05922000012091855,
 'gates': {'em_converged': True,
  'bbse_identifiable': True,
  'bbse_raw_feasible': True,
  'em_bbse_agree_within_0p10': True,
  'f1_gain_at_least_0p01': False,
  'beats_train_only_baseline_by_0p01': False,
  'at_least_three_positive_rows': False,
  'nonzero_alpha_wins': True},
 'passed': False,
 'test_predictions_touched': False,
 'note': '56 hidden labels were read only after priors and target probabilities were saved.'}

{'selected': None,
 'eligible': [],
 'thresholds': {'residual': 0.85, 'text': 0.45},
 'no_new_model_if_failed': True,
 'kaggle_uploaded': False,
 'original_final_replaced': False,
 'label_changes': 0,
 'co_training_attempted': False,
 'reason_for_no_co_training': 'require evidence of complementary text errors before adding pseudo labels'}

... (2 outputs left out)

{'protocol': {'seed': 42,
  'folds': 4,
  'unique_gold_embedding_groups': 150,
  'oof_held_features_excluded_from_all_fits': True,
  'temporal_fit': 'train rows start_year <= 2006 only',
  'temporal_primary': '2007-2009 observed labels in train CSV only',
  'temporal_56_row_check': 'includes previously inspected validation labels; diagnostic only',
  'fallback': 'argmax reason on every empty predicted set',
  'historical_thresholds': {'supervised': 0.5, 'semi': 0.35},
  'common_thresholds': [0.35, 0.5],
  'full_refit_max_absolute_validation_probability_gap': {'supervised': 5.21249710061511e-14,
   'semi': 0.0}},
 'validation': {'metrics': {'supervised': {'n': 78,
    'sample_f1': 0.8782051282051284,
    'exact_sets': 61,
    'per_reason': {'Safety': {'support': 9,
      'predicted': 12,
      'precision': 0.75,
      'recall': 1.0,
      'f1': 0.8571428571428571},
     'Funding': {'support': 4,
      'predicted': 3,
      'precision': 0.3333333333333333,
      'recall': 0.25,
      'f1

{'overall': {'baseline': {'sample_f1': 0.9026004728132389,
   'exact_set_rate': 0.8581560283687943},
  'ensemble': {'sample_f1': 0.9026004728132389,
   'exact_set_rate': 0.8609929078014185},
  'other': {'sample_f1': 0.9026004728132389,
   'exact_set_rate': 0.8609929078014185},
  'final': {'sample_f1': 0.9002364066193853,
   'exact_set_rate': 0.8482269503546099}},
 'by_repeat': {'0': {'baseline': {'sample_f1': 0.8978723404255319,
    'exact_set_rate': 0.8468085106382979},
   'ensemble': {'sample_f1': 0.8907801418439716,
    'exact_set_rate': 0.8425531914893617},
   'other': {'sample_f1': 0.8907801418439716,
    'exact_set_rate': 0.8425531914893617},
   'final': {'sample_f1': 0.8879432624113475,
    'exact_set_rate': 0.825531914893617}},
  '1': {'baseline': {'sample_f1': 0.899290780141844,
    'exact_set_rate': 0.8595744680851064},
   'ensemble': {'sample_f1': 0.902127659574468,
    'exact_set_rate': 0.8595744680851064},
   'other': {'sample_f1': 0.902127659574468,
    'exact_set_rate': 

{'overall': {'baseline': {'sample_f1': 0.9026004728132389,
   'exact_set_rate': 0.8581560283687943},
  'tuned': {'sample_f1': 0.8983451536643025,
   'exact_set_rate': 0.8539007092198582},
  'final': {'sample_f1': 0.8983451536643025,
   'exact_set_rate': 0.8539007092198582}},
 'by_repeat': {'0': {'baseline': {'sample_f1': 0.8978723404255319,
    'exact_set_rate': 0.8468085106382979},
   'tuned': {'sample_f1': 0.8936170212765957,
    'exact_set_rate': 0.8425531914893617},
   'final': {'sample_f1': 0.8936170212765957,
    'exact_set_rate': 0.8425531914893617}},
  '1': {'baseline': {'sample_f1': 0.899290780141844,
    'exact_set_rate': 0.8595744680851064},
   'tuned': {'sample_f1': 0.8964539007092199,
    'exact_set_rate': 0.8595744680851064},
   'final': {'sample_f1': 0.8964539007092199,
    'exact_set_rate': 0.8595744680851064}},
  '2': {'baseline': {'sample_f1': 0.9106382978723404,
    'exact_set_rate': 0.8680851063829788},
   'tuned': {'sample_f1': 0.9049645390070922,
    'exact_set_ra

{'overall': {'current_fixed': {'sample_f1': 0.8877068557919621,
   'exact_set_rate': 0.8141843971631205},
  'current_retuned': {'sample_f1': 0.8847281323877069,
   'exact_set_rate': 0.8241134751773049},
  'local': {'sample_f1': 0.8964539007092198,
   'exact_set_rate': 0.8425531914893617},
  'after_consensus': {'sample_f1': 0.8964539007092198,
   'exact_set_rate': 0.8425531914893617},
  'final': {'sample_f1': 0.8983451536643027,
   'exact_set_rate': 0.8453900709219858}},
 'by_repeat': {'0': {'current_fixed': {'sample_f1': 0.8794326241134751,
    'exact_set_rate': 0.8},
   'current_retuned': {'sample_f1': 0.8836879432624113,
    'exact_set_rate': 0.8382978723404255},
   'local': {'sample_f1': 0.8822695035460993,
    'exact_set_rate': 0.8170212765957446},
   'after_consensus': {'sample_f1': 0.8822695035460993,
    'exact_set_rate': 0.8170212765957446},
   'final': {'sample_f1': 0.8822695035460993,
    'exact_set_rate': 0.8170212765957446}},
  '1': {'current_fixed': {'sample_f1': 0.8985815

## Explicit optional replay

Choose **one** study from the preceding table, set a new output directory, and only then change `RUN_HISTORICAL_TRAINING` to `True`. Some nested studies may take substantial time. All current model, prediction, data and evidence bytes are protected. Existing output directories are rejected. The replay does not upload to Kaggle or update the selection registry.

Core-family recipes are in `tools/src/train_core.py`; this section covers the historical experiments and the earlier nested local-selection study. A fresh `models/new_experiments/NAME` directory is the recommended destination. Default Run All skips this cell's training branch.


In [5]:
HISTORICAL_STUDY = 'label_shift'
HISTORICAL_OUTPUT = None  # Set e.g. ROOT/'models/new_experiments/label_shift_replay'
if RUN_HISTORICAL_TRAINING:
    if HISTORICAL_OUTPUT is None:
        raise ValueError('Set an explicit, new HISTORICAL_OUTPUT directory before training.')
    display(replay.run(HISTORICAL_STUDY, ROOT, HISTORICAL_OUTPUT))
else:
    print('Historical training skipped; preserved experiment sources and evidence verified only.')


## Full source appendix

These are the migrated `.py` sources used by the callable entries above, not pseudocode. Each original archive member and original SHA-256 is recorded. The ported SHA-256 identifies the included file. Only path/import handling and safe explicit-output entry points changed; fitting and model-selection calculations remain in the restored functions. The archive files themselves are not required to review or execute these sources.


### _historical_support.py

New filesystem/replay support only. SHA-256: `2e0d3fafef571890ce3eccb9016973d26dd2a127658fe9f1a8ec2dc8dbf3993d`

```python
"""Filesystem/import adapters for preserved historical experiments; no model code."""
from pathlib import Path
import sys

CORE = Path(__file__).resolve().parent.parent
if str(CORE) not in sys.path:
    sys.path.insert(0, str(CORE))


def validate_output(root, output):
    """Allow only a new output directory, outside curated data/model/report/tools trees."""
    root, output = Path(root).resolve(), Path(output).resolve()
    if not (root / 'models/selection.json').is_file():
        raise FileNotFoundError('Expected a complete midterm project with models/selection.json.')
    if output.exists():
        raise FileExistsError(f'Historical replay requires a new, nonexistent directory: {output}')
    allowed = root / 'models/new_experiments'
    for name in ('data', 'models', 'report', 'tools'):
        if output.is_relative_to(root / name) and not output.is_relative_to(allowed):
            raise ValueError('Do not write into curated evidence. Use models/new_experiments/NEW_NAME or a new external directory.')
    return root, output


def prepare_output(root, output):
    root, output = validate_output(root, output)
    output.mkdir(parents=True, exist_ok=False)
    return root, output

```


### midterm_graph_refinement.py

Original member: `archive/midterm_before_cleanup_2026-11-02.zip ! midterm/midterm_graph_refinement.py`

Original SHA-256: `e91a5c403fb9d8dc6d9fec3a98949044135e9b49a99597fcf5f7fef97a7308da`

Ported SHA-256: `0cf800ef7722898f4ae6a5b1e1654ce3f0e0cabb4820f50ca70394a361379230`

```python
"""Local-geometry comparison selected by training-only cross-validation."""

from _historical_support import prepare_output
from pathlib import Path
import json,warnings
import joblib
import numpy as np
import pandas as pd
from sklearn.exceptions import ConvergenceWarning
from sklearn.model_selection import StratifiedKFold
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.semi_supervised import LabelSpreading
from threadpoolctl import threadpool_limits

from midterm_augmentation import observed_label_mask,predict_reason_probabilities
from midterm_evaluation import REASONS,dataset_path,evaluate_part1,file_sha256
from midterm_part1_finalization import binary_decisions,build_submission,build_reason_lookup
from midterm_part2_improvement import validate_submission
from midterm_refinement import EMBEDDINGS,fit_binary_targets,save_run,write_json

SETTINGS=[{'method':m,'neighbors':k} for m in ['knn','graph','graph_lr'] for k in [5,15]]

def key(setting):return f'p1geo_{setting["method"]}_k{setting["neighbors"]}_v1'


def safe_positive_probability(probabilities,classes,prior):
    values=np.asarray(probabilities,dtype=float)
    valid=np.isfinite(values).all(axis=1)&(np.nan_to_num(values).sum(axis=1)>0)
    result=np.full(len(values),prior,dtype=float)
    result[valid]=values[valid,list(classes).index(1)]
    return result,~valid


def fit_geometry(frame,setting):
    observed=observed_label_mask(frame[list(REASONS)].to_numpy())
    scaler=StandardScaler().fit(frame[EMBEDDINGS]);x=scaler.transform(frame[EMBEDDINGS])
    y=frame[list(REASONS)].to_numpy();prior=y[observed].mean(axis=0)
    models=[];q=[];iterations=[];fallbacks=[]
    for j in range(5):
        if setting['method']=='knn':
            model=KNeighborsClassifier(n_neighbors=setting['neighbors'],weights='distance',metric='euclidean',n_jobs=1)
            with threadpool_limits(limits=1):model.fit(x[observed],y[observed,j])
        else:
            targets=np.where(observed,y[:,j],-1).astype(int)
            model=LabelSpreading(kernel='knn',n_neighbors=setting['neighbors'],alpha=.2,max_iter=200,tol=1e-4,n_jobs=1)
            with threadpool_limits(limits=1),warnings.catch_warnings(record=True) as recorded:
                warnings.simplefilter('always');model.fit(x,targets)
            if any(issubclass(w.category,ConvergenceWarning) for w in recorded) or model.n_iter_>=200:
                raise RuntimeError('Graph label spreading did not converge.')
            scores,missing=safe_positive_probability(model.label_distributions_,model.classes_,prior[j])
            q.append(scores[~observed]);iterations.append(int(model.n_iter_));fallbacks.append(int(missing[~observed].sum()))
        models.append(model)
    bundle={'setting':setting,'scaler':scaler,'geometry_models':models,'prior':prior,'fit_ids':frame.ID.tolist(),
        'seed_ids':frame.loc[observed,'ID'].tolist(),'threshold':.35,'empty_policy':'argmax',
        'iterations':iterations,'unconnected_pseudo_fallbacks':fallbacks}
    if setting['method']=='graph_lr':
        q=np.column_stack(q)
        bundle['classifier_models']=fit_binary_targets(x[observed],y[observed],x[~observed],q,C=.1,pseudo_weight=.1)
        bundle['pseudo_q']=q
    elif setting['method']=='graph':bundle['pseudo_q']=np.column_stack(q)
    return bundle


def predict_geometry(bundle,frame):
    x=bundle['scaler'].transform(frame[EMBEDDINGS])
    if bundle['setting']['method']=='graph_lr':
        scores=predict_reason_probabilities(bundle['classifier_models'],x);fallbacks=np.zeros_like(scores,dtype=bool)
    else:
        scores=[];fallbacks=[]
        for j,model in enumerate(bundle['geometry_models']):
            with threadpool_limits(limits=1),np.errstate(invalid='ignore',divide='ignore'):
                distribution=model.predict_proba(x)
            score,missing=safe_positive_probability(distribution,model.classes_,bundle['prior'][j])
            scores.append(score);fallbacks.append(missing)
        scores=np.column_stack(scores);fallbacks=np.column_stack(fallbacks)
    result=pd.DataFrame(scores,columns=REASONS);result.insert(0,'ID',frame.ID.to_numpy())
    return result,fallbacks


def run_geometry(root, output):
    root, folder = prepare_output(root, output)
    plan={'settings':SETTINGS,'seed':42,'cv_folds':4,'threshold':.35,'empty_set_policy':'argmax',
        'label_spreading':{'kernel':'knn','alpha':.2,'max_iter':200,'tol':1e-4,'disconnected_fallback':'observed-training reason prior'},
        'graph_lr':{'C':.1,'pseudo_weight':.1,'targets':'soft training-graph distributions'},
        'selection':'One graph winner and one kNN control by mean training-CV sample F1, then provided validation confirmation',
        'input_sha256':{s:file_sha256(dataset_path(root,'part1',s)) for s in ['train','val']},
        'public_score_used_for_selection':False,'test_used_for_selection':False}
    path=folder/'experiment_plan.json'
    if path.exists():
        if json.loads(path.read_text())!=plan:raise ValueError('Local geometry plan changed.')
    else:write_json(path,plan)
    if (folder/'completed_checksums.json').exists():
        for name,digest in json.loads((folder/'completed_checksums.json').read_text()).items():
            if file_sha256(folder/name)!=digest:raise ValueError(f'Changed geometry result: {name}')
        return json.loads((folder/'summary.json').read_text())
    train=pd.read_csv(dataset_path(root,'part1','train'));observed=observed_label_mask(train[list(REASONS)].to_numpy())
    positions=np.flatnonzero(observed);strata=train.loc[observed,list(REASONS)].to_numpy().argmax(axis=1)
    cv=StratifiedKFold(n_splits=4,shuffle=True,random_state=42)
    metrics=[];provenance=[]
    for fold,(_,hold) in enumerate(cv.split(positions,strata)):
        held=positions[hold];fit=train.drop(index=train.index[held]).reset_index(drop=True);reference=train.iloc[held]
        assert set(fit.ID).isdisjoint(reference.ID)
        provenance.append({'fold':fold,'fit_ids':fit.ID.tolist(),'held_out_ids':reference.ID.tolist(),
            'seed_ids':fit.loc[observed_label_mask(fit[list(REASONS)].to_numpy()),'ID'].tolist()})
        for setting in SETTINGS:
            bundle=fit_geometry(fit,setting);probabilities,fallback=predict_geometry(bundle,reference)
            result=evaluate_part1(reference,probabilities,.35,empty_set_policy='argmax')
            metrics.append({'setting_id':key(setting),'fold':fold,**setting,**result.summary,'graph_unconnected_predictions':int(fallback.sum())})
        print(f'Completed local-geometry CV fold {fold+1}/4',flush=True)
    folds=pd.DataFrame(metrics);folds.to_csv(folder/'cv_fold_metrics.csv',index=False)
    comparison=folds.groupby(['setting_id','method','neighbors'],as_index=False).agg(
        mean_f1_samples=('f1_samples','mean'),std_f1_samples=('f1_samples','std'),mean_f1_macro=('f1_macro','mean'))
    comparison=comparison.sort_values(['mean_f1_samples','setting_id'],ascending=[False,True]).reset_index(drop=True)
    comparison.to_csv(folder/'cv_comparison.csv',index=False);write_json(folder/'cv_fit_scope.json',provenance)
    graph_id=comparison.loc[comparison.method.ne('knn'),'setting_id'].iloc[0]
    knn_id=comparison.loc[comparison.method.eq('knn'),'setting_id'].iloc[0]
    write_json(folder/'cv_selection_locked.json',{'graph_id':graph_id,'knn_id':knn_id,'validation_used':False})
    validation=pd.read_csv(dataset_path(root,'part1','val'))
    current=pd.read_csv(root/'models/part1/02_kmeans_lr/evidence/selected/selected_validation/validation_predictions.csv')
    current_scores=current[['ID',*[f'prob_{r}' for r in REASONS]]].rename(columns={f'prob_{r}':r for r in REASONS})
    original=evaluate_part1(validation,current_scores,.35,empty_set_policy='argmax')
    results={'current_submission':original};models={};summary_rows=[{'setting_id':'current_submission',**original.summary}]
    for run_id in [graph_id,knn_id]:
        setting=next(s for s in SETTINGS if key(s)==run_id)
        bundle=fit_geometry(train,setting);probabilities,fallbacks=predict_geometry(bundle,validation)
        result=evaluate_part1(validation,probabilities,.35,empty_set_policy='argmax')
        config={'task':'part1','setting':setting,'threshold':.35,'empty_policy':'argmax','input_files':plan['input_sha256'],
            'CV_selection_locked_before_validation':True}
        save_run(folder/'runs'/run_id,config,bundle,result,{'fit_ids':bundle['fit_ids'],'seed_ids':bundle['seed_ids'],
            'graph_iterations':bundle['iterations'],'unconnected_pseudo_rows_by_reason':bundle['unconnected_pseudo_fallbacks'],
            'validation_unconnected_predictions':int(fallbacks.sum())})
        results[run_id]=result;models[run_id]=bundle;summary_rows.append({'setting_id':run_id,**result.summary})
    pd.DataFrame(summary_rows).to_csv(folder/'validation_comparison.csv',index=False)
    pd.concat([r.by_group.assign(setting_id=k) for k,r in results.items()]).to_csv(folder/'validation_by_reason.csv',index=False)
    graph_result=results[graph_id];better=graph_result.summary['f1_samples']>original.summary['f1_samples']+1e-12
    selection={'graph_cv_winner':graph_id,'knn_cv_winner':knn_id,'graph_validation_f1':graph_result.summary['f1_samples'],
        'current_validation_f1':original.summary['f1_samples'],'improved_official_validation':better,
        'selected_submission':graph_id if better else 'current_submission','test_used':False}
    write_json(folder/'submission_selection_locked.json',selection)
    def row_f1(result):
        y=result.predictions[[f'true_{r}' for r in REASONS]].to_numpy();d=result.predictions[[f'pred_{r}' for r in REASONS]].to_numpy()
        return 2*(y*d).sum(axis=1)/(y.sum(axis=1)+d.sum(axis=1))
    differences=row_f1(graph_result)-row_f1(original);rng=np.random.default_rng(42)
    draws=differences[rng.integers(0,len(differences),size=(5000,len(differences)))].mean(axis=1)
    write_json(folder/'paired_validation_uncertainty.json',{'delta_sample_f1':float(differences.mean()),
        'descriptive_95_percentile_interval':np.quantile(draws,[.025,.975]).tolist(),
        'warning':'Descriptive interval does not correct for model selection or nonrandom seeds.'})
    if better:
        test=pd.read_csv(dataset_path(root,'part1','test'));bundle=models[graph_id]
        probabilities,fallbacks=predict_geometry(bundle,test)
        decisions=binary_decisions(probabilities[list(REASONS)],dict.fromkeys(REASONS,.35),empty_set_policy='argmax')
        pred=pd.DataFrame(decisions,columns=REASONS);pred.insert(0,'ID',test.ID.to_numpy())
        sample=pd.read_csv(dataset_path(root,'part1','test').parent/'part1_sample_submission.csv')
        submission=build_submission(sample,pred);dest=folder/'candidate_submissions';dest.mkdir(exist_ok=True)
        validate_submission(submission,sample,1);submission.to_csv(dest/'part1_graph_candidate.csv',index=False)
        probabilities.to_csv(dest/'part1_graph_candidate_probabilities.csv',index=False)
        frames={'train':train,'val':validation,'test':test}
        lookup=build_reason_lookup(frames,{s:predict_geometry(bundle,f)[0][list(REASONS)].to_numpy() for s,f in frames.items()},
            dict.fromkeys(REASONS,.35),graph_id,empty_set_policy='argmax')
        lookup.to_csv(folder/'reason_lookup_candidate.csv',index=False)
        filled=train.copy();q=bundle.get('pseudo_q')
        if q is not None:
            masked=train.loc[~observed,['ID']].copy()
            for j,r in enumerate(REASONS):masked[f'prob_{r}']=q[:,j]
            masked.to_csv(folder/'masked_training_graph_probabilities.csv',index=False)
            targets=binary_decisions(q,dict.fromkeys(REASONS,.35),empty_set_policy='argmax')
            filled.loc[~observed,list(REASONS)]=targets
            assert np.array_equal(filled.loc[observed,list(REASONS)],train.loc[observed,list(REASONS)])
            filled.to_csv(folder/'part1_train_graph_augmented.csv',index=False)
    write_json(folder/'summary.json',selection)
    write_json(folder/'completed_checksums.json',{str(p.relative_to(folder)):file_sha256(p) for p in folder.rglob('*') if p.is_file()})
    return selection


if __name__ == "__main__":
    from replay import main
    main(default_study='graph')

```


### midterm_part1_label_shift.py

Original member: `archive/midterm_before_cleanup_2026-11-02.zip ! midterm/midterm_part1_label_shift.py`

Original SHA-256: `e1c8a5d7d787e225ddd73a9e002dcc52cbf879d7bdd027feaa8e410054ffe51e`

Ported SHA-256: `ce8df2a12be4349d812047955423ba071ad9b3a8555f32e215b5f999e4d11373`

```python
"""Year-held-out binary label-shift test for Part 1 reasons.

The EM and BBSE estimators see only target embeddings. Target reason labels
are read for scoring only after the prior estimates have been saved.
"""
from __future__ import annotations

from _historical_support import prepare_output

import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from scipy.special import expit, logit
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedKFold

from midterm_augmentation import observed_label_mask
from midterm_evaluation import REASONS, dataset_path, file_sha256
from midterm_part1_round2 import fit_control, control_scores, make_decisions, score
from midterm_part1_finalization import build_submission
from midterm_part2_improvement import validate_submission

ROOT = None  # Set only by explicit replay(root, output).
OUT = None
EPS = 1e-6
ALPHAS = (0., .25, .5, 1.)
DECISION_THRESHOLD = .35
RECRUITMENT = REASONS.index("Recruitment")


def write_json(path, obj):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(obj, indent=2, allow_nan=False) + "\n")


def em_prior(scores, source_prior, *, tolerance=1e-10, max_iter=2000):
    """Binary Saerens-style EM fixed point with clipped numerical boundaries."""
    p = np.clip(np.asarray(scores, dtype=float), EPS, 1-EPS)
    source = float(np.clip(source_prior, EPS, 1-EPS))
    target = source
    for iteration in range(1, max_iter+1):
        corrected = expit(logit(p) + logit(target) - logit(source))
        updated = float(np.clip(corrected.mean(), EPS, 1-EPS))
        if abs(updated-target) < tolerance:
            return {"prior": updated, "iterations": iteration, "converged": True}
        target = updated
    return {"prior": target, "iterations": max_iter, "converged": False}


def bbse_prior(oof_scores, oof_labels, target_scores, *, threshold=DECISION_THRESHOLD):
    """Binary black-box shift estimate using held-out source predictions."""
    y = np.asarray(oof_labels, dtype=int)
    pred = np.asarray(oof_scores) >= threshold
    target_pred_rate = float((np.asarray(target_scores) >= threshold).mean())
    positives, negatives = y == 1, y == 0
    if not positives.any() or not negatives.any():
        raise ValueError("BBSE requires positive and negative source gold labels.")
    tpr = float(pred[positives].mean())
    fpr = float(pred[negatives].mean())
    determinant = tpr-fpr
    raw = (target_pred_rate-fpr)/determinant if abs(determinant) > 1e-12 else None
    return {"estimate_raw": None if raw is None else float(raw),
            "estimate_clipped": None if raw is None else float(np.clip(raw, 0, 1)),
            "tpr": tpr, "fpr": fpr, "tpr_minus_fpr": determinant,
            "target_predicted_positive_rate": target_pred_rate,
            "source_positive_count": int(positives.sum()),
            "source_negative_count": int(negatives.sum())}


def source_oof(source):
    observed = observed_label_mask(source[list(REASONS)].to_numpy())
    gold = source.loc[observed].reset_index(drop=True)
    y = gold.Recruitment.to_numpy(dtype=int)
    folds = StratifiedKFold(n_splits=4, shuffle=True, random_state=42)
    result = np.full((len(gold), len(REASONS)), np.nan)
    assignment = np.full(len(gold), -1)
    for fold, (_, held_idx) in enumerate(folds.split(gold, y)):
        held = gold.iloc[held_idx]
        fit = source.loc[~source.ID.isin(held.ID)].reset_index(drop=True)
        if set(fit.ID) & set(held.ID):
            raise RuntimeError("Held-out source gold entered the OOF model.")
        result[held_idx] = control_scores(fit_control(fit), held)
        assignment[held_idx] = fold
    if not np.isfinite(result).all() or (assignment < 0).any():
        raise RuntimeError("Incomplete source OOF predictions.")
    return gold, result, assignment


def apply_shift(base, source_prior, target_prior, alpha):
    scores = np.asarray(base, dtype=float).copy()
    source = np.clip(source_prior, EPS, 1-EPS)
    target = np.clip(target_prior, EPS, 1-EPS)
    scores[:, RECRUITMENT] = expit(logit(np.clip(scores[:, RECRUITMENT], EPS, 1-EPS))
                                     + alpha*(logit(target)-logit(source)))
    return scores


def row_f1(y, d):
    intersection = (y*d).sum(axis=1)
    denominator = y.sum(axis=1)+d.sum(axis=1)
    return np.where(denominator > 0, 2*intersection/denominator, 0.)


def run(root, output):
    global ROOT, OUT
    ROOT, OUT = prepare_output(root, output)
    OUT.mkdir(parents=True, exist_ok=True)
    train = pd.read_csv(dataset_path(ROOT, "part1", "train"))
    val = pd.read_csv(dataset_path(ROOT, "part1", "val"))
    combined = pd.concat([train, val], ignore_index=True)
    if combined.ID.duplicated().any():
        raise ValueError("Train and validation IDs overlap.")
    source = combined.loc[combined.start_year <= 2006].reset_index(drop=True)
    target_all = combined.loc[combined.start_year.between(2007, 2009)].reset_index(drop=True)
    target_features = target_all.drop(columns=list(REASONS))
    if len(source) != 667 or len(target_all) != 556:
        raise ValueError("Unexpected year split sizes.")
    write_json(OUT / "plan.json", {"source_year_max": 2006,
        "target_years": [2007, 2008, 2009], "source_rows": len(source),
        "target_unlabeled_rows": len(target_features), "alpha_grid": ALPHAS,
        "reason_applied": "Recruitment", "other_reasons_unchanged": True,
        "source_prior": "observed source-gold prevalence, with all-source mean model probability diagnostic",
        "target_prior": "binary EM on all 556 unlabeled target embeddings",
        "cross_check": "BBSE from four-fold source-gold OOF hard decisions at 0.35",
        "pass_gate": "EM converged, BBSE feasible with TPR-FPR >= 0.1, prior disagreement <= 0.10, sample F1 gain >= 0.01 over source-model alpha0 and train-only baseline, >=3 positive rows",
        "input_sha256": {split: file_sha256(dataset_path(ROOT, "part1", split)) for split in ("train", "val", "test")}})
    source_model = fit_control(source)
    target_scores = control_scores(source_model, target_features)
    source_gold, oof_scores, oof_fold = source_oof(source)
    source_y = source_gold[list(REASONS)].to_numpy(dtype=int)
    source_prior = source_y.mean(axis=0)
    source_model_mean = control_scores(source_model, source.drop(columns=list(REASONS))).mean(axis=0)
    priors = []
    for j, reason in enumerate(REASONS):
        em = em_prior(target_scores[:, j], source_prior[j])
        bbse = bbse_prior(oof_scores[:, j], source_y[:, j], target_scores[:, j])
        priors.append({"reason": reason, "source_gold_prior": float(source_prior[j]),
                       "source_model_mean_probability": float(source_model_mean[j]),
                       "target_mean_uncorrected_probability": float(target_scores[:, j].mean()),
                       "em_prior": em["prior"], "em_iterations": em["iterations"],
                       "em_converged": em["converged"], **{f"bbse_{k}": v for k, v in bbse.items()}})
    # Persist predictions and priors before reading hidden target reason labels.
    pd.DataFrame(priors).to_csv(OUT / "prior_estimates_before_reveal.csv", index=False)
    pd.DataFrame({"ID": source_gold.ID, "fold": oof_fold,
                  **{f"true_{r}": source_y[:, j] for j, r in enumerate(REASONS)},
                  **{f"prob_{r}": oof_scores[:, j] for j, r in enumerate(REASONS)}}).to_csv(OUT / "source_gold_oof.csv", index=False)
    pd.DataFrame({"ID": target_features.ID,
                  **{f"prob_{r}": target_scores[:, j] for j, r in enumerate(REASONS)}}).to_csv(OUT / "unlabeled_target_probabilities.csv", index=False)
    joblib.dump(source_model, OUT / "early_source_model.joblib", compress=3)
    # Reveal only the 56 labeled target rows for evaluation after prior estimation.
    hidden = target_all.loc[observed_label_mask(target_all[list(REASONS)].to_numpy())].reset_index(drop=True)
    if len(hidden) != 56:
        raise ValueError("Expected 56 hidden target gold labels.")
    held_positions = target_features.set_index("ID").index.get_indexer(hidden.ID)
    if (held_positions < 0).any():
        raise RuntimeError("Hidden gold IDs missing from target features.")
    y = hidden[list(REASONS)].to_numpy(dtype=int)
    recruitment_prior = next(row for row in priors if row["reason"] == "Recruitment")
    baseline = make_decisions("binary", target_scores[held_positions])
    baseline_f1 = score(y, baseline)
    train_only_early = train.loc[train.start_year <= 2006].reset_index(drop=True)
    train_only_scores = control_scores(fit_control(train_only_early), target_features)
    train_only_f1 = score(y, make_decisions("binary", train_only_scores[held_positions]))
    comparison = []
    decisions = {}
    for alpha in ALPHAS:
        updated = apply_shift(target_scores, recruitment_prior["source_gold_prior"],
                              recruitment_prior["em_prior"], alpha)
        decision = make_decisions("binary", updated[held_positions])
        decisions[alpha] = decision
        delta = row_f1(y, decision)-row_f1(y, baseline)
        comparison.append({"alpha": alpha, "sample_f1": score(y, decision),
                           "gain_vs_alpha0": float(score(y, decision)-baseline_f1),
                           "recruitment_f1": float(f1_score(y[:, RECRUITMENT], decision[:, RECRUITMENT], zero_division=0)),
                           "changed_decision_rows": int((decision != baseline).any(axis=1).sum()),
                           "positive_row_deltas": int((delta > 1e-12).sum()),
                           "negative_row_deltas": int((delta < -1e-12).sum()),
                           "predicted_multi": int((decision.sum(axis=1) > 1).sum())})
    pd.DataFrame(comparison).to_csv(OUT / "alpha_backtest.csv", index=False)
    detail = hidden[["ID", "start_year", *REASONS]].copy()
    for alpha, decision in decisions.items():
        key = str(alpha).replace(".", "p")
        for j, reason in enumerate(REASONS):
            detail[f"alpha_{key}_pred_{reason}"] = decision[:, j]
        detail[f"alpha_{key}_row_f1"] = row_f1(y, decision)
    detail.to_csv(OUT / "heldout_row_diagnostics.csv", index=False)
    best = sorted(comparison, key=lambda row: (-row["sample_f1"], row["alpha"]))[0]
    bbse_est = recruitment_prior["bbse_estimate_clipped"]
    disagreement = None if bbse_est is None else abs(recruitment_prior["em_prior"]-bbse_est)
    gates = {"em_converged": bool(recruitment_prior["em_converged"]),
             "bbse_identifiable": recruitment_prior["bbse_tpr_minus_fpr"] >= .1,
             "bbse_raw_feasible": recruitment_prior["bbse_estimate_raw"] is not None
                 and -.05 <= recruitment_prior["bbse_estimate_raw"] <= 1.05,
             "em_bbse_agree_within_0p10": disagreement is not None and disagreement <= .1,
             "f1_gain_at_least_0p01": best["gain_vs_alpha0"] >= .01,
             "beats_train_only_baseline_by_0p01": best["sample_f1"] >= train_only_f1 + .01,
             "at_least_three_positive_rows": best["positive_row_deltas"] >= 3,
             "nonzero_alpha_wins": best["alpha"] > 0}
    passed = all(gates.values())
    write_json(OUT / "backtest_decision.json", {"baseline_f1": baseline_f1,
        "train_only_early_baseline_f1": train_only_f1,
        "best": best, "recruitment_source_prior": recruitment_prior["source_gold_prior"],
        "recruitment_em_target_prior": recruitment_prior["em_prior"],
        "recruitment_bbse_target_prior": bbse_est, "em_bbse_difference": disagreement,
        "gates": gates, "passed": passed,
        "test_predictions_touched": False,
        "note": "56 hidden labels were read only after priors and target probabilities were saved."})
    if passed:
        # Apply the locked alpha to the 312-row test only after the backtest gate.
        final_model = fit_control(combined)
        test = pd.read_csv(dataset_path(ROOT, "part1", "test"))
        final_prior = float(combined.loc[observed_label_mask(combined[list(REASONS)].to_numpy()), "Recruitment"].mean())
        test_scores = control_scores(final_model, test.drop(columns=list(REASONS)))
        final_em = em_prior(test_scores[:, RECRUITMENT], final_prior)
        full_gold, full_oof, _ = source_oof(combined)
        full_bbse = bbse_prior(full_oof[:, RECRUITMENT], full_gold.Recruitment.to_numpy(dtype=int),
                               test_scores[:, RECRUITMENT])
        test_agreement = full_bbse["estimate_clipped"] is not None and abs(final_em["prior"]-full_bbse["estimate_clipped"]) <= .1
        if not (final_em["converged"] and full_bbse["tpr_minus_fpr"] >= .1
                and full_bbse["estimate_raw"] is not None
                and -.05 <= full_bbse["estimate_raw"] <= 1.05 and test_agreement):
            write_json(OUT / "test_candidate_rejected.json", {"reason": "final EM/BBSE cross-check failed",
                "em_test_prior": final_em["prior"], "bbse_test": full_bbse, "agreement": test_agreement})
            print(json.dumps(json.loads((OUT / "backtest_decision.json").read_text()), indent=2), flush=True)
            return
        shifted = apply_shift(test_scores, final_prior, final_em["prior"], best["alpha"])
        d = make_decisions("binary", shifted)
        pred = pd.DataFrame(d, columns=REASONS)
        pred.insert(0, "ID", test.ID.to_numpy())
        sample = pd.read_csv(dataset_path(ROOT, "part1", "test").parent / "part1_sample_submission.csv")
        submission = build_submission(sample, pred)
        submission.to_csv(OUT / "part1_label_shift_candidate.csv", index=False)
        validate_submission(submission, sample, 1)
        joblib.dump(final_model, OUT / "final_model.joblib", compress=3)
        write_json(OUT / "test_candidate_metadata.json", {"alpha": best["alpha"],
            "source_prior": final_prior, "em_test_prior": final_em["prior"],
            "bbse_test": full_bbse,
            "em_converged": final_em["converged"], "test_predicted_multi": int((d.sum(axis=1)>1).sum()),
            "submission_sha256": file_sha256(OUT / "part1_label_shift_candidate.csv"),
            "kaggle_uploaded": False})
    print(json.dumps(json.loads((OUT / "backtest_decision.json").read_text()), indent=2), flush=True)




if __name__ == "__main__":
    from replay import main
    main(default_study='label_shift')

```


### midterm_part1_local_followup.py

Original member: `archive/midterm_before_report_cleanup_2026-11-02.zip ! midterm/midterm_part1_local_followup.py`

Original SHA-256: `03652f39d3f9bbaee479028f1a96438197dc4d0c0737abb12b7dbcfe0db8c68a`

Ported SHA-256: `b4580da93ade55cf8fd96f126a96638ac9122b847d4506a318b2049509feb4af`

```python
"""Bounded local-ensemble, Other-kernel, and expected-sample-F1 study.

The fixed reference is the exact kernel recipe that scored 0.88568 publicly.
Selection is nested; the recorded plan cannot change in an existing output dir.
"""
from __future__ import annotations

from _historical_support import prepare_output
import argparse
from dataclasses import dataclass
import importlib.metadata
import json
from pathlib import Path
import time

import joblib
import numpy as np
import pandas as pd
from scipy.linalg import solve
from scipy.spatial.distance import cdist, pdist
from threadpoolctl import threadpool_limits

import midterm_part1_local_consensus as previous
from midterm_evaluation import REASONS, dataset_path

EMB = previous.EMB
EPS = 1e-12
STAGES = ('baseline', 'ensemble', 'other', 'final')
REFERENCE_RUN = 'part1_local_consensus_2026_11_02'
REFERENCE_FILE = 'submission.csv'
REFERENCE_SHA = '6a6949390a85a5d4e767672fc718b2727b9633094574f4c1fcd90c5fee55487c'
BASE_CONFIG = dict(geometry='global', family='ridge', bandwidth=.5, strength=.1)
CONFIGS = {
    'baseline': BASE_CONFIG,
    'cosine': dict(geometry='unit', family='ridge', bandwidth=.5, strength=.1),
    'svm': dict(geometry='global', family='svm', bandwidth=.5, strength=.3),
    'other_narrow': dict(geometry='global', family='ridge', bandwidth=2., strength=.1),
    'other_mixed': dict(geometry='global', family='mixed_ridge', bandwidth=[.5, 2.], strength=.1),
}
ENSEMBLES = [('baseline', 'cosine'), ('baseline', 'svm'), ('baseline', 'cosine', 'svm')]
PLAN = {
    'version': 1, 'reference_public_score': .88568, 'reference_submission_sha256': REFERENCE_SHA,
    'fixed_reference': BASE_CONFIG, 'threshold': .5,
    'outer_folds': 4, 'repeat_seeds': list(previous.SEEDS), 'inner_folds': 3,
    'grouping': 'same <=1e-4 raw-distance connected components as previous experiment',
    'scope': 'held groups excluded from gold and unlabeled feature scopes; test used only for parity and final prediction',
    'calibration': 'three grouped raw-score cross-fits, sigmoid C=1; matching seeds for all components',
    'components': CONFIGS, 'equal_probability_ensembles': ENSEMBLES,
    'component_eligibility': {'improved_inner_groups_minimum': 2, 'F1_deficit_minimum': -.02},
    'stage2': 'replace ONLY Other with narrow or 50:50 mixed RBF kernel; four columns frozen',
    'stage3': 'threshold .5+argmax versus exact expected sample-F1 under independent Bernoulli marginals; 32 truth / 31 action states',
    'selection': 'best eligible candidate within each stage by inner OOF whole-sample F1, then require gain > .002 over incoming stage; ties keep first listed option',
    'candidate_gate': {'mean_oof_gain_minimum': .005, 'positive_repeats_minimum': 2,
                       'worst_repeat_gain_minimum': -.005, 'source_gain_minimum': -.01,
                       'late56_gain_minimum': -.01, 'historical_validation_gain_minimum': -.01},
    'export': 'only final sequential procedure may be exported if all gates pass and final test decisions differ; no outer-selected stage fallback',
    'caveat': 'All 235 labels and prior experiment/public results informed this study. OOF is resampling evidence, not a fresh holdout.',
}


def mixed_rbf_kernel(a, b, gammas):
    """Convex kernel mean, with unit diagonal on its training square matrix."""
    dist = cdist(a, b, 'sqeuclidean')
    return np.mean([np.exp(-g * dist) for g in gammas], axis=0)


@dataclass
class MixedRidge:
    geometry: previous.Geometry
    train_x: np.ndarray
    gammas: tuple
    coefficients: np.ndarray

    def raw(self, x):
        return mixed_rbf_kernel(self.geometry.transform(x), self.train_x, self.gammas) @ self.coefficients


def fit_base(scope, config):
    if config['family'] != 'mixed_ridge':
        return previous.fit_base(scope, config)
    geom = previous.Geometry.fit(scope.data.x[scope.eligible], 'global')
    z = geom.transform(scope.data.x[scope.gold])
    distances = pdist(z, 'sqeuclidean')
    nonzero = distances[distances > 1e-12]
    median = float(np.median(nonzero)) if len(nonzero) else 1.
    gammas = tuple(b / median for b in config['bandwidth'])
    kernel = mixed_rbf_kernel(z, z, gammas)
    coef = solve(kernel + config['strength'] * np.eye(len(z)), scope.data.y[scope.gold], assume_a='pos')
    return MixedRidge(geom, z, gammas, coef)


def fit_calibrated(scope, config, seed):
    raw = np.full((len(scope.gold), 5), np.nan)
    members = []
    for held, inner in scope.folds(seed):
        model = fit_base(inner, config)
        raw[held] = model.raw(scope.data.x[scope.gold[held]])
        members.append(model)
    if not np.isfinite(raw).all():
        raise ValueError('Incomplete grouped calibration predictions')
    return previous.Calibrated(fit_base(scope, config), previous.Sigmoid.fit(raw, scope.data.y[scope.gold]), members)


TRUTH_SETS = np.array([[(mask >> j) & 1 for j in range(5)] for mask in range(32)], dtype=int)
ACTION_SETS = np.array(sorted(TRUTH_SETS[1:].tolist(), key=lambda row: (sum(row), sum(int(v) << j for j, v in enumerate(row)))), dtype=int)
F1_PAYOFF = 2 * (ACTION_SETS @ TRUTH_SETS.T) / (ACTION_SETS.sum(1)[:, None] + TRUTH_SETS.sum(1)[None, :])


def expected_f1_decisions(probabilities):
    """Bayes action for the SPECIFIED independence approximation, not general joint truth."""
    p = np.asarray(probabilities, dtype=float)
    if p.ndim != 2 or p.shape[1] != 5 or not np.isfinite(p).all() or np.any((p < 0) | (p > 1)):
        raise ValueError('Expected finite probabilities with shape (n, 5), in [0, 1]')
    joint = np.prod(np.where(TRUTH_SETS[None, :, :], p[:, None, :], 1 - p[:, None, :]), axis=2)
    utility = joint @ F1_PAYOFF.T
    return ACTION_SETS[np.argmax(utility, axis=1)].copy()


def replace_other(probabilities, replacement):
    p = np.asarray(probabilities, float).copy()
    replacement = np.asarray(replacement, float)
    p[:, REASONS.index('Other')] = replacement
    return p


def decode(p, decoder):
    if decoder == 'threshold':
        return previous.decide(p, .5)
    if decoder == 'expected_f1':
        return expected_f1_decisions(p)
    raise ValueError(decoder)


def blend(probabilities, members):
    return np.mean([probabilities[name] for name in members], axis=0)


def score(y, p, decoder='threshold'):
    return float(previous.row_f1(y, decode(p, decoder)).mean())


def eligibility(y, baseline, other, groups):
    delta = previous.row_f1(y, decode(other, 'threshold')) - previous.row_f1(y, decode(baseline, 'threshold'))
    by_group = pd.DataFrame({'group': groups, 'delta': delta}).groupby('group').delta.mean()
    info = {'improved_groups': int((by_group > EPS).sum()), 'worsened_groups': int((by_group < -EPS).sum()),
            'sample_f1_delta': float(delta.mean())}
    return info['improved_groups'] >= 2 and info['sample_f1_delta'] >= -.02 - EPS, info


@dataclass
class FittedFollowup:
    models: dict
    recipes: dict
    selection: dict

    def probabilities(self, x):
        components = {name: model.predict(x) for name, model in self.models.items()}
        result = {}
        for stage, recipe in self.recipes.items():
            p = blend(components, recipe['members'])
            if recipe['other'] is not None:
                p = replace_other(p, components[recipe['other']][:, REASONS.index('Other')])
            result[stage] = p
        return result

    def predict_sets(self, x):
        return {stage: decode(p, self.recipes[stage]['decoder']) for stage, p in self.probabilities(x).items()}


def fit_procedure(scope, seed):
    y = scope.data.y[scope.gold]
    folds = list(scope.folds(seed))
    inner_predictions = {}
    for name, config in CONFIGS.items():
        p = np.full((len(y), 5), np.nan)
        for k, (held, inner) in enumerate(folds):
            model = fit_calibrated(inner, config, seed + 100 + k)
            p[held] = model.predict(scope.data.x[scope.gold[held]])
        inner_predictions[name] = p
    baseline = inner_predictions['baseline']
    recipes = {'baseline': {'members': ['baseline'], 'other': None, 'decoder': 'threshold'}}
    inner_scores = {'baseline': score(y, baseline)}
    candidate_records, complement = [], {}
    permitted = set()
    for name in ('cosine', 'svm'):
        allow, details = eligibility(y, baseline, inner_predictions[name], scope.data.groups[scope.gold])
        complement[name] = dict(eligible=bool(allow), **details)
        if allow:
            permitted.add(name)
    best_recipe, best_score, best_p = recipes['baseline'].copy(), inner_scores['baseline'], baseline
    for members in ENSEMBLES:
        p = blend(inner_predictions, members)
        value = score(y, p)
        allowed = set(members).difference({'baseline'}).issubset(permitted)
        candidate_records.append({'stage': 'ensemble', 'members': members, 'eligible': allowed, 'f1': value})
        if allowed and value > best_score + EPS:
            best_recipe, best_score, best_p = {'members': list(members), 'other': None, 'decoder': 'threshold'}, value, p
    if best_score <= inner_scores['baseline'] + .002 + EPS:
        best_recipe, best_score, best_p = recipes['baseline'].copy(), inner_scores['baseline'], baseline
    recipes['ensemble'], inner_scores['ensemble'] = best_recipe.copy(), best_score
    stage1_p = best_p.copy()
    for name in ('other_narrow', 'other_mixed'):
        p = replace_other(stage1_p, inner_predictions[name][:, REASONS.index('Other')])
        value = score(y, p)
        candidate_records.append({'stage': 'other', 'model': name, 'f1': value})
        if value > best_score + EPS:
            best_recipe, best_score, best_p = {**recipes['ensemble'], 'other': name}, value, p
    if best_score <= inner_scores['ensemble'] + .002 + EPS:
        best_recipe, best_score, best_p = recipes['ensemble'].copy(), inner_scores['ensemble'], stage1_p
    recipes['other'], inner_scores['other'] = best_recipe.copy(), best_score
    expected_score = score(y, best_p, 'expected_f1')
    candidate_records.append({'stage': 'decoder', 'decoder': 'expected_f1', 'f1': expected_score})
    if expected_score > best_score + .002 + EPS:
        best_recipe, best_score = {**best_recipe, 'decoder': 'expected_f1'}, expected_score
    recipes['final'], inner_scores['final'] = best_recipe.copy(), best_score
    # Matching seed across all components is essential to isolate the model change.
    models = {name: fit_calibrated(scope, config, seed + 900) for name, config in CONFIGS.items()}
    selection = {'recipes': recipes, 'inner_scores': inner_scores, 'candidates': candidate_records,
                 'component_complementarity': complement,
                 'fit_gold_ids': scope.data.frame.iloc[scope.gold].ID.tolist(),
                 'fit_feature_ids': scope.data.frame.iloc[scope.eligible].ID.tolist()}
    return FittedFollowup(models, recipes, selection)


def prediction_frame(data, held, model, evaluation, repeat=-1, fold=-1):
    frame = data.frame.iloc[held][['ID', 'source', 'start_year']].reset_index(drop=True).copy()
    frame['group'], frame['evaluation'], frame['repeat'], frame['fold'] = data.groups[held], evaluation, repeat, fold
    y = data.y[held]
    for j, reason in enumerate(REASONS):
        frame['true_' + reason] = y[:, j]
    for stage, p in model.probabilities(data.x[held]).items():
        pred = decode(p, model.recipes[stage]['decoder'])
        frame[stage + '_f1'] = previous.row_f1(y, pred)
        frame[stage + '_exact'] = (y == pred).all(1)
        frame[stage + '_set'] = [' '.join(r for r, bit in zip(REASONS, row) if bit) for row in pred]
        for j, reason in enumerate(REASONS):
            frame[f'{stage}_prob_{reason}'] = p[:, j]
            frame[f'{stage}_pred_{reason}'] = pred[:, j]
    return frame


def bootstrap(frame):
    grouped = frame.assign(delta=frame.final_f1-frame.baseline_f1).groupby('group').delta.agg(['sum', 'count'])
    samples = np.random.default_rng(42).integers(len(grouped), size=(2000, len(grouped)))
    deltas = grouped['sum'].to_numpy()[samples].sum(1) / grouped['count'].to_numpy()[samples].sum(1)
    return np.quantile(deltas, [.025, .975]).tolist()


def run(root, output):
    root, out = prepare_output(root, output)
    # JSON round-trip turns tuples into lists for a stable repeated-run comparison.
    plan = json.loads(json.dumps(PLAN))
    if (out/'experiment_plan.json').exists() and json.loads((out/'experiment_plan.json').read_text()) != plan:
        raise ValueError('Recorded plan differs; use a new output directory')
    previous.write_json(out/'experiment_plan.json', plan)
    before = previous.protected_hashes(root)
    ref = root/'models/part1/03_kernel_ridge'
    for name in (REFERENCE_FILE, 'model.joblib'):
        before[str(ref/name)] = previous.digest(ref/name)
    previous.write_json(out/'protected_before.json', before)
    data, test = previous.load_data(root)
    full_scope = data.scope(data.gold, np.arange(len(data.frame)))
    ref_model = joblib.load(ref/'model.joblib')
    p_saved = ref_model.predict(test[EMB].to_numpy())
    p_refit = fit_calibrated(full_scope, BASE_CONFIG, 942).predict(test[EMB].to_numpy())
    gap = float(np.max(np.abs(p_saved-p_refit)))
    if gap > 1e-12 or previous.digest(ref/REFERENCE_FILE) != REFERENCE_SHA:
        raise ValueError(f'Fixed reference parity failed: {gap}')
    print(f'Fixed .88568 reference parity maximum gap: {gap:.3g}', flush=True)
    previous.write_json(out/'metadata.json', {'source_sha256': previous.digest(__file__),
        'dependency_sha256': previous.digest(Path(previous.__file__)),
        'data_hashes': {s: previous.digest(dataset_path(root, 'part1', s)) for s in ('train', 'val', 'test')},
        'versions': {p: importlib.metadata.version(p) for p in ('numpy', 'pandas', 'scipy', 'scikit-learn')},
        'reference_probability_gap': gap, 'gold_rows': len(data.gold), 'gold_groups': len(np.unique(data.groups[data.gold]))})
    pieces, selections = [], []
    started = time.time()
    for repeat, seed in enumerate(previous.SEEDS):
        for fold, (held, scope) in enumerate(full_scope.folds(seed, n=4)):
            fitted = fit_procedure(scope, seed + fold * 17)
            frame = prediction_frame(data, data.gold[held], fitted, 'grouped_oof', repeat, fold)
            pieces.append(frame)
            selections.append(dict(repeat=repeat, fold=fold, **fitted.selection))
            print(f'repeat {repeat+1} fold {fold+1}: {json.dumps(previous.summarize(frame))}', flush=True)
    oof = pd.concat(pieces, ignore_index=True)
    assert len(oof) == 705 and oof.groupby(['repeat','ID']).size().eq(1).all()
    oof.to_csv(out/'oof_predictions.csv', index=False)
    previous.write_json(out/'fold_selections.json', selections)
    source, years = data.frame.source.to_numpy(), data.frame.start_year.to_numpy()
    specs = {
        'historical_validation': (np.flatnonzero(source=='train'), np.flatnonzero(source!='validation'), np.flatnonzero(source=='validation')),
        'late_train_20': (np.flatnonzero((source=='train')&(years<=2006)), np.flatnonzero((source!='validation')&(years<=2006)), np.flatnonzero((source=='train')&(years>=2007)&(years<=2009))),
        'late_combined_56': (np.flatnonzero((source!='unlabeled')&(years<=2006)), np.flatnonzero(years<=2006), np.flatnonzero((source!='unlabeled')&(years>=2007)&(years<=2009))),
    }
    diagnostics, tables = {}, [oof]
    for name, (gold, eligible, held) in specs.items():
        model = fit_procedure(data.scope(gold, eligible), 42)
        frame = prediction_frame(data, held, model, name)
        frame['near_early_gold_match'] = np.isin(data.groups[held], data.groups[gold])
        frame.to_csv(out/f'{name}_predictions.csv', index=False)
        diagnostics[name] = {'metrics': previous.summarize(frame), 'novel_group_metrics': previous.summarize(frame.loc[~frame.near_early_gold_match]),
                             'gold_rows': len(gold), 'rows': len(held), 'selection': model.selection}
        tables.append(frame)
        print(name + ': ' + json.dumps(diagnostics[name]['metrics']), flush=True)
    previous.write_json(out/'diagnostics.json', diagnostics)
    table = pd.concat(tables, ignore_index=True)
    changed = table.loc[table.final_set != table.baseline_set].copy()
    changed['sample_f1_delta'] = changed.final_f1-changed.baseline_f1
    changed.to_csv(out/'changed_trials.csv', index=False)
    reason_records, year_records, set_records = [], [], []
    for name, frame in table.groupby('evaluation'):
        for stage in STAGES:
            for reason in REASONS:
                y, p = frame['true_'+reason].to_numpy(), frame[f'{stage}_pred_{reason}'].to_numpy()
                tp, fp, fn = int((y*p).sum()), int(((1-y)*p).sum()), int((y*(1-p)).sum())
                reason_records.append(dict(evaluation=name, stage=stage, reason=reason, tp=tp, fp=fp, fn=fn,
                                           f1=2*tp/(2*tp+fp+fn) if 2*tp+fp+fn else 0.))
        for year, sub in frame.groupby('start_year'):
            year_records.append(dict(evaluation=name, year=int(year), observations=len(sub), **{m:v['sample_f1'] for m,v in previous.summarize(sub).items()}))
        true_sets = frame[['true_'+r for r in REASONS]].apply(lambda row:' '.join(r for r,v in zip(REASONS,row) if v), axis=1)
        for true_set, sub in frame.groupby(true_sets):
            set_records.append(dict(evaluation=name, true_set=true_set, unique_trials=sub.ID.nunique(), observations=len(sub),
                                    **{m:v['sample_f1'] for m,v in previous.summarize(sub).items()}))
    pd.DataFrame(reason_records).to_csv(out/'metrics_by_reason.csv',index=False)
    pd.DataFrame(year_records).to_csv(out/'metrics_by_year.csv',index=False)
    pd.DataFrame(set_records).to_csv(out/'metrics_by_true_set.csv',index=False)
    overall = previous.summarize(oof)
    repeats = {str(k):previous.summarize(v) for k,v in oof.groupby('repeat')}
    sources = {str(k):previous.summarize(v) for k,v in oof.groupby('source')}
    delta = lambda scores:scores['final']['sample_f1']-scores['baseline']['sample_f1']
    gains = [delta(s) for s in repeats.values()]
    gates = {'mean_oof_gain':delta(overall)>=.005-EPS, 'positive_repeats':sum(g>EPS for g in gains)>=2,
             'worst_repeat':min(gains)>=-.005-EPS, 'both_sources':all(delta(s)>=-.01-EPS for s in sources.values()),
             'late56':delta(diagnostics['late_combined_56']['metrics'])>=-.01-EPS,
             'historical_validation':delta(diagnostics['historical_validation']['metrics'])>=-.01-EPS}
    summary = {'overall':overall, 'by_repeat':repeats, 'by_source':sources, 'repeat_gains':gains,
               'mean_gain':delta(overall), 'group_bootstrap_95_percent':bootstrap(oof),
               'diagnostic_metrics':{k:v['metrics'] for k,v in diagnostics.items()},
               'candidate_gate':gates, 'candidate_allowed':all(gates.values()), 'candidate':None,
               'selection_counts':{stage:{key:sum(json.dumps(s['recipes'][stage],sort_keys=True)==key for s in selections)
                                          for key in sorted({json.dumps(s['recipes'][stage],sort_keys=True) for s in selections})} for stage in STAGES},
               'elapsed_seconds':time.time()-started, 'caveat':PLAN['caveat']}
    if all(gates.values()):
        fitted = fit_procedure(full_scope,42)
        predictions = fitted.predict_sets(test[EMB].to_numpy())
        changed_count = int((predictions['final']!=predictions['baseline']).any(1).sum())
        previous.write_json(out/'tools/archive/records/final_selection_nov01.json',fitted.selection)
        if changed_count:
            submission = pd.DataFrame({'ID':test.ID,'Predicted':[' '.join(r for r,b in zip(REASONS,row) if b) for row in predictions['final']]})
            contents = submission.to_csv(index=False).encode()
            import hashlib
            sha = hashlib.sha256(contents).hexdigest()
            name = f'part1_local_followup_{sha[:12]}.csv'
            (out/name).write_bytes(contents)
            joblib.dump(fitted,out/'model.joblib')
            summary['candidate'] = {'filename':name,'sha256':sha,'model_sha256':previous.digest(out/'model.joblib'),
                                    'rows':len(submission),'changed_vs_0_88568':changed_count,'kaggle_uploaded':False}
        else:
            summary['export_note'] = 'Full-data selection produced unchanged reference predictions; no duplicate submission exported.'
    previous.write_json(out/'summary.json',summary)
    for path, sha in before.items():
        if previous.digest(path)!=sha:
            raise RuntimeError(f'Protected file changed: {path}')
    previous.write_json(out/'verification.json',{'protected_files_unchanged':True,'gold_rows':235,'oof_rows':len(oof),'outer_folds':12,
        'output_hashes':{p.name:previous.digest(p) for p in out.glob('*') if p.is_file() and p.name!='verification.json'}})
    print(json.dumps(summary,indent=2),flush=True)




if __name__ == "__main__":
    from replay import main
    main(default_study='local_followup')

```


### midterm_part1_mixture_followup.py

Original member: `archive/midterm_before_cleanup_2026-11-02.zip ! midterm/midterm_part1_mixture_followup.py`

Original SHA-256: `7dd18ac24c2bb1e3a60ada09db9d93cc21fa55665496e851018829a84db08425`

Ported SHA-256: `d4d6ce1092b788292d287dd157d31b419b74b5079f36b56c2f51298e5fac68b9`

```python
"""Fixed small Gaussian-mixture comparison after the locked round-2 selection.

Exploratory only: it cannot retroactively change selection or claim independent
validation, because the first round-2 validation results were already inspected.
"""
from __future__ import annotations

from _historical_support import prepare_output

from pathlib import Path
import json

import numpy as np
import pandas as pd
from scipy.special import softmax
from sklearn.mixture import GaussianMixture
from threadpoolctl import threadpool_limits

from midterm_augmentation import observed_label_mask
from midterm_evaluation import REASONS, dataset_path
from midterm_next_round import p1_assignment
from midterm_part1_round2 import CovarianceGeometry, make_decisions, metrics_rows, score
from midterm_refinement import EMBEDDINGS

ROOT = None  # Set only by explicit replay(root, output).
OUT = None
DIMS = (30, 60, 100)


def fit_mixture(frame, dimensions):
    labels = frame[list(REASONS)].to_numpy()
    observed = observed_label_mask(labels)
    single = observed & ((labels == 1).sum(axis=1) == 1)
    transform = CovarianceGeometry(dimensions).fit(frame[EMBEDDINGS].to_numpy())
    x = transform.transform(frame[EMBEDDINGS].to_numpy())[single]
    y = labels[single].argmax(axis=1)
    models = []
    for j in range(len(REASONS)):
        # Rare classes need one Gaussian when a fold has fewer than four seeds.
        n_components = 2 if (y == j).sum() >= 4 else 1
        model = GaussianMixture(n_components=n_components, covariance_type="diag", reg_covar=.1,
                                n_init=3, max_iter=200, random_state=42)
        with threadpool_limits(limits=1):
            model.fit(x[y == j])
        if not model.converged_:
            raise RuntimeError("Class mixture failed to converge.")
        models.append(model)
    return {"transform": transform, "models": models,
            "prior": np.bincount(y, minlength=len(REASONS)) / len(y)}


def predict(bundle, frame):
    x = bundle["transform"].transform(frame[EMBEDDINGS].to_numpy())
    logp = np.column_stack([model.score_samples(x) for model in bundle["models"]])
    return softmax(logp + np.log(bundle["prior"])[None, :], axis=1)


def run(root, output):
    global ROOT, OUT
    ROOT, OUT = prepare_output(root, output)
    OUT.mkdir(parents=True, exist_ok=True)
    train = pd.read_csv(dataset_path(ROOT, "part1", "train"))
    val = pd.read_csv(dataset_path(ROOT, "part1", "val"))
    gold, assignment = p1_assignment(train)
    folds = assignment.set_index("ID").loc[gold.ID, "fold"].to_numpy()
    oof = {dimension: np.full((len(gold), len(REASONS)), np.nan) for dimension in DIMS}
    for fold in range(4):
        held = gold.loc[folds == fold]
        fit = train.loc[~train.ID.isin(held.ID)]
        assert set(fit.ID).isdisjoint(held.ID)
        for dimension in DIMS:
            oof[dimension][folds == fold] = predict(fit_mixture(fit, dimension), held)
        print(f"Mixture OOF {fold + 1}/4", flush=True)
    temporal_fit = train.loc[train.start_year <= 2006]
    temporal_hold = gold.loc[gold.start_year.between(2007, 2009)]
    rows = []
    for dimension in DIMS:
        name = f"mixture_{dimension}"
        od = make_decisions("primary", oof[dimension])
        td = make_decisions("primary", predict(fit_mixture(temporal_fit, dimension), temporal_hold))
        vd = make_decisions("primary", predict(fit_mixture(train, dimension), val))
        rows.extend(metrics_rows(name, "oof", gold, od))
        rows.extend(metrics_rows(name, "temporal", temporal_hold, td))
        rows.extend(metrics_rows(name, "validation", val, vd))
        print(name, "OOF", score(gold[list(REASONS)], od),
              "time", score(temporal_hold[list(REASONS)], td),
              "val", score(val[list(REASONS)], vd), flush=True)
    pd.DataFrame(rows).to_csv(OUT / "metrics.csv", index=False)
    (OUT / "status.json").write_text(json.dumps({
        "fixed_model": "up to two diagonal Gaussians per single-label class (one if fewer than four seeds), empirical class prior",
        "geometry": "Ledoit-Wolf covariance whitening", "dimensions": DIMS,
        "selection_status": "exploratory_after_round2_validation_inspection",
        "validation_is_independent_for_this_comparison": False,
        "selected_model_changed": False, "test_prediction_written": False}, indent=2) + "\n")




if __name__ == "__main__":
    from replay import main
    main(default_study='mixture_followup')

```


### midterm_part1_other_views.py

Original member: `archive/midterm_before_cleanup_2026-11-02.zip ! midterm/midterm_part1_other_views.py`

Original SHA-256: `11eaa037a272c2f88e0864420ba9e4d4d97916281eed4b1aa404af4a96c4920b`

Ported SHA-256: `c7dd870cdcec879a1f86311c5b51e499fd4b7e076d291801fe7006ac8c0af4b2`

```python
"""Targeted Part 1 Other correction and independent text-view audit.

The baseline four named reason decisions are preserved. New candidates only
add Other. Outer OOF folds, inner residual-model folds, and time holdouts all
exclude held-out labels before fitting every transform and classifier.
"""
from __future__ import annotations

from _historical_support import prepare_output

import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from scipy.sparse import hstack
from scipy.spatial.distance import cdist
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, f1_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from threadpoolctl import threadpool_limits

from midterm_augmentation import logistic_parameters, observed_label_mask
from midterm_evaluation import REASONS, dataset_path, file_sha256
from midterm_next_round import p1_assignment
from midterm_part1_round2 import fit_control, control_scores, make_decisions, score
from midterm_part1_finalization import build_submission
from midterm_part2_improvement import validate_submission
from midterm_refinement import EMBEDDINGS

ROOT = None  # Set only by explicit replay(root, output).
OUT = None
OTHER = REASONS.index("Other")
TEXT_FIELDS = ("criteria", "diseases_cleaned", "drugs")
TEXT_LIMITS = (12000, 4000, 4000)
RESIDUAL_THRESHOLDS = (.65, .75, .85)
TEXT_THRESHOLDS = (.25, .35, .45)


def write_json(path, obj):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(obj, indent=2, allow_nan=False) + "\n")


def write_locked_json(path, obj):
    path = Path(path)
    if path.exists() and json.loads(path.read_text()) != obj:
        raise ValueError(f"Locked selection changed: {path}")
    write_json(path, obj)


def row_f1(y, decision):
    numerator = 2 * (y * decision).sum(axis=1)
    denominator = y.sum(axis=1) + decision.sum(axis=1)
    return np.divide(numerator, denominator, out=np.zeros(len(y), dtype=float), where=denominator > 0)


def text_values(frame, field):
    return frame[field].fillna("").astype(str).tolist()


def fit_text(frame):
    observed = observed_label_mask(frame[list(REASONS)].to_numpy())
    vectorizers, blocks = {}, []
    for field, limit in zip(TEXT_FIELDS, TEXT_LIMITS):
        vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=3, max_features=limit,
                                    lowercase=True, strip_accents="unicode", sublinear_tf=True,
                                    norm="l2", stop_words="english")
        blocks.append(vectorizer.fit_transform(text_values(frame, field)))
        vectorizers[field] = vectorizer
    features = hstack(blocks, format="csr") / np.sqrt(len(TEXT_FIELDS))
    y = frame.loc[observed, list(REASONS)].to_numpy(dtype=int)
    models = []
    for j in range(len(REASONS)):
        model = LogisticRegression(**{**logistic_parameters(42), "C": 1.0})
        with threadpool_limits(limits=1):
            model.fit(features[observed], y[:, j])
        models.append(model)
    return {"vectorizers": vectorizers, "models": models,
            "fit_ids": frame.ID.tolist(), "gold_ids": frame.loc[observed, "ID"].tolist()}


def text_scores(bundle, frame):
    blocks = [bundle["vectorizers"][field].transform(text_values(frame, field)) for field in TEXT_FIELDS]
    features = hstack(blocks, format="csr") / np.sqrt(len(TEXT_FIELDS))
    return np.column_stack([model.predict_proba(features)[:, 1] for model in bundle["models"]])


def residual_features(bundle, fit_frame, query, base_scores):
    observed = observed_label_mask(fit_frame[list(REASONS)].to_numpy())
    gold = fit_frame.loc[observed]
    scaler = bundle["scaler"]
    x_gold = scaler.transform(gold[EMBEDDINGS])
    x_query = scaler.transform(query[EMBEDDINGS])
    distances = cdist(x_query, x_gold, metric="euclidean")
    other_mask = gold.Other.to_numpy(dtype=int) == 1
    if not other_mask.any() or other_mask.all():
        raise ValueError("Residual distance features need Other and non-Other gold.")
    named = np.asarray(base_scores)[:, :OTHER]
    top_two = np.sort(named, axis=1)[:, -2:]
    return np.column_stack([base_scores[:, OTHER], top_two[:, 1],
                            top_two[:, 1]-top_two[:, 0],
                            distances[:, other_mask].min(axis=1),
                            distances[:, ~other_mask].min(axis=1)])


def fit_residual_meta(frame):
    observed = observed_label_mask(frame[list(REASONS)].to_numpy())
    gold = frame.loc[observed].reset_index(drop=True)
    labels = gold.Other.to_numpy(dtype=int)
    inner = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    meta_x = np.full((len(gold), 5), np.nan)
    for _, held_idx in inner.split(gold, labels):
        held = gold.iloc[held_idx]
        inner_fit = frame.loc[~frame.ID.isin(held.ID)].reset_index(drop=True)
        assert set(inner_fit.ID).isdisjoint(held.ID)
        bundle = fit_control(inner_fit)
        p = control_scores(bundle, held)
        meta_x[held_idx] = residual_features(bundle, inner_fit, held, p)
    if not np.isfinite(meta_x).all():
        raise RuntimeError("Missing inner OOF residual features.")
    scaler = StandardScaler().fit(meta_x)
    classifier = LogisticRegression(C=.25, solver="liblinear", max_iter=500,
                                    class_weight="balanced", random_state=42)
    with threadpool_limits(limits=1):
        classifier.fit(scaler.transform(meta_x), labels)
    return {"scaler": scaler, "classifier": classifier,
            "training_gold_ids": gold.ID.tolist(), "inner_oof_features": meta_x}


def residual_scores(meta, base_bundle, fit_frame, query, base_prob):
    features = residual_features(base_bundle, fit_frame, query, base_prob)
    return meta["classifier"].predict_proba(meta["scaler"].transform(features))[:, 1]


def add_other(base_prob, other_signal, threshold, *, method):
    decision = make_decisions("binary", base_prob)
    if method == "residual":
        eligible = (other_signal >= threshold) & (base_prob[:, OTHER] >= .08)
    elif method == "text":
        eligible = (other_signal >= threshold) & (base_prob[:, OTHER] >= .10)
    else:
        raise ValueError(method)
    decision[eligible, OTHER] = 1
    return decision


def fit_and_score(fit, query):
    base = fit_control(fit)
    base_prob = control_scores(base, query)
    text = fit_text(fit)
    text_prob = text_scores(text, query)
    meta = fit_residual_meta(fit)
    residual_prob = residual_scores(meta, base, fit, query, base_prob)
    return {"base": base_prob, "text": text_prob, "residual": residual_prob,
            "models": {"base": base, "text": text, "meta": meta}}


def split_scores(fit, query, *, include_models=False):
    result = fit_and_score(fit, query)
    if not include_models:
        result.pop("models")
    return result


def candidate_decisions(scores, residual_threshold, text_threshold):
    return {
        "baseline": make_decisions("binary", scores["base"]),
        "residual_other": add_other(scores["base"], scores["residual"], residual_threshold, method="residual"),
        "text_other": add_other(scores["base"], scores["text"][:, OTHER], text_threshold, method="text"),
    }


def difference_rows(split, reference, baseline, candidate, method):
    y = reference[list(REASONS)].to_numpy(dtype=int)
    before = row_f1(y, baseline)
    after = row_f1(y, candidate)
    changed = np.flatnonzero((baseline != candidate).any(axis=1))
    rows = []
    for i in changed:
        reason_set = lambda d: " ".join(r for j, r in enumerate(REASONS) if d[i, j])
        rows.append({"split": split, "method": method, "ID": reference.iloc[i].ID,
                     "start_year": int(reference.iloc[i].start_year),
                     "truth": reason_set(y), "before": reason_set(baseline),
                     "after": reason_set(candidate), "row_f1_before": float(before[i]),
                     "row_f1_after": float(after[i]), "delta": float(after[i]-before[i]),
                     "outcome": "fixed" if after[i] > before[i] else "new_error" if after[i] < before[i] else "unchanged_f1"})
    return rows


def evaluate_rows(split, reference, decisions):
    y = reference[list(REASONS)].to_numpy(dtype=int)
    baseline = decisions["baseline"]
    rows = []
    for name, d in decisions.items():
        rows.append({"split": split, "candidate": name, "n": len(reference),
                     "sample_f1": score(y, d),
                     "exact_sets": int((y == d).all(axis=1).sum()),
                     "other_f1": float(f1_score(y[:, OTHER], d[:, OTHER], zero_division=0)),
                     "other_false_negatives": int(((y[:, OTHER] == 1) & (d[:, OTHER] == 0)).sum()),
                     "other_false_positives": int(((y[:, OTHER] == 0) & (d[:, OTHER] == 1)).sum()),
                     "fixed_rows": int((row_f1(y,d) > row_f1(y,baseline)+1e-12).sum()),
                     "new_error_rows": int((row_f1(y,d) < row_f1(y,baseline)-1e-12).sum())})
    return rows


def audit_labels(train, val, out):
    combined = pd.concat([train, val], ignore_index=True)
    gold = combined.loc[observed_label_mask(combined[list(REASONS)].to_numpy())].reset_index(drop=True)
    x = gold[EMBEDDINGS].to_numpy(dtype=float)
    distances = cdist(x, x, metric="euclidean")
    np.fill_diagonal(distances, np.inf)
    finite = distances[np.triu_indices(len(gold), k=1)]
    quantile = np.quantile(finite, .02)
    y = gold[list(REASONS)].to_numpy(dtype=int)
    upper_i, upper_j = np.triu_indices(len(gold), k=1)
    conflict = (y[upper_i] != y[upper_j]).any(axis=1)
    very_close = finite <= quantile
    exact_duplicate = finite < 1e-8
    pairs = []
    for i in range(len(gold)):
        for j in np.argsort(distances[i])[:5]:
            if j <= i:
                continue
            intersection = np.logical_and(y[i], y[j]).sum()
            union = np.logical_or(y[i], y[j]).sum()
            overlap = intersection/union if union else 1.
            if overlap >= 1.:
                continue
            pairs.append({"ID_a": gold.iloc[i].ID, "ID_b": gold.iloc[j].ID,
                          "year_a": int(gold.iloc[i].start_year), "year_b": int(gold.iloc[j].start_year),
                          "raw_embedding_distance": float(distances[i, j]),
                          "below_pairwise_2pct_distance": bool(distances[i, j] <= quantile),
                          "label_jaccard": float(overlap),
                          "labels_a": " ".join(np.array(REASONS)[y[i] == 1]),
                          "labels_b": " ".join(np.array(REASONS)[y[j] == 1])})
    pd.DataFrame(pairs).sort_values(["raw_embedding_distance", "label_jaccard"]).head(30).to_csv(out / "nearest_discordant_gold_pairs.csv", index=False)
    oof = pd.read_csv(ROOT / 'models/part1/exploration/replay_inputs/oof_weight_char/oof_weight_010.csv')
    val_pred = pd.read_csv(ROOT / 'models/part1/02_kmeans_lr/evidence/selected/selected_validation/validation_predictions.csv')
    review = []
    for source, frame in (("train_gold_oof", oof), ("validation_holdout", val_pred)):
        for reason in REASONS:
            y_r = frame[f"true_{reason}"].to_numpy(dtype=int)
            p_r = frame[f"prob_{reason}"].to_numpy(dtype=float)
            high_wrong = (y_r == 0) & (p_r >= .75)
            low_wrong = (y_r == 1) & (p_r <= .15)
            for i in np.flatnonzero(high_wrong | low_wrong):
                review.append({"source": source, "ID": frame.iloc[i].ID,
                               "reason": reason, "observed_label": int(y_r[i]),
                               "oof_or_holdout_probability": float(p_r[i]),
                               "flag": "high_probability_negative" if high_wrong[i] else "low_probability_positive",
                               "action": "manual_review_only_no_label_change"})
    pd.DataFrame(review, columns=["source", "ID", "reason", "observed_label",
                                  "oof_or_holdout_probability", "flag", "action"]).to_csv(out / "high_confidence_label_review.csv", index=False)
    return {"gold_rows": len(gold), "discordant_pairs_saved": min(30, len(pairs)),
            "pairwise_distance_2pct": float(quantile), "high_confidence_flags": len(review),
            "all_pairs_within_2pct": int(very_close.sum()),
            "discordant_pairs_within_2pct": int((very_close & conflict).sum()),
            "exact_duplicate_pairs": int(exact_duplicate.sum()),
            "discordant_exact_duplicate_pairs": int((exact_duplicate & conflict).sum()),
            "labels_modified": 0}


def save_scores(path, reference, scores, fold=None):
    result = reference[["ID", "start_year", *REASONS]].reset_index(drop=True).copy()
    if fold is not None:
        result["fold"] = fold
    for j, reason in enumerate(REASONS):
        result[f"base_prob_{reason}"] = scores["base"][:, j]
        result[f"text_prob_{reason}"] = scores["text"][:, j]
    result["residual_other_score"] = scores["residual"]
    result.to_csv(path, index=False)


def run(root, output):
    global ROOT, OUT
    ROOT, OUT = prepare_output(root, output)
    OUT.mkdir(parents=True, exist_ok=True)
    train = pd.read_csv(dataset_path(ROOT, "part1", "train"))
    val = pd.read_csv(dataset_path(ROOT, "part1", "val"))
    gold, assignment = p1_assignment(train)
    fold = assignment.set_index("ID").loc[gold.ID, "fold"].to_numpy()
    write_json(OUT / "plan.json", {"candidate_scope": "add Other only; four named reasons preserved",
        "residual_features": ["base_other_probability", "max_named_probability", "top_two_named_gap",
                              "nearest_other_gold_distance", "nearest_nonother_gold_distance"],
        "residual_thresholds": RESIDUAL_THRESHOLDS, "text_fields": TEXT_FIELDS,
        "text_thresholds": TEXT_THRESHOLDS, "text_min_df": 3, "text_ngram": [1,2],
        "selection": "OOF sample F1 threshold winner per method; then require both OOF and 56-row temporal gain and no validation loss",
        "time_fit": "train+validation rows <=2006, time holdout 2007-2009 labeled rows (56)",
        "label_changes_allowed": False,
        "input_sha256": {split: file_sha256(dataset_path(ROOT, "part1", split)) for split in ("train", "val", "test")}})
    oof_base = np.full((len(gold), len(REASONS)), np.nan)
    oof_text = np.full((len(gold), len(REASONS)), np.nan)
    oof_meta = np.full(len(gold), np.nan)
    for k in range(4):
        held = gold.loc[fold == k]
        fit = train.loc[~train.ID.isin(held.ID)].reset_index(drop=True)
        assert set(fit.ID).isdisjoint(held.ID)
        result = split_scores(fit, held)
        oof_base[fold == k] = result["base"]
        oof_text[fold == k] = result["text"]
        oof_meta[fold == k] = result["residual"]
        print(f"Other-view outer OOF fold {k+1}/4 done", flush=True)
    if not np.isfinite(oof_base).all() or not np.isfinite(oof_text).all() or not np.isfinite(oof_meta).all():
        raise RuntimeError("Incomplete OOF predictions.")
    saved_oof = pd.read_csv(ROOT / 'models/part1/exploration/replay_inputs/oof_weight_char/oof_weight_010.csv').set_index("ID").loc[gold.ID]
    saved_prob = saved_oof[[f"prob_{r}" for r in REASONS]].to_numpy()
    if not np.allclose(oof_base, saved_prob, atol=1e-12, rtol=0):
        raise RuntimeError("Baseline OOF no longer reproduces the locked model.")
    oof_scores = {"base": oof_base, "text": oof_text, "residual": oof_meta}
    base_decision = make_decisions("binary", oof_base)
    y_oof = gold[list(REASONS)].to_numpy(dtype=int)
    grid = []
    for method, thresholds in (("residual", RESIDUAL_THRESHOLDS), ("text", TEXT_THRESHOLDS)):
        signal = oof_meta if method == "residual" else oof_text[:, OTHER]
        for threshold in thresholds:
            d = add_other(oof_base, signal, threshold, method=method)
            grid.append({"method": method, "threshold": threshold, "sample_f1": score(y_oof, d),
                         "fixed_rows": int((row_f1(y_oof, d) > row_f1(y_oof, base_decision)+1e-12).sum()),
                         "new_error_rows": int((row_f1(y_oof, d) < row_f1(y_oof, base_decision)-1e-12).sum())})
    pd.DataFrame(grid).to_csv(OUT / "oof_threshold_grid.csv", index=False)
    selected_threshold = {method: sorted((row for row in grid if row["method"] == method),
                                         key=lambda row: (-row["sample_f1"], -row["threshold"]))[0]["threshold"]
                          for method in ("residual", "text")}
    write_locked_json(OUT / "selection_locked_before_time_validation.json", {"thresholds": selected_threshold,
        "oof_baseline_f1": score(y_oof, base_decision), "oof_grid": grid,
        "time_and_validation_labels_used_for_threshold_selection": False})
    oof_d = candidate_decisions(oof_scores, selected_threshold["residual"], selected_threshold["text"])
    save_scores(OUT / "oof_scores.csv", gold, oof_scores, fold=fold)
    combined = pd.concat([train, val], ignore_index=True)
    early = combined.loc[combined.start_year <= 2006].reset_index(drop=True)
    late = combined.loc[combined.start_year.between(2007, 2009)].reset_index(drop=True)
    late_gold = late.loc[observed_label_mask(late[list(REASONS)].to_numpy())].reset_index(drop=True)
    if len(late_gold) != 56 or set(early.ID) & set(late_gold.ID):
        raise RuntimeError("Unexpected time holdout or overlap.")
    time_scores = split_scores(early, late_gold)
    time_d = candidate_decisions(time_scores, selected_threshold["residual"], selected_threshold["text"])
    save_scores(OUT / "temporal_scores.csv", late_gold, time_scores)
    val_scores = split_scores(train, val)
    val_d = candidate_decisions(val_scores, selected_threshold["residual"], selected_threshold["text"])
    save_scores(OUT / "validation_scores.csv", val, val_scores)
    all_metrics = (evaluate_rows("oof", gold, oof_d)
                   + evaluate_rows("temporal_56", late_gold, time_d)
                   + evaluate_rows("validation_78", val, val_d))
    pd.DataFrame(all_metrics).to_csv(OUT / "comparison.csv", index=False)
    text_standalone = []
    for split, reference, scores in (("oof", gold, oof_scores),
                                     ("temporal_56", late_gold, time_scores),
                                     ("validation_78", val, val_scores)):
        d = make_decisions("binary", scores["text"])
        y = reference[list(REASONS)].to_numpy(dtype=int)
        text_standalone.append({"split": split, "n": len(reference),
                                "text_only_sample_f1": score(y, d),
                                "text_only_other_f1": float(f1_score(y[:, OTHER], d[:, OTHER], zero_division=0))})
    pd.DataFrame(text_standalone).to_csv(OUT / "text_standalone_metrics.csv", index=False)
    changes = []
    for split, ref, decisions in (("oof", gold, oof_d), ("temporal_56", late_gold, time_d),
                                  ("validation_78", val, val_d)):
        for method in ("residual_other", "text_other"):
            changes.extend(difference_rows(split, ref, decisions["baseline"], decisions[method], method))
    pd.DataFrame(changes, columns=["split", "method", "ID", "start_year", "truth", "before", "after",
                                   "row_f1_before", "row_f1_after", "delta", "outcome"]).to_csv(OUT / "changed_trials.csv", index=False)
    # Error complementarity: text probability on baseline Other false negatives versus negatives.
    complement = []
    for split, ref, scores, decisions in (("oof", gold, oof_scores, oof_d),
                                           ("temporal_56", late_gold, time_scores, time_d),
                                           ("validation_78", val, val_scores, val_d)):
        truth = ref.Other.to_numpy(dtype=int)
        miss = (truth == 1) & (decisions["baseline"][:, OTHER] == 0)
        nonother = truth == 0
        complement.append({"split": split, "baseline_other_false_negatives": int(miss.sum()),
                           "mean_text_other_p_on_misses": float(scores["text"][miss, OTHER].mean()) if miss.any() else None,
                           "mean_text_other_p_on_nonother": float(scores["text"][nonother, OTHER].mean()),
                           "base_other_average_precision": float(average_precision_score(truth, scores["base"][:, OTHER])),
                           "text_other_average_precision": float(average_precision_score(truth, scores["text"][:, OTHER]))})
    pd.DataFrame(complement).to_csv(OUT / "text_complementarity.csv", index=False)
    audit = audit_labels(train, val, OUT)
    write_json(OUT / "label_audit_summary.json", audit)
    table = pd.DataFrame(all_metrics)
    eligible = []
    for method in ("residual_other", "text_other"):
        metrics = {split: table.loc[(table.split == split) & (table.candidate == method), "sample_f1"].item()
                   for split in ("oof", "temporal_56", "validation_78")}
        baseline = {split: table.loc[(table.split == split) & (table.candidate == "baseline"), "sample_f1"].item()
                    for split in ("oof", "temporal_56", "validation_78")}
        if (metrics["oof"] > baseline["oof"] + 1e-12
                and metrics["temporal_56"] > baseline["temporal_56"] + 1e-12
                and metrics["validation_78"] >= baseline["validation_78"] - 1e-12):
            eligible.append(method)
    selected = sorted(eligible, key=lambda name: (-table.loc[(table.split == "oof") & (table.candidate == name), "sample_f1"].item(), name))[0] if eligible else None
    write_json(OUT / "decision.json", {"selected": selected, "eligible": eligible,
        "thresholds": selected_threshold, "no_new_model_if_failed": True,
        "kaggle_uploaded": False, "original_final_replaced": False,
        "label_changes": 0, "co_training_attempted": False,
        "reason_for_no_co_training": "require evidence of complementary text errors before adding pseudo labels"})
    if selected is not None:
        # Keep a separate candidate; never replace the locked final.
        test = pd.read_csv(dataset_path(ROOT, "part1", "test"))
        full = split_scores(combined, test, include_models=True)
        final_d = candidate_decisions(full, selected_threshold["residual"], selected_threshold["text"])[selected]
        pred = pd.DataFrame(final_d, columns=REASONS)
        pred.insert(0, "ID", test.ID.to_numpy())
        sample = pd.read_csv(dataset_path(ROOT, "part1", "test").parent / "part1_sample_submission.csv")
        submission = build_submission(sample, pred)
        submission.to_csv(OUT / "part1_other_candidate.csv", index=False)
        validate_submission(submission, sample, 1)
        joblib.dump(full["models"], OUT / "selected_models.joblib", compress=3)
        write_json(OUT / "candidate_metadata.json", {"selected": selected,
            "submission_sha256": file_sha256(OUT / "part1_other_candidate.csv"),
            "test_changed_rows": int((final_d != make_decisions("binary", full["base"])).any(axis=1).sum()),
            "kaggle_uploaded": False})
    print(pd.DataFrame(all_metrics).to_string(index=False), flush=True)
    print(json.dumps(json.loads((OUT / "decision.json").read_text()), indent=2), flush=True)




if __name__ == "__main__":
    from replay import main
    main(default_study='other_views')

```


### midterm_part1_parity.py

Original member: `archive/midterm_before_report_cleanup_2026-11-02.zip ! midterm/midterm_part1_parity.py`

Original SHA-256: `eaec41010ec6834b831e2457e029f84b0e1ee6548be0929d36163b22bbd35877`

Ported SHA-256: `a6c7eb905172acebcba8aa781979ab43d72ebb97e1191d88df65eba8d3f67552`

```python
"""Reproducible, label-leakage-aware comparison of the two retained Part 1 models.

Run from the curated midterm directory with ``python midterm_part1_parity.py``.
This script only writes a new analysis directory; it never replaces a submission.
"""

from __future__ import annotations

from _historical_support import prepare_output

import json
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, precision_recall_fscore_support
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.multiclass import OneVsRestClassifier
from sklearn.preprocessing import StandardScaler

from midterm_augmentation import (
    cluster_memberships, estimate_cluster_reason_rates, logistic_parameters,
    observed_label_mask, predict_reason_probabilities, recover_labels,
)
from midterm_evaluation import REASONS, dataset_path, part1_decisions
from midterm_part1_finalization import fit_classifiers


ROOT = None  # Set only by explicit replay(root, output).
OUT = None
EMB = [f"emb_{i}" for i in range(150)]
SEED = 42
THRESHOLDS = {"supervised": 0.50, "semi": 0.35}
COMMON_THRESHOLDS = (0.35, 0.50)
EPS = 1e-12


def train_models(features: pd.DataFrame, labels: np.ndarray):
    """Fit scaler on available features and classifiers on permitted labels only."""
    labels = np.asarray(labels, dtype=int)
    visible = observed_label_mask(labels)
    scaler = StandardScaler().fit(features[EMB])
    x = scaler.transform(features[EMB])

    baseline = OneVsRestClassifier(
        LogisticRegression(**logistic_parameters(SEED)), n_jobs=1
    ).fit(x[visible], labels[visible])

    clusterer = KMeans(
        n_clusters=10, n_init=20, tol=1e-4, max_iter=500,
        algorithm="lloyd", random_state=SEED,
    ).fit(x)
    membership = cluster_memberships(clusterer, x, "kmeans")
    mapping = estimate_cluster_reason_rates(membership, labels, smoothing_strength=2.0)
    recovered = recover_labels(labels, membership, mapping, threshold=0.5, pseudo_weight=0.1)
    semi, _ = fit_classifiers(
        x, recovered["labels"], recovered["sample_weight"], C=0.1, seed=SEED,
    )
    return scaler, baseline, semi


def score_models(models, frame: pd.DataFrame):
    scaler, baseline, semi = models
    x = scaler.transform(frame[EMB])
    return {"supervised": baseline.predict_proba(x),
            "semi": predict_reason_probabilities(semi, x)}


def decisions(probabilities, threshold=None):
    return {name: part1_decisions(p, THRESHOLDS[name] if threshold is None else threshold,
                                  empty_set_policy="argmax")
            for name, p in probabilities.items()}


def sample_scores(truth, predicted):
    intersections = (truth & predicted).sum(axis=1)
    denominators = truth.sum(axis=1) + predicted.sum(axis=1)
    return np.divide(2 * intersections, denominators,
                     out=np.zeros(len(truth), dtype=float), where=denominators != 0)


def metrics(truth, predicted):
    p, r, f, support = precision_recall_fscore_support(
        truth, predicted, average=None, zero_division=0,
    )
    return {"n": len(truth), "sample_f1": float(f1_score(truth, predicted, average="samples", zero_division=0)),
            "exact_sets": int(np.all(truth == predicted, axis=1).sum()),
            "per_reason": {reason: {"support": int(support[j]), "predicted": int(predicted[:, j].sum()),
                                    "precision": float(p[j]), "recall": float(r[j]), "f1": float(f[j])}
                           for j, reason in enumerate(REASONS)}}


def sets(values):
    return [";".join(reason for reason, value in zip(REASONS, row) if value) for row in values]


def detail_frame(frame, truth, probs, pred, *, split, folds=None, early_embeddings=None):
    result = frame[["ID", "start_year"]].reset_index(drop=True).copy()
    result.insert(1, "split", split)
    if folds is not None:
        result["fold"] = folds
    if early_embeddings is not None:
        early_keys = {tuple(row) for row in early_embeddings.to_numpy()}
        result["exact_early_embedding_match"] = [tuple(row) in early_keys for row in frame[EMB].to_numpy()]
    result["truth"] = sets(truth)
    result["supervised"] = sets(pred["supervised"])
    result["semi"] = sets(pred["semi"])
    for name in ("supervised", "semi"):
        result[f"{name}_row_f1"] = sample_scores(truth, pred[name])
        result[f"{name}_exact"] = np.all(truth == pred[name], axis=1)
        for j, reason in enumerate(REASONS):
            result[f"{name}_prob_{reason}"] = probs[name][:, j]
    result["winner"] = np.where(result["supervised_row_f1"] > result["semi_row_f1"] + EPS,
                                "supervised", np.where(result["semi_row_f1"] > result["supervised_row_f1"] + EPS,
                                                       "semi", "tie"))
    result["different_sets"] = np.any(pred["supervised"] != pred["semi"], axis=1)
    return result


def comparison(frame, truth, probs, *, split, folds=None, early_embeddings=None):
    pred = decisions(probs)
    details = detail_frame(frame, truth, probs, pred, split=split, folds=folds,
                           early_embeddings=early_embeddings)
    by_year = []
    for year in sorted(frame["start_year"].unique()):
        sel = frame["start_year"].to_numpy() == year
        by_year.append({"split": split, "start_year": int(year), "n": int(sel.sum()),
                        **{f"{name}_sample_f1": metrics(truth[sel], pred[name][sel])["sample_f1"]
                           for name in pred}})
    return {"metrics": {name: metrics(truth, pred[name]) for name in pred},
            "common_thresholds": {str(t): {name: metrics(truth, chosen)["sample_f1"]
                                          for name, chosen in decisions(probs, t).items()}
                                  for t in COMMON_THRESHOLDS},
            "wins": {name: int((details["winner"] == name).sum()) for name in ("supervised", "semi", "tie")},
            "different_sets": int(details["different_sets"].sum()),
            "by_year": by_year}, details


def verify_saved_predictions(train, val, labels):
    """Check the refit recipe against immutable historical validation outputs."""
    models = train_models(train, labels)
    fitted = score_models(models, val)
    records = {
        "supervised": ROOT / 'models/part1/01_supervised_lr/evidence/original_run/validation_predictions.csv',
        "semi": ROOT / 'models/part1/02_kmeans_lr/evidence/selected/selected_validation/validation_predictions.csv',
    }
    gaps = {}
    for name, path in records.items():
        saved = pd.read_csv(path).set_index("ID").loc[val["ID"]]
        actual = saved[[f"prob_{reason}" for reason in REASONS]].to_numpy()
        gaps[name] = float(np.max(np.abs(fitted[name] - actual)))
        if gaps[name] > 1e-10:
            raise RuntimeError(f"{name} refit failed historical parity: {gaps[name]}")
    # Also check that the saved model artifacts themselves reproduce the records.
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=UserWarning)
        base_bundle = joblib.load(ROOT / 'models/part1/01_supervised_lr/model.joblib')
    saved_base = base_bundle["pipeline"].predict_proba(val[EMB])
    if not np.allclose(saved_base, fitted["supervised"], atol=1e-10, rtol=0):
        raise RuntimeError("Saved supervised model disagrees with the refit recipe")
    return models, fitted, gaps


def grouped_oof(train, labels):
    visible = observed_label_mask(labels)
    gold = train.loc[visible].reset_index(drop=True)
    y = labels[visible]
    keys = pd.factorize(pd.Series([tuple(row) for row in gold[EMB].to_numpy()]), sort=False)[0]
    primary = y.argmax(axis=1)
    splitter = StratifiedGroupKFold(n_splits=4, shuffle=True, random_state=SEED)
    probs = {name: np.full((len(gold), 5), np.nan) for name in THRESHOLDS}
    folds = np.full(len(gold), -1, dtype=int)
    gold_positions = np.flatnonzero(visible)
    for fold, (_, held) in enumerate(splitter.split(gold, primary, keys)):
        # Held rows must be absent even from scaler and K-Means fitting:
        # the original selected model did not fit on competition test features.
        fit_mask = np.ones(len(train), dtype=bool)
        fit_mask[gold_positions[held]] = False
        models = train_models(train.loc[fit_mask], labels[fit_mask])
        scores = score_models(models, gold.iloc[held])
        for name in probs:
            probs[name][held] = scores[name]
        folds[held] = fold
    if np.isnan(probs["supervised"]).any() or (folds < 0).any():
        raise RuntimeError("OOF assignments are incomplete")
    if any(len(set(keys[folds == fold]) & set(keys[folds != fold])) for fold in range(4)):
        raise RuntimeError("Identical gold embeddings crossed a fold boundary")
    return gold, y, probs, folds, int(len(np.unique(keys)))


def temporal_backtest(train, val, labels):
    """Strict pre-2007 fit; later labels are never passed to fitting functions."""
    early = train["start_year"].to_numpy() <= 2006
    early_train = train.loc[early].reset_index(drop=True)
    early_labels = labels[early]
    if not observed_label_mask(early_labels).any():
        raise RuntimeError("No early labels")
    models = train_models(early_train, early_labels)
    late_train = train.loc[(train["start_year"].between(2007, 2009)) & observed_label_mask(labels)]
    late_val = val.loc[val["start_year"].between(2007, 2009)]
    late = pd.concat([late_train, late_val], ignore_index=True)
    truth = late[list(REASONS)].to_numpy(dtype=int)
    probs = score_models(models, late)
    exact_early = early_train.loc[observed_label_mask(early_labels), EMB]
    return late, truth, probs, len(late_train), exact_early


def run(root, output):
    global ROOT, OUT
    ROOT, OUT = prepare_output(root, output)
    train = pd.read_csv(dataset_path(ROOT, "part1", "train"))
    val = pd.read_csv(dataset_path(ROOT, "part1", "val"))
    test = pd.read_csv(dataset_path(ROOT, "part1", "test"))
    labels = train[list(REASONS)].to_numpy(dtype=int)
    assert len(train) == 1171 and observed_label_mask(labels).sum() == 157 and len(val) == 78
    OUT.mkdir(parents=True, exist_ok=True)

    full_models, val_probs, parity = verify_saved_predictions(train, val, labels)
    test_probs = score_models(full_models, test)
    test_pred = decisions(test_probs)
    test_disagreements = test[["ID", "start_year"]].copy()
    for name in THRESHOLDS:
        test_disagreements[name] = sets(test_pred[name])
    test_disagreements["different_sets"] = np.any(
        test_pred["supervised"] != test_pred["semi"], axis=1,
    )
    test_disagreements.loc[test_disagreements["different_sets"]].to_csv(
        OUT / "test_disagreements.csv", index=False,
    )
    validation, val_details = comparison(val, val[list(REASONS)].to_numpy(dtype=int), val_probs,
                                         split="validation")
    gold, truth, oof_probs, folds, unique_groups = grouped_oof(train, labels)
    oof, oof_details = comparison(gold, truth, oof_probs, split="grouped_oof", folds=folds)
    late, late_truth, late_probs, n_train_gold, early_emb = temporal_backtest(train, val, labels)
    late_result, late_details = comparison(late, late_truth, late_probs, split="late_2007_2009",
                                           early_embeddings=early_emb)
    # Primary time check uses only gold rows that originated in the training CSV.
    late_primary, _ = comparison(late.iloc[:n_train_gold], late_truth[:n_train_gold],
                                 {name: p[:n_train_gold] for name, p in late_probs.items()},
                                 split="late_train_only", early_embeddings=early_emb)
    novel = ~late_details.iloc[:n_train_gold]["exact_early_embedding_match"].to_numpy()
    late_novel, _ = comparison(
        late.iloc[:n_train_gold].iloc[novel], late_truth[:n_train_gold][novel],
        {name: p[:n_train_gold][novel] for name, p in late_probs.items()},
        split="late_train_no_exact_early_embedding", early_embeddings=early_emb,
    )

    for name, details in (("validation", val_details), ("grouped_oof", oof_details),
                          ("late_2007_2009", late_details)):
        details.to_csv(OUT / f"{name}_predictions.csv", index=False)
    pd.concat([d.loc[d["different_sets"]] for d in (val_details, oof_details, late_details)],
              ignore_index=True).to_csv(OUT / "changed_trials.csv", index=False)
    pd.DataFrame(validation["by_year"] + oof["by_year"] + late_result["by_year"]).to_csv(
        OUT / "metrics_by_year.csv", index=False,
    )
    per_reason = []
    for split, result in (("validation", validation), ("grouped_oof", oof),
                          ("late_train_only", late_primary), ("late_train_no_exact_early_embedding", late_novel),
                          ("late_2007_2009", late_result)):
        for model, m in result["metrics"].items():
            for reason, values in m["per_reason"].items():
                per_reason.append({"split": split, "model": model, "reason": reason, **values})
    pd.DataFrame(per_reason).to_csv(OUT / "metrics_by_reason.csv", index=False)

    # Selection gate fixed before inspecting the run: a new candidate needs a
    # meaningful OOF gain, a meaningful strict-time gain, and no val regression.
    a, b = "supervised", "semi"
    oof_delta = oof["metrics"][a]["sample_f1"] - oof["metrics"][b]["sample_f1"]
    time_delta = late_primary["metrics"][a]["sample_f1"] - late_primary["metrics"][b]["sample_f1"]
    val_delta = validation["metrics"][a]["sample_f1"] - validation["metrics"][b]["sample_f1"]
    unique_wins = late_details.iloc[:n_train_gold]["winner"].value_counts().to_dict()
    gate = {"oof_gain_at_least_0_01": bool(oof_delta >= 0.01),
            "strict_time_gain_at_least_0_02": bool(time_delta >= 0.02),
            "strict_time_at_least_two_net_row_wins": bool(unique_wins.get(a, 0) - unique_wins.get(b, 0) >= 2),
            "validation_nonworse": bool(val_delta >= -1e-12)}
    result = {"protocol": {"seed": SEED, "folds": 4, "unique_gold_embedding_groups": unique_groups,
                           "oof_held_features_excluded_from_all_fits": True,
                           "temporal_fit": "train rows start_year <= 2006 only",
                           "temporal_primary": "2007-2009 observed labels in train CSV only",
                           "temporal_56_row_check": "includes previously inspected validation labels; diagnostic only",
                           "fallback": "argmax reason on every empty predicted set",
                           "historical_thresholds": THRESHOLDS, "common_thresholds": list(COMMON_THRESHOLDS),
                           "full_refit_max_absolute_validation_probability_gap": parity},
              "validation": validation, "grouped_oof": oof,
              "late_train_only": late_primary,
              "late_train_no_exact_early_embedding": late_novel,
              "late_2007_2009": late_result,
              "supervised_minus_semi": {"oof_sample_f1": oof_delta, "strict_time_sample_f1": time_delta,
                                        "validation_sample_f1": val_delta},
              "unlabeled_test_disagreements": int(test_disagreements["different_sets"].sum()),
              "candidate_gate": gate, "candidate_approved_for_local_generation": all(gate.values())}
    (OUT / "summary.json").write_text(json.dumps(result, indent=2, allow_nan=False) + "\n")
    print(json.dumps({"parity": parity, "oof_delta": oof_delta, "strict_time_delta": time_delta,
                      "validation_delta": val_delta, "gate": gate}, indent=2))




if __name__ == "__main__":
    from replay import main
    main(default_study='parity')

```


### midterm_part1_ridge_boundary.py

Original member: `archive/midterm_before_report_cleanup_2026-11-02.zip ! midterm/midterm_part1_ridge_boundary.py`

Original SHA-256: `f8d8e3b45528e71906613b84d18a2bc3d4cf102a66f6dde8944fbfdb1f58187d`

Ported SHA-256: `6e167a7196f9086db7ade215b3806d524fed7601a87947d3809ac2162ec39166`

```python
"""Prespecified ridge boundary grid followed by scope-local group weighting."""
from __future__ import annotations

from _historical_support import prepare_output
import argparse
from dataclasses import dataclass
import hashlib
import importlib.metadata
import json
from pathlib import Path
import time

import joblib
import numpy as np
import pandas as pd
from scipy.linalg import solve
from scipy.spatial.distance import cdist, pdist
from threadpoolctl import threadpool_limits

import midterm_part1_local_consensus as previous
import midterm_part1_local_followup as recent
from midterm_evaluation import REASONS, dataset_path

EMB = previous.EMB
EPS = 1e-12
BASE_CONFIG = dict(bandwidth=.5, strength=.1, beta=0.)
BANDWIDTHS = (.125, .25, .5, 1.)
STRENGTHS = (.01, .03, .1, .3)
BETAS = (0., .5, 1.)
GRID = [BASE_CONFIG.copy()] + [dict(bandwidth=b, strength=a, beta=0.) for b in BANDWIDTHS for a in STRENGTHS if (b,a)!=(.5,.1)]
STAGES = ('baseline', 'tuned', 'final')
PLAN = {
    'version':1, 'reference_public_score':.88568, 'reference_submission_sha256':recent.REFERENCE_SHA,
    'fixed_reference':BASE_CONFIG, 'geometry':'raw Euclidean, single global scale',
    'kernel':'exp(-b * squared_distance / median_nonzero_training_gold_squared_distance)',
    'boundary_grid':GRID, 'weight_exponents':BETAS,
    'weight_formula':'n_gold_in_current_fitting_group ** (-beta), normalized to mean 1; unlabeled and held rows do not count',
    'weighted_objective':'sum_i w_i * ||y_i-f(x_i)||^2 + alpha*||f||_H^2',
    'weighting_scope':'ridge loss only; gamma normalization and sigmoid calibration remain unweighted',
    'outer_folds':4, 'repeat_seeds':list(previous.SEEDS), 'inner_folds':3,
    'grouping':'same <=1e-4 raw Euclidean connected components; held groups removed from gold and feature pools',
    'calibration':'three grouped raw-score cross-fits; unweighted sigmoid C=1; same folds and seeds for all candidates',
    'selection':'stage1 best of16 inner OOF sample-F1; stage2 best beta for retained stage1 config; each stage needs gain >.002 over incoming stage',
    'tie_break':'reference first, then listed b/alpha order; beta0 first, then .5, then1',
    'decision':'fixed threshold .5 plus argmax for empty set; all evaluation is unweighted per trial',
    'candidate_gate':recent.PLAN['candidate_gate'],
    'export':recent.PLAN['export'],
    'caveat':recent.PLAN['caveat'],
}


def group_weights(groups, beta):
    groups = np.asarray(groups)
    if groups.ndim != 1 or not len(groups) or not np.isfinite(beta) or beta < 0:
        raise ValueError('Expected nonempty group vector and finite nonnegative beta')
    _, inverse, counts = np.unique(groups, return_inverse=True, return_counts=True)
    weights = counts[inverse].astype(float) ** (-beta)
    return weights / weights.mean()


def fit_base(scope, config):
    if config['beta'] == 0:
        return previous.fit_base(scope, {**recent.BASE_CONFIG, 'bandwidth':config['bandwidth'], 'strength':config['strength']})
    geom = previous.Geometry.fit(scope.data.x[scope.eligible], 'global')
    z = geom.transform(scope.data.x[scope.gold])
    distances = pdist(z, 'sqeuclidean')
    nonzero = distances[distances > 1e-12]
    median = float(np.median(nonzero)) if len(nonzero) else 1.
    gamma = config['bandwidth'] / median
    kernel = np.exp(-gamma * cdist(z, z, 'sqeuclidean'))
    weights = group_weights(scope.data.groups[scope.gold], config['beta'])
    coefficients = solve(kernel + np.diag(config['strength'] / weights), scope.data.y[scope.gold], assume_a='pos')
    return previous.BaseModel(geom, 'ridge', z, gamma, [], coefficients)


def fit_calibrated(scope, config, seed):
    raw = np.full((len(scope.gold), 5), np.nan)
    members = []
    for held, inner in scope.folds(seed):
        base = fit_base(inner, config)
        raw[held] = base.raw(scope.data.x[scope.gold[held]])
        members.append(base)
    if not np.isfinite(raw).all():
        raise ValueError('Incomplete calibration OOF scores')
    return previous.Calibrated(fit_base(scope, config), previous.Sigmoid.fit(raw, scope.data.y[scope.gold]), members)


def inner_predictions(scope, config, folds, seed):
    p = np.full((len(scope.gold), 5), np.nan)
    for k, (held, inner) in enumerate(folds):
        model = fit_calibrated(inner, config, seed + 100 + k)
        p[held] = model.predict(scope.data.x[scope.gold[held]])
    return p


@dataclass
class FittedBoundary:
    models: dict
    recipes: dict
    selection: dict

    def probabilities(self, x):
        return {stage:model.predict(x) for stage,model in self.models.items()}

    def predict(self, x):
        return self.models['final'].predict(x)

    def predict_sets(self, x):
        return {stage:previous.decide(p,.5) for stage,p in self.probabilities(x).items()}


def fit_procedure(scope, seed):
    y = scope.data.y[scope.gold]
    folds = list(scope.folds(seed))
    grid_scores, weight_scores = [], []
    best_config = BASE_CONFIG.copy()
    best_score = -np.inf
    baseline_score = None
    for config in GRID:
        p = inner_predictions(scope, config, folds, seed)
        value = previous.f1(y,p,.5)
        grid_scores.append(dict(config=config, sample_f1=value))
        if config == BASE_CONFIG:
            baseline_score = value
        if value > best_score + EPS:
            best_score, best_config = value, config.copy()
    if best_score <= baseline_score + .002 + EPS:
        best_score, best_config = baseline_score, BASE_CONFIG.copy()
    tuned_config, tuned_score = best_config.copy(), best_score
    weight_scores.append(dict(config=tuned_config.copy(), sample_f1=tuned_score))
    for beta in BETAS[1:]:
        config = {**tuned_config, 'beta':beta}
        p = inner_predictions(scope, config, folds, seed)
        value = previous.f1(y,p,.5)
        weight_scores.append(dict(config=config, sample_f1=value))
        if value > best_score + EPS:
            best_score, best_config = value, config.copy()
    if best_score <= tuned_score + .002 + EPS:
        best_score, best_config = tuned_score, tuned_config.copy()
    configs = {'baseline':BASE_CONFIG.copy(), 'tuned':tuned_config, 'final':best_config}
    recipes = {stage:{**config,'threshold':.5,'decoder':'threshold'} for stage,config in configs.items()}
    cache, models = {}, {}
    for stage,config in configs.items():
        key = json.dumps(config,sort_keys=True)
        if key not in cache:
            cache[key] = fit_calibrated(scope, config, seed+900)
        models[stage] = cache[key]
    selection = {'recipes':recipes, 'grid_scores':grid_scores, 'weight_scores':weight_scores,
        'inner_scores':{'baseline':baseline_score,'tuned':tuned_score,'final':best_score},
        'fit_gold_ids':scope.data.frame.iloc[scope.gold].ID.tolist(),
        'fit_feature_ids':scope.data.frame.iloc[scope.eligible].ID.tolist(),
        'selected_weights':group_weights(scope.data.groups[scope.gold],best_config['beta']).tolist()}
    return FittedBoundary(models,recipes,selection)


def save_breakdowns(table, out):
    reason_rows, year_rows, set_rows = [], [], []
    for evaluation, frame in table.groupby('evaluation'):
        for stage in STAGES:
            for reason in REASONS:
                y, p = frame['true_'+reason].to_numpy(), frame[f'{stage}_pred_{reason}'].to_numpy()
                tp, fp, fn = int((y*p).sum()), int(((1-y)*p).sum()), int((y*(1-p)).sum())
                reason_rows.append(dict(evaluation=evaluation,stage=stage,reason=reason,tp=tp,fp=fp,fn=fn,
                    f1=2*tp/(2*tp+fp+fn) if 2*tp+fp+fn else 0.))
        for year, sub in frame.groupby('start_year'):
            year_rows.append(dict(evaluation=evaluation,year=int(year),observations=len(sub),
                **{s:m['sample_f1'] for s,m in previous.summarize(sub).items()}))
        true_sets=frame[['true_'+r for r in REASONS]].apply(lambda row:' '.join(r for r,v in zip(REASONS,row) if v),axis=1)
        for true_set, sub in frame.groupby(true_sets):
            set_rows.append(dict(evaluation=evaluation,true_set=true_set,unique_trials=sub.ID.nunique(),observations=len(sub),
                **{s:m['sample_f1'] for s,m in previous.summarize(sub).items()}))
    pd.DataFrame(reason_rows).to_csv(out/'metrics_by_reason.csv',index=False)
    pd.DataFrame(year_rows).to_csv(out/'metrics_by_year.csv',index=False)
    pd.DataFrame(set_rows).to_csv(out/'metrics_by_true_set.csv',index=False)


def run(root, output):
    root, out = prepare_output(root, output)
    plan=json.loads(json.dumps(PLAN))
    if (out/'experiment_plan.json').exists() and json.loads((out/'experiment_plan.json').read_text())!=plan:
        raise ValueError('Recorded plan differs; use a new output directory')
    previous.write_json(out/'experiment_plan.json',plan)
    reference=root/'models/part1/03_kernel_ridge'
    before=previous.protected_hashes(root)
    for name in (recent.REFERENCE_FILE,'model.joblib'):
        before[str(reference/name)]=previous.digest(reference/name)
    previous.write_json(out/'protected_before.json',before)
    data,test=previous.load_data(root)
    full_scope=data.scope(data.gold,np.arange(len(data.frame)))
    saved=joblib.load(reference/'model.joblib').predict(test[EMB].to_numpy())
    refit=fit_calibrated(full_scope,BASE_CONFIG,942).predict(test[EMB].to_numpy())
    gap=float(np.max(np.abs(saved-refit)))
    if gap>1e-12 or previous.digest(reference/recent.REFERENCE_FILE)!=recent.REFERENCE_SHA:
        raise ValueError('Reference parity failed')
    print(f'Fixed .88568 reference probability gap: {gap}',flush=True)
    previous.write_json(out/'metadata.json',{'source_sha256':previous.digest(__file__),
        'dependencies':{Path(m.__file__).name:previous.digest(m.__file__) for m in (previous,recent)},
        'versions':{p:importlib.metadata.version(p) for p in ('numpy','pandas','scipy','scikit-learn')},
        'data_hashes':{s:previous.digest(dataset_path(root,'part1',s)) for s in ('train','val','test')},
        'reference_probability_gap':gap,'gold_rows':235,'gold_groups':len(np.unique(data.groups[data.gold]))})
    audit=data.frame.iloc[data.gold][['ID','source','start_year']].reset_index(drop=True)
    audit['group']=data.groups[data.gold]
    for beta in BETAS:
        audit[f'weight_beta_{beta}']=group_weights(data.groups[data.gold],beta)
    audit.to_csv(out/'group_weight_audit.csv',index=False)
    pieces,selections=[],[]
    started=time.time()
    for repeat,seed in enumerate(previous.SEEDS):
        for fold,(held,scope) in enumerate(full_scope.folds(seed,n=4)):
            fitted=fit_procedure(scope,seed+fold*17)
            frame=recent.prediction_frame(data,data.gold[held],fitted,'grouped_oof',repeat,fold)
            pieces.append(frame)
            selections.append(dict(repeat=repeat,fold=fold,**fitted.selection))
            print(f'repeat{repeat+1} fold{fold+1}: {json.dumps(previous.summarize(frame))}',flush=True)
    oof=pd.concat(pieces,ignore_index=True)
    if len(oof)!=705 or not oof.groupby(['repeat','ID']).size().eq(1).all():
        raise ValueError('Incomplete OOF')
    # The unchanged fixed reference must reproduce the last round on identical folds.
    last=pd.read_csv(root/'models/part1/exploration/local_followup/oof_predictions.csv').set_index(['repeat','fold','ID'])
    check=oof.set_index(['repeat','fold','ID']).loc[last.index]
    np.testing.assert_allclose(check[[f'baseline_prob_{r}' for r in REASONS]],last[[f'baseline_prob_{r}' for r in REASONS]],atol=1e-12,rtol=0)
    oof.to_csv(out/'oof_predictions.csv',index=False)
    previous.write_json(out/'fold_selections.json',selections)
    source,years=data.frame.source.to_numpy(),data.frame.start_year.to_numpy()
    specs={
        'historical_validation':(np.flatnonzero(source=='train'),np.flatnonzero(source!='validation'),np.flatnonzero(source=='validation')),
        'late_train_20':(np.flatnonzero((source=='train')&(years<=2006)),np.flatnonzero((source!='validation')&(years<=2006)),np.flatnonzero((source=='train')&(years>=2007)&(years<=2009))),
        'late_combined_56':(np.flatnonzero((source!='unlabeled')&(years<=2006)),np.flatnonzero(years<=2006),np.flatnonzero((source!='unlabeled')&(years>=2007)&(years<=2009))),
    }
    diagnostics,tables={},[oof]
    for name,(gold,eligible,held) in specs.items():
        fitted=fit_procedure(data.scope(gold,eligible),42)
        frame=recent.prediction_frame(data,held,fitted,name)
        frame['near_early_gold_match']=np.isin(data.groups[held],data.groups[gold])
        frame.to_csv(out/f'{name}_predictions.csv',index=False)
        diagnostics[name]={'metrics':previous.summarize(frame),'novel_group_metrics':previous.summarize(frame.loc[~frame.near_early_gold_match]),
            'rows':len(held),'gold_rows':len(gold),'selection':fitted.selection}
        tables.append(frame)
        print(name+': '+json.dumps(diagnostics[name]['metrics']),flush=True)
    previous.write_json(out/'diagnostics.json',diagnostics)
    table=pd.concat(tables,ignore_index=True)
    changed=table.loc[table.final_set!=table.baseline_set].copy()
    changed['sample_f1_delta']=changed.final_f1-changed.baseline_f1
    changed.to_csv(out/'changed_trials.csv',index=False)
    save_breakdowns(table,out)
    overall=previous.summarize(oof)
    repeats={str(k):previous.summarize(v) for k,v in oof.groupby('repeat')}
    sources={str(k):previous.summarize(v) for k,v in oof.groupby('source')}
    delta=lambda s:s['final']['sample_f1']-s['baseline']['sample_f1']
    gains=[delta(v) for v in repeats.values()]
    gates={'mean_oof_gain':delta(overall)>=.005-EPS,'positive_repeats':sum(g>EPS for g in gains)>=2,
        'worst_repeat':min(gains)>=-.005-EPS,'both_sources':all(delta(v)>=-.01-EPS for v in sources.values()),
        'late56':delta(diagnostics['late_combined_56']['metrics'])>=-.01-EPS,
        'historical_validation':delta(diagnostics['historical_validation']['metrics'])>=-.01-EPS}
    summary={'overall':overall,'by_repeat':repeats,'by_source':sources,'repeat_gains':gains,'mean_gain':delta(overall),
        'group_bootstrap_95_percent':recent.bootstrap(oof),'diagnostic_metrics':{k:v['metrics'] for k,v in diagnostics.items()},
        'candidate_gate':gates,'candidate_allowed':all(gates.values()),'candidate':None,
        'selection_counts':{stage:{key:sum(json.dumps(s['recipes'][stage],sort_keys=True)==key for s in selections)
            for key in sorted({json.dumps(s['recipes'][stage],sort_keys=True) for s in selections})} for stage in STAGES},
        'elapsed_seconds':time.time()-started,'caveat':PLAN['caveat']}
    if all(gates.values()):
        fitted=fit_procedure(full_scope,42)
        predictions=fitted.predict_sets(test[EMB].to_numpy())
        previous.write_json(out/'tools/archive/records/final_selection_nov01.json',fitted.selection)
        changed_count=int((predictions['final']!=predictions['baseline']).any(1).sum())
        if changed_count:
            submission=pd.DataFrame({'ID':test.ID,'Predicted':[' '.join(r for r,b in zip(REASONS,row) if b) for row in predictions['final']]})
            contents=submission.to_csv(index=False).encode()
            sha=hashlib.sha256(contents).hexdigest()
            filename=f'part1_ridge_boundary_{sha[:12]}.csv'
            (out/filename).write_bytes(contents)
            joblib.dump(fitted,out/'model.joblib')
            summary['candidate']={'filename':filename,'sha256':sha,'model_sha256':previous.digest(out/'model.joblib'),
                'rows':len(submission),'changed_vs_0_88568':changed_count,'kaggle_uploaded':False}
        else:
            summary['export_note']='Final decisions unchanged; no duplicate submission exported.'
    previous.write_json(out/'summary.json',summary)
    for path,sha in before.items():
        if previous.digest(path)!=sha:
            raise RuntimeError(f'Protected file changed: {path}')
    previous.write_json(out/'verification.json',{'protected_files_unchanged':True,'baseline_oof_matches_previous':True,
        'gold_rows':235,'oof_rows':len(oof),'outer_folds':12,
        'output_hashes':{p.name:previous.digest(p) for p in out.glob('*') if p.is_file() and p.name!='verification.json'}})
    print(json.dumps(summary,indent=2),flush=True)


if __name__=='__main__':
    from midterm_part1_ridge_boundary import main as run_main
    with threadpool_limits(limits=1):
        run_main()


if __name__ == "__main__":
    from replay import main
    main(default_study='ridge_boundary')

```


### midterm_part1_round2.py

Original member: `archive/midterm_before_cleanup_2026-11-02.zip ! midterm/midterm_part1_round2.py`

Original SHA-256: `3cf21506640e06895a2442c8732f460d08d75fb3482b70c6b2bdab02db505462`

Ported SHA-256: `c8550d6744e7cbdb02f287dc8443b08888413f092d3afbc3e211e5c4e6a7069c`

```python
"""Part 1 structure, embedding geometry, and temporal backtest study.

This writes a separate candidate study. The existing final is never replaced.
Held-out labels are excluded before any fitted transform, K-Means, or classifier.
"""
from __future__ import annotations

from _historical_support import prepare_output

import argparse
import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from scipy.special import softmax
from sklearn.cluster import KMeans
from sklearn.covariance import LedoitWolf
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.preprocessing import StandardScaler
from threadpoolctl import threadpool_limits

from midterm_augmentation import estimate_cluster_reason_rates, logistic_parameters, observed_label_mask
from midterm_evaluation import REASONS, dataset_path, file_sha256, part1_decisions
from midterm_next_round import p1_assignment
from midterm_part1_finalization import build_submission, predict_bundle
from midterm_part2_improvement import validate_submission
from midterm_refinement import EMBEDDINGS, prepare_reason_context

STUDY = Path("refinement/part1_structure_geometry_2026_11_02")
THRESHOLDS = (.35, .45, .55, .65)
DIMS = (30, 60, 100)
SEED = 42


def write_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(value, indent=2, allow_nan=False) + "\n")


def write_locked_json(path, value):
    path = Path(path)
    if path.exists() and json.loads(path.read_text()) != value:
        raise ValueError(f"Locked selection changed: {path}")
    write_json(path, value)


def score(y, decision):
    return float(f1_score(y, decision, average="samples", zero_division=0))


def make_decisions(kind, probabilities, secondary=None, threshold=None):
    p = np.asarray(probabilities)
    if kind == "binary":
        return part1_decisions(p, .35, empty_set_policy="argmax")
    decision = np.zeros((len(p), len(REASONS)), dtype=int)
    first = p.argmax(axis=1)
    decision[np.arange(len(p)), first] = 1
    if kind == "primary_secondary":
        other = np.asarray(secondary).copy()
        other[np.arange(len(p)), first] = -1
        second = other.argmax(axis=1)
        take = other[np.arange(len(p)), second] >= threshold
        decision[np.flatnonzero(take), second[take]] = 1
    return decision


def fit_primary(fit):
    labels = fit[list(REASONS)].to_numpy()
    observed = observed_label_mask(labels)
    single = observed & ((labels == 1).sum(axis=1) == 1)
    x = fit[EMBEDDINGS].to_numpy(dtype=float)
    scaler = StandardScaler().fit(x)
    y = labels[single].argmax(axis=1)
    if set(y) != set(range(len(REASONS))):
        raise ValueError("Every primary class must have a single-label gold example.")
    model = LogisticRegression(C=.1, solver="lbfgs", max_iter=1000, random_state=SEED)
    with threadpool_limits(limits=1):
        model.fit(scaler.transform(x[single]), y)
    return {"kind": "primary", "scaler": scaler, "model": model,
            "gold_single_ids": fit.loc[single, "ID"].tolist()}


def primary_scores(bundle, frame):
    p = bundle["model"].predict_proba(bundle["scaler"].transform(frame[EMBEDDINGS].to_numpy()))
    if not np.array_equal(bundle["model"].classes_, np.arange(len(REASONS))):
        raise ValueError("Unexpected primary class order.")
    return p


class CovarianceGeometry:
    """Rotation-aware PCA projection with Ledoit-Wolf shrunk whitening."""

    def __init__(self, dimensions):
        self.dimensions = dimensions

    def fit(self, x):
        x = np.asarray(x, dtype=float)
        estimator = LedoitWolf(assume_centered=False).fit(x)
        eigenvalues, eigenvectors = np.linalg.eigh(estimator.covariance_)
        order = np.argsort(eigenvalues)[::-1][:self.dimensions]
        self.mean_ = estimator.location_
        self.rotation_ = eigenvectors[:, order] / np.sqrt(eigenvalues[order])[None, :]
        self.eigenvalues_ = eigenvalues[order]
        self.shrinkage_ = float(estimator.shrinkage_)
        return self

    def transform(self, x):
        return (np.asarray(x, dtype=float) - self.mean_) @ self.rotation_


def fit_geometry(fit, dimensions):
    labels = fit[list(REASONS)].to_numpy()
    observed = observed_label_mask(labels)
    transform = CovarianceGeometry(dimensions).fit(fit[EMBEDDINGS].to_numpy())
    x = transform.transform(fit[EMBEDDINGS].to_numpy())
    clusterer = KMeans(n_clusters=10, n_init=20, max_iter=500, random_state=SEED)
    with threadpool_limits(limits=1):
        clusterer.fit(x)
    membership = np.eye(10)[clusterer.labels_]
    mapping = estimate_cluster_reason_rates(membership, labels, 2.0)
    pseudo = (membership[~observed] @ mapping["rates"] >= .5).astype(int)
    recovered = labels.copy().astype(int)
    recovered[~observed] = pseudo
    weights = np.where(observed, 1., .1)
    models = []
    for j in range(len(REASONS)):
        model = LogisticRegression(**{**logistic_parameters(SEED), "C": .1})
        with threadpool_limits(limits=1):
            model.fit(x, recovered[:, j], sample_weight=weights)
        models.append(model)
    return {"kind": "geometry", "dimensions": dimensions, "transform": transform,
            "clusterer": clusterer, "mapping": mapping, "models": models,
            "gold_ids": fit.loc[observed, "ID"].tolist()}


def geometry_scores(bundle, frame):
    x = bundle["transform"].transform(frame[EMBEDDINGS].to_numpy())
    return np.column_stack([m.predict_proba(x)[:, 1] for m in bundle["models"]])


def fit_centroid(fit, dimensions):
    labels = fit[list(REASONS)].to_numpy()
    observed = observed_label_mask(labels)
    single = observed & ((labels == 1).sum(axis=1) == 1)
    transform = CovarianceGeometry(dimensions).fit(fit[EMBEDDINGS].to_numpy())
    x = transform.transform(fit[EMBEDDINGS].to_numpy())
    y = labels[single].argmax(axis=1)
    if set(y) != set(range(len(REASONS))):
        raise ValueError("Every class needs a centroid.")
    centers = np.vstack([x[single][y == j].mean(axis=0) for j in range(len(REASONS))])
    prior = np.bincount(y, minlength=len(REASONS)) / len(y)
    return {"kind": "centroid", "dimensions": dimensions, "transform": transform,
            "centers": centers, "prior": prior, "gold_single_ids": fit.loc[single, "ID"].tolist()}


def centroid_scores(bundle, frame):
    x = bundle["transform"].transform(frame[EMBEDDINGS].to_numpy())
    distance = ((x[:, None, :] - bundle["centers"][None, :, :]) ** 2).sum(axis=2)
    return softmax(-distance / 2 + np.log(bundle["prior"])[None, :], axis=1)


def fit_control(fit):
    from midterm_next_round import fit_reason_weight
    context = prepare_reason_context(fit, 10)
    return fit_reason_weight(fit, context, .1)


def control_scores(bundle, frame):
    x = bundle["scaler"].transform(frame[EMBEDDINGS])
    return np.column_stack([m.predict_proba(x)[:, 1] for m in bundle["models"]])


def fit_candidate(name, fit):
    if name == "control":
        return fit_control(fit)
    if name == "primary":
        return fit_primary(fit)
    method, dimensions = name.split("_")
    return fit_geometry(fit, int(dimensions)) if method == "geometry" else fit_centroid(fit, int(dimensions))


def predict_candidate(name, bundle, frame):
    if name == "control":
        return control_scores(bundle, frame)
    if name == "primary":
        return primary_scores(bundle, frame)
    if name.startswith("geometry_"):
        return geometry_scores(bundle, frame)
    return centroid_scores(bundle, frame)


def metrics_rows(name, split, reference, decisions, probabilities=None):
    y = reference[list(REASONS)].to_numpy(dtype=int)
    rows = [{"model": name, "split": split, "group": "all", "n": len(y), "f1_samples": score(y, decisions),
             "predicted_multi": int((decisions.sum(axis=1) > 1).sum())}]
    for label, mask in (("single_truth", y.sum(axis=1) == 1), ("multi_truth", y.sum(axis=1) > 1),
                        ("late_2007_2009", reference.start_year.between(2007, 2009).to_numpy())):
        if mask.any():
            rows.append({"model": name, "split": split, "group": label, "n": int(mask.sum()),
                         "f1_samples": score(y[mask], decisions[mask]),
                         "predicted_multi": int((decisions[mask].sum(axis=1) > 1).sum())})
    for j, reason in enumerate(REASONS):
        rows.append({"model": name, "split": split, "group": reason, "n": len(y),
                     "f1_samples": float(f1_score(y[:, j], decisions[:, j], zero_division=0)),
                     "predicted_multi": None})
    return rows


def run(root, output):
    root, out = prepare_output(root, output)
    train = pd.read_csv(dataset_path(root, "part1", "train"))
    val = pd.read_csv(dataset_path(root, "part1", "val"))
    test = pd.read_csv(dataset_path(root, "part1", "test"))
    gold, assignment = p1_assignment(train)
    names = ["control", "primary", *[f"geometry_{d}" for d in DIMS], *[f"centroid_{d}" for d in DIMS]]
    write_json(out / "plan.json", {"models": names, "secondary_thresholds": THRESHOLDS,
        "folds": 4, "seed": SEED, "primary_training": "single-label gold only",
        "secondary_training": "existing K-Means augmented binary reason model",
        "geometry": "Ledoit-Wolf shrunk covariance eigenvectors, top 30/60/100, inverse-square-root scaling",
        "temporal_fit": "train rows start_year <= 2006, held-out train gold 2007-2009",
        "validation_in_selection": False, "test_labels_used": False,
        "input_sha256": {split: file_sha256(dataset_path(root, "part1", split)) for split in ("train", "val", "test")}})
    oof = {name: np.full((len(gold), len(REASONS)), np.nan) for name in names}
    fold_by_id = assignment.set_index("ID").loc[gold.ID, "fold"].to_numpy()
    for fold in range(4):
        held = gold.loc[fold_by_id == fold].reset_index(drop=True)
        fit = train.loc[~train.ID.isin(held.ID)].reset_index(drop=True)
        assert set(fit.ID).isdisjoint(held.ID)
        for name in names:
            bundle = fit_candidate(name, fit)
            oof[name][fold_by_id == fold] = predict_candidate(name, bundle, held)
        print(f"OOF fold {fold + 1}/4 done", flush=True)
    for name in names:
        assert np.isfinite(oof[name]).all()
    y = gold[list(REASONS)].to_numpy(dtype=int)
    decisions = {"control": make_decisions("binary", oof["control"]),
                 "primary_only": make_decisions("primary", oof["primary"])}
    structure_grid = []
    for t in THRESHOLDS:
        d = make_decisions("primary_secondary", oof["primary"], oof["control"], t)
        structure_grid.append({"threshold": t, "oof_f1_samples": score(y, d),
                               "oof_predicted_multi": int((d.sum(axis=1) > 1).sum())})
    threshold = max(structure_grid, key=lambda row: (row["oof_f1_samples"], row["threshold"]))["threshold"]
    decisions["primary_secondary"] = make_decisions("primary_secondary", oof["primary"], oof["control"], threshold)
    for name in names:
        if name.startswith("geometry_"):
            decisions[name] = make_decisions("binary", oof[name])
        elif name.startswith("centroid_"):
            decisions[name] = make_decisions("primary", oof[name])
    oof_rows = []
    for name, d in decisions.items():
        oof_rows.extend(metrics_rows(name, "oof", gold, d))
    pd.DataFrame(oof_rows).to_csv(out / "oof_metrics.csv", index=False)
    pd.DataFrame(structure_grid).to_csv(out / "primary_secondary_threshold_oof.csv", index=False)
    flat_oof = gold[["ID", "start_year", *REASONS]].copy()
    flat_oof["fold"] = fold_by_id
    for name in names:
        for j, reason in enumerate(REASONS):
            flat_oof[f"{name}_prob_{reason}"] = oof[name][:, j]
    for name, d in decisions.items():
        for j, reason in enumerate(REASONS):
            flat_oof[f"{name}_pred_{reason}"] = d[:, j]
    flat_oof.to_csv(out / "oof_predictions.csv", index=False)
    # Pre-register the OOF winner before evaluating validation.
    oof_all = {name: score(y, d) for name, d in decisions.items()}
    ranked = sorted(decisions, key=lambda name: (-oof_all[name], name))
    write_locked_json(out / "selection_locked_before_validation.json", {
        "oof_ranked": ranked, "oof_sample_f1": oof_all, "secondary_threshold": threshold,
        "validation_labels_seen": False, "selection_rule": "highest pooled OOF sample F1; time backtest and validation are directional checks"})
    # Temporal backtest excludes every row later than 2006 from fitting.
    temporal_fit = train.loc[train.start_year <= 2006].reset_index(drop=True)
    temporal_hold = gold.loc[gold.start_year.between(2007, 2009)].reset_index(drop=True)
    temporal_scores = {name: predict_candidate(name, fit_candidate(name, temporal_fit), temporal_hold) for name in names}
    td = {"control": make_decisions("binary", temporal_scores["control"]),
          "primary_only": make_decisions("primary", temporal_scores["primary"]),
          "primary_secondary": make_decisions("primary_secondary", temporal_scores["primary"], temporal_scores["control"], threshold)}
    for name in names:
        if name.startswith("geometry_"):
            td[name] = make_decisions("binary", temporal_scores[name])
        elif name.startswith("centroid_"):
            td[name] = make_decisions("primary", temporal_scores[name])
    pd.DataFrame([row for name, d in td.items() for row in metrics_rows(name, "temporal", temporal_hold, d)]).to_csv(out / "temporal_metrics.csv", index=False)
    # Provided validation is a single pre-refit check, not an independent test after refit.
    full_models = {name: fit_candidate(name, train) for name in names}
    val_scores = {name: predict_candidate(name, full_models[name], val) for name in names}
    vd = {"control": make_decisions("binary", val_scores["control"]),
          "primary_only": make_decisions("primary", val_scores["primary"]),
          "primary_secondary": make_decisions("primary_secondary", val_scores["primary"], val_scores["control"], threshold)}
    for name in names:
        if name.startswith("geometry_"):
            vd[name] = make_decisions("binary", val_scores[name])
        elif name.startswith("centroid_"):
            vd[name] = make_decisions("primary", val_scores[name])
    pd.DataFrame([row for name, d in vd.items() for row in metrics_rows(name, "validation", val, d)]).to_csv(out / "validation_metrics_before_refit.csv", index=False)
    frozen = joblib.load(root / 'models/part1/02_kmeans_lr/model.joblib')
    frozen_val = make_decisions("binary", predict_bundle(frozen, val))
    frozen_score = score(val[list(REASONS)].to_numpy(dtype=int), frozen_val)
    frozen_test_hash = file_sha256(root / 'models/part1/02_kmeans_lr/submission.csv')
    # Gate requires improvement in OOF, time backtest, and original validation.
    temporal_all = {name: score(temporal_hold[list(REASONS)].to_numpy(dtype=int), d) for name, d in td.items()}
    val_all = {name: score(val[list(REASONS)].to_numpy(dtype=int), d) for name, d in vd.items()}
    eligible = [name for name in ranked if name != "control"
                and oof_all[name] > oof_all["control"] + 1e-12
                and temporal_all[name] >= temporal_all["control"] - 1e-12
                and val_all[name] >= frozen_score - 1e-12]
    selected = eligible[0] if eligible else "control"
    write_locked_json(out / "pre_refit_selection.json", {"selected": selected, "eligible": eligible,
        "oof_sample_f1": oof_all, "temporal_sample_f1": temporal_all,
        "validation_sample_f1": val_all, "frozen_validation_f1": frozen_score,
        "validation_labels_used_in_fitting_at_this_point": False,
        "old_final_submission_sha256": frozen_test_hash})
    # The final candidate uses all 235 visible labels only after the selection is frozen.
    combined = pd.concat([train, val], ignore_index=True)
    if combined.ID.duplicated().any() or observed_label_mask(combined[list(REASONS)].to_numpy()).sum() != 235:
        raise ValueError("Expected 235 unique visible gold labels in train+validation.")
    final_name = "primary" if selected == "primary_secondary" else selected
    refit = {final_name: fit_candidate(final_name, combined)}
    if selected == "primary_secondary":
        refit["control"] = fit_candidate("control", combined)
    test_scores = {name: predict_candidate(name, model, test) for name, model in refit.items()}
    if selected == "primary_secondary":
        test_decision = make_decisions("primary_secondary", test_scores["primary"], test_scores["control"], threshold)
    elif selected == "control" or selected.startswith("geometry_"):
        test_decision = make_decisions("binary", test_scores[selected])
    else:
        test_decision = make_decisions("primary", test_scores[selected])
    predictions = pd.DataFrame(test_decision, columns=REASONS)
    predictions.insert(0, "ID", test.ID.to_numpy())
    sample = pd.read_csv(dataset_path(root, "part1", "test").parent / "part1_sample_submission.csv")
    submission = build_submission(sample, predictions)
    submission_path = out / "part1_refit_235_candidate.csv"
    submission.to_csv(submission_path, index=False)
    validate_submission(submission, sample, 1)
    joblib.dump({"selected": selected, "models": refit, "threshold": threshold,
                "label_source": "part1_train observed + part1_val", "gold_count": 235},
                out / "refit_model.joblib", compress=3)
    predictions.to_csv(out / "test_decisions.csv", index=False)
    if file_sha256(root / 'models/part1/02_kmeans_lr/submission.csv') != frozen_test_hash:
        raise RuntimeError("Original Part 1 final was modified.")
    write_json(out / "summary.json", {"selected": selected, "oof_f1": oof_all[selected],
        "time_f1": temporal_all[selected], "pre_refit_validation_f1": val_all[selected],
        "frozen_validation_f1": frozen_score, "refit_gold_count": 235,
        "test_predicted_multi": int((test_decision.sum(axis=1) > 1).sum()),
        "candidate_path": str(submission_path.relative_to(out)),
        "candidate_sha256": file_sha256(submission_path), "original_final_replaced": False,
        "kaggle_uploaded": False,
        "warning": "Validation labels entered final refit; validation score is no longer an independent estimate for that fitted model."})
    print(json.dumps(json.loads((out / "summary.json").read_text()), indent=2), flush=True)




if __name__ == "__main__":
    from replay import main
    main(default_study='structure_geometry')

```


### midterm_part1_val_refit_weight.py

Original member: `archive/midterm_before_cleanup_2026-11-02.zip ! midterm/midterm_part1_val_refit_weight.py`

Original SHA-256: `c794e8982181d2b529fb2146a85788e8b6225bf26e1f406cda84bbaeb43f8505`

Ported SHA-256: `0f83e26ec643f2ac3b86ee075cec8db657fabfd8e609ba66c3d57b9de307a920`

```python
"""Post-selection sensitivity: include all 78 validation labels with bounded weight.

Cluster mapping stays fitted on train only. Validation labels enter only the
final classifier. Each OOF/temporal held-out ID is excluded before any fitting.
"""
from __future__ import annotations

from _historical_support import prepare_output

import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from threadpoolctl import threadpool_limits

from midterm_augmentation import logistic_parameters, observed_label_mask
from midterm_evaluation import REASONS, dataset_path, file_sha256
from midterm_next_round import p1_assignment
from midterm_part1_finalization import build_submission
from midterm_part1_round2 import make_decisions, metrics_rows, score
from midterm_part2_improvement import validate_submission
from midterm_refinement import EMBEDDINGS, prepare_reason_context

ROOT = None  # Set only by explicit replay(root, output).
OUT = None
WEIGHTS = (.25, .5, 1.)


def fit_with_extra(train, extra, weight):
    if set(train.ID) & set(extra.ID):
        raise ValueError("Training and extra label IDs overlap.")
    if not observed_label_mask(extra[list(REASONS)].to_numpy()).all():
        raise ValueError("Extra validation labels must be observed.")
    context = prepare_reason_context(train, 10)
    observed = context["observed"]
    x = context["x"]
    labels = train[list(REASONS)].to_numpy(dtype=int).copy()
    labels[~observed] = (context["pseudo_q"] >= .5).astype(int)
    extra_x = context["scaler"].transform(extra[EMBEDDINGS])
    both_x = np.vstack([x, extra_x])
    both_y = np.vstack([labels, extra[list(REASONS)].to_numpy(dtype=int)])
    sample_weights = np.concatenate([np.where(observed, 1., .1), np.full(len(extra), weight)])
    models = []
    for j in range(len(REASONS)):
        model = LogisticRegression(**{**logistic_parameters(42), "C": .1})
        with threadpool_limits(limits=1):
            model.fit(both_x, both_y[:, j], sample_weight=sample_weights)
        models.append(model)
    return {"scaler": context["scaler"], "clusterer": context["clusterer"],
            "mapping": context["mapping"], "models": models, "val_weight": weight,
            "fit_train_ids": train.ID.tolist(), "fit_extra_ids": extra.ID.tolist()}


def predict(bundle, frame):
    x = bundle["scaler"].transform(frame[EMBEDDINGS])
    return np.column_stack([model.predict_proba(x)[:, 1] for model in bundle["models"]])


def run(root, output):
    global ROOT, OUT
    ROOT, OUT = prepare_output(root, output)
    OUT.mkdir(parents=True, exist_ok=True)
    train = pd.read_csv(dataset_path(ROOT, "part1", "train"))
    val = pd.read_csv(dataset_path(ROOT, "part1", "val"))
    test = pd.read_csv(dataset_path(ROOT, "part1", "test"))
    gold, assignment = p1_assignment(train)
    fold = assignment.set_index("ID").loc[gold.ID, "fold"].to_numpy()
    combined = pd.concat([train, val], ignore_index=True)
    all_gold = combined.loc[observed_label_mask(combined[list(REASONS)].to_numpy())]
    temporal_hold = all_gold.loc[all_gold.start_year.between(2007, 2009)]
    temporal_train = train.loc[train.start_year <= 2006]
    temporal_extra = val.loc[val.start_year <= 2006]
    assert len(temporal_hold) == 56 and set(temporal_train.ID).isdisjoint(temporal_hold.ID)
    assert set(temporal_extra.ID).isdisjoint(temporal_hold.ID)
    rows = []
    checks = []
    for weight in WEIGHTS:
        oof = np.full((len(gold), len(REASONS)), np.nan)
        for k in range(4):
            held = gold.loc[fold == k]
            fit_train = train.loc[~train.ID.isin(held.ID)]
            assert set(fit_train.ID).isdisjoint(held.ID) and set(val.ID).isdisjoint(held.ID)
            oof[fold == k] = predict(fit_with_extra(fit_train, val, weight), held)
        d_oof = make_decisions("binary", oof)
        d_time = make_decisions("binary", predict(fit_with_extra(temporal_train, temporal_extra, weight), temporal_hold))
        name = f"val_weight_{weight:g}"
        rows.extend(metrics_rows(name, "train_gold_oof", gold, d_oof))
        rows.extend(metrics_rows(name, "temporal_56", temporal_hold, d_time))
        checks.append({"weight": weight, "oof_157_f1": score(gold[list(REASONS)], d_oof),
                       "time_56_f1": score(temporal_hold[list(REASONS)], d_time),
                       "time_predicted_multi": int((d_time.sum(axis=1) > 1).sum())})
        print(name, checks[-1], flush=True)
    pd.DataFrame(rows).to_csv(OUT / "metrics.csv", index=False)
    pd.DataFrame(checks).to_csv(OUT / "comparison.csv", index=False)
    # Fixed controls from identical folds/splits; no post-refit validation metric.
    baseline_oof = .8895966029723991
    baseline_time = .869047619047619
    qualified = [row for row in checks if row["oof_157_f1"] > baseline_oof + 1e-12
                 and row["time_56_f1"] >= baseline_time - 1e-12]
    selected = sorted(qualified, key=lambda row: (-row["time_56_f1"], -row["oof_157_f1"], row["weight"]))[0] if qualified else None
    summary = {"baseline_oof_157_f1": baseline_oof, "baseline_time_56_f1": baseline_time,
               "selected": selected, "all_labels_235_used_in_final_fit": bool(selected),
               "original_final_replaced": False, "kaggle_uploaded": False,
               "caveat": "Exploratory after earlier validation inspection; no post-refit independent validation score."}
    if selected:
        bundle = fit_with_extra(train, val, selected["weight"])
        d_test = make_decisions("binary", predict(bundle, test))
        pred = pd.DataFrame(d_test, columns=REASONS)
        pred.insert(0, "ID", test.ID.to_numpy())
        sample = pd.read_csv(dataset_path(ROOT, "part1", "test").parent / "part1_sample_submission.csv")
        submission = build_submission(sample, pred)
        path = OUT / "part1_val_weight_candidate.csv"
        submission.to_csv(path, index=False)
        validate_submission(submission, sample, 1)
        joblib.dump(bundle, OUT / "selected_model.joblib", compress=3)
        summary["candidate_path"] = str(path.relative_to(OUT))
        summary["candidate_sha256"] = file_sha256(path)
        summary["test_predicted_multi"] = int((d_test.sum(axis=1) > 1).sum())
    (OUT / "summary.json").write_text(json.dumps(summary, indent=2, allow_nan=False) + "\n")
    print(json.dumps(summary, indent=2), flush=True)




if __name__ == "__main__":
    from replay import main
    main(default_study='val_refit_weight')

```


### midterm_rank_part1.py

Original member: `archive/midterm_before_cleanup_2026-11-02.zip ! midterm/midterm_rank_part1.py`

Original SHA-256: `151f2c2cba72a8a834191e92764caf663bb988eb2fa4f357b6a072fc324370ed`

Ported SHA-256: `972b59415afd73c0d0caf5ecd3851367fff3becd628793c5586f8ee86fb9297b`

```python
"""New Part 1 candidate studies; never replaces the locked final submission.

Thresholds use frozen gold out-of-fold probabilities. Selective pseudo labels
are recomputed inside every gold holdout, so held-out reasons never enter K-Means
mapping, scaling, or classifier fitting.
"""
from __future__ import annotations

from _historical_support import prepare_output

import argparse
import itertools
import json
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from threadpoolctl import threadpool_limits

from midterm_augmentation import logistic_parameters, observed_label_mask
from midterm_evaluation import REASONS, dataset_path, evaluate_part1, file_sha256
from midterm_next_round import verify_oof
from midterm_part1_finalization import build_submission, predict_bundle
from midterm_part2_improvement import validate_submission
from midterm_refinement import EMBEDDINGS, prepare_reason_context

STUDY = Path("refinement/rank_improvement_2026_11_02/part1")
BASE_THRESHOLD = .35
THRESHOLD_GRID = (.25, .30, .35, .40, .45)
PSEUDO_SETTINGS = tuple(
    {"positive_cutoff": pos, "negative_cutoff": neg, "pseudo_weight": weight}
    for pos, neg, weight in itertools.product((.50, .60), (.10, .20), (.03, .10))
)


def write_json(path: Path, obj) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(obj, indent=2, allow_nan=False) + "\n")


def scores_from_saved(frame: pd.DataFrame) -> pd.DataFrame:
    return frame[["ID", *[f"prob_{r}" for r in REASONS]]].rename(
        columns={f"prob_{r}": r for r in REASONS}
    )


def load_gold_oof(root: Path):
    train = pd.read_csv(dataset_path(root, "part1", "train"))
    gold = train.loc[observed_label_mask(train[list(REASONS)].to_numpy())].reset_index(drop=True)
    assignment = pd.read_csv(root / 'models/part1/exploration/replay_inputs/oof_weight_char/fold_assignments.csv')
    assignment = assignment.set_index("ID").loc[gold.ID].reset_index()
    saved = pd.read_csv(root / 'models/part1/exploration/replay_inputs/oof_weight_char/oof_weight_010.csv')
    oof = verify_oof(gold, saved, assignment, part=1)
    return train, gold, assignment, oof


def choose_reason_thresholds(gold: pd.DataFrame, scores: pd.DataFrame):
    """Five one-dimensional OOF checks, with Funding fixed for low support.

    A reason changes only if its OOF F1 improves by at least .02. This is a
    simple predeclared guard against shifting a threshold on an exact tie.
    """
    thresholds = dict.fromkeys(REASONS, BASE_THRESHOLD)
    rows = []
    aligned = scores.set_index("ID").loc[gold.ID]
    for reason in REASONS:
        truth = gold[reason].to_numpy(dtype=int)
        base = f1_score(truth, aligned[reason].to_numpy() >= BASE_THRESHOLD, zero_division=0)
        candidates = []
        for threshold in THRESHOLD_GRID:
            value = f1_score(truth, aligned[reason].to_numpy() >= threshold, zero_division=0)
            candidates.append((threshold, value))
            rows.append({"reason": reason, "threshold": threshold, "oof_reason_f1": value,
                         "positive_support": int(truth.sum())})
        if reason != "Funding":
            best = sorted(candidates, key=lambda pair: (-pair[1], abs(pair[0] - BASE_THRESHOLD), pair[0]))[0]
            if best[1] >= base + .02:
                thresholds[reason] = best[0]
    return thresholds, pd.DataFrame(rows)


def threshold_study(root: Path, folder: Path):
    train, gold, assignment, oof = load_gold_oof(root)
    source = scores_from_saved(oof)
    selected, grid = choose_reason_thresholds(gold, source)
    control_oof = evaluate_part1(gold, source, BASE_THRESHOLD, empty_set_policy="argmax")
    candidate_oof = evaluate_part1(gold, source, selected, empty_set_policy="argmax")
    out = folder / "thresholds"
    out.mkdir(parents=True, exist_ok=True)
    grid.to_csv(out / "per_reason_oof_grid.csv", index=False)
    control_oof.save(out / "oof_control")
    candidate_oof.save(out / "oof_candidate")
    fold_rows = []
    for fold in sorted(assignment.fold.unique()):
        mask = assignment.fold.eq(fold).to_numpy()
        for name, rule in (("control", BASE_THRESHOLD), ("candidate", selected)):
            result = evaluate_part1(gold.loc[mask], source.loc[mask], rule, empty_set_policy="argmax")
            fold_rows.append({"fold": int(fold), "setting": name, **result.summary})
    pd.DataFrame(fold_rows).to_csv(out / "oof_by_fold.csv", index=False)
    write_json(out / "selection_locked_before_validation.json", {
        "thresholds": selected, "grid": THRESHOLD_GRID, "funding_fixed": True,
        "selection": "per-reason gold OOF F1, minimum +0.02; then sample F1 gate",
        "validation_labels_used": False, "test_labels_used": False,
    })
    validation = pd.read_csv(dataset_path(root, "part1", "val"))
    current = scores_from_saved(pd.read_csv(root / 'models/part1/02_kmeans_lr/evidence/selected/selected_validation/validation_predictions.csv'))
    model = joblib.load(root / 'models/part1/02_kmeans_lr/model.joblib')
    actual = pd.DataFrame(predict_bundle(model, validation), columns=REASONS)
    actual.insert(0, "ID", validation.ID.to_numpy())
    if not np.allclose(actual[list(REASONS)], current.set_index("ID").loc[validation.ID, list(REASONS)], atol=1e-12, rtol=0):
        raise ValueError("Frozen final model no longer matches saved validation probabilities.")
    control_val = evaluate_part1(validation, current, BASE_THRESHOLD, empty_set_policy="argmax")
    candidate_val = evaluate_part1(validation, current, selected, empty_set_policy="argmax")
    control_val.save(out / "validation_control")
    candidate_val.save(out / "validation_candidate")
    oof_gain = candidate_oof.summary["f1_samples"] - control_oof.summary["f1_samples"]
    val_gain = candidate_val.summary["f1_samples"] - control_val.summary["f1_samples"]
    passes = bool(selected != dict.fromkeys(REASONS, BASE_THRESHOLD) and oof_gain > 1e-12 and val_gain > 1e-12)
    summary = {"selected_thresholds": selected, "oof_control_f1": control_oof.summary["f1_samples"],
               "oof_candidate_f1": candidate_oof.summary["f1_samples"],
               "validation_control_f1": control_val.summary["f1_samples"],
               "validation_candidate_f1": candidate_val.summary["f1_samples"],
               "passes_local_gate": passes, "final_replaced": False, "kaggle_uploaded": False}
    if passes:
        test = pd.read_csv(dataset_path(root, "part1", "test"))
        prob = pd.DataFrame(predict_bundle(model, test), columns=REASONS)
        prob.insert(0, "ID", test.ID.to_numpy())
        from midterm_evaluation import part1_decisions
        pred = pd.DataFrame(part1_decisions(prob[list(REASONS)], selected, empty_set_policy="argmax"), columns=REASONS)
        pred.insert(0, "ID", test.ID.to_numpy())
        sample = pd.read_csv(dataset_path(root, "part1", "test").parent / "part1_sample_submission.csv")
        submission = build_submission(sample, pred)
        submission.to_csv(out / "part1_threshold_candidate.csv", index=False)
        validate_submission(submission, sample, 1)
        summary["candidate_submission"] = str((out / "part1_threshold_candidate.csv").relative_to(out))
    write_json(out / "summary.json", summary)
    return summary


def fit_selective(train: pd.DataFrame, context: dict, setting: dict):
    observed = context["observed"]
    q = context["pseudo_q"]
    x = context["x"]
    gold_y = train.loc[observed, list(REASONS)].to_numpy(dtype=int)
    pseudo_x = x[~observed]
    models, counts = [], []
    for j, reason in enumerate(REASONS):
        high = q[:, j] >= setting["positive_cutoff"]
        low = q[:, j] <= setting["negative_cutoff"]
        # Do not teach a rare class from pseudo negatives alone.
        if not high.any():
            low[:] = False
        selected = high | low
        target = np.concatenate([gold_y[:, j], high[selected].astype(int)])
        features = np.vstack([x[observed], pseudo_x[selected]])
        weights = np.concatenate([np.ones(len(gold_y)), np.full(int(selected.sum()), setting["pseudo_weight"])])
        if len(np.unique(target)) != 2:
            raise ValueError(f"No two observed classes for {reason}.")
        model = LogisticRegression(**{**logistic_parameters(42), "C": .1})
        with threadpool_limits(limits=1), warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always")
            model.fit(features, target, sample_weight=weights)
        if any(issubclass(w.category, ConvergenceWarning) for w in caught) or model.n_iter_.max() >= model.max_iter:
            raise RuntimeError(f"Selective pseudo model did not converge for {reason}.")
        models.append(model)
        counts.append({"reason": reason, "pseudo_positive": int(high.sum()),
                       "pseudo_negative": int(low.sum()), "pseudo_used": int(selected.sum()),
                       "gold_positive": int(gold_y[:, j].sum())})
    return {"scaler": context["scaler"], "models": models, "setting": setting,
            "fit_ids": train.ID.tolist(), "gold_ids": train.loc[observed, "ID"].tolist()}, pd.DataFrame(counts)


def predict_selective(bundle: dict, frame: pd.DataFrame) -> pd.DataFrame:
    x = bundle["scaler"].transform(frame[EMBEDDINGS])
    values = np.column_stack([m.predict_proba(x)[:, 1] for m in bundle["models"]])
    result = pd.DataFrame(values, columns=REASONS)
    result.insert(0, "ID", frame.ID.to_numpy())
    return result


def pseudo_study(root: Path, folder: Path):
    train, gold, assignment, saved_control = load_gold_oof(root)
    out = folder / "selective_pseudo"
    out.mkdir(parents=True, exist_ok=True)
    write_json(out / "experiment_plan.json", {
        "settings": PSEUDO_SETTINGS, "kmeans_clusters": 10, "lr_C": .1,
        "decision_threshold": BASE_THRESHOLD, "funding_or_other_reason_without_confident_positive": "gold_only",
        "selection": "pooled gold OOF sample F1", "folds": sorted(int(x) for x in assignment.fold.unique()),
        "validation_labels_used": False, "test_labels_used": False,
    })
    frames = {i: [] for i in range(len(PSEUDO_SETTINGS))}
    count_rows = []
    fold_rows = []
    for fold in sorted(assignment.fold.unique()):
        held_ids = set(assignment.loc[assignment.fold.eq(fold), "ID"])
        fit = train.loc[~train.ID.isin(held_ids)].reset_index(drop=True)
        held = gold.loc[gold.ID.isin(held_ids)].reset_index(drop=True)
        if set(fit.ID) & held_ids:
            raise RuntimeError("A held-out gold trial entered a pseudo fit.")
        context = prepare_reason_context(fit, 10)
        for i, setting in enumerate(PSEUDO_SETTINGS):
            bundle, counts = fit_selective(fit, context, setting)
            counts["setting_id"] = i
            counts["fold"] = int(fold)
            count_rows.append(counts)
            probabilities = predict_selective(bundle, held)
            result = evaluate_part1(held, probabilities, BASE_THRESHOLD, empty_set_policy="argmax")
            frames[i].append(result.predictions.assign(fold=int(fold)))
            fold_rows.append({"setting_id": i, "fold": int(fold), **result.summary})
        print(f"Part 1 selective pseudo fold {int(fold)+1}/4 complete", flush=True)
    pd.concat(count_rows).to_csv(out / "pseudo_counts_by_fold.csv", index=False)
    pd.DataFrame(fold_rows).to_csv(out / "oof_by_fold.csv", index=False)
    control_oof = evaluate_part1(gold, scores_from_saved(saved_control), BASE_THRESHOLD, empty_set_policy="argmax")
    results, rows = {}, [{"setting_id": "current", **control_oof.summary}]
    for i in frames:
        oof = verify_oof(gold, pd.concat(frames[i], ignore_index=True), assignment, part=1)
        oof.to_csv(out / f"oof_setting_{i}.csv", index=False)
        result = evaluate_part1(gold, scores_from_saved(oof), BASE_THRESHOLD, empty_set_policy="argmax")
        results[i] = result
        rows.append({"setting_id": i, **PSEUDO_SETTINGS[i], **result.summary})
    pd.DataFrame(rows).to_csv(out / "oof_comparison.csv", index=False)
    winner = max(results, key=lambda i: (results[i].summary["f1_samples"], -i))
    control_oof.save(out / "oof_control")
    results[winner].save(out / "oof_candidate")
    write_json(out / "selection_locked_before_validation.json", {
        "winner": winner, "setting": PSEUDO_SETTINGS[winner],
        "oof_f1": results[winner].summary["f1_samples"], "validation_labels_used": False,
    })
    full_context = prepare_reason_context(train, 10)
    full_bundle, full_counts = fit_selective(train, full_context, PSEUDO_SETTINGS[winner])
    joblib.dump(full_bundle, out / "selected_model.joblib", compress=3)
    full_counts.to_csv(out / "selected_full_training_pseudo_counts.csv", index=False)
    validation = pd.read_csv(dataset_path(root, "part1", "val"))
    current = scores_from_saved(pd.read_csv(root / 'models/part1/02_kmeans_lr/evidence/selected/selected_validation/validation_predictions.csv'))
    control_val = evaluate_part1(validation, current, BASE_THRESHOLD, empty_set_policy="argmax")
    candidate_val = evaluate_part1(validation, predict_selective(full_bundle, validation), BASE_THRESHOLD, empty_set_policy="argmax")
    control_val.save(out / "validation_control")
    candidate_val.save(out / "validation_candidate")
    oof_gain = results[winner].summary["f1_samples"] - control_oof.summary["f1_samples"]
    val_gain = candidate_val.summary["f1_samples"] - control_val.summary["f1_samples"]
    passes = bool(oof_gain > 1e-12 and val_gain > 1e-12)
    summary = {"winner": winner, "setting": PSEUDO_SETTINGS[winner],
               "oof_control_f1": control_oof.summary["f1_samples"],
               "oof_candidate_f1": results[winner].summary["f1_samples"],
               "validation_control_f1": control_val.summary["f1_samples"],
               "validation_candidate_f1": candidate_val.summary["f1_samples"],
               "passes_local_gate": passes, "final_replaced": False, "kaggle_uploaded": False}
    if passes:
        from midterm_evaluation import part1_decisions
        test = pd.read_csv(dataset_path(root, "part1", "test"))
        prob = predict_selective(full_bundle, test)
        pred = pd.DataFrame(part1_decisions(prob[list(REASONS)], BASE_THRESHOLD, empty_set_policy="argmax"), columns=REASONS)
        pred.insert(0, "ID", test.ID.to_numpy())
        sample = pd.read_csv(dataset_path(root, "part1", "test").parent / "part1_sample_submission.csv")
        submission = build_submission(sample, pred)
        submission.to_csv(out / "part1_selective_pseudo_candidate.csv", index=False)
        validate_submission(submission, sample, 1)
        summary["candidate_submission"] = str((out / "part1_selective_pseudo_candidate.csv").relative_to(out))
    write_json(out / "summary.json", summary)
    return summary


def run(root, output, study="all"):
    root, folder = prepare_output(root, output)
    frozen = {name: file_sha256(root / name) for name in (
        'tools/archive/records/final_selection_nov01.json', 'models/part1/02_kmeans_lr/model.joblib',
        'models/part1/02_kmeans_lr/submission.csv', 'models/part2/02_text_blend/submission.csv')}
    summary_path = folder / "summary.json"
    summaries = json.loads(summary_path.read_text()) if summary_path.exists() else {}
    if study in ("thresholds", "all"):
        summaries["thresholds"] = threshold_study(root, folder)
    if study in ("pseudo", "all"):
        summaries["selective_pseudo"] = pseudo_study(root, folder)
    for name, digest in frozen.items():
        if file_sha256(root / name) != digest:
            raise RuntimeError(f"A locked final artifact changed: {name}")
    write_json(summary_path, summaries)
    print(json.dumps(summaries, indent=2), flush=True)




if __name__ == "__main__":
    from replay import main
    main(default_study='selective_pseudo')

```


### replay.py

New filesystem/replay support only. SHA-256: `32728f68ec6d1705bdc425d8a1546b8c86b3012fb281a3570990977e6855de42`

```python
"""Review or explicitly replay preserved historical experiments, without promotion.

Default CLI action checks source and input availability without fitting:
  python tools/src/historical/replay.py --check
Explicit training (may take substantial time):
  python tools/src/historical/replay.py --study label_shift --run --output models/new_experiments/label_shift_replay
"""
from __future__ import annotations
from pathlib import Path
import argparse
import hashlib
import importlib
import json
import subprocess
import sys
from threadpoolctl import threadpool_limits

from _historical_support import validate_output

ROOT = Path(__file__).resolve().parents[3]
DIRECTORY = Path(__file__).resolve().parent
STUDIES = {
    'graph': ('midterm_graph_refinement', 'run_geometry', 'models/part1/exploration/graph/summary.json'),
    'selective_pseudo': ('midterm_rank_part1', 'run', 'models/part1/exploration/selective_pseudo/summary.json'),
    'structure_geometry': ('midterm_part1_round2', 'run', 'models/part1/exploration/structure_geometry/summary.json'),
    'label_shift': ('midterm_part1_label_shift', 'run', 'models/part1/exploration/label_shift/backtest_decision.json'),
    'other_views': ('midterm_part1_other_views', 'run', 'models/part1/exploration/other_views/decision.json'),
    'mixture_followup': ('midterm_part1_mixture_followup', 'run', 'models/part1/exploration/structure_geometry/mixture_followup/status.json'),
    'val_refit_weight': ('midterm_part1_val_refit_weight', 'run', 'models/part1/exploration/structure_geometry/validation_refit_weight/summary.json'),
    'parity': ('midterm_part1_parity', 'run', 'models/part1/02_kmeans_lr/evidence/baseline_parity/summary.json'),
    'local_followup': ('midterm_part1_local_followup', 'run', 'models/part1/exploration/local_followup/summary.json'),
    'ridge_boundary': ('midterm_part1_ridge_boundary', 'run', 'models/part1/exploration/ridge_boundary/summary.json'),
    'local_selection': ('midterm_part1_local_consensus', 'main', 'models/part1/03_kernel_ridge/evidence/selection_procedure/summary.json'),
}
REQUIRED_INPUTS = (
    'data/part1/part1_train.csv', 'data/part1/part1_val.csv', 'data/part1/part1_test.csv',
    'data/part1/part1_sample_submission.csv', 'models/selection.json',
    'models/part1/01_supervised_lr/model.joblib',
    'models/part1/01_supervised_lr/evidence/original_run/validation_predictions.csv',
    'models/part1/02_kmeans_lr/model.joblib', 'models/part1/02_kmeans_lr/submission.csv',
    'models/part1/02_kmeans_lr/evidence/selected/selected_validation/validation_predictions.csv',
    'models/part1/03_kernel_ridge/model.joblib', 'models/part1/03_kernel_ridge/submission.csv',
    'models/part2/02_text_blend/submission.csv',
    'models/part1/exploration/local_followup/oof_predictions.csv',
    'models/part1/exploration/replay_inputs/oof_weight_char/fold_assignments.csv',
    'models/part1/exploration/replay_inputs/oof_weight_char/oof_weight_010.csv',
    'tools/archive/records/final_selection_nov01.json',
)


def digest(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


def check(root=ROOT):
    """Compile and import sources, and verify required saved inputs; never fit models."""
    root = Path(root).resolve()
    provenance = json.loads((DIRECTORY / 'source_provenance.json').read_text())
    for rec in provenance['source_files'].values():
        source = root / rec['ported_path']
        if digest(source) != rec['ported_sha256']:
            raise AssertionError(f'Historical source differs from migration record: {source.name}')
        compile(source.read_text(), str(source), 'exec')
    for rec in provenance['restored_evidence']:
        if digest(root / rec['path']) != rec['original_sha256']:
            raise AssertionError(f'Restored historical input differs: {rec["path"]}')
    for relative in REQUIRED_INPUTS:
        if not (root / relative).is_file():
            raise FileNotFoundError(f'Missing replay input: {relative}')
    for study, (module, function, summary) in STUDIES.items():
        target = importlib.import_module(module)
        if not callable(getattr(target, function, None)):
            raise TypeError(f'Missing replay function for {study}')
        if not (root / summary).is_file():
            raise FileNotFoundError(f'Missing recorded summary: {summary}')
    return {'status': 'PASS', 'ported_runners': len(provenance['source_files']),
            'replay_studies': len(STUDIES), 'training_performed': False,
            'check_scope': 'compile/import, source hashes, required saved inputs; no experimental refits'}


def recorded_summaries(root=ROOT):
    return {name: json.loads((Path(root) / item[2]).read_text()) for name, item in STUDIES.items()}


def run(study, root, output):
    """Explicit training API; keeps selected model/CSV/data bytes unchanged."""
    if study not in STUDIES:
        raise ValueError(f'Unknown historical study: {study}')
    root, output = validate_output(root, output)
    check(root)
    protected = {p: digest(p) for directory in ('data', 'models')
                 for p in (root / directory).rglob('*') if p.is_file()}
    protected[root / 'models/selection.json'] = digest(root / 'models/selection.json')
    module, function, _ = STUDIES[study]
    try:
        if study == 'local_selection':
            subprocess.run([sys.executable, str(root / 'tools/src/midterm_part1_local_consensus.py'),
                            '--root', str(root), '--out', str(output)], check=True, cwd=root)
        else:
            replay_function = getattr(importlib.import_module(module), function)
            if study in ('local_followup', 'ridge_boundary'):
                # Preserve these two original CLI wrappers' single-thread scope.
                with threadpool_limits(limits=1):
                    replay_function(root, output)
            else:
                replay_function(root, output)
    finally:
        changed = [str(p.relative_to(root)) for p, sha in protected.items()
                   if not p.is_file() or digest(p) != sha]
        if changed:
            raise RuntimeError(f'Curated inputs changed during historical replay: {changed}')
    return {'study': study, 'output': str(output), 'production_selection_changed': False}


def main(default_study=None):
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument('--root', type=Path, default=ROOT)
    parser.add_argument('--study', choices=tuple(STUDIES), default=default_study)
    parser.add_argument('--check', action='store_true', help='Check source and saved inputs only (default).')
    parser.add_argument('--run', action='store_true', help='Explicitly fit the selected historical experiment.')
    parser.add_argument('--output', type=Path, help='Required with --run; must not exist.')
    args = parser.parse_args()
    if args.run:
        if args.check or not args.study or args.output is None:
            parser.error('--run requires --study and --output, without --check.')
        result = run(args.study, args.root, args.output)
    else:
        result = check(args.root)
    print(json.dumps(result, indent=2))


if __name__ == '__main__':
    main()

```


## Earlier frozen source record

The following exact source copies preserve earlier independent 157-label L2/SVM/threshold studies, diagnostics, and superseded delivery/report utilities. They are source records, **not current replay entry points**: their paths and saved-model dependencies belong to the original layout. Current supported replays are listed above. This audit did not rerun historical model searches. These originals remain readable inside this submitted notebook even though the large machine backup ZIPs are excluded.

Historical author wording inside these exact code copies is superseded by the current two-author attribution: Pei-Chen Chung (pc786) and Yu-Jye Hu (yh2444), with equal contributions. The code blocks remain byte-faithful source records.


### Original midterm/midterm_closeout.py

Original archive member: `archive/midterm_before_cleanup_2026-11-02.zip ! midterm/midterm_closeout.py`

SHA-256: `ffc313e38fa735763e080f37dfa80c201b897f5db8f377c2b963c882e69fdc30`

```python
"""Promote the validated 25/25/50 outcome blend without refitting components."""
from pathlib import Path
import json
import shutil
import joblib
import numpy as np
import pandas as pd

from midterm_evaluation import PHASES, REASONS, ExperimentLogger, dataset_path, evaluate_part2, file_sha256, validate_fit_scope
from midterm_finalization import FINAL_RUN, WORD_RUN, FIELD_RUN, check_run, predict_final_outcomes
from midterm_history import baseline_model_path
from midterm_next_round import verify_hashes, paired_diagnostics
from midterm_part2_baseline import TEXT_COLUMNS, predict_outcome_probabilities, analyze_failure_reasons
from midterm_part2_improvement import BASELINE_RUN, validate_submission

CHARACTER_RUN='character_tfidf_lr_c1_v1'


def write_json(path,value):
    Path(path).write_text(json.dumps(value,indent=2,allow_nan=False)+'\n')


def integrate_closeout(root):
    root=Path(root).resolve(); final=root/'part2_final';final.mkdir(exist_ok=True)
    prior=root/'history/pre_character_closeout'
    if not prior.is_dir():
        raise FileNotFoundError('Archive prior final outputs before promoting the character blend.')
    study=root/'refinement/oof_weight_char'
    gate=json.loads((study/'part2/candidate_gate.json').read_text())
    lock=json.loads((study/'part2/selection_locked_before_validation.json').read_text())
    if not gate['passes_candidate_gate'] or lock['setting']!='current_plus_character':
        raise ValueError('The frozen study does not support the requested final candidate.')
    # Earlier study files and their original sources remain verifiable by exact digest.
    verify_hashes(study,json.loads((study/'completed_checksums.json').read_text()))
    train=pd.read_csv(dataset_path(root,'part2','train'))
    val=pd.read_csv(dataset_path(root,'part2','val'))
    if set(train.trial_id)&set(val.trial_id):raise ValueError('Training/validation trials overlap.')
    word_dir=root/'experiments/runs'/WORD_RUN
    base_dir=baseline_model_path(root).parent
    field_dir=root/'refinement/part2/runs'/FIELD_RUN
    check_run(word_dir,'artifact_checksums.json');check_run(base_dir,'artifact_checksums.json');check_run(field_dir,'checksums.json')
    candidate_path=study/'part2/oof_selected_model.joblib'
    candidate=joblib.load(candidate_path)
    if candidate['kind']!='current_plus_character' or candidate['weights']!=[.5,.5] or candidate['reason_features_used'] is not False:
        raise ValueError('Unexpected frozen word/character candidate.')
    word=joblib.load(word_dir/'model.joblib');character=candidate['character']
    if character['reason_features_used'] is not False:raise ValueError('Forbidden outcome features.')
    np.testing.assert_allclose(predict_final_outcomes(word,val).p_success,
                               predict_final_outcomes(candidate['current'],val).p_success,atol=1e-15,rtol=0)
    inputs={f'part2_{split}':file_sha256(dataset_path(root,'part2',split)) for split in ['train','val']}
    settings={'kind':'word_character_probability_blend','weights':[.5,.5],
              'flat_weights':{'concatenated_word':.25,'separate_field_word':.25,'character':.5},
              'positive_class':1,'refit_performed':False,'input_sha256':inputs,
              'components':{'word_blend':{'run_id':WORD_RUN,'model_sha256':file_sha256(word_dir/'model.joblib')},
                            'character':{'study':'refinement/oof_weight_char','candidate_sha256':file_sha256(candidate_path)}},
              'selection':'Fixed candidate selected on grouped training OOF and confirmed on validation before public submission'}
    bundle={'kind':'word_character_probability_blend','weights':[.5,.5],
            'flat_weights':settings['flat_weights'],'positive_class':1,'reason_features_used':False,
            'thresholds':dict.fromkeys(PHASES,.5),'word_blend':word,'character':character,
            'selection_metadata':settings}
    char_scope=json.loads((study/'part2/full_character_fit_scope.json').read_text())['fit_ids']
    base_scope=json.loads((base_dir/'config.json').read_text())['declared_fit_scope']
    fit_scope={}
    for phase in PHASES:
        ids=train.loc[train.phase.eq(phase),'trial_id'].tolist()
        if word['field']['models'][phase]['fit_trial_ids']!=ids:raise ValueError('Field model fit scope differs.')
        for prefix in ['tfidf_phase_','classifier_phase_']:
            if char_scope[prefix+phase]!=ids:raise ValueError('Character model fit scope differs.')
            if validate_fit_scope(ids,train.trial_id)!=base_scope[prefix+phase]:raise ValueError('Baseline fit scope differs.')
        for component in ['concatenated_word','separate_field_word','character']:
            fit_scope[f'inherited_{component}_{phase}']=ids
    result=evaluate_part2(val,predict_final_outcomes(bundle,val),.5)
    expected=pd.read_csv(study/'part2/validation/current_plus_character/validation_predictions.csv').set_index('ID').loc[val.ID]
    np.testing.assert_allclose(result.predictions.p_success,expected.p_success,atol=1e-15,rtol=0)
    run_dir=root/'experiments/runs'/FINAL_RUN
    if run_dir.exists():
        check_run(run_dir,'artifact_checksums.json')
        if json.loads((run_dir/'config.json').read_text())['hyperparameters']!=settings:
            raise ValueError('The registered final blend settings changed.')
        bundle=joblib.load(run_dir/'model.joblib')
        np.testing.assert_allclose(predict_final_outcomes(bundle,val).p_success,result.predictions.p_success,atol=1e-15,rtol=0)
    else:
        run_dir=ExperimentLogger(root).log_validation_run(task='part2',probabilities=result.predictions[['ID','p_success']],
            model_name='Frozen 25/25/50 word-field-character probability blend',
            feature_spec={'input_columns':[*TEXT_COLUMNS,'phase'],'representation':'Word/field TF-IDF plus char_wb 3-5 TF-IDF',
                          'reason_features_used':False},hyperparameters=settings,fit_trial_ids=fit_scope,seed=42,
            thresholds=.5,run_id=FINAL_RUN,
            hypothesis='A fixed character-text component may complement two word-based models.',
            notes='Derived artifact only. No fitting. Grouped OOF and validation selected the candidate before public submission; user screenshot later reports 0.59696.')
        joblib.dump(bundle,run_dir/'model.joblib',compress=3)
        write_json(run_dir/'component_sources.json',settings)
        write_json(run_dir/'artifact_checksums.json',{p.name:file_sha256(p) for p in run_dir.iterdir() if p.is_file()})
    shutil.copy2(run_dir/'model.joblib',final/'selected_model.joblib')
    result.save(final/'selected_validation')
    base=word['baseline']
    baseline_result=evaluate_part2(val,predict_outcome_probabilities(base,val),.5)
    word_result=evaluate_part2(val,predict_final_outcomes(word,val),.5)
    char_result=evaluate_part2(val,predict_outcome_probabilities(character,val),.5)
    old=pd.read_csv(prior/'part2_final/validation_comparison.csv')
    old['selected']=False
    additions=[]
    for run,label,evaluation in [(CHARACTER_RUN,'Character TF-IDF',char_result),(FINAL_RUN,'Word + character blend (final)',result)]:
        additions.append({'run_id':run,'feature_setting':label,'structured':False,'C':1.,**evaluation.summary,
                          'mean_phase_f1':float(evaluation.by_group.f1.mean()),'mean_phase_roc_auc':float(evaluation.by_group.roc_auc.mean()),
                          'mean_phase_ap':float(evaluation.by_group.average_precision.mean()),
                          'delta_log_loss_vs_baseline':evaluation.summary['log_loss_equal_phase']-baseline_result.summary['log_loss_equal_phase'],
                          'study':'Grouped-OOF character follow-up','selected':run==FINAL_RUN})
    comparison=pd.concat([old,pd.DataFrame(additions)],ignore_index=True).drop_duplicates('run_id',keep='last').sort_values('log_loss_equal_phase')
    comparison.to_csv(final/'validation_comparison.csv',index=False)
    old_phase=pd.read_csv(prior/'part2_final/validation_metrics_all_candidates.csv')
    pd.concat([old_phase,char_result.by_group.assign(run_id=CHARACTER_RUN),result.by_group.assign(run_id=FINAL_RUN)],ignore_index=True).drop_duplicates(['run_id','phase'],keep='last').to_csv(final/'validation_metrics_all_candidates.csv',index=False)
    lookup=pd.read_csv(root/'part1_final/reason_lookup.csv')
    analyses={name:analyze_failure_reasons(evaluation.predictions,lookup) for name,evaluation in
              [('baseline',baseline_result),('word',word_result),('final',result)]}
    for name,frame in analyses['final'].items():frame.to_csv(final/f'selected_{name}.csv',index=False)
    for before_name in ['baseline','word']:
        for key,stem in [('reason_rates','reason_error_rate_comparison'),('source_rates','reason_error_rate_comparison_by_source'),
                         ('overall_failed_rates','overall_failure_error_rate_comparison')]:
            on=['phase','reason_group']+(['source_type'] if key=='source_rates' else [])
            merged=analyses[before_name][key].merge(analyses['final'][key],on=on,suffixes=(f'_{before_name}','_final'),validate='one_to_one')
            if not np.array_equal(merged[f'n_trials_{before_name}'],merged.n_trials_final):raise ValueError('Reason-group support changed.')
            merged['delta_false_success_rate']=merged.false_success_rate_final-merged[f'false_success_rate_{before_name}']
            merged.to_csv(final/(stem+('_vs_word' if before_name=='word' else '')+'.csv'),index=False)
    failures=analyses['final']['failed_rows']
    provenance={'reason_model_source':'part1_s5_kmeans_threshold035_seed42_v1',
                'reason_lookup_sha256':file_sha256(root/'part1_final/reason_lookup.csv'),'reason_features_used':False,
                'failure_trial_phase_rows':len(failures),'observed_reason_rows':int(failures.source_type.eq('observed').sum()),
                'predicted_reason_rows':int(failures.source_type.eq('predicted').sum()),'unmatched_rows':int(failures.source_type.eq('unmatched').sum()),
                'training_resubstitution_reason_rows':int(failures.prediction_is_training_resubstitution.eq(True).sum())}
    if [provenance[k] for k in ['failure_trial_phase_rows','observed_reason_rows','predicted_reason_rows','unmatched_rows']]!=[452,27,171,254]:
        raise ValueError('The fixed Part 1 reason linkage changed.')
    write_json(final/'analysis_provenance.json',provenance)
    write_json(final/'paired_uncertainty_vs_word.json',paired_diagnostics(word_result,result,part=2))
    selection={'selected_run_id':FINAL_RUN,'selection_metric':'log_loss_equal_phase','selection_score':result.summary['log_loss_equal_phase'],
               'weights':[.5,.5],'flat_weights':settings['flat_weights'],'threshold':.5,'refit_performed':False,
               'validation_refit':False,'blend_weight_tuned_on_public':False,
               'candidate_selected_on_training_OOF_and_validation_before_public_submission':True,
               'selected_model_sha256':file_sha256(final/'selected_model.joblib'),'components':settings['components'],
               'user_reported_public_score':.59696,'public_score_source':'User-provided submission screenshot'}
    write_json(final/'selection_locked.json',selection)
    write_json(final/'comparison_roles.json',{'baseline':BASELINE_RUN,'previous_final':WORD_RUN,'final_submission':FINAL_RUN})
    test=pd.read_csv(dataset_path(root,'part2','test'))
    probability=predict_final_outcomes(bundle,test)
    sample=pd.read_csv(dataset_path(root,'part2','test').parent/'part2_sample_submission.csv')
    submission=sample[['ID']].merge(probability,on='ID',validate='one_to_one',sort=False).rename(columns={'p_success':'label'})
    source_csv=study/'submissions/part2_candidate.csv'
    pd.testing.assert_frame_equal(submission,pd.read_csv(source_csv),check_exact=False,atol=1e-15,rtol=0)
    validate_submission(submission,sample,2)
    probability.to_csv(final/'test_probabilities.csv',index=False)
    test[['ID','trial_id','phase']].merge(probability,on='ID',validate='one_to_one').to_csv(final/'test_predictions.csv',index=False)
    shutil.copy2(source_csv,final/'part2_submission.csv')
    submissions=root/'submissions';submissions.mkdir(exist_ok=True)
    shutil.copy2(source_csv,submissions/'part2_submission.csv')
    shutil.copy2(root/'part1_final/part1_submission.csv',submissions/'part1_submission.csv')
    sample1=pd.read_csv(dataset_path(root,'part1','test').parent/'part1_sample_submission.csv')
    checks=[validate_submission(pd.read_csv(submissions/'part1_submission.csv'),sample1,1),validate_submission(pd.read_csv(submissions/'part2_submission.csv'),sample,2)]
    write_json(submissions/'validation_checks.json',{'checks':checks,'part1_sha256':file_sha256(submissions/'part1_submission.csv'),
               'part2_sha256':file_sha256(submissions/'part2_submission.csv'),'part2_selected_model':FINAL_RUN,
               'user_reported_public_scores':{'part1':.85405,'part2':.59696},'uploads_performed_by_this_integration':False})
    followups=[]
    retained_oof=pd.read_csv(study/'part1/oof_comparison.csv').set_index('setting').loc['weight_010','f1_samples']
    followups.append({'method':'Retained K-Means + LR','OOF_sample_F1':retained_oof,'validation_sample_F1':.8782051282051282})
    for folder_name,setting,method in [('rbf_svm','gold_linear_svm_c1','Gold linear SVM'),('rbf_svm','gold_rbf_svm_c10','Gold RBF SVM C=10'),('l2_geometry','l2_c10p0','Row L2 C=10')]:
        path=root/'refinement'/folder_name
        cv=pd.read_csv(path/'oof_comparison.csv').set_index('setting').loc[setting,'f1_samples']
        valid=pd.read_csv(path/'validation_comparison.csv').set_index('setting').loc[setting,'f1_samples']
        followups.append({'method':method,'OOF_sample_F1':cv,'validation_sample_F1':valid})
    path=root/'refinement/threshold_blend'
    followups.append({'method':'75/25 LR + linear SVM','OOF_sample_F1':pd.read_csv(path/'oof_comparison.csv').set_index('policy').loc['blend_075_025_t035','f1_samples'],
                      'validation_sample_F1':pd.read_csv(path/'validation_comparison.csv').set_index('policy').loc['blend_075_025_t035','f1_samples']})
    pd.DataFrame(followups).to_csv(final/'part1_followup_summary.csv',index=False)
    summary={'part1':{'model':'part1_s5_kmeans_threshold035_seed42_v1','sample_f1':.8782051282051282,
                     'macro_f1':.7648422333984687,'public_score':.85405,'decision':'Retain original after graph, weight, SVM, L2, and policy follow-ups'},
             'part2':{'model':FINAL_RUN,**result.summary,'public_score':.59696,'previous_public_score':.59962,
                      'baseline_public_score':.60149,'baseline_equal_phase_log_loss':baseline_result.summary['log_loss_equal_phase'],
                      'previous_equal_phase_log_loss':word_result.summary['log_loss_equal_phase'],
                      'flat_weights':settings['flat_weights'],'selected_phase_metrics':result.by_group.to_dict('records')},
             'reason_provenance':provenance,'component_refitting_performed':False,'private_scores_known':False,
             'public_scores_source':'User-provided submission screenshots','agent_uploads_performed':False,
             'notebooks_in_workflow':15,'registered_runs':len(list((root/'experiments/runs').glob('*/config.json')))}
    write_json(root/'final_selection.json',summary);write_json(final/'results_summary.json',summary)
    verification={'status':'PASS','selected_run_artifacts_verified':True,'models_refitted':0,
                  'component_models_verified':True,'reason_group_support_unchanged':True,
                  'test_submission_matches_candidate_byte_for_byte':file_sha256(source_csv)==file_sha256(submissions/'part2_submission.csv'),
                  'part1_submission_unchanged':file_sha256(submissions/'part1_submission.csv')==file_sha256(root/'part1_final/part1_submission.csv'),
                  'final_submission_sha256':file_sha256(submissions/'part2_submission.csv'),
                  'selected_model_sha256':file_sha256(final/'selected_model.joblib'),'checks':checks}
    write_json(final/'final_verification.json',verification)
    (final/'final_experiments_report.html').write_text(
        '<!doctype html><html lang="en"><meta charset="utf-8"><title>Final Outcome Model</title>'
        '<style>body{font:16px system-ui;margin:36px}table{border-collapse:collapse}td,th{padding:7px;border:1px solid #ccd}</style>'
        '<h1>Final Outcome Model: 25/25/50 Word and Character Blend</h1>'
        '<p>Validation equal-phase log loss: 0.580900. User-reported public log loss: 0.59696. '
        'Private scores are unknown. Components were not refitted during final integration.</p>'
        '<h2>Validation comparisons</h2>'+comparison.to_html(index=False)+
        '<h2>Selected phase metrics</h2>'+result.by_group.to_html(index=False)+
        '<p>Reasons and reason embeddings are used only for error analysis. '
        'See notebook 15 and report/main.pdf for the final methods, limitations, and complete delivery.</p></html>')
    return summary

```


### Original midterm/midterm_delivery.py

Original archive member: `archive/midterm_before_cleanup_2026-11-02.zip ! midterm/midterm_delivery.py`

SHA-256: `8afc2e162200a9f0221281706aaf8fd53a0ea24534b38a62897e13db67074c9d`

```python
"""Build the current integrated midterm package and verify its ZIP."""
from pathlib import Path
from midterm_delivery_final import build_delivery

if __name__ == '__main__':
    import argparse,json
    parser=argparse.ArgumentParser(description=__doc__)
    parser.add_argument('--project-root',type=Path,default=Path(__file__).resolve().parent)
    parser.add_argument('--output-dir',type=Path,help='Delivery folder outside the source project; defaults to ../deliverables.')
    args=parser.parse_args()
    print(json.dumps(build_delivery(args.project_root,args.output_dir),indent=2))

```


### Original midterm/midterm_delivery_final.py

Original archive member: `archive/midterm_before_cleanup_2026-11-02.zip ! midterm/midterm_delivery_final.py`

SHA-256: `14ce26b200cfeec0c9f62ffacf6055d7effae4ddac749992c0eb4238c6e65290`

```python
"""Build and verify the final fifteen-notebook delivery."""
from pathlib import Path
import hashlib
import json
import shutil
import tempfile
import zipfile
from midterm_evaluation import file_sha256

FOLDERS=['cs-5785-midterm-2026-clinical-trial-reason-prediction',
    'cs-5785-midterm-2026-part-2-trial-outcome-prediction','data_audit','evaluation_protocol',
    'experiments','label_augmentation','part1_final','part2_baseline','part2_final','submissions',
    'report','tests','reproducibility','refinement','history']


def build_delivery(root, output_dir=None):
    root=Path(root).resolve()
    outputs=Path(output_dir).resolve() if output_dir is not None else root.parent/'deliverables'
    if outputs.is_relative_to(root):
        raise ValueError('Delivery output must be outside the source project.')
    layout=json.loads((root/'report/layout_verification.json').read_text())
    for key,path in [('pdf_sha256','main.pdf'),('main_tex_sha256','main.tex'),('methods_tex_sha256','methods.tex'),('cvpr_style_sha256','cvpr.sty')]:
        if file_sha256(root/'report'/path)!=layout[key]:raise ValueError(f'Report layout verification is stale: {path}')
    if layout['body_pages']>5 or not layout['all_pages_visually_reviewed']:raise ValueError('Report layout has not passed review.')
    notebooks=sorted(root.glob('[0-9][0-9]_*.ipynb'))
    if [int(p.name[:2]) for p in notebooks]!=list(range(1,16)):raise ValueError('Expected notebooks 01 through 15.')
    for path in notebooks:
        nb=json.loads(path.read_text())
        for cell in nb['cells']:
            if cell['cell_type']=='code' and (not cell.get('execution_count') or any(o.get('output_type')=='error' for o in cell.get('outputs',[]))):
                raise ValueError(f'Notebook must be successfully executed: {path.name}')
    outputs.mkdir(parents=True,exist_ok=True)
    staging=Path(tempfile.mkdtemp(prefix='.midterm_build_',dir=outputs.parent))
    try:
        files=notebooks+list(root.glob('midterm_*.py'))+list(root.glob('reproduce*.py'))
        files += [root/name for name in ['CS5785_Midterm_Starter.ipynb','README.md','requirements.txt','requirements-report.txt','final_selection.json']]
        for source in sorted(set(files)):
            if not source.is_file():raise FileNotFoundError(source)
            shutil.copy2(source,staging/source.name)
        ignore=shutil.ignore_patterns('__pycache__','.DS_Store','.ipynb_checkpoints','.reproduction_cache')
        for name in FOLDERS:
            shutil.copytree(root/name,staging/name,ignore=ignore)
        digest={str(p.relative_to(staging)):file_sha256(p) for p in sorted(staging.rglob('*')) if p.is_file()}
        manifest={'files':digest,'file_count':len(digest),'notebooks':[p.name for p in notebooks],
            'author':'PEI-CHEN CHUNG','report_body_pages':layout['body_pages'],'report_reference_pages':layout['reference_pages'],
            'selected_models':json.loads((root/'final_selection.json').read_text()),'uploads_performed':False}
        (staging/'package_manifest.json').write_text(json.dumps(manifest,indent=2)+'\n')
        archive=outputs/'midterm_code.zip';pending=outputs/'.midterm_code.tmp.zip'
        with zipfile.ZipFile(pending,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=6) as z:
            for p in sorted(staging.rglob('*')):
                if p.is_file():z.write(p,arcname=str(Path('midterm')/p.relative_to(staging)))
        with zipfile.ZipFile(pending) as z:
            if z.testzip() is not None:raise RuntimeError('ZIP integrity check failed.')
            if any(not n.startswith('midterm/') for n in z.namelist()):raise ValueError('Expected one midterm archive root.')
            if any(n.endswith('.zip') for n in z.namelist()):raise ValueError('Nested delivery archives are not included.')
            for name,digest in manifest['files'].items():
                if hashlib.sha256(z.read('midterm/'+name)).hexdigest()!=digest:raise ValueError(f'ZIP hash differs: {name}')
        # Publish the verified ZIP; the retained source project is never replaced.
        pending.replace(archive)
        shutil.copy2(staging/'package_manifest.json',root/'package_manifest.json')
        shutil.copy2(root/'report/main.pdf',outputs/'midterm_report.pdf')
        for name in ['part1_submission.csv','part2_submission.csv']:shutil.copy2(root/'submissions'/name,outputs/name)
        result={'status':'PASS','source_project':str(root),'package_folder':str(root),
            'persistent_extracted_copy_created':False,'archive':str(archive),'archive_sha256':file_sha256(archive),
            'archive_bytes':archive.stat().st_size,'manifest_files_verified':len(manifest['files']),
            'notebooks':manifest['notebooks'],'report_body_pages':layout['body_pages'],'report_reference_pages':layout['reference_pages'],
            'report_sha256':file_sha256(outputs/'midterm_report.pdf'),
            'part1_sha256':file_sha256(outputs/'part1_submission.csv'),'part2_sha256':file_sha256(outputs/'part2_submission.csv'),
            'uploads_performed':False}
        (outputs/'delivery_checks.json').write_text(json.dumps(result,indent=2)+'\n')
        return result
    finally:
        if staging.exists():shutil.rmtree(staging)

```


### Original midterm/midterm_l2_geometry.py

Original archive member: `archive/midterm_before_cleanup_2026-11-02.zip ! midterm/midterm_l2_geometry.py`

SHA-256: `048602a431134e88af078f43f8a50cfb77d200229def221bf6091d22af3caa32`

```python
"""Part 1 row-L2 ablation with fixed K-Means pseudo targets and a shared C grid."""
from __future__ import annotations

import argparse
import hashlib
import html
import json
import os
from pathlib import Path
import platform
import warnings

import joblib
import numpy as np
import pandas as pd
import sklearn
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import normalize
from threadpoolctl import threadpool_limits

from midterm_augmentation import logistic_parameters, observed_label_mask
from midterm_evaluation import REASONS, dataset_path, evaluate_part1, file_sha256, part1_decisions
from midterm_next_round import (
    cache_complete, finish_cache, paired_diagnostics, read_json,
    reason_error_counts, validate_fold_scope, verify_hashes, verify_oof, write_json,
)
from midterm_part1_finalization import build_submission
from midterm_part2_improvement import validate_submission
from midterm_refinement import EMBEDDINGS, prepare_reason_context

STUDY = "refinement/l2_geometry"
C_GRID = (.1, 1., 10., 100.)
CURRENT = "retained_kmeans_lr"
THRESHOLD = .35
SETTINGS = {
    f"{'l2' if use_l2 else 'standard'}_c{str(C).replace('.', 'p')}": {"use_l2": use_l2, "C": C}
    for use_l2 in [False, True] for C in C_GRID
}


def classifier_features(values, use_l2):
    """Normalize each standardized row; leave a zero vector at zero."""
    x = np.asarray(values, dtype=float)
    if x.ndim != 2 or not np.isfinite(x).all():
        raise ValueError("Classifier features must be a finite two-dimensional matrix.")
    return normalize(x, norm="l2", axis=1, copy=True) if use_l2 else x.copy()


def fixed_targets(frame, context):
    """Return identical gold/pseudo targets and weights for both geometries."""
    if frame.ID.tolist() != context["fit_ids"]:
        raise ValueError("Feature context and target rows must have identical IDs and order.")
    observed = observed_label_mask(frame[list(REASONS)].to_numpy())
    if not np.array_equal(observed, context["observed"]):
        raise ValueError("The observed-label mask differs from the preprocessing context.")
    q = np.asarray(context["pseudo_q"])
    if q.shape != (int((~observed).sum()), len(REASONS)) or not np.isfinite(q).all() or ((q < 0) | (q > 1)).any():
        raise ValueError("Invalid cluster reason rates.")
    targets = frame[list(REASONS)].to_numpy(dtype=np.int64).copy()
    targets[~observed] = (q >= .5).astype(np.int64)
    if not np.isin(targets, [0, 1]).all():
        raise ValueError("Every classifier target must be binary.")
    weights = np.where(observed, 1.0, .1)
    digest = hashlib.sha256(targets.tobytes() + weights.tobytes()).hexdigest()
    table = frame[["ID"]].copy().reset_index(drop=True)
    table["source"] = np.where(observed, "observed_gold", "fixed_kmeans_pseudo")
    table["sample_weight"] = weights
    for index, reason in enumerate(REASONS):
        table[f"target_{reason}"] = targets[:, index]
    return targets, weights, digest, table


def fit_setting(frame, context, setting):
    if not np.isfinite(setting["C"]) or setting["C"] <= 0:
        raise ValueError("C must be finite and positive.")
    targets, weights, digest, _ = fixed_targets(frame, context)
    x = classifier_features(context["x"], setting["use_l2"])
    models, iterations = [], []
    for index in range(len(REASONS)):
        estimator = LogisticRegression(**{**logistic_parameters(42), "C": setting["C"]})
        with warnings.catch_warnings(record=True) as caught, threadpool_limits(limits=1):
            warnings.simplefilter("always")
            estimator.fit(x, targets[:, index], sample_weight=weights)
        if any(issubclass(w.category, ConvergenceWarning) for w in caught) or estimator.n_iter_.max() >= 500:
            raise RuntimeError("A classifier did not converge under the declared limit.")
        if not np.array_equal(estimator.classes_, [0, 1]):
            raise ValueError("Every reason classifier requires both classes.")
        models.append(estimator); iterations.append(int(estimator.n_iter_.max()))
    return {"kind": "fixed_pseudo_l2_ablation", "setting": setting,
            "scaler": context["scaler"], "clusterer": context["clusterer"],
            "mapping": context["mapping"], "models": models,
            "feature_columns": EMBEDDINGS, "threshold": THRESHOLD, "empty_set_policy": "argmax",
            "fit_ids": frame.ID.tolist(), "seed_ids": context["seed_ids"],
            "target_weight_sha256": digest, "classifier_iterations": iterations,
            "cluster_geometry": "StandardScaler only; unchanged by classifier row normalization"}


def predict_l2_study(bundle, frame):
    if bundle.get("kind") != "fixed_pseudo_l2_ablation":
        raise ValueError("Unexpected L2-study model bundle.")
    x = classifier_features(bundle["scaler"].transform(frame[EMBEDDINGS]), bundle["setting"]["use_l2"])
    values = np.column_stack([m.predict_proba(x)[:, list(m.classes_).index(1)] for m in bundle["models"]])
    if not np.isfinite(values).all() or ((values < 0) | (values > 1)).any():
        raise ValueError("Invalid reason probabilities.")
    scores = pd.DataFrame(values, columns=REASONS)
    scores.insert(0, "ID", frame.ID.to_numpy())
    return scores


def scores_from_saved(frame):
    return frame[["ID", *[f"prob_{r}" for r in REASONS]]].rename(columns={f"prob_{r}": r for r in REASONS})


def protected_hashes(root):
    paths = [root / "final_selection.json"]
    for folder in ["part1_final", "part2_final", "experiments", "submissions", "deliverables", "report", "refinement"]:
        paths.extend(p for p in (root / folder).rglob("*") if p.is_file()
                     and not p.is_relative_to(root / STUDY) and "__pycache__" not in p.parts)
    paths.extend(p for p in root.glob("[0-9][0-9]_*.ipynb") if int(p.name[:2]) < 13)
    return {p.relative_to(root).as_posix(): file_sha256(p) for p in sorted(set(paths))}


def norm_summary(context, setting):
    before = np.linalg.norm(context["x"], axis=1)
    after = np.linalg.norm(classifier_features(context["x"], setting["use_l2"]), axis=1)
    return {"use_l2": setting["use_l2"], "C": setting["C"],
            "before_mean_norm": float(before.mean()), "before_median_norm": float(np.median(before)),
            "after_mean_norm": float(after.mean()), "after_min_norm": float(after.min()),
            "after_max_norm": float(after.max()), "zero_rows": int((before == 0).sum())}


def run_oof(root, folder, train):
    observed = observed_label_mask(train[list(REASONS)].to_numpy())
    gold = train.loc[observed].reset_index(drop=True)
    assignment = pd.read_csv(root / "refinement/oof_weight_char/part1/fold_assignments.csv")
    if not assignment.ID.is_unique or set(assignment.ID) != set(gold.ID) or set(assignment.fold) != set(range(4)):
        raise ValueError("The saved four-fold assignment does not match the gold rows.")
    assignment = assignment.set_index("ID").loc[gold.ID].reset_index()
    assignment.to_csv(folder / "fold_assignments.csv", index=False)
    frames, fold_metrics, norms = {name: [] for name in SETTINGS}, [], []
    for fold in range(4):
        cache = folder / "fold_cache" / f"fold_{fold}"
        if not cache_complete(cache):
            cache.mkdir(parents=True, exist_ok=True)
            held_ids = assignment.loc[assignment.fold.eq(fold), "ID"]
            held = gold.loc[gold.ID.isin(held_ids)].reset_index(drop=True)
            fit = train.loc[~train.ID.isin(held_ids)].reset_index(drop=True)
            scope = validate_fold_scope(fit, held, part=1)
            context = prepare_reason_context(fit, 10)
            _, _, digest, targets = fixed_targets(fit, context)
            targets.to_csv(cache / "shared_classifier_targets.csv", index=False)
            scope.update(label_seed_ids=context["seed_ids"], target_weight_sha256=digest,
                         cluster_fit_ids=context["fit_ids"],
                         kmeans_iterations=int(context["clusterer"].n_iter_),
                         labels_and_weights_shared_across_all_settings=True)
            write_json(cache / "fit_scope.json", scope)
            for name, setting in SETTINGS.items():
                bundle = fit_setting(fit, context, setting)
                if bundle["target_weight_sha256"] != digest:
                    raise AssertionError("A setting changed the shared targets or weights.")
                result = evaluate_part1(held, predict_l2_study(bundle, held), THRESHOLD, empty_set_policy="argmax")
                result.predictions.assign(fold=fold).to_csv(cache / f"{name}.csv", index=False)
                write_json(cache / f"{name}_metrics.json", result.summary)
                write_json(cache / f"{name}_norms.json", norm_summary(context, setting))
            finish_cache(cache)
        for name in SETTINGS:
            frames[name].append(pd.read_csv(cache / f"{name}.csv"))
            fold_metrics.append({"setting": name, "fold": fold, **read_json(cache / f"{name}_metrics.json")})
            norms.append({"setting": name, "fold": fold, **read_json(cache / f"{name}_norms.json")})
        print(f"Shared-target L2 OOF fold {fold + 1}/4 complete", flush=True)
    folds = pd.DataFrame(fold_metrics)
    folds.to_csv(folder / "fold_metrics.csv", index=False)
    pd.DataFrame(norms).to_csv(folder / "feature_norms_by_fold.csv", index=False)
    evaluations, comparison, by_reason = {}, [], []
    for name, setting in SETTINGS.items():
        oof = verify_oof(gold, pd.concat(frames[name], ignore_index=True), assignment, part=1)
        result = evaluate_part1(gold, scores_from_saved(oof), THRESHOLD, empty_set_policy="argmax")
        evaluations[name] = result
        oof.to_csv(folder / f"oof_{name}.csv", index=False)
        selected_folds = folds.loc[folds.setting.eq(name)]
        comparison.append({"setting": name, **setting, **result.summary,
                           "mean_fold_f1": selected_folds.f1_samples.mean(), "std_fold_f1": selected_folds.f1_samples.std()})
        by_reason.append(reason_error_counts(result).assign(setting=name))
    frozen = verify_oof(gold, pd.read_csv(root / "refinement/oof_weight_char/part1/oof_weight_010.csv"), assignment, part=1)
    retained = evaluate_part1(gold, scores_from_saved(frozen), THRESHOLD, empty_set_policy="argmax")
    evaluations[CURRENT] = retained
    reproduced = evaluations["standard_c0p1"].predictions[[f"prob_{r}" for r in REASONS]].to_numpy()
    reference = frozen[[f"prob_{r}" for r in REASONS]].to_numpy()
    error = float(np.abs(reproduced - reference).max())
    if error > 1e-10:
        raise ValueError(f"The matched standard control does not reproduce frozen OOF probabilities: {error}")
    table = pd.DataFrame(comparison)
    table["delta_f1_vs_retained"] = table.f1_samples - retained.summary["f1_samples"]
    table.to_csv(folder / "oof_comparison.csv", index=False)
    pd.concat(by_reason).to_csv(folder / "oof_by_reason.csv", index=False)
    same_c = table.loc[table.use_l2].merge(table.loc[~table.use_l2], on="C", suffixes=("_l2", "_standard"), validate="one_to_one")
    same_c["delta_f1_l2_minus_standard"] = same_c.f1_samples_l2 - same_c.f1_samples_standard
    same_c.to_csv(folder / "same_C_oof_comparison.csv", index=False)
    chosen = {}
    for use_l2, label in [(False, "standard_setting"), (True, "l2_setting")]:
        candidates = [name for name, setting in SETTINGS.items() if setting["use_l2"] == use_l2]
        chosen[label] = max(candidates, key=lambda name: (evaluations[name].summary["f1_samples"], -SETTINGS[name]["C"]))
    lock = {**chosen, "metric": "pooled four-fold gold OOF sample F1", "tie_policy": "Prefer smaller C",
            "official_validation_labels_used": False, "test_predictions_used": False,
            "public_feedback_used_to_choose_C": False, "frozen_control_max_probability_difference": error}
    write_json(folder / "selection_locked_before_validation.json", lock)
    return lock, evaluations


def validation_and_candidate(root, folder, train, lock, oof):
    context = prepare_reason_context(train, 10)
    _, _, digest, targets = fixed_targets(train, context)
    targets.to_csv(folder / "full_training_shared_targets.csv", index=False)
    scope = {"fit_ids": context["fit_ids"], "gold_seed_ids": context["seed_ids"],
             "target_weight_sha256": digest, "label_mapping_uses_gold_only": True,
             "scaler_and_cluster_fit_on_training_only": True, "validation_rows_in_fit": 0}
    write_json(folder / "full_training_fit_scope.json", scope)
    selected_names = list(dict.fromkeys([lock["standard_setting"], lock["l2_setting"], "standard_c0p1"]))
    bundles = {name: fit_setting(train, context, SETTINGS[name]) for name in selected_names}
    for name, bundle in bundles.items():
        joblib.dump(bundle, folder / f"{name}.joblib", compress=3)
    # Read official validation only after locking one C for each geometry.
    validation = pd.read_csv(dataset_path(root, "part1", "val"))
    frozen = pd.read_csv(root / "part1_final/selected_validation/validation_predictions.csv")
    results = {CURRENT: evaluate_part1(validation, scores_from_saved(frozen), THRESHOLD, empty_set_policy="argmax")}
    results.update({name: evaluate_part1(validation, predict_l2_study(bundle, validation), THRESHOLD, empty_set_policy="argmax")
                    for name, bundle in bundles.items()})
    error = float(np.abs(results["standard_c0p1"].predictions[[f"prob_{r}" for r in REASONS]].to_numpy() -
                         results[CURRENT].predictions[[f"prob_{r}" for r in REASONS]].to_numpy()).max())
    if error > 1e-10:
        raise ValueError("The standard control does not reproduce the retained validation probabilities.")
    for name, result in results.items():
        result.save(folder / "validation" / name)
    pd.DataFrame([{"setting": name, **result.summary} for name, result in results.items()]).to_csv(folder / "validation_comparison.csv", index=False)
    pd.concat([reason_error_counts(result).assign(setting=name) for name, result in results.items()]).to_csv(folder / "validation_by_reason.csv", index=False)
    l2 = lock["l2_setting"]; standard = lock["standard_setting"]
    oof_gain = oof[l2].summary["f1_samples"] - oof[CURRENT].summary["f1_samples"]
    val_gain = results[l2].summary["f1_samples"] - results[CURRENT].summary["f1_samples"]
    gate = {"oof_selected_l2_setting": l2, "oof_selected_standard_setting": standard,
            "l2_C": SETTINGS[l2]["C"], "standard_C": SETTINGS[standard]["C"],
            "oof_f1_gain_vs_retained": oof_gain, "validation_f1_gain_vs_retained": val_gain,
            "oof_f1_gain_vs_selected_standard": oof[l2].summary["f1_samples"] - oof[standard].summary["f1_samples"],
            "validation_f1_gain_vs_selected_standard": results[l2].summary["f1_samples"] - results[standard].summary["f1_samples"],
            "passes_candidate_gate": bool(oof_gain > 1e-12 and val_gain > 1e-12),
            "validation_control_max_probability_difference": error,
            "shared_training_targets_and_weights_verified": all(b["target_weight_sha256"] == digest for b in bundles.values()),
            "final_model_replaced": False, "public_or_private_score_known": False}
    write_json(folder / "candidate_gate.json", gate)
    for label, control in [("retained", CURRENT), ("selected_standard", standard)]:
        write_json(folder / f"paired_oof_vs_{label}.json", paired_diagnostics(oof[control], oof[l2], part=1))
        write_json(folder / f"paired_validation_vs_{label}.json", paired_diagnostics(results[control], results[l2], part=1))
    path = None
    if gate["passes_candidate_gate"]:
        test = pd.read_csv(dataset_path(root, "part1", "test"))
        scores = predict_l2_study(bundles[l2], test)
        decisions = pd.DataFrame(part1_decisions(scores[list(REASONS)], THRESHOLD, empty_set_policy="argmax"), columns=REASONS)
        decisions.insert(0, "ID", test.ID.to_numpy())
        sample = pd.read_csv(dataset_path(root, "part1", "test").parent / "part1_sample_submission.csv")
        output = folder / "submissions"; output.mkdir(exist_ok=True)
        scores.to_csv(output / "part1_l2_candidate_probabilities.csv", index=False)
        path = output / "part1_l2_candidate.csv"
        build_submission(sample, decisions).to_csv(path, index=False)
        validate_submission(pd.read_csv(path), sample, 1)
        path = path.relative_to(root).as_posix()
    return gate, path


def report(folder, summary):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    table = pd.read_csv(folder / "oof_comparison.csv")
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), constrained_layout=True)
    for use_l2, label, color in [(False, "StandardScaler", "#58748c"), (True, "StandardScaler + row L2", "#007f8b")]:
        subset = table.loc[table.use_l2.eq(use_l2)].sort_values("C")
        axes[0].plot(subset.C, subset.f1_samples, marker="o", label=label, color=color)
    axes[0].set_xscale("log"); axes[0].set_xlabel("LR C (log scale)")
    axes[0].set_ylabel("Sample F1"); axes[0].set_title("Training OOF: shared labels and weights")
    axes[0].legend(fontsize=9)
    validation = pd.read_csv(folder / "validation_comparison.csv")
    selected = [CURRENT, summary["oof_selected_standard_setting"], summary["oof_selected_l2_setting"]]
    validation = validation.set_index("setting").loc[list(dict.fromkeys(selected))].reset_index()
    bars = axes[1].barh(validation.setting.str.replace("_", " "), validation.f1_samples,
                        color=["#007f8b" if s == summary["oof_selected_l2_setting"] else "#a4b9c7" for s in validation.setting])
    axes[1].bar_label(bars, fmt="%.4f", padding=4, fontsize=9)
    axes[1].invert_yaxis(); axes[1].set_xlim(0, 1.08)
    axes[1].set_title("Official validation sample F1")
    for ax in axes:
        ax.spines[["top", "right"]].set_visible(False)
    fig.savefig(folder / "comparison.png", dpi=180); plt.close(fig)
    blocks = []
    for name in ["oof_comparison", "same_C_oof_comparison", "validation_comparison", "validation_by_reason"]:
        blocks.append("<h2>" + html.escape(name.replace("_", " ").title()) + "</h2>" +
                      pd.read_csv(folder / f"{name}.csv").to_html(index=False, float_format=lambda v: f"{v:.6f}"))
    decision = ("The selected L2 pipeline improves both training OOF and official validation over the retained model. A separate candidate CSV is available."
                if summary["passes_candidate_gate"] else
                "The selected L2 pipeline does not pass the joint improvement gate. Retain the current final model; no candidate CSV is generated.")
    page = """<!doctype html><html lang="en"><meta charset="utf-8"><title>Part 1 L2 Geometry Study</title>
<style>body{font:16px system-ui;margin:36px;max-width:1450px;color:#203047}table{border-collapse:collapse;font-size:13px}td,th{padding:7px;border:1px solid #cbd5df}th{background:#edf2f6}pre{background:#f3f5f7;padding:16px;white-space:pre-wrap}img{max-width:100%}</style>
<h1>Part 1: StandardScaler versus StandardScaler + L2</h1>
<p>This study changes only classifier geometry and LR C. K-Means K=10, mapping smoothing=2, hard pseudo-target threshold=0.5, pseudo weight=0.1, prediction threshold=0.35, and the argmax fallback stay fixed. Both geometries share the same training-fitted scaler, clusterer, gold labels, pseudo targets, and sample weights within each fold. K-Means is not refitted on L2-normalized vectors.</p>
<p>The common C grid is 0.1, 1, 10, 100. Row L2 normalization changes feature magnitude and therefore effective regularization. Scores at the same C are reported, and one C per geometry is selected with pooled four-fold gold OOF F1. This finite grid cannot establish that either geometry is universally better. Zero vectors remain zero.</p>
<p>Held-out gold rows are excluded before fitting the scaler, clustering, and label mapping. Official validation is evaluated only after both choices are locked. Earlier validation/public feedback has been inspected, and gold labels are nonrandom. Paired intervals are descriptive and do not correct for selection or shared fitted OOF models.</p>"""
    page += "<h2>Decision</h2><p>" + decision + "</p><pre>" + html.escape(json.dumps(summary, indent=2)) + "</pre>"
    page += '<img src="comparison.png" alt="OOF C curves and validation comparison">' + "".join(blocks)
    page += "<h2>Paired validation difference versus retained model</h2><pre>" + html.escape(json.dumps(read_json(folder / "paired_validation_vs_retained.json"), indent=2)) + "</pre>"
    page += "<p>Final models, earlier studies/notebooks, Part 2, report PDF, and delivery ZIP are unchanged. No files were uploaded.</p></html>"
    (folder / "study_report.html").write_text(page)


def run_study(root, *, allow_training=False):
    root = Path(root).resolve(); folder = root / STUDY
    if cache_complete(folder):
        verify_hashes(root, read_json(folder / "protected_artifacts.json"))
        verify_hashes(root, read_json(folder / "input_and_source_hashes.json"))
        return read_json(folder / "summary.json")
    if not allow_training:
        raise RuntimeError("Missing or incomplete L2 study. Run midterm_l2_geometry.py --allow-training intentionally.")
    os.environ.setdefault("LOKY_MAX_CPU_COUNT", "1")
    folder.mkdir(parents=True, exist_ok=True)
    plan = {"study": STUDY, "settings": SETTINGS, "seed": 42,
            "folds": "Study-11 saved four-fold first-positive gold assignment",
            "K": 10, "smoothing": 2., "pseudo_threshold": .5, "pseudo_weight": .1,
            "prediction_threshold": THRESHOLD, "empty_set_policy": "argmax",
            "label_mapping": "Training gold only; fixed across classifier settings",
            "clustering_input": "StandardScaler output, never L2-normalized",
            "classifier_input": "StandardScaler output, optionally row L2-normalized",
            "C_grid": list(C_GRID), "selection": "One C per geometry using pooled OOF sample F1; ties prefer smaller C",
            "validation_policy": "Retained model, standard C=0.1 reproduction check, and OOF-selected C per geometry",
            "candidate_gate": "L2 must strictly improve OOF and official validation sample F1 over retained model",
            "threshold_tuning": False, "SVM_or_ensemble_added": False,
            "validation_and_public_feedback_previously_seen": True,
            "public_scores_used_for_new_selection": False,
            "python": platform.python_version(), "sklearn": sklearn.__version__}
    path = folder / "experiment_plan.json"
    if path.exists() and read_json(path) != plan:
        raise ValueError("The predeclared L2 study plan changed.")
    write_json(path, plan)
    files = [root / p for p in ["midterm_l2_geometry.py", "midterm_next_round.py", "midterm_refinement.py",
             "midterm_augmentation.py", "midterm_evaluation.py", "midterm_part1_finalization.py",
             "midterm_part2_improvement.py", "refinement/oof_weight_char/part1/fold_assignments.csv"]]
    files += [dataset_path(root, "part1", split) for split in ["train", "val", "test"]]
    hashes = {p.relative_to(root).as_posix(): file_sha256(p) for p in files}
    if (folder / "input_and_source_hashes.json").exists():
        if read_json(folder / "input_and_source_hashes.json") != hashes:
            raise ValueError("Input or source changed during the L2 study.")
        verify_hashes(root, read_json(folder / "protected_artifacts.json"))
    else:
        write_json(folder / "input_and_source_hashes.json", hashes)
        write_json(folder / "protected_artifacts.json", protected_hashes(root))
    train = pd.read_csv(dataset_path(root, "part1", "train"))
    lock, oof = run_oof(root, folder, train)
    gate, path = validation_and_candidate(root, folder, train, lock, oof)
    summary = {**gate, "candidate_submission": path, "gold_oof_rows": 157,
               "outer_folds": 4, "settings_evaluated_in_OOF": len(SETTINGS),
               "pseudo_labels_recomputed_for_L2": False, "all_original_artifacts_unchanged": True,
               "uploads_performed": False, "report_pdf_and_delivery_zip_updated": False}
    report(folder, summary)
    verify_hashes(root, read_json(folder / "protected_artifacts.json")); verify_hashes(root, hashes)
    write_json(folder / "summary.json", summary)
    finish_cache(folder)
    return summary


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--project-root", type=Path, default=Path(__file__).resolve().parent)
    parser.add_argument("--allow-training", action="store_true")
    args = parser.parse_args()
    print(json.dumps(run_study(args.project_root, allow_training=args.allow_training), indent=2))


if __name__ == "__main__":
    main()

```


### Original midterm/midterm_part1_diagnosis.py

Original archive member: `archive/midterm_before_cleanup_2026-11-02.zip ! midterm/midterm_part1_diagnosis.py`

SHA-256: `8a1ac57b638e17f008ab7ca8a726358fcade4117e7d0040793633bedf60a9d58`

```python
"""Diagnose frozen Part 1 predictions without training or inferring test labels."""
from __future__ import annotations

from pathlib import Path
from midterm_history import verify_snapshot
import hashlib
import json
import numpy as np
import pandas as pd

REASONS = ('Safety', 'Funding', 'Efficacy', 'Recruitment', 'Other')
THRESHOLD = .35
NEAR_THRESHOLD_BAND = .05
DATA_FOLDER = 'cs-5785-midterm-2026-clinical-trial-reason-prediction'
BASE_CV_ID = 'p1ref_hard_k10_c01_v1'
GRAPH_ID = 'p1geo_graph_lr_k15_v1'


def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()


def write_json(path, value):
    Path(path).write_text(json.dumps(value, indent=2, allow_nan=False) + '\n')


def aligned_probabilities(reference, predictions):
    """Join by exact ID sets rather than relying on row positions."""
    for frame in [reference, predictions]:
        if 'ID' not in frame or frame.ID.isna().any() or not frame.ID.is_unique:
            raise ValueError('IDs must be unique and nonmissing.')
        if not frame.ID.map(lambda v: isinstance(v, str) and bool(v.strip())).all():
            raise ValueError('IDs must be nonempty strings.')
    if set(reference.ID) != set(predictions.ID):
        raise ValueError('Reference and probability ID sets differ.')
    columns = list(REASONS) if set(REASONS) <= set(predictions) else [f'prob_{r}' for r in REASONS]
    if not set(columns) <= set(predictions):
        raise ValueError('Five probability columns are required.')
    values = predictions.set_index('ID').loc[reference.ID, columns].to_numpy(dtype=float)
    if not np.isfinite(values).all() or ((values < 0) | (values > 1)).any():
        raise ValueError('Probabilities must be finite and in [0, 1].')
    return values


def decisions(probabilities, *, fallback):
    """Keep >= 0.35 and use the first maximum in the recorded reason order."""
    values = np.asarray(probabilities, dtype=float)
    if values.ndim != 2 or values.shape[1] != len(REASONS):
        raise ValueError('Expected an n-by-five probability matrix.')
    if not np.isfinite(values).all() or ((values < 0) | (values > 1)).any():
        raise ValueError('Invalid probability values.')
    result = (values >= THRESHOLD).astype(int)
    if fallback:
        positions = np.flatnonzero(result.sum(axis=1) == 0)
        result[positions, values[positions].argmax(axis=1)] = 1
    return result


def reason_sets(values):
    """Use an explicit empty marker in analysis tables, never in submissions."""
    return [' '.join(r for r, flag in zip(REASONS, row) if flag) or '(empty)' for row in values]


def row_f1(truth, prediction):
    y = np.asarray(truth, dtype=int)
    d = np.asarray(prediction, dtype=int)
    denominator = y.sum(axis=1) + d.sum(axis=1)
    return np.divide(2 * (y * d).sum(axis=1), denominator,
                     out=np.zeros(len(y), dtype=float), where=denominator != 0)


def reason_metrics(truth, prediction):
    tp = (truth & prediction).sum(axis=0)
    fp = ((1 - truth) & prediction).sum(axis=0)
    fn = (truth & (1 - prediction)).sum(axis=0)
    def divide(a, b):
        return np.divide(a, b, out=np.zeros(len(a), dtype=float), where=b != 0)
    return pd.DataFrame({'reason': REASONS, 'n_positive': truth.sum(axis=0), 'tp': tp, 'fp': fp, 'fn': fn,
                         'precision': divide(tp, tp + fp), 'recall': divide(tp, tp + fn),
                         'f1': divide(2 * tp, 2 * tp + fp + fn)})


def compare_trial_predictions(reference, base_scores, graph_scores, split):
    if split not in ['val', 'test']:
        raise ValueError('This comparison is restricted to validation and test.')
    a = aligned_probabilities(reference, base_scores)
    b = aligned_probabilities(reference, graph_scores)
    raw_a, raw_b = decisions(a, fallback=False), decisions(b, fallback=False)
    final_a, final_b = decisions(a, fallback=True), decisions(b, fallback=True)
    result = reference[['ID', 'start_year']].reset_index(drop=True).copy()
    result.insert(1, 'split', split)
    result['truth_available'] = split == 'val'
    for name, scores, raw, final in [('baseline', a, raw_a, final_a), ('graph', b, raw_b, final_b)]:
        result[f'{name}_raw_reasons'] = reason_sets(raw)
        result[f'{name}_final_reasons'] = reason_sets(final)
        result[f'{name}_fallback_used'] = raw.sum(axis=1) == 0
        result[f'{name}_fallback_reason'] = np.where(raw.sum(axis=1) == 0, np.array(REASONS)[scores.argmax(axis=1)], '(not used)')
        result[f'{name}_reason_count'] = final.sum(axis=1)
        result[f'{name}_max_probability'] = scores.max(axis=1)
        result[f'{name}_top_two_probability_gap'] = np.sort(scores, axis=1)[:, -1] - np.sort(scores, axis=1)[:, -2]
        result[f'{name}_minimum_threshold_distance'] = np.abs(scores - THRESHOLD).min(axis=1)
        for j, reason in enumerate(REASONS):
            result[f'{name}_prob_{reason}'] = scores[:, j]
            result[f'{name}_raw_pred_{reason}'] = raw[:, j]
            result[f'{name}_pred_{reason}'] = final[:, j]
    result['final_prediction_changed'] = (final_a != final_b).any(axis=1)
    result['raw_prediction_changed'] = (raw_a != raw_b).any(axis=1)
    result['added_reasons'] = reason_sets((1 - final_a) & final_b)
    result['removed_reasons'] = reason_sets(final_a & (1 - final_b))
    result['n_added_reasons'] = ((1 - final_a) & final_b).sum(axis=1)
    result['n_removed_reasons'] = (final_a & (1 - final_b)).sum(axis=1)
    for j, reason in enumerate(REASONS):
        result[f'delta_prob_{reason}'] = b[:, j] - a[:, j]
    if split == 'val':
        truth = reference[list(REASONS)].to_numpy()
        if not np.isin(truth, [0, 1]).all():
            raise ValueError('Validation truth must contain observed binary labels.')
        truth = truth.astype(int)
        result['true_reasons'] = reason_sets(truth)
        for j, reason in enumerate(REASONS):
            result[f'true_{reason}'] = truth[:, j]
        for name, raw, final in [('baseline', raw_a, final_a), ('graph', raw_b, final_b)]:
            result[f'{name}_raw_sample_f1'] = row_f1(truth, raw)
            result[f'{name}_sample_f1'] = row_f1(truth, final)
            result[f'{name}_fallback_f1_gain'] = row_f1(truth, final) - row_f1(truth, raw)
            result[f'{name}_tp'] = (truth & final).sum(axis=1)
            result[f'{name}_fp'] = ((1 - truth) & final).sum(axis=1)
            result[f'{name}_fn'] = (truth & (1 - final)).sum(axis=1)
        result['delta_sample_f1'] = result.graph_sample_f1 - result.baseline_sample_f1
        result['delta_raw_sample_f1'] = result.graph_raw_sample_f1 - result.baseline_raw_sample_f1
        result['delta_fallback_f1_gain'] = result.graph_fallback_f1_gain - result.baseline_fallback_f1_gain
        result['contribution_to_mean_f1_delta'] = result.delta_sample_f1 / len(result)
        result['change_outcome'] = np.select([result.delta_sample_f1 > 1e-12, result.delta_sample_f1 < -1e-12],
                                              ['improved', 'worsened'], default='equal')
    return result


def transitions(trials):
    records = []
    for reason in REASONS:
        a = trials[f'baseline_pred_{reason}'].to_numpy()
        b = trials[f'graph_pred_{reason}'].to_numpy()
        add, remove = (a == 0) & (b == 1), (a == 1) & (b == 0)
        row = {'split': trials.split.iloc[0], 'reason': reason, 'n_trials': len(trials),
               'baseline_positive': int(a.sum()), 'graph_positive': int(b.sum()),
               'baseline_positive_rate': float(a.mean()), 'graph_positive_rate': float(b.mean()),
               'positive_rate_delta': float(b.mean() - a.mean()), 'added': int(add.sum()),
               'removed': int(remove.sum()), 'unchanged_negative': int(((a == 0) & (b == 0)).sum()),
               'unchanged_positive': int(((a == 1) & (b == 1)).sum()),
               'mean_probability_delta': float(trials[f'delta_prob_{reason}'].mean())}
        if trials.truth_available.all():
            y = trials[f'true_{reason}'].to_numpy()
            row.update(added_true_positive=int(y[add].sum()), added_false_positive=int((1 - y[add]).sum()),
                       removed_false_positive=int((1 - y[remove]).sum()), removed_true_positive=int(y[remove].sum()))
        records.append(row)
    return pd.DataFrame(records)


def validation_changes(trials):
    records = []
    for reason in REASONS:
        mask = trials[f'baseline_pred_{reason}'].ne(trials[f'graph_pred_{reason}'])
        for _, row in trials.loc[mask].iterrows():
            old, new, true = int(row[f'baseline_pred_{reason}']), int(row[f'graph_pred_{reason}']), int(row[f'true_{reason}'])
            category = ('added_true_positive' if true else 'added_false_positive') if new else (
                'removed_true_positive' if true else 'removed_false_positive')
            records.append({'ID': row.ID, 'start_year': row.start_year, 'reason': reason, 'truth': true,
                            'baseline_prediction': old, 'graph_prediction': new, 'change_type': category,
                            'baseline_probability': row[f'baseline_prob_{reason}'],
                            'graph_probability': row[f'graph_prob_{reason}']})
    return pd.DataFrame(records)


def paired_cv(root):
    base = Path(root) / 'refinement'
    a_scope = json.loads((base / 'part1/cv_fit_scope.json').read_text())
    b_scope = json.loads((base / 'local_geometry/cv_fit_scope.json').read_text())
    if len(a_scope) != 4 or len(b_scope) != 4:
        raise ValueError('Expected four folds in both studies.')
    for a, b in zip(a_scope, b_scope):
        if a['fold'] != b['fold'] or set(a['held_out_ids']) != set(b['held_out_ids']) or set(a['fit_ids']) != set(b['fit_ids']):
            raise ValueError('The two studies do not use identical held-out folds.')
    a = pd.read_csv(base / 'part1/cv_fold_metrics.csv').query('setting_id == @BASE_CV_ID')
    b = pd.read_csv(base / 'local_geometry/cv_fold_metrics.csv').query('setting_id == @GRAPH_ID')
    columns = ['fold', 'n_rows', 'f1_samples', 'f1_macro', 'n_empty_set_fallbacks']
    result = a[columns].merge(b[columns], on='fold', suffixes=('_baseline', '_graph'), validate='one_to_one').sort_values('fold')
    if len(result) != 4 or not result.n_rows_baseline.eq(result.n_rows_graph).all():
        raise ValueError('Fold sizes do not match.')
    result['delta_sample_f1'] = result.f1_samples_graph - result.f1_samples_baseline
    result['delta_macro_f1'] = result.f1_macro_graph - result.f1_macro_baseline
    result['outcome'] = np.select([result.delta_sample_f1 > 1e-12, result.delta_sample_f1 < -1e-12],
                                  ['improved', 'worsened'], default='equal')
    return result


def seed_coverage(root, train, frames):
    """Measure distances using saved transforms; no estimator is fitted."""
    import joblib
    from scipy.spatial.distance import cdist
    root = Path(root)
    graph = joblib.load(root / f'refinement/local_geometry/runs/{GRAPH_ID}/model.joblib')
    original = joblib.load(root / 'part1_final/selected_model.joblib')
    features = [f'emb_{i}' for i in range(150)]
    mask = train[list(REASONS)].isin([0, 1]).all(axis=1).to_numpy()
    if set(graph['fit_ids']) != set(train.ID) or set(graph['seed_ids']) != set(train.loc[mask, 'ID']):
        raise AssertionError('Graph fit scope differs from original training data.')
    scaler = graph['scaler']
    for name in ['mean_', 'scale_']:
        np.testing.assert_allclose(getattr(scaler, name), getattr(original['scaler'], name), rtol=1e-12, atol=1e-12)
    train_x = scaler.transform(train[features])
    seed_x = train_x[mask]
    seed_ids = train.loc[mask, 'ID'].to_numpy()
    seed_y = train.loc[mask, list(REASONS)].to_numpy(dtype=int)
    seed_strings = np.array(reason_sets(seed_y))
    result = []
    for split, frame in frames.items():
        x = scaler.transform(frame[features])
        distances = cdist(x, seed_x, metric='euclidean')
        order = np.argsort(distances, axis=1)[:, :15]
        closest = order[:, 0]
        selected = np.take_along_axis(distances, order, axis=1)
        rows = pd.DataFrame({'ID': frame.ID.to_numpy(), 'split': split,
                             'nearest_visible_seed_ID': seed_ids[closest],
                             'nearest_visible_seed_reasons': seed_strings[closest],
                             'distance_to_nearest_visible_seed': selected[:, 0],
                             'mean_distance_to_15_visible_seeds': selected.mean(axis=1),
                             'distance_to_nearest_training_trial': cdist(x, train_x, metric='euclidean').min(axis=1)})
        for j, reason in enumerate(REASONS):
            rows[f'fraction_15_visible_seeds_{reason}'] = seed_y[order, j].mean(axis=1)
        result.append(rows)
    return pd.concat(result, ignore_index=True)


def run_diagnosis(root):
    root = Path(root)
    out = root / 'refinement/part1_diagnosis'
    out.mkdir(parents=True, exist_ok=True)
    protected_path = out / 'protected_before.json'
    if not protected_path.exists():
        raise FileNotFoundError('Restore the original-output hash snapshot before diagnosis.')
    protected = json.loads(protected_path.read_text())
    def verify_originals():
        return verify_snapshot(root, protected)
    snapshot_audit = verify_originals()
    source_paths = {
        'train': root / DATA_FOLDER / 'part1_train.csv', 'val': root / DATA_FOLDER / 'part1_val.csv',
        'test': root / DATA_FOLDER / 'part1_test.csv',
        'baseline_val': root / 'part1_final/selected_validation/validation_predictions.csv',
        'baseline_test': root / 'part1_final/test_probabilities.csv',
        'graph_val': root / f'refinement/local_geometry/runs/{GRAPH_ID}/validation_predictions.csv',
        'graph_test': root / 'refinement/submissions/part1_graph_candidate_probabilities.csv',
        'baseline_submission': root / 'part1_final/part1_submission.csv',
        'graph_submission': root / 'refinement/submissions/part1_graph_candidate.csv',
        'baseline_model': root / 'part1_final/selected_model.joblib',
        'graph_model': root / f'refinement/local_geometry/runs/{GRAPH_ID}/model.joblib',
        'baseline_cv': root / 'refinement/part1/cv_fold_metrics.csv',
        'graph_cv': root / 'refinement/local_geometry/cv_fold_metrics.csv',
        'baseline_cv_scope': root / 'refinement/part1/cv_fit_scope.json',
        'graph_cv_scope': root / 'refinement/local_geometry/cv_fit_scope.json',
    }
    manifest = {name: {'path': str(path.relative_to(root)), 'sha256': sha256(path)} for name, path in source_paths.items()}
    plan = {'threshold': THRESHOLD, 'empty_set_policy': 'argmax', 'reason_order': list(REASONS),
            'near_threshold_display_band': NEAR_THRESHOLD_BAND, 'model_fitting': False,
            'inputs': manifest, 'purpose': 'Post-submission diagnosis, not hidden-label inference or new model selection'}
    plan_path = out / 'diagnosis_plan.json'
    if plan_path.exists() and json.loads(plan_path.read_text()) != plan:
        raise ValueError('Diagnostic inputs or settings changed.')
    write_json(plan_path, plan)
    train = pd.read_csv(source_paths['train'])
    frames = {split: pd.read_csv(source_paths[split]) for split in ['val', 'test']}
    trials = {}
    for split, frame in frames.items():
        a = pd.read_csv(source_paths[f'baseline_{split}'])
        b = pd.read_csv(source_paths[f'graph_{split}'])
        trials[split] = compare_trial_predictions(frame, a, b, split)
        if split == 'val':
            for label, saved in [('baseline', a), ('graph', b)]:
                if set(f'pred_{r}' for r in REASONS) <= set(saved):
                    expected = saved.set_index('ID').loc[frame.ID, [f'pred_{r}' for r in REASONS]].to_numpy()
                    np.testing.assert_array_equal(trials[split][[f'{label}_pred_{r}' for r in REASONS]], expected)
        else:
            for label in ['baseline', 'graph']:
                saved = pd.read_csv(source_paths[f'{label}_submission'])
                expected = saved.set_index('ID').loc[frame.ID, 'Predicted'].tolist()
                if trials[split][f'{label}_final_reasons'].tolist() != expected:
                    raise AssertionError('Reconstructed test decisions differ from submitted CSV.')
    coverage = seed_coverage(root, train, frames)
    coverage_rows = []
    split_rows = []
    probability_rows = []
    cardinality_rows = []
    transition_tables = []
    pattern_tables = []
    for split, table in trials.items():
        table = table.merge(coverage.loc[coverage.split.eq(split)].drop(columns='split'), on='ID', validate='one_to_one', sort=False)
        trials[split] = table
        table.to_csv(out / f'{"validation" if split == "val" else "test"}_trial_comparison.csv', index=False)
        fallback = table.baseline_fallback_used | table.graph_fallback_used
        table.loc[fallback].to_csv(out / f'{"validation" if split == "val" else "test"}_fallback_cases.csv', index=False)
        transition_tables.append(transitions(table))
        changed = table.final_prediction_changed
        row = {'split': split, 'n_trials': len(table), 'n_final_changed': int(changed.sum()),
               'changed_rate': float(changed.mean()), 'n_raw_changed': int(table.raw_prediction_changed.sum()),
               'baseline_fallbacks': int(table.baseline_fallback_used.sum()), 'graph_fallbacks': int(table.graph_fallback_used.sum()),
               'baseline_multilabel_trials': int(table.baseline_reason_count.gt(1).sum()),
               'graph_multilabel_trials': int(table.graph_reason_count.gt(1).sum()),
               'baseline_mean_reason_count': float(table.baseline_reason_count.mean()),
               'graph_mean_reason_count': float(table.graph_reason_count.mean())}
        split_rows.append(row)
        for name, subgroup in [('all', table), ('changed', table.loc[changed]), ('unchanged', table.loc[~changed])]:
            coverage_rows.append({'split': split, 'prediction_group': name, 'n_trials': len(subgroup),
                                  'median_nearest_seed_distance': float(subgroup.distance_to_nearest_visible_seed.median()),
                                  'mean_nearest_seed_distance': float(subgroup.distance_to_nearest_visible_seed.mean()),
                                  'p90_nearest_seed_distance': float(subgroup.distance_to_nearest_visible_seed.quantile(.9)),
                                  'median_distance_to_15_seeds': float(subgroup.mean_distance_to_15_visible_seeds.median())})
        for model in ['baseline', 'graph']:
            for count in range(1, 6):
                n = int(table[f'{model}_reason_count'].eq(count).sum())
                cardinality_rows.append({'split': split, 'model': model, 'n_predicted_reasons': count,
                                         'n_trials': n, 'fraction': n / len(table)})
            for reason in REASONS:
                scores = table[f'{model}_prob_{reason}']
                probability_rows.append({'split': split, 'model': model, 'reason': reason,
                                         'mean_probability': float(scores.mean()), 'median_probability': float(scores.median()),
                                         'near_threshold_trials': int((scores - THRESHOLD).abs().le(NEAR_THRESHOLD_BAND).sum()),
                                         'near_threshold_fraction': float((scores - THRESHOLD).abs().le(NEAR_THRESHOLD_BAND).mean())})
        pattern = table.loc[changed].groupby(['baseline_final_reasons', 'graph_final_reasons'], as_index=False).agg(n_trials=('ID', 'size'))
        pattern.insert(0, 'split', split)
        pattern_tables.append(pattern)
        aggregates = {'n_trials': ('ID', 'size'), 'n_changed': ('final_prediction_changed', 'sum'),
                      'changed_rate': ('final_prediction_changed', 'mean'),
                      'baseline_fallbacks': ('baseline_fallback_used', 'sum'), 'graph_fallbacks': ('graph_fallback_used', 'sum')}
        if split == 'val':
            aggregates.update(baseline_sample_f1=('baseline_sample_f1', 'mean'), graph_sample_f1=('graph_sample_f1', 'mean'),
                              contribution_to_total_delta=('contribution_to_mean_f1_delta', 'sum'))
        year = table.groupby('start_year', dropna=False, as_index=False).agg(**aggregates)
        year.to_csv(out / f'{"validation" if split == "val" else "test"}_year_summary.csv', index=False)
    # Test reason transitions intentionally have no truth or correctness fields.
    transition_tables[0].to_csv(out / 'validation_reason_transitions.csv', index=False)
    transition_tables[1].to_csv(out / 'test_reason_transitions.csv', index=False)
    pd.concat(pattern_tables, ignore_index=True).to_csv(out / 'prediction_pattern_changes.csv', index=False)
    pd.DataFrame(split_rows).to_csv(out / 'split_summary.csv', index=False)
    pd.DataFrame(cardinality_rows).to_csv(out / 'prediction_set_size_summary.csv', index=False)
    pd.DataFrame(probability_rows).to_csv(out / 'probability_margin_summary.csv', index=False)
    pd.DataFrame(coverage_rows).to_csv(out / 'seed_coverage_summary.csv', index=False)
    coverage.to_csv(out / 'seed_coverage_by_trial.csv', index=False)
    val = trials['val']
    truth = val[[f'true_{r}' for r in REASONS]].to_numpy(dtype=int)
    metric_tables = []
    decision_rows = []
    for model in ['baseline', 'graph']:
        for policy in ['raw', 'final']:
            prediction = val[[f'{model}_{"raw_pred" if policy == "raw" else "pred"}_{r}' for r in REASONS]].to_numpy(dtype=int)
            metrics = reason_metrics(truth, prediction)
            decision_rows.append({'model': model, 'decision_policy': policy,
                                  'sample_f1': float(row_f1(truth, prediction).mean()), 'macro_f1': float(metrics.f1.mean()),
                                  'empty_predictions': int((prediction.sum(axis=1) == 0).sum())})
            if policy == 'final':
                metric_tables.append(metrics.assign(model=model))
    metrics = metric_tables[0].drop(columns='model').merge(metric_tables[1].drop(columns='model'), on=['reason', 'n_positive'], suffixes=('_baseline', '_graph'), validate='one_to_one')
    for name in ['tp', 'fp', 'fn', 'precision', 'recall', 'f1']:
        metrics[f'delta_{name}'] = metrics[f'{name}_graph'] - metrics[f'{name}_baseline']
    metrics.to_csv(out / 'validation_reason_metrics.csv', index=False)
    pd.DataFrame(decision_rows).to_csv(out / 'decision_rule_comparison.csv', index=False)
    validation_changes(val).to_csv(out / 'validation_label_change_events.csv', index=False)
    val.loc[val.final_prediction_changed | val.graph_fallback_used | val.baseline_fallback_used].sort_values(['delta_sample_f1', 'ID']).to_csv(out / 'validation_cases_to_review.csv', index=False)
    cv = paired_cv(root)
    cv.to_csv(out / 'cv_fold_comparison.csv', index=False)
    public = pd.DataFrame([
        {'task': 'Part 1', 'model': 'baseline', 'public_score': .85405, 'higher_is_better': True,
         'screenshot': 'codex-clipboard-c0d5d575-939e-4f27-843f-4d85a706417a.png'},
        {'task': 'Part 1', 'model': 'graph', 'public_score': .84081, 'higher_is_better': True,
         'screenshot': 'codex-clipboard-c0d5d575-939e-4f27-843f-4d85a706417a.png'},
        {'task': 'Part 2', 'model': 'baseline', 'public_score': .60149, 'higher_is_better': False,
         'screenshot': 'codex-clipboard-0cde248b-0ee6-453c-9e75-1e6e3ebd4b4c.png'},
        {'task': 'Part 2', 'model': 'blend', 'public_score': .59962, 'higher_is_better': False,
         'screenshot': 'codex-clipboard-0cde248b-0ee6-453c-9e75-1e6e3ebd4b4c.png'},
    ])
    public['source_type'] = 'Manually transcribed from user-provided submission screenshots'
    public.to_csv(out / 'observed_public_scores.csv', index=False)
    delta = float(val.delta_sample_f1.mean())
    raw_delta = float(val.delta_raw_sample_f1.mean())
    fallback_delta = float(val.delta_fallback_f1_gain.mean())
    np.testing.assert_allclose(delta, raw_delta + fallback_delta, atol=1e-12)
    np.testing.assert_allclose(delta, val.contribution_to_mean_f1_delta.sum(), atol=1e-12)
    positive = val.delta_sample_f1 > 1e-12
    negative = val.delta_sample_f1 < -1e-12
    fallback_correct = val.graph_fallback_used & val.graph_fallback_f1_gain.gt(0)
    summary = {
        'scope': 'Frozen Part 1 prediction diagnosis; no fitting or new submission creation',
        'validation': {'n_trials': len(val), 'n_changed': int(val.final_prediction_changed.sum()),
                       'improved_trials': int(positive.sum()), 'worsened_trials': int(negative.sum()),
                       'equal_f1_trials': int((~positive & ~negative).sum()),
                       'baseline_sample_f1': float(val.baseline_sample_f1.mean()),
                       'graph_sample_f1': float(val.graph_sample_f1.mean()), 'delta_sample_f1': delta,
                       'raw_threshold_delta': raw_delta, 'fallback_interaction_delta': fallback_delta,
                       'fallback_fraction_of_net_gain': fallback_delta / delta if abs(delta) > 1e-12 else None,
                       'gross_gain_contribution': float(val.loc[positive, 'contribution_to_mean_f1_delta'].sum()),
                       'gross_loss_contribution': float(val.loc[negative, 'contribution_to_mean_f1_delta'].sum()),
                       'graph_fallback_trials': int(val.graph_fallback_used.sum()),
                       'graph_fallback_correct_trials': int(fallback_correct.sum()),
                       'correct_graph_fallback_IDs': val.loc[fallback_correct, 'ID'].tolist()},
        'test': split_rows[1],
        'cv': {'matched_fold_scope': True, 'improved_folds': int(cv.outcome.eq('improved').sum()),
               'worsened_folds': int(cv.outcome.eq('worsened').sum()), 'equal_folds': int(cv.outcome.eq('equal').sum()),
               'baseline_mean_fold_sample_f1': float(cv.f1_samples_baseline.mean()),
               'graph_mean_fold_sample_f1': float(cv.f1_samples_graph.mean()),
               'mean_fold_delta': float(cv.delta_sample_f1.mean()),
               'pooled_oof_delta_from_fold_means': float(np.average(cv.delta_sample_f1, weights=cv.n_rows_baseline)),
               'individual_graph_OOF_predictions_available': False},
        'observed_public_score_delta': {'Part 1': float(.84081 - .85405), 'Part 2': float(.59962 - .60149)},
        'interpretation': {
            'confirmed': [
                'The graph validation advantage is small and driven mostly by one correct fallback.',
                'Other predictions increase in validation and test, with six new validation false positives.',
                'Graph sample F1 improves in two training folds, ties in one, and worsens in one.',
                'Graph public Part 1 F1 is lower in the supplied screenshot.',
            ],
            'not_identifiable': [
                'Which hidden test predictions are wrong or which reason caused the public decline.',
                'Private leaderboard performance or the causal effect of temporal shift.',
            ],
            'recommendation': 'Retain the original Part 1 model as the preferred current submission. Keep graph results as a comparison. No new model experiment is justified solely by these diagnostics.'},
    }
    write_json(out / 'summary.json', summary)
    verify_originals()
    test_columns = trials['test'].columns.tolist()
    assert not any(c.startswith('true_') or 'f1' in c or c == 'change_outcome' for c in test_columns)
    verification = {'status': 'PASS', 'protected_original_files': len(protected), 'original_files_unchanged': True, 'historical_snapshot_verified': True, 'snapshot_resolution': snapshot_audit,
                    'id_alignment': 'Exact unique ID sets, aligned by key', 'test_labels_used': False,
                    'test_correctness_inferred': False, 'models_fitted': 0, 'new_submissions_created': 0,
                    'stored_submission_decisions_reproduced': True, 'same_cv_fold_scope_verified': True,
                    'validation_f1_decomposition_reconciled': True,
                    'diagnosis_source_sha256': sha256(Path(__file__)),
                    'output_sha256': {p.name: sha256(p) for p in sorted(out.iterdir()) if p.suffix in ['.csv', '.json'] and p.name != 'verification.json'}}
    write_json(out / 'verification.json', verification)
    return summary


if __name__ == '__main__':
    import argparse
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument('--project-root', type=Path, default=Path(__file__).resolve().parent)
    print(json.dumps(run_diagnosis(parser.parse_args().project_root), indent=2))

```


### Original midterm/midterm_rbf_svm.py

Original archive member: `archive/midterm_before_cleanup_2026-11-02.zip ! midterm/midterm_rbf_svm.py`

SHA-256: `52234022ba10fcc30f5e0f32d9e8ff7110b692e8a0068e0147d2609f108289de`

```python
"""Gold-label RBF SVM comparisons with nested, training-only calibration.

The frozen four-fold assignment and existing Part 1 model are preserved.
Only the predeclared RBF C value is selected with training OOF predictions.
"""
from __future__ import annotations

import argparse
import html
import json
import os
from pathlib import Path
import platform
import warnings

import joblib
import numpy as np
import pandas as pd
import sklearn
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from threadpoolctl import threadpool_limits

from midterm_augmentation import logistic_parameters, observed_label_mask
from midterm_evaluation import REASONS, dataset_path, evaluate_part1, file_sha256, part1_decisions
from midterm_next_round import (
    cache_complete, finish_cache, paired_diagnostics, read_json,
    reason_error_counts, validate_fold_scope, verify_hashes, verify_oof, write_json,
)
from midterm_part1_finalization import build_submission
from midterm_part2_improvement import validate_submission

STUDY = "refinement/rbf_svm"
EMBEDDINGS = [f"emb_{i}" for i in range(150)]
SETTINGS = {
    "gold_lr_c1": {"family": "logistic", "C": 1.0},
    "gold_linear_svm_c1": {"family": "svm", "kernel": "linear", "C": 1.0},
    "gold_rbf_svm_c1": {"family": "svm", "kernel": "rbf", "C": 1.0},
    "gold_rbf_svm_c10": {"family": "svm", "kernel": "rbf", "C": 10.0},
}
RBF_SETTINGS = ("gold_rbf_svm_c1", "gold_rbf_svm_c10")
CURRENT = "retained_kmeans_lr"
THRESHOLD = .35


def check_fit(model, caught):
    if any(issubclass(w.category, ConvergenceWarning) for w in caught):
        raise RuntimeError("An estimator failed to converge under the declared limit.")
    if getattr(model, "fit_status_", 0) != 0 or np.max(model.n_iter_) >= model.max_iter:
        raise RuntimeError("An estimator reached its iteration limit.")
    if not np.array_equal(model.classes_, [0, 1]):
        raise ValueError("Every fitted reason model requires both gold-label classes.")


def fit_svc_models(x, y, setting):
    """Fit five independent SVMs; no built-in or pseudo-label calibration."""
    if setting.get("kernel") not in {"linear", "rbf"} or len(x) != len(y):
        raise ValueError("Unexpected SVM setting or target shape.")
    if not np.isin(y, [0, 1]).all() or y.shape != (len(x), len(REASONS)):
        raise ValueError("SVM targets must contain five binary gold labels.")
    models = []
    for column in range(len(REASONS)):
        if np.unique(y[:, column]).size != 2:
            raise ValueError("Each reason needs both gold-label classes.")
        model = SVC(C=setting["C"], kernel=setting["kernel"], gamma="scale",
                    class_weight=None, tol=1e-3, max_iter=10000,
                    cache_size=200, random_state=42)
        with warnings.catch_warnings(record=True) as caught, threadpool_limits(limits=1):
            warnings.simplefilter("always")
            model.fit(x, y[:, column])
        check_fit(model, caught)
        models.append(model)
    return models


def margins(models, x):
    return np.column_stack([model.decision_function(x) for model in models])


def inner_assignment(gold):
    strata = gold[list(REASONS)].to_numpy().argmax(axis=1)
    counts = np.bincount(strata)
    if counts[counts > 0].min() < 3:
        raise ValueError("Three-fold first-positive calibration lacks gold-label support.")
    assignment = np.full(len(gold), -1)
    splitter = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    for fold, (_, held) in enumerate(splitter.split(gold, strata)):
        assignment[held] = fold
    return assignment


def fit_family(frame, svm_settings):
    """Fit LR and SVMs with an identical outer training context.

    Scalers may use the original masked training embeddings. Classifiers and
    calibrators use observed labels only. Inner-held gold rows are excluded
    from both scaling and SVM fitting before generating calibration margins.
    """
    if not frame.ID.is_unique:
        raise ValueError("Training row IDs must be unique.")
    mask = observed_label_mask(frame[list(REASONS)].to_numpy())
    gold = frame.loc[mask].reset_index(drop=True)
    y = gold[list(REASONS)].to_numpy(dtype=int)
    assignment = inner_assignment(gold)
    oof_margins = {name: np.full(y.shape, np.nan) for name in svm_settings}
    inner_scopes = []
    for fold in range(3):
        held = gold.loc[assignment == fold].reset_index(drop=True)
        fit = frame.loc[~frame.ID.isin(held.ID)].reset_index(drop=True)
        scope = validate_fold_scope(fit, held, part=1)
        seed_mask = observed_label_mask(fit[list(REASONS)].to_numpy())
        seeds = fit.loc[seed_mask]
        scope.update(inner_fold=fold, scaler_fit_ids=fit.ID.tolist(),
                     classifier_fit_ids=seeds.ID.tolist(), calibration_hold_ids=held.ID.tolist())
        inner_scopes.append(scope)
        scaler = StandardScaler().fit(fit[EMBEDDINGS])
        x = scaler.transform(seeds[EMBEDDINGS])
        target = seeds[list(REASONS)].to_numpy(dtype=int)
        for name, setting in svm_settings.items():
            models = fit_svc_models(x, target, setting)
            oof_margins[name][assignment == fold] = margins(models, scaler.transform(held[EMBEDDINGS]))
    scaler = StandardScaler().fit(frame[EMBEDDINGS])
    gold_x = scaler.transform(gold[EMBEDDINGS])
    base_scope = {"scaler_fit_ids": frame.ID.tolist(), "classifier_fit_ids": gold.ID.tolist(),
                  "calibrator_fit_ids": gold.ID.tolist(), "inner_splits": inner_scopes,
                  "masked_rows_used_as_classifier_or_calibration_targets": 0,
                  "outer_held_rows_used_in_any_fit": 0,
                  "preprocessing": "StandardScaler; no row L2 normalization"}
    bundles = {}
    lr_models = []
    for column in range(len(REASONS)):
        model = LogisticRegression(**logistic_parameters(42))
        with warnings.catch_warnings(record=True) as caught, threadpool_limits(limits=1):
            warnings.simplefilter("always")
            model.fit(gold_x, y[:, column])
        check_fit(model, caught)
        lr_models.append(model)
    bundles["gold_lr_c1"] = {"kind": "gold_lr", "scaler": scaler, "models": lr_models,
                              "setting": SETTINGS["gold_lr_c1"], "threshold": THRESHOLD,
                              "empty_set_policy": "argmax", "fit_scope": base_scope}
    for name, setting in svm_settings.items():
        raw = oof_margins[name]
        if not np.isfinite(raw).all():
            raise AssertionError("Every calibration row requires one inner-OOF margin per reason.")
        calibrators = []
        for column in range(len(REASONS)):
            # A regularized sigmoid on gold-only OOF margins; not SVC's built-in calibration.
            calibrator = LogisticRegression(**{**logistic_parameters(42), "solver": "lbfgs",
                                               "C": 1.0, "max_iter": 1000, "tol": 1e-6})
            with warnings.catch_warnings(record=True) as caught, threadpool_limits(limits=1):
                warnings.simplefilter("always")
                calibrator.fit(raw[:, [column]], y[:, column])
            check_fit(calibrator, caught)
            calibrators.append(calibrator)
        bundles[name] = {"kind": "gold_calibrated_svm", "scaler": scaler,
                         "models": fit_svc_models(gold_x, y, setting), "calibrators": calibrators,
                         "setting": setting, "threshold": THRESHOLD, "empty_set_policy": "argmax",
                         "calibration": "3-fold gold OOF margins -> sigmoid LR C=1, unpenalized intercept",
                         "fit_scope": base_scope}
    calibration_rows = gold[["ID", *REASONS]].copy().assign(inner_fold=assignment)
    for name, scores in oof_margins.items():
        for index, reason in enumerate(REASONS):
            calibration_rows[f"{name}_margin_{reason}"] = scores[:, index]
    return bundles, base_scope, calibration_rows


def predict_svm_study(bundle, frame):
    """Return independent, ID-aligned probabilities without fitting."""
    if bundle.get("kind") not in {"gold_lr", "gold_calibrated_svm"}:
        raise ValueError("Unexpected SVM-study model bundle.")
    x = bundle["scaler"].transform(frame[EMBEDDINGS])
    if bundle["kind"] == "gold_lr":
        probabilities = np.column_stack([m.predict_proba(x)[:, list(m.classes_).index(1)] for m in bundle["models"]])
    else:
        raw = margins(bundle["models"], x)
        probabilities = np.column_stack([m.predict_proba(raw[:, [j]])[:, list(m.classes_).index(1)]
                                         for j, m in enumerate(bundle["calibrators"])])
    if probabilities.shape != (len(frame), len(REASONS)) or not np.isfinite(probabilities).all() or ((probabilities < 0) | (probabilities > 1)).any():
        raise ValueError("Invalid reason probabilities.")
    output = pd.DataFrame(probabilities, columns=REASONS)
    output.insert(0, "ID", frame.ID.to_numpy())
    return output


def scores_from_saved(frame):
    return frame[["ID", *[f"prob_{r}" for r in REASONS]]].rename(columns={f"prob_{r}": r for r in REASONS})


def protected_hashes(root):
    paths = [root / "final_selection.json"]
    study = root / STUDY
    for folder in ["part1_final", "part2_final", "experiments", "submissions", "deliverables", "report", "refinement"]:
        paths.extend(p for p in (root / folder).rglob("*") if p.is_file()
                     and not p.is_relative_to(study) and "__pycache__" not in p.parts)
    paths.extend(p for p in root.glob("[0-9][0-9]_*.ipynb") if int(p.name[:2]) < 12)
    return {p.relative_to(root).as_posix(): file_sha256(p) for p in sorted(set(paths))}


def run_oof(root, folder, train):
    gold = train.loc[observed_label_mask(train[list(REASONS)].to_numpy())].reset_index(drop=True)
    assignment = pd.read_csv(root / "refinement/oof_weight_char/part1/fold_assignments.csv")
    if not assignment.ID.is_unique or set(assignment.ID) != set(gold.ID) or set(assignment.fold) != set(range(4)):
        raise ValueError("The original four-fold gold assignment is invalid.")
    assignment = assignment.set_index("ID").loc[gold.ID].reset_index()
    assignment.to_csv(folder / "fold_assignments.csv", index=False)
    settings = {name: setting for name, setting in SETTINGS.items() if setting["family"] == "svm"}
    frames, fold_rows = {name: [] for name in SETTINGS}, []
    for fold in range(4):
        cache = folder / "fold_cache" / f"fold_{fold}"
        if not cache_complete(cache):
            cache.mkdir(parents=True, exist_ok=True)
            held_ids = assignment.loc[assignment.fold.eq(fold), "ID"]
            held = gold.loc[gold.ID.isin(held_ids)].reset_index(drop=True)
            fit = train.loc[~train.ID.isin(held_ids)].reset_index(drop=True)
            outer_scope = validate_fold_scope(fit, held, part=1)
            bundles, scope, calibration = fit_family(fit, settings)
            if set(scope["scaler_fit_ids"]) & set(held_ids) or set(scope["calibrator_fit_ids"]) & set(held_ids):
                raise AssertionError("An outer-held row entered a fitted component.")
            write_json(cache / "fit_scope.json", {"outer": outer_scope, "nested": scope})
            calibration.to_csv(cache / "inner_oof_calibration_margins.csv", index=False)
            for name, bundle in bundles.items():
                result = evaluate_part1(held, predict_svm_study(bundle, held), THRESHOLD, empty_set_policy="argmax")
                result.predictions.assign(fold=fold).to_csv(cache / f"{name}.csv", index=False)
                write_json(cache / f"{name}_metrics.json", result.summary)
            finish_cache(cache)
        for name in SETTINGS:
            frames[name].append(pd.read_csv(cache / f"{name}.csv"))
            fold_rows.append({"setting": name, "fold": fold, **read_json(cache / f"{name}_metrics.json")})
        print(f"Nested-calibration SVM OOF fold {fold + 1}/4 complete", flush=True)
    folds = pd.DataFrame(fold_rows)
    folds.to_csv(folder / "fold_metrics.csv", index=False)
    evaluations, comparisons, reason_rows = {}, [], []
    for name in SETTINGS:
        oof = verify_oof(gold, pd.concat(frames[name], ignore_index=True), assignment, part=1)
        result = evaluate_part1(gold, scores_from_saved(oof), THRESHOLD, empty_set_policy="argmax")
        evaluations[name] = result
        oof.to_csv(folder / f"oof_{name}.csv", index=False)
        subset = folds.loc[folds.setting.eq(name)]
        comparisons.append({"setting": name, **result.summary, "mean_fold_f1": subset.f1_samples.mean(),
                            "std_fold_f1": subset.f1_samples.std()})
        reason_rows.append(reason_error_counts(result).assign(setting=name))
    frozen = pd.read_csv(root / "refinement/oof_weight_char/part1/oof_weight_010.csv")
    frozen = verify_oof(gold, frozen, assignment, part=1)
    retained = evaluate_part1(gold, scores_from_saved(frozen), THRESHOLD, empty_set_policy="argmax")
    evaluations[CURRENT] = retained
    comparisons.append({"setting": CURRENT, **retained.summary,
                        "mean_fold_f1": pd.read_csv(root / "refinement/oof_weight_char/part1/oof_comparison.csv").set_index("setting").loc["weight_010", "mean_fold_f1"],
                        "std_fold_f1": pd.read_csv(root / "refinement/oof_weight_char/part1/oof_comparison.csv").set_index("setting").loc["weight_010", "std_fold_f1"]})
    reason_rows.append(reason_error_counts(retained).assign(setting=CURRENT))
    table = pd.DataFrame(comparisons)
    table["delta_f1_vs_retained"] = table.f1_samples - retained.summary["f1_samples"]
    table.to_csv(folder / "oof_comparison.csv", index=False)
    pd.concat(reason_rows).to_csv(folder / "oof_by_reason.csv", index=False)
    winner = max(RBF_SETTINGS, key=lambda name: (evaluations[name].summary["f1_samples"], -SETTINGS[name]["C"]))
    lock = {"rbf_setting": winner, "C": SETTINGS[winner]["C"], "gamma": "scale",
            "metric": "pooled four-fold gold OOF sample F1", "threshold": THRESHOLD,
            "tie_policy": "Prefer C=1", "official_validation_labels_used": False,
            "test_predictions_used": False, "public_feedback_used_to_select_C": False}
    write_json(folder / "selection_locked_before_validation.json", lock)
    return lock, evaluations


def validation_and_candidate(root, folder, train, lock, oof):
    svm_names = ["gold_linear_svm_c1", lock["rbf_setting"]]
    bundles, scope, calibration = fit_family(train, {name: SETTINGS[name] for name in svm_names})
    write_json(folder / "full_training_fit_scope.json", scope)
    calibration.to_csv(folder / "full_training_inner_oof_margins.csv", index=False)
    for name, bundle in bundles.items():
        joblib.dump(bundle, folder / f"{name}.joblib", compress=3)
    # This study first reads official validation labels after locking the RBF C.
    validation = pd.read_csv(dataset_path(root, "part1", "val"))
    current = pd.read_csv(root / "part1_final/selected_validation/validation_predictions.csv")
    results = {CURRENT: evaluate_part1(validation, scores_from_saved(current), THRESHOLD, empty_set_policy="argmax")}
    results.update({name: evaluate_part1(validation, predict_svm_study(bundle, validation), THRESHOLD, empty_set_policy="argmax")
                    for name, bundle in bundles.items()})
    for name, result in results.items():
        result.save(folder / "validation" / name)
    pd.DataFrame([{"setting": name, **r.summary} for name, r in results.items()]).to_csv(folder / "validation_comparison.csv", index=False)
    pd.concat([reason_error_counts(r).assign(setting=name) for name, r in results.items()]).to_csv(folder / "validation_by_reason.csv", index=False)
    selected = lock["rbf_setting"]
    oof_gain = oof[selected].summary["f1_samples"] - oof[CURRENT].summary["f1_samples"]
    val_gain = results[selected].summary["f1_samples"] - results[CURRENT].summary["f1_samples"]
    gate = {"oof_selected_rbf_setting": selected, "oof_f1_gain_vs_retained": oof_gain,
            "validation_f1_gain_vs_retained": val_gain,
            "passes_candidate_gate": bool(oof_gain > 1e-12 and val_gain > 1e-12),
            "final_model_replaced": False, "public_or_private_score_known": False}
    write_json(folder / "candidate_gate.json", gate)
    for label, control in [("retained", CURRENT), ("matched_gold_lr", "gold_lr_c1"), ("linear_svm", "gold_linear_svm_c1")]:
        write_json(folder / f"paired_oof_vs_{label}.json", paired_diagnostics(oof[control], oof[selected], part=1))
        write_json(folder / f"paired_validation_vs_{label}.json", paired_diagnostics(results[control], results[selected], part=1))
    path = None
    if gate["passes_candidate_gate"]:
        test = pd.read_csv(dataset_path(root, "part1", "test"))
        scores = predict_svm_study(bundles[selected], test)
        decisions = pd.DataFrame(part1_decisions(scores[list(REASONS)], THRESHOLD, empty_set_policy="argmax"), columns=REASONS)
        decisions.insert(0, "ID", test.ID.to_numpy())
        sample = pd.read_csv(dataset_path(root, "part1", "test").parent / "part1_sample_submission.csv")
        output = folder / "submissions"
        output.mkdir(exist_ok=True)
        scores.to_csv(output / "part1_rbf_candidate_probabilities.csv", index=False)
        submission = build_submission(sample, decisions)
        path = output / "part1_rbf_candidate.csv"
        submission.to_csv(path, index=False)
        validate_submission(pd.read_csv(path), sample, 1)
        path = path.relative_to(root).as_posix()
    return gate, path


def report(folder, summary):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    fig, axes = plt.subplots(1, 2, figsize=(13, 5), constrained_layout=True)
    for ax, split, title in zip(axes, ["oof", "validation"], ["Training OOF sample F1", "Official validation sample F1"]):
        table = pd.read_csv(folder / f"{split}_comparison.csv")
        colors = ["#007f8b" if s == summary["oof_selected_rbf_setting"] else "#a4b9c7" for s in table.setting]
        bars = ax.barh(table.setting.str.replace("_", " "), table.f1_samples, color=colors)
        ax.bar_label(bars, fmt="%.4f", padding=4, fontsize=9)
        ax.set_title(title + " (higher is better)", fontsize=11)
        ax.invert_yaxis(); ax.set_xlim(0, 1.08); ax.spines[["top", "right"]].set_visible(False)
    fig.savefig(folder / "comparison.png", dpi=180)
    plt.close(fig)
    tables = []
    for filename in ["oof_comparison", "validation_comparison", "oof_by_reason", "validation_by_reason"]:
        tables.append("<h2>" + html.escape(filename.replace("_", " ").title()) + "</h2>" +
                      pd.read_csv(folder / f"{filename}.csv").to_html(index=False, float_format=lambda v: f"{v:.6f}"))
    decision = ("The selected RBF passes the predeclared OOF and validation improvement gate; a candidate submission is available."
                if summary["passes_candidate_gate"] else
                "The selected RBF does not pass the joint improvement gate. Retain the existing final model; no new submission is generated.")
    page = """<!doctype html><html lang="en"><meta charset="utf-8"><title>Gold-label RBF SVM Study</title>
<style>body{font:16px system-ui;margin:36px;max-width:1450px;color:#203047}table{border-collapse:collapse;font-size:13px}td,th{padding:7px;border:1px solid #cbd5df}th{background:#edf2f6}pre{background:#f3f5f7;padding:16px;white-space:pre-wrap}img{max-width:100%}</style>
<h1>Gold-label RBF SVM Study</h1>
<p>Five independent binary classifiers use only observed labels. RBF C is selected from 1 and 10, gamma is scale, and class weights are unchanged. A gold LR C=1 and gold linear SVM C=1 provide controls. All settings use the original four folds, prediction threshold 0.35, and the argmax fallback.</p>
<p>StandardScaler is fitted only on the training context, including originally masked embeddings. Classifier and calibration targets are gold only. SVM probabilities use a fixed regularized sigmoid fitted to three-fold inner-OOF gold margins; inner-held rows are excluded from scaling and SVM fitting. Calibration slopes and intercepts are learned only inside the outer training fold. This is not SVC's built-in calibration. LR uses its native probabilities, so an LR/SVM difference includes both model and calibration differences. The matched linear SVM helps assess the effect of the kernel.</p>
<p>RBF C is locked before reading validation labels in this study. Prior validation and public feedback have already been inspected. The 157 gold labels are nonrandom and Funding has only eight positives. OOF selection scores and paired intervals are descriptive, with no correction for model selection or shared fitted models.</p>"""
    page += "<h2>Decision</h2><p>" + decision + "</p><pre>" + html.escape(json.dumps(summary, indent=2)) + "</pre>"
    page += '<img src="comparison.png" alt="OOF and validation model comparisons">' + "".join(tables)
    page += "<h2>Paired validation difference versus retained model</h2><pre>" + html.escape(json.dumps(read_json(folder / "paired_validation_vs_retained.json"), indent=2)) + "</pre>"
    page += "<p>Current final artifacts, Part 2, the report PDF, and the delivery ZIP are unchanged. No submission was uploaded. Existing K-Means/GMM missing-label recovery remains part of the overall assignment; this study alone is a supervised comparison.</p></html>"
    (folder / "study_report.html").write_text(page)


def run_study(root, *, allow_training=False):
    root = Path(root).resolve(); folder = root / STUDY
    if cache_complete(folder):
        verify_hashes(root, read_json(folder / "protected_artifacts.json"))
        verify_hashes(root, read_json(folder / "input_and_source_hashes.json"))
        return read_json(folder / "summary.json")
    if not allow_training:
        raise RuntimeError("Missing or incomplete SVM study. Run midterm_rbf_svm.py --allow-training intentionally.")
    os.environ.setdefault("LOKY_MAX_CPU_COUNT", "1")
    folder.mkdir(parents=True, exist_ok=True)
    plan = {"study": STUDY, "settings": SETTINGS, "selected_family": "RBF SVM",
            "outer_folds": "Saved study-11 first-positive gold four-fold assignment",
            "calibration_inner_folds": 3, "calibration_seed": 42,
            "calibration": "Gold OOF margins -> sigmoid LR C=1, solver=lbfgs, unpenalized intercept",
            "scaler_context": "All training-context embeddings, excluding every current held-out row",
            "supervised_and_calibration_targets": "Observed gold labels only",
            "prediction_threshold": THRESHOLD, "empty_set_policy": "argmax",
            "gamma": "scale", "class_weight": None, "row_L2_normalization": False,
            "pseudo_labels_used_as_targets": False, "ensembling": False,
            "validation_policy": "Evaluate retained control, gold LR, linear SVM, and one OOF-selected RBF",
            "candidate_gate": "Strict improvement over retained model in both OOF and validation sample F1",
            "validation_and_public_feedback_previously_seen": True, "public_feedback_selects_C": False,
            "python": platform.python_version(), "sklearn": sklearn.__version__}
    plan_path = folder / "experiment_plan.json"
    if plan_path.exists() and read_json(plan_path) != plan:
        raise ValueError("The predeclared SVM study plan changed.")
    write_json(plan_path, plan)
    source_files = [root / p for p in ["midterm_rbf_svm.py", "midterm_next_round.py", "midterm_evaluation.py",
                     "midterm_augmentation.py", "midterm_part1_finalization.py", "midterm_part2_improvement.py",
                     "refinement/oof_weight_char/part1/fold_assignments.csv"]]
    source_files += [dataset_path(root, "part1", split) for split in ["train", "val", "test"]]
    hashes = {p.relative_to(root).as_posix(): file_sha256(p) for p in source_files}
    if (folder / "input_and_source_hashes.json").exists():
        if read_json(folder / "input_and_source_hashes.json") != hashes:
            raise ValueError("Source or data changed during the SVM study.")
        verify_hashes(root, read_json(folder / "protected_artifacts.json"))
    else:
        write_json(folder / "input_and_source_hashes.json", hashes)
        write_json(folder / "protected_artifacts.json", protected_hashes(root))
    train = pd.read_csv(dataset_path(root, "part1", "train"))
    lock, oof = run_oof(root, folder, train)
    gate, candidate_path = validation_and_candidate(root, folder, train, lock, oof)
    summary = {**gate, "candidate_submission": candidate_path,
               "gold_training_rows": int(observed_label_mask(train[list(REASONS)].to_numpy()).sum()),
               "masked_rows_used_as_targets": 0, "outer_folds": 4, "inner_calibration_folds": 3,
               "all_original_artifacts_unchanged": True, "uploads_performed": False,
               "report_pdf_and_delivery_zip_updated": False}
    report(folder, summary)
    verify_hashes(root, read_json(folder / "protected_artifacts.json"))
    verify_hashes(root, hashes)
    write_json(folder / "summary.json", summary)
    finish_cache(folder)
    return summary


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--project-root", type=Path, default=Path(__file__).resolve().parent)
    parser.add_argument("--allow-training", action="store_true")
    args = parser.parse_args()
    print(json.dumps(run_study(args.project_root, allow_training=args.allow_training), indent=2))


if __name__ == "__main__":
    main()

```


### Original midterm/midterm_refinement_review.py

Original archive member: `archive/midterm_before_cleanup_2026-11-02.zip ! midterm/midterm_refinement_review.py`

SHA-256: `2bde49877661ea3fb12c96a0d1b0f7c07da3666e3a9b9f02ae37a9ce038c63e0`

```python
"""Audit frozen candidate studies without fitting or modifying submitted outputs."""
from pathlib import Path
from midterm_history import verify_snapshot, baseline_model_path
from collections import Counter
import json
import joblib
import numpy as np
import pandas as pd
from midterm_evaluation import REASONS, PHASES, dataset_path, evaluate_part1, file_sha256
from midterm_augmentation import observed_label_mask
from midterm_part1_finalization import binary_decisions, build_submission
from midterm_part2_baseline import predict_outcome_probabilities
from midterm_part2_improvement import validate_submission
from midterm_refinement import predict_reasons, predict_field_outcomes, write_json
from midterm_graph_refinement import predict_geometry


def _saved_scores(path):
    frame = pd.read_csv(path)
    return frame[['ID', *[f'prob_{r}' for r in REASONS]]].rename(
        columns={f'prob_{r}': r for r in REASONS})


def audit_refinement(root):
    root = Path(root)
    folder = root / 'refinement'
    protected = json.loads((folder / 'original_output_checksums.json').read_text())
    snapshot_audit = verify_snapshot(root, protected)
    train = pd.read_csv(dataset_path(root, 'part1', 'train'))
    val = pd.read_csv(dataset_path(root, 'part1', 'val'))
    test = pd.read_csv(dataset_path(root, 'part1', 'test'))
    train_ids = set(train.ID)
    visible_ids = set(train.loc[observed_label_mask(train[list(REASONS)].to_numpy()), 'ID'])
    scope_checks = []
    for path in [folder / 'part1/cv_fit_scope.json', folder / 'local_geometry/cv_fit_scope.json']:
        held_counts = Counter()
        for fold in json.loads(path.read_text()):
            fit = set(fold['fit_ids'])
            held = set(fold['held_out_ids'])
            seeds = set(fold.get('seed_ids', fold.get('label_seed_ids', [])))
            assert fit.isdisjoint(held)
            assert fit | held == train_ids
            assert held <= visible_ids
            assert seeds == visible_ids - held
            assert seeds <= fit
            held_counts.update(held)
        assert set(held_counts) == visible_ids and set(held_counts.values()) == {1}
        scope_checks.append({'study': path.parent.name, 'folds': 4,
                             'observed_trials_held_once': len(held_counts), 'status': 'PASS'})
    reloaded = []
    for runs, predict in [(folder / 'part1/runs', predict_reasons),
                          (folder / 'local_geometry/runs', lambda b, f: predict_geometry(b, f)[0])]:
        for directory in sorted(runs.iterdir()):
            bundle = joblib.load(directory / 'model.joblib')
            assert set(bundle['fit_ids']) == train_ids
            assert set(bundle['seed_ids']) == visible_ids
            actual = predict(bundle, val)
            expected = _saved_scores(directory / 'validation_predictions.csv')
            assert actual.ID.tolist() == expected.ID.tolist()
            error = float(np.max(np.abs(actual[list(REASONS)].to_numpy() - expected[list(REASONS)].to_numpy())))
            assert error < 1e-12
            reloaded.append({'model': directory.name, 'max_probability_error': error, 'status': 'PASS'})
    graph_dir = folder / 'local_geometry/runs/p1geo_graph_lr_k15_v1'
    graph = joblib.load(graph_dir / 'model.joblib')
    graph_scores = predict_geometry(graph, test)[0]
    binary = binary_decisions(graph_scores[list(REASONS)], dict.fromkeys(REASONS, .35), empty_set_policy='argmax')
    decisions = pd.DataFrame(binary, columns=REASONS)
    decisions.insert(0, 'ID', test.ID.to_numpy())
    p1_sample = pd.read_csv(dataset_path(root, 'part1', 'test').parent / 'part1_sample_submission.csv')
    rebuilt_p1 = build_submission(p1_sample, decisions)
    saved_p1 = pd.read_csv(folder / 'submissions/part1_graph_candidate.csv')
    validate_submission(saved_p1, p1_sample, 1)
    pd.testing.assert_frame_equal(rebuilt_p1, saved_p1)
    augmented = pd.read_csv(folder / 'local_geometry/part1_train_graph_augmented.csv')
    assert augmented.ID.tolist() == train.ID.tolist()
    observed = observed_label_mask(train[list(REASONS)].to_numpy())
    np.testing.assert_array_equal(augmented.loc[observed, list(REASONS)], train.loc[observed, list(REASONS)])
    assert augmented[list(REASONS)].isin([0, 1]).all().all()
    field = joblib.load(folder / 'part2/runs/field_tfidf_lr_c1_v1/model.joblib')
    base = joblib.load(baseline_model_path(root))
    outcome_train = pd.read_csv(dataset_path(root, 'part2', 'train'))
    outcome_val = pd.read_csv(dataset_path(root, 'part2', 'val'))
    outcome_test = pd.read_csv(dataset_path(root, 'part2', 'test'))
    assert set(outcome_train.trial_id).isdisjoint(outcome_val.trial_id)
    assert set(outcome_train.trial_id).isdisjoint(outcome_test.trial_id)
    for phase, bundle in field['models'].items():
        assert set(bundle['fit_trial_ids']) == set(outcome_train.loc[outcome_train.phase.eq(phase), 'trial_id'])
    expected_field = pd.read_csv(folder / 'part2/field_tfidf_lr_c1/validation_predictions.csv')
    actual_field = predict_field_outcomes(field, outcome_val)
    assert actual_field.ID.tolist() == expected_field.ID.tolist()
    field_error = float(np.max(np.abs(actual_field.p_success - expected_field.p_success)))
    assert field_error < 1e-12
    reloaded.append({'model': 'field_tfidf_lr_c1_v1', 'max_probability_error': field_error, 'status': 'PASS'})
    p_base = predict_outcome_probabilities(base, outcome_test)
    p_field = predict_field_outcomes(field, outcome_test)
    assert p_base.ID.tolist() == p_field.ID.tolist()
    probabilities = p_base.copy()
    probabilities['p_success'] = .5 * p_base.p_success + .5 * p_field.p_success
    p2_sample = pd.read_csv(dataset_path(root, 'part2', 'test').parent / 'part2_sample_submission.csv')
    rebuilt_p2 = p2_sample[['ID']].merge(probabilities, on='ID', validate='one_to_one', sort=False).rename(columns={'p_success': 'label'})
    saved_p2 = pd.read_csv(folder / 'submissions/part2_candidate.csv')
    validate_submission(saved_p2, p2_sample, 2)
    pd.testing.assert_frame_equal(rebuilt_p2, saved_p2, check_exact=False, rtol=1e-12, atol=1e-12)
    policy_rows = []
    sources = {'current_submission': root / 'part1_final/selected_validation/validation_predictions.csv',
               'p1geo_graph_lr_k15_v1': graph_dir / 'validation_predictions.csv'}
    for name, path in sources.items():
        scores = _saved_scores(path)
        empty_rows = int((scores[list(REASONS)] < .35).all(axis=1).sum())
        for policy in ['allow', 'argmax']:
            result = evaluate_part1(val, scores, .35, empty_set_policy=policy)
            policy_rows.append({'setting_id': name, 'empty_set_policy': policy, 'raw_empty_rows': empty_rows, **result.summary})
    pd.DataFrame(policy_rows).to_csv(folder / 'local_geometry/decision_policy_comparison.csv', index=False)
    # Resample whole trials, retaining all phases of a combined-phase trial together.
    before = pd.read_csv(folder / 'part2/baseline/validation_predictions.csv')
    after = pd.read_csv(folder / 'part2/blend_equal/validation_predictions.csv')
    for field_name in ['ID', 'trial_id', 'phase', 'label']:
        assert before[field_name].tolist() == after[field_name].tolist()
    y = before.label.to_numpy()
    def row_loss(frame):
        p = np.clip(frame.p_success.to_numpy(), 1e-15, 1 - 1e-15)
        return -(y * np.log(p) + (1 - y) * np.log1p(-p))
    delta = row_loss(after) - row_loss(before)
    codes, unique_trials = pd.factorize(before.trial_id, sort=True)
    phase_codes = before.phase.map({v: i for i, v in enumerate(PHASES)}).to_numpy()
    sums = np.zeros((len(unique_trials), 3))
    counts = np.zeros_like(sums)
    np.add.at(sums, (codes, phase_codes), delta)
    np.add.at(counts, (codes, phase_codes), 1)
    rng = np.random.default_rng(42)
    multiplicities = rng.multinomial(len(unique_trials), np.full(len(unique_trials), 1 / len(unique_trials)), size=2000)
    draws = ((multiplicities @ sums) / (multiplicities @ counts)).mean(axis=1)
    uncertainty = {'delta_equal_phase_log_loss': float((sums.sum(axis=0) / counts.sum(axis=0)).mean()),
                   'descriptive_95_percentile_interval': np.quantile(draws, [.025, .975]).tolist(),
                   'resampling_unit': 'trial_id, preserving all phases together', 'draws': 2000, 'seed': 42,
                   'warning': 'Descriptive only; does not correct for model selection or temporal shift.'}
    write_json(folder / 'part2/paired_validation_uncertainty.json', uncertainty)
    report = {'status': 'PASS', 'protected_original_files': len(protected), 'original_outputs_unchanged': True, 'historical_snapshot_verified': True, 'snapshot_resolution': snapshot_audit,
              'cv_scope': scope_checks, 'reloaded_models': reloaded,
              'gold_training_labels_preserved': int(observed.sum()),
              'candidate_submissions': [
                  {'file': 'submissions/part1_graph_candidate.csv', 'rows': len(saved_p1),
                   'null_cells': int(saved_p1.isna().sum().sum()),
                   'test_argmax_fallbacks': int((graph_scores[list(REASONS)] < .35).all(axis=1).sum()),
                   'matches_saved_model': True, 'sha256': file_sha256(folder / 'submissions/part1_graph_candidate.csv')},
                  {'file': 'submissions/part2_candidate.csv', 'rows': len(saved_p2),
                   'null_cells': int(saved_p2.isna().sum().sum()), 'matches_saved_model': True,
                   'sha256': file_sha256(folder / 'submissions/part2_candidate.csv')}],
              'source_sha256': {p.name: file_sha256(p) for p in [root / 'midterm_refinement.py', root / 'midterm_graph_refinement.py', Path(__file__)]},
              'test_input_sha256': {part: file_sha256(dataset_path(root, part, 'test')) for part in ['part1', 'part2']},
              'new_public_scores_known': False}
    write_json(folder / 'verification.json', report)
    return report

```


### Original midterm/midterm_report.py

Original archive member: `archive/midterm_before_cleanup_2026-11-02.zip ! midterm/midterm_report.py`

SHA-256: `da35380e100e869b0e5acd7cd0a4c148e580574af24ea40341918a21f0376e7a`

```python
"""Generate the current integrated English report source."""
from pathlib import Path
from midterm_report_final import generate_report

if __name__ == '__main__':
    import argparse
    parser=argparse.ArgumentParser(description=__doc__)
    parser.add_argument('--project-root',type=Path,default=Path.cwd())
    print(generate_report(parser.parse_args().project_root))

```


### Original midterm/midterm_report_final.py

Original archive member: `archive/midterm_before_cleanup_2026-11-02.zip ! midterm/midterm_report_final.py`

SHA-256: `44ebe44e694814d9aa8625d19c38543f6d13e9578e96150ca2552b3c0bbad7f8`

```python
"""Generate the integrated CVPR report from final and historical result tables."""
from pathlib import Path
import json
import shutil
import pandas as pd
from midterm_history import historical_path
from midterm_finalization import FINAL_RUN, WORD_RUN, FIELD_RUN
from midterm_closeout import CHARACTER_RUN
from midterm_part2_improvement import BASELINE_RUN


def generate_report(root):
    root=Path(root);report=root/'report';report.mkdir(exist_ok=True)
    images=report/'images';images.mkdir(exist_ok=True)
    final=root/'part2_final'
    p1=pd.read_csv(root/'part1_final/validation_comparison.csv')
    labels=pd.read_csv(root/'part1_final/validation_reason_comparison.csv')
    graph=pd.read_csv(root/'refinement/part1_diagnosis/validation_reason_metrics.csv')
    comparison=pd.read_csv(final/'validation_comparison.csv')
    phase=pd.read_csv(final/'validation_metrics_all_candidates.csv')
    reasons=pd.read_csv(final/'reason_error_rate_comparison.csv')
    shutil.copy2(final/'figures/final_comparison.png',images/'final_comparison.png')
    names=['Supervised LR','KM + LR','GMM + LR','KM + pathways','KM + LR','KM + LR','KM + LR (retained)','KM + LR']
    rows=[]
    for row,name in zip(p1.itertuples(),names):
        per=labels.loc[labels.run_id.eq(row.run_id)].set_index('reason').loc[['Safety','Funding','Efficacy','Recruitment','Other'],'f1']
        rows.append(' & '.join([name,f'{row.C:g}',f'{row.threshold:.2f}',f'{row.f1_samples:.4f}',f'{row.f1_macro:.4f}',*[f'{v:.4f}' for v in per]])+r' \\')
    rows.append(' & '.join(['Graph soft + LR','0.1','0.35','0.8932','0.7777',*[f'{v:.4f}' for v in graph.f1_graph]])+r' \\')
    overall=[]
    for row in comparison.itertuples():
        name=('Text + metadata' if row.structured else 'Text')
        if row.run_id==FIELD_RUN:name='Separate fields'
        if row.run_id==WORD_RUN:name='Word blend'
        if row.run_id==CHARACTER_RUN:name='Character text'
        if row.run_id==FINAL_RUN:name='Word + char (final)'
        overall.append(' & '.join([name,f'{row.C:g}',f'{row.log_loss_equal_phase:.6f}',f'{row.mean_phase_f1:.4f}'])+r' \\')
    phase_rows=[]
    for run_id,name in [(BASELINE_RUN,'Baseline'),(FINAL_RUN,'Final blend')]:
        for row in phase.loc[phase.run_id.eq(run_id)].itertuples():
            phase_rows.append(' & '.join([name,row.phase,f'{row.f1:.4f}',f'{row.roc_auc:.4f}',f'{row.average_precision:.4f}',f'{row.log_loss:.4f}'])+r' \\')
    reason_rows=[]
    for row in reasons.itertuples():
        if row.reason_group=='matched_no_predicted_reason':continue
        n=int(row.n_trials_baseline)
        name='No recorded reason' if row.reason_group=='no_recorded_stopping_reason' else row.reason_group
        source='--' if row.reason_group=='no_recorded_stopping_reason' else f'{int(row.n_observed_reasons_baseline)}/{int(row.n_predicted_reasons_baseline)}'
        before=f'{int(row.n_predicted_success_baseline)}/{n} ({100*row.false_success_rate_baseline:.1f}\\%)' if n else '--'
        after=f'{int(row.n_predicted_success_final)}/{n} ({100*row.false_success_rate_final:.1f}\\%)' if n else '--'
        reason_rows.append(' & '.join([row.phase,name,str(n),source,before,after])+r' \\')
    follow_rows=[]
    for row in pd.read_csv(final/'part1_followup_summary.csv').itertuples():
        follow_rows.append(' & '.join([row.method,f'{row.OOF_sample_F1:.4f}',f'{row.validation_sample_F1:.4f}'])+r' \\')
    methods=r"""\begin{abstract}
K-Means label recovery and logistic regression give validation sample F1 0.8782 for stopping
reasons. A fixed text-model blend
improves outcome validation log loss to 0.580900 and reported public loss to 0.59696. We report
failure errors and selection limits.
\end{abstract}
\section{Methods}
\subsection{Data and evaluation}
The course data and rules define both tasks~\cite{handout}. Part 1 contains 1,171 training trials:
157 observed five-label vectors and 1,014 masked as $-100$; validation/test contain 78/312 trials.
Reasons may overlap. Observed seeds favor early trials and rare reasons; Funding has only 8/4
training/validation positives. We report sample, per-reason and macro F1, with undefined F1 set
to zero. No validation refitting is used; IDs align rows.

Part 2 contains 8,037/1,013 training/validation trial-phase rows. Phase I/II/III validation support
is 149/501/363, with 85/236/240 successes. Combined-phase copies remain in the same split.
Success is class 1. F1 uses threshold 0.5; ROC-AUC and average precision (AP) use probabilities.
Selection minimizes equal-phase mean log loss, matching 500 test rows per phase. All fitted
preprocessing uses training data only. We use scikit-learn~\cite{sklearn}, seed 42, without deep learning.

\subsection{Part 1: unsupervised label recovery}
StandardScaler fits all 150 training embedding dimensions~\cite{lectures}.
Five supervised L2 logistic regressions use observed labels
($C=1$, threshold 0.5). K-Means uses 10 clusters, k-means++, 20 starts, Lloyd updates, maximum
500 iterations, tolerance $10^{-4}$; it takes 19 iterations. GMM uses 10 diagonal components,
five K-Means initializations, maximum 500 EM iterations, tolerance $10^{-3}$, covariance
regularization $10^{-4}$; it converges in 23 iterations. Hard memberships or responsibilities
define $w_{ik}$. For observed seeds $L$,
\begin{equation}
q_{kr}=\frac{\sum_{i\in L}w_{ik}y_{ir}+2\pi_r}{\sum_{i\in L}w_{ik}+2},\quad
\pi_r=|L|^{-1}\sum_{i\in L}y_{ir}.
\end{equation}
Masked targets are $\mathbb{1}[\sum_kw_{ik}q_{kr}\geq0.5]$; seed-empty clusters use the prior.
All masked rows receive vectors and observed targets stay fixed. Gold/pseudo weights are 1/0.1.
We compare augmentation, broad pathways, $C=0.1,1,10$, then common thresholds 0.35/0.5/0.65 at
the best $C$. Pathways use binary level-0 terms with training frequency at least five and
availability flags. Logistic fits use liblinear, maximum 500 iterations, tolerance $10^{-4}$,
and no class weights. The retained model uses K-Means targets, $C=0.1$, threshold 0.35.
Final decisions retain all qualifying reasons, or select the maximum probability if empty;
ties follow Safety, Funding, Efficacy, Recruitment, Other. The fallback affects four test
and 13 masked-training rows.

\subsection{Part 2: baseline, controls, and final blend}
The baseline concatenates criteria, disease and drug text; missing fields become empty.
Each phase fits its own TF-IDF~\cite{tfidf} and LR ($C=1$). Word unigrams/bigrams use
min\_df=3, max\_features=20,000, lowercase Unicode accent normalization, sublinear TF, smoothed
IDF, L2 normalization, and no stopword removal. Optimizer limits match Part 1.

Controls compare text/metadata at $C=0.1,1,10$~\cite{lr}. Metadata uses ICD/pathway terms,
matching flags, training-fitted year imputation/scaling/clipping and missingness. Later,
separate criteria/disease/drug TF-IDF blocks use limits 20,000/5,000/5,000 and scale $1/\sqrt3$.
Their fixed 50/50 blend is combined equally with a character TF-IDF LR model (details in
Section 4.2), giving weights 0.25 word, 0.25 field, 0.50 character. Reasons and reason embeddings
are excluded. Selection uses grouped training OOF and supplied validation; integration refits nothing.
"""
    tex=r"""\documentclass[10pt,twocolumn,letterpaper]{article}
\usepackage[pagenumbers]{cvpr}
\usepackage{amsmath,amssymb,booktabs}
\usepackage[breaklinks,colorlinks,citecolor=blue,urlcolor=blue,linkcolor=blue]{hyperref}
\title{Clinical Trial Stopping Reasons and Outcome Prediction\\
\large CS5785 / ORIE5750 / ECE5414: Fall 2026 Midterm Project}
\author{PEI-CHEN CHUNG\\Cornell University\\
\small Individual project: data audit, experiments, evaluation, interpretation, and report}
\begin{document}
\maketitle
\input{methods}
\clearpage
\twocolumn[{
\section{Results}
\subsection{Stopping-reason prediction}
\begin{center}\small
\begin{tabular}{lrrrrrrrrr}
\toprule
Method & $C$ & Thr. & Sample F1 & Macro F1 & Safety & Funding & Efficacy & Recruit. & Other\\
\midrule
@P1@
\bottomrule
\end{tabular}
\captionof{table}{Part 1 validation (78 trials). Positive support is 9, 4, 14, 50, 10. Graph uses the
same final fallback policy. Its higher validation F1 did not transfer to the reported public subset.}
\end{center}
\subsection{Outcome models and selected phase metrics}
\begin{center}\small
\begin{tabular}{lrrr}
\toprule
Representation & $C$ & Equal-phase loss & Mean phase F1\\
\midrule
@OVERALL@
\bottomrule
\end{tabular}
\captionof{table}{Six original controls and subsequent word/character comparisons. All blend components
use $C=1$; final weights are fixed at 25/25/50, without a new classifier fit.}
\end{center}
\begin{center}\small
\begin{tabular}{llrrrr}
\toprule
Model & Phase & F1 & ROC-AUC & AP & Log loss\\
\midrule
@PHASE@
\bottomrule
\end{tabular}
\captionof{table}{Required validation metrics for baseline and selected outcome model. F1 threshold
stays at 0.5. All 30 phase-setting rows are included in the accompanying CSV.}
\end{center}
}]
\paragraph{Interpretation.}
At unchanged $C=1$ and threshold 0.5, both augmentation methods underperform supervised LR.
GMM hard pseudo-labels contain no Safety, Funding or Efficacy positives; K-Means contains no
Funding positives. Pathways also reduce F1. The retained model changes regularization and
threshold as well as training labels, so its small gain over the supervised control does not
isolate a benefit from unlabeled data. The word-only blend loss is 0.582270; character-only loss is 0.582757. Their fixed blend
lowers loss to 0.580900, versus baseline 0.583773. F1 improves only in Phase II versus baseline.

\clearpage
\twocolumn[{
\subsection{Failure errors by phase, stopping reason, and source}
\begin{center}\small
\begin{tabular}{llrrll}
\toprule
Phase & Reason group & $n$ & Observed / predicted & Baseline errors & Final blend errors\\
\midrule
@REASONS@
\bottomrule
\end{tabular}
\captionof{table}{True validation failures predicted successful: $k/n$ (percent), using the same
original Part 1 lookup in both columns. No recorded reason means no Part 1 match. Categories
overlap. Phase III Funding has no trials. Matched empty-reason groups have zero support in all phases
after the declared fallback and are omitted. A dash means no applicable source or no trials.}
\end{center}
}]
\paragraph{Denominators and provenance.}
Only true outcome-label-0 rows enter this analysis. For phase/reason group $g$,
\begin{equation}
\mathrm{FSR}_g=\frac{\sum_{i\in g}\mathbb{1}[p_i\geq0.5]}{|g|}.
\end{equation}
The 452 failed trial-phase rows represent 398 unique trials. Of 198 matched rows, 27 have
observed reasons and 171 have predicted reasons; 254 rows have no Part 1 match. Among predicted
rows, 131 are Part 1 training resubstitution estimates. These are not confirmed reason labels.
The three formerly empty matched rows inherit flagged argmax predictions. Trials may appear
in several reasons/phases but occur once within each phase/group.

\paragraph{Overall errors.}
The baseline calls 34/64 Phase I failures successful, 73/265 Phase II failures, and 80/123
Phase III failures. The final blend changes these to 33/64, 65/265 and 80/123. Compared
with the prior word blend (33/64, 70/265, 79/123), it removes five Phase II false positives
but adds one in Phase III. Phase I true-success false negatives rise from baseline 9 to 12;
Phase III rises from 21 to 23. Probability loss, thresholded F1 and failure-group errors
therefore need separate interpretation.

\paragraph{Interpretation limits.}
Reason-specific changes are conditional on observed or estimated Part 1 categories. Small
Funding and Other groups cannot support strong conclusions. The CSV outputs also separate
observed, predicted and unmatched sources, retain resubstitution flags, and give descriptive
Wilson intervals. These intervals exclude reason-label uncertainty. We make no causal claims.

\clearpage
\twocolumn[{
\section{Follow-up Studies and Public Generalization}
\begin{center}
\includegraphics[width=.95\textwidth]{images/final_comparison.png}
\captionof{figure}{Left: final character blending lowers log loss in every phase versus the prior
word blend. Right: graph Part 1 predictions shift toward Other on test. These are predicted
rates, not true test prevalence or error rates. Source: notebook 15.}
\end{center}
\begin{center}\small
\begin{tabular}{lrr}
\toprule
Part 1 follow-up & Pooled gold OOF F1 & Validation F1\\
\midrule
@FOLLOWUPS@
\bottomrule
\end{tabular}
\captionof{table}{Later Part 1 studies reuse 157 gold rows and the same four outer folds. OOF
selects settings before validation; it is not independent after selection. The retained model wins validation.}
\end{center}
}]
\subsection{Graph and pseudo-label comparisons}
Initial follow-ups compare supervised, hard and soft K-Means targets at $C=0.1/1$ with
10/30 clusters where applicable. Soft targets do not win. A six-setting local study tests
weighted kNN, direct LabelSpreading, and graph soft targets plus LR, each with 5/15 neighbors.
LabelSpreading~\cite{graph} uses $\alpha=0.2$, 200 iterations and tolerance $10^{-4}$;
disconnected mass uses the observed prior. Selected graph+LR uses 15 neighbors, $C=0.1$,
and gold/pseudo weights 1/0.1. Mean fold F1 is 0.89447 versus 0.89006 for hard K-Means.
Held-out rows are excluded before scaling, clustering/graph fitting and label mapping.

Graph changes 16 validation sets: seven improve, seven worsen, two tie. One correct argmax
fallback supplies 85.7\% of its net validation gain. Other recall rises 0.40 to 0.70 but
precision falls 0.80 to 0.5385. Test Other positives rise 57 to 84; hidden labels are needed
to identify actual errors. Pseudo weights 0/0.03/0.1 give pooled OOF F1
0.81975/0.87516/0.88960; retain 0.1.

\subsection{Classifier geometry and decision policies}
Gold-only SVMs~\cite{svm} compare linear $C=1$ and RBF $C=1/10$, gamma=scale, no class weights,
tolerance $10^{-3}$, maximum 10,000 iterations. Three inner folds generate gold-only margins
for sigmoid LR calibration ($C=1$, lbfgs, unpenalized intercept, tolerance $10^{-6}$,
maximum 1,000 iterations). Inner-held rows are excluded from scaling/SVM fitting; outer-held
rows never enter calibration. OOF selects RBF $C=10$, but it fails validation.

A row-L2 study~\cite{normalizer} keeps K-Means targets fixed and compares common LR grids
$C=0.1,1,10,100$. OOF selects L2 $C=10$; its small gain fails validation. Four final policies
compare retained-LR thresholds 0.30/0.35/0.40 and a fixed 75/25 LR/linear-SVM blend at 0.35.
The blend wins OOF, but two validation rows worsen and none improve. No alternative policy
is substituted after validation. These sequential studies preserve the original final Part 1 model.

\clearpage
\section{Discussion, Limitations, and Reproducibility}
\subsection{Observed public results and final choices}
\begin{center}\small
\begin{tabular}{llr}
\toprule
Task & Model & Public score\\
\midrule
Part 1, F1 & K-Means + LR & 0.85405\\
 & Graph candidate & 0.84081\\
Part 2, log loss & Text LR & 0.60149\\
 & Word blend & 0.59962\\
 & Word + character & 0.59696\\
\bottomrule
\end{tabular}
\captionof{table}{Scores transcribed from user-provided submission screenshots, not a live
leaderboard query. Higher F1 and lower log loss are better. Private scores are unknown.}
\end{center}
We retain the original Part 1 pipeline after the graph public decline. This rollback uses
aggregate public feedback and is disclosed; it is not a new independent validation result.
The final outcome blend was selected by grouped training OOF and confirmed on validation
before its public submission. OOF loss moves from 0.597089 (word blend) to 0.596486; validation
moves from 0.582270 to 0.580900. The reported public loss also decreases. Public scores did not
tune its fixed 25/25/50 component weights.
The integration performs no new fitting or upload and reproduces the exact successful
candidate CSV for Part 2.

\subsection{What did and did not help}
Cluster geometry need not align with five overlapping reasons. Hard pseudo-targets suppress
rare positives, while softer alternatives do not reliably fix the problem. Broad pathways
can be redundant, sparsely matched, or poorly supported by the small labeled set. Exact drug
matching is incomplete, so missing matches do not mean no relevant biological mechanism.
These are plausible explanations, not isolated effects.

The character model uses char\_wb 3--5 grams, min\_df=3, at most 30,000 features per phase,
LR $C=1$, and the baseline TF/IDF normalization. Five saved folds group all copies of each
trial, with vocabularies and classifiers fitted inside each training fold. The fixed blend
is selected by pooled OOF equal-phase loss and confirmed on supplied validation.

For outcomes, uniform stronger regularization and the joint metadata block fail to improve
loss. Text at $C=10$ helps some thresholded errors but harms Phase II probabilities. Separating
text fields alone also fails. Character text alone is slightly worse than the word blend, but their fixed mixture
has better loss in every phase versus the word blend. This is consistent with complementary
errors; improved calibration is not separately established. Phase I F1 is unchanged and Phase III
F1 decreases versus the word blend, so the final model is not better in every metric.

\subsection{Selection uncertainty and distribution differences}
The 78-trial reason validation set is reused for feature, regularization, threshold and
follow-up selection. Studies were expanded sequentially after earlier results. Training
CV helps check stability but does not create a fresh holdout. The Part 1 descriptive paired
bootstrap interval for final-policy F1 difference is [-0.0385, 0.0705]. The Part 2 interval
for final-minus-word equal-phase loss is [-0.00522, 0.00249], resampling whole trials so combined
phases stay together. Both include zero and exclude selection and temporal uncertainty.

Part 1 validation years are 1999--2009 and test years 2010--2016. Observed Part 2 test years
are 2014--2021, versus training through 2013. Biased reason seeds, later cohorts and possible
outcome label noise limit generalization. Differences in embedding seed coverage and predicted
reason rates do not identify a causal explanation for the public decline. Public membership
and hidden labels were not recovered, and no external fully labeled data were used.

\subsection{Reproduction and delivery}
The \texttt{midterm} folder includes fifteen executed notebooks, original course data, helper
modules, figures and their source, fitted artifacts, submissions and report sources. The 14
original registered runs remain immutable. Two subsequent registered artifacts package the frozen word and word/character blends without
fitting (16 registered runs in total). Full-training follow-ups and OOF evidence remain under
\texttt{refinement}. Exact-digest versioned source/output archives support earlier audits.
The former generated ZIP is excluded from history checks to avoid recursively nested archives.

Run \texttt{python reproduce.py} in the project folder. It executes notebooks in fresh
processes; later notebooks reuse saved components with fitting blocked by default. Initial
baseline/augmentation notebooks can recompute their original controls while preserving
registered runs. Data, model, input-scope, submission and package hashes are checked.
Part 1 submits 312 nonempty reason strings; Part 2 submits 1,500 success probabilities,
500 per phase, with exact sample ID order. Reasons are used solely for outcome error analysis.

\paragraph{Authorship and assistance.}
PEI-CHEN CHUNG completed the project individually. OpenAI Codex assisted with implementation,
checks and drafting. The author is responsible for the submitted work.

\clearpage
{\small
\begin{thebibliography}{9}
\bibitem{handout} CS5785 / ORIE5750 / ECE5414 teaching staff.
\emph{Midterm Project 2026}. Cornell University, Fall 2026. Course assignment and datasets.
\bibitem{lectures} CS5785 / ORIE5750 / ECE5414 teaching staff.
\emph{L6: MLE and Classification Evaluation; L7: Classification Evaluation, Choice of Divergence,
Regularization; L8: Regularization and Model Selection; L11: K-Means, Unsupervised GMM; L12: EM
Algorithm and Density Estimation}. Cornell University, Fall 2026. Supplied course slides.
\bibitem{sklearn} F. Pedregosa et al. Scikit-learn: Machine Learning in Python.
\emph{Journal of Machine Learning Research}, 12:2825--2830, 2011.
\url{https://jmlr.org/papers/v12/pedregosa11a.html}.
\bibitem{tfidf} Scikit-learn developers. \emph{Text feature extraction}.
\url{https://scikit-learn.org/stable/modules/feature_extraction.html}.
\bibitem{lr} Scikit-learn developers. \emph{Logistic regression}.
\url{https://scikit-learn.org/stable/modules/linear_model.html}.
\bibitem{graph} Scikit-learn developers. \emph{LabelSpreading}.
\url{https://scikit-learn.org/stable/modules/generated/sklearn.semi_supervised.LabelSpreading.html}.
\bibitem{svm} Scikit-learn developers. \emph{Support vector machines}.
\url{https://scikit-learn.org/stable/modules/svm.html}.
\bibitem{normalizer} Scikit-learn developers. \emph{Normalizer}.
\url{https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.Normalizer.html}.
\bibitem{template} CVPR organizers. \emph{CVPR 2026 Author Kit}, commit
\texttt{12909ae437f6dbc7435069cfdb4ca44c18e6a02f}. Template linked from the course handout.
\url{https://github.com/cvpr-org/author-kit}.
\end{thebibliography}
}
\end{document}
"""
    for tag,values in [('@P1@',rows),('@OVERALL@',overall),('@PHASE@',phase_rows),('@REASONS@',reason_rows),('@FOLLOWUPS@',follow_rows)]:
        tex=tex.replace(tag,'\n'.join(values))
    (report/'methods.tex').write_text(methods)
    (report/'main.tex').write_text(tex)
    provenance=json.loads(historical_path(root,'report/source_provenance.json').read_text())
    provenance.update({'body_page_limit':5,'final_selection':'final_selection.json',
        'results_sources':['part1_final/validation_comparison.csv','part1_final/validation_reason_comparison.csv',
            'part2_final/validation_comparison.csv','part2_final/validation_metrics_all_candidates.csv',
            'part2_final/reason_error_rate_comparison.csv','refinement/part1_diagnosis/summary.json',
            'refinement/part1_diagnosis/observed_public_scores.csv','part2_final/part1_followup_summary.csv',
            'refinement/oof_weight_char/part2/oof_comparison.csv','refinement/rbf_svm/summary.json',
            'refinement/l2_geometry/summary.json','refinement/threshold_blend/summary.json']})
    (report/'source_provenance.json').write_text(json.dumps(provenance,indent=2)+'\n')
    return report/'main.tex'

```


### Original midterm/midterm_threshold_blend.py

Original archive member: `archive/midterm_before_cleanup_2026-11-02.zip ! midterm/midterm_threshold_blend.py`

SHA-256: `9625a292b6e0ee5803a7c1f2921ec7e1e907f8a54b02867b809a749a6db1972a`

```python
"""Four fixed Part 1 decision policies using frozen OOF scores and models.

No classifier, scaler, clusterer, or calibrator is fitted in this study.
Select one policy on training OOF, then evaluate it once on validation.
"""
from __future__ import annotations

import argparse
import html
import json
from pathlib import Path
import platform

import joblib
import numpy as np
import pandas as pd
import sklearn

from midterm_augmentation import observed_label_mask
from midterm_evaluation import REASONS, dataset_path, evaluate_part1, file_sha256, part1_decisions
from midterm_next_round import (
    cache_complete, finish_cache, paired_diagnostics, read_json,
    reason_error_counts, verify_hashes, verify_oof, write_json,
)
from midterm_part1_finalization import build_submission, predict_bundle
from midterm_part2_improvement import validate_submission
from midterm_rbf_svm import predict_svm_study

STUDY = "refinement/threshold_blend"
CONTROL = "current_t035"
POLICIES = {
    CONTROL: {"current_weight": 1.0, "linear_svm_weight": 0.0, "threshold": .35},
    "current_t030": {"current_weight": 1.0, "linear_svm_weight": 0.0, "threshold": .30},
    "current_t040": {"current_weight": 1.0, "linear_svm_weight": 0.0, "threshold": .40},
    "blend_075_025_t035": {"current_weight": .75, "linear_svm_weight": .25, "threshold": .35},
}


def aligned_scores(reference, scores):
    """Align five independent reason probabilities by unique row ID."""
    if reference.ID.isna().any() or not reference.ID.is_unique:
        raise ValueError("Reference IDs must be unique and nonmissing.")
    if scores.ID.isna().any() or not scores.ID.is_unique or set(scores.ID) != set(reference.ID):
        raise ValueError("Probability IDs must match the reference exactly once.")
    values = scores.set_index("ID").loc[reference.ID, list(REASONS)].to_numpy(dtype=float)
    if not np.isfinite(values).all() or ((values < 0) | (values > 1)).any():
        raise ValueError("Reason probabilities must be finite and in [0, 1].")
    return values


def policy_scores(reference, current, linear_svm, policy):
    weights = np.array([policy["current_weight"], policy["linear_svm_weight"]], dtype=float)
    if not np.isfinite(weights).all() or (weights < 0).any() or not np.isclose(weights.sum(), 1, atol=1e-12, rtol=0):
        raise ValueError("Policy weights must be finite, nonnegative, and sum to one.")
    if not np.isfinite(policy["threshold"]) or not 0 <= policy["threshold"] <= 1:
        raise ValueError("The common threshold must be in [0, 1].")
    first = aligned_scores(reference, current)
    if weights[1]:
        if linear_svm is None:
            raise ValueError("A positive SVM weight requires calibrated SVM probabilities.")
        second = aligned_scores(reference, linear_svm)
        values = weights[0] * first + weights[1] * second
    else:
        values = first.copy()
    # These are independent labels: never divide by the sum across reasons.
    result = pd.DataFrame(values, columns=REASONS)
    result.insert(0, "ID", reference.ID.to_numpy())
    return result


def select_policy(evaluations):
    """Predeclared order preserves the current setting on an exact tie."""
    if set(evaluations) != set(POLICIES):
        raise ValueError("Evaluate exactly the four predeclared policies before selection.")
    values = np.array([evaluations[name].summary["f1_samples"] for name in POLICIES])
    if not np.isfinite(values).all():
        raise ValueError("Policy selection requires finite OOF F1 scores.")
    return list(POLICIES)[int(np.argmax(values))]


def scores_from_saved(frame):
    return frame[["ID", *[f"prob_{r}" for r in REASONS]]].rename(columns={f"prob_{r}": r for r in REASONS})


def predict_policy_probabilities(bundle, frame):
    """Apply the frozen model(s), then the declared probability weighting."""
    if bundle.get("kind") != "frozen_part1_decision_policy":
        raise ValueError("Unexpected frozen-policy bundle.")
    current = pd.DataFrame(predict_bundle(bundle["current_model"], frame), columns=REASONS)
    current.insert(0, "ID", frame.ID.to_numpy())
    linear = None
    if bundle["policy"]["linear_svm_weight"]:
        linear = predict_svm_study(bundle["linear_svm_model"], frame)
    return policy_scores(frame, current, linear, bundle["policy"])


def policy_decisions(bundle, probabilities):
    if bundle.get("kind") != "frozen_part1_decision_policy":
        raise ValueError("Unexpected frozen-policy bundle.")
    return part1_decisions(probabilities[list(REASONS)], bundle["policy"]["threshold"], empty_set_policy="argmax")


def protected_hashes(root):
    paths = [root / "final_selection.json"]
    for folder in ["part1_final", "part2_final", "experiments", "submissions", "deliverables", "report", "refinement"]:
        paths.extend(p for p in (root / folder).rglob("*") if p.is_file()
                     and not p.is_relative_to(root / STUDY) and "__pycache__" not in p.parts)
    paths.extend(p for p in root.glob("[0-9][0-9]_*.ipynb") if int(p.name[:2]) < 14)
    return {p.relative_to(root).as_posix(): file_sha256(p) for p in sorted(set(paths))}


def run_oof(root, folder):
    train = pd.read_csv(dataset_path(root, "part1", "train"))
    gold = train.loc[observed_label_mask(train[list(REASONS)].to_numpy())].reset_index(drop=True)
    assignment = pd.read_csv(root / "refinement/oof_weight_char/part1/fold_assignments.csv")
    assignment = assignment.set_index("ID").loc[gold.ID].reset_index()
    current = verify_oof(gold, pd.read_csv(root / "refinement/oof_weight_char/part1/oof_weight_010.csv"), assignment, part=1)
    linear = verify_oof(gold, pd.read_csv(root / "refinement/rbf_svm/oof_gold_linear_svm_c1.csv"), assignment, part=1)
    assignment.to_csv(folder / "fold_assignments.csv", index=False)
    provenance = []
    for fold in range(4):
        expected_held = set(assignment.loc[assignment.fold.eq(fold), "ID"])
        first = read_json(root / f"refinement/oof_weight_char/part1/fold_cache/fold_{fold}/fit_scope.json")
        second = read_json(root / f"refinement/rbf_svm/fold_cache/fold_{fold}/fit_scope.json")
        if set(first["held_out_row_ids"]) != expected_held or set(second["outer"]["held_out_row_ids"]) != expected_held:
            raise ValueError("Component models used different OOF holdouts.")
        for fit_ids in [first["fit_row_ids"], first["label_seed_ids"],
                        second["nested"]["scaler_fit_ids"], second["nested"]["calibrator_fit_ids"]]:
            if expected_held & set(fit_ids):
                raise ValueError("An OOF row entered a component model fit.")
        for inner in second["nested"]["inner_splits"]:
            if set(inner["calibration_hold_ids"]) & set(inner["scaler_fit_ids"]):
                raise ValueError("An inner calibration holdout entered scaler fitting.")
            if set(inner["calibration_hold_ids"]) & set(inner["classifier_fit_ids"]):
                raise ValueError("An inner calibration holdout entered SVM fitting.")
        provenance.append({"fold": fold, "identical_component_holdouts": True,
                           "outer_holdout_fit_overlap": 0, "held_out_rows": len(expected_held)})
    write_json(folder / "component_oof_lineage.json", provenance)
    evaluations, rows, reasons, fold_rows = {}, [], [], []
    for name, policy in POLICIES.items():
        probabilities = policy_scores(gold, scores_from_saved(current), scores_from_saved(linear), policy)
        result = evaluate_part1(gold, probabilities, policy["threshold"], empty_set_policy="argmax")
        evaluations[name] = result
        result.predictions.assign(fold=assignment.fold.to_numpy()).to_csv(folder / f"oof_{name}.csv", index=False)
        rows.append({"policy": name, **policy, **result.summary})
        reasons.append(reason_error_counts(result).assign(policy=name))
        for fold in range(4):
            mask = assignment.fold.eq(fold).to_numpy()
            sub = evaluate_part1(gold.loc[mask], probabilities.loc[mask], policy["threshold"], empty_set_policy="argmax")
            fold_rows.append({"policy": name, "fold": fold, **sub.summary})
    comparison = pd.DataFrame(rows)
    comparison["delta_f1_vs_current"] = comparison.f1_samples - evaluations[CONTROL].summary["f1_samples"]
    comparison.to_csv(folder / "oof_comparison.csv", index=False)
    pd.concat(reasons).to_csv(folder / "oof_by_reason.csv", index=False)
    pd.DataFrame(fold_rows).to_csv(folder / "fold_metrics.csv", index=False)
    selected = select_policy(evaluations)
    lock = {"selected_policy": selected, "parameters": POLICIES[selected],
            "selection_metric": "sample F1 across all 157 training OOF rows",
            "tie_order": list(POLICIES), "validation_labels_used": False,
            "test_predictions_used": False, "new_public_feedback_used_for_selection": False,
            "classifiers_or_calibrators_fitted": False}
    write_json(folder / "selection_locked_before_validation.json", lock)
    return lock, evaluations


def validation_and_candidate(root, folder, lock, oof):
    selected = lock["selected_policy"]
    current_model = joblib.load(root / "part1_final/selected_model.joblib")
    linear_model = (joblib.load(root / "refinement/rbf_svm/gold_linear_svm_c1.joblib")
                    if POLICIES[selected]["linear_svm_weight"] else None)
    bundle = {"kind": "frozen_part1_decision_policy", "policy_name": selected,
              "policy": POLICIES[selected].copy(), "current_model": current_model,
              "linear_svm_model": linear_model, "empty_set_policy": "argmax",
              "refit_performed": False,
              "current_component_sha256": file_sha256(root / "part1_final/selected_model.joblib"),
              "linear_component_sha256": file_sha256(root / "refinement/rbf_svm/gold_linear_svm_c1.joblib") if linear_model else None}
    joblib.dump(bundle, folder / "selected_policy.joblib", compress=3)
    # Only now read official validation, after the four-policy OOF selection.
    validation = pd.read_csv(dataset_path(root, "part1", "val"))
    current_saved = scores_from_saved(pd.read_csv(root / "part1_final/selected_validation/validation_predictions.csv"))
    model_current = pd.DataFrame(predict_bundle(current_model, validation), columns=REASONS)
    model_current.insert(0, "ID", validation.ID.to_numpy())
    current_error = float(np.abs(aligned_scores(validation, model_current) - aligned_scores(validation, current_saved)).max())
    if current_error > 1e-12:
        raise ValueError("The current frozen model does not reproduce its saved validation scores.")
    linear_error = None
    if linear_model is not None:
        linear_actual = predict_svm_study(linear_model, validation)
        linear_saved = scores_from_saved(pd.read_csv(root / "refinement/rbf_svm/validation/gold_linear_svm_c1/validation_predictions.csv"))
        linear_error = float(np.abs(aligned_scores(validation, linear_actual) - aligned_scores(validation, linear_saved)).max())
        if linear_error > 1e-12:
            raise ValueError("The calibrated SVM does not reproduce its saved validation scores.")
    control = evaluate_part1(validation, current_saved, .35, empty_set_policy="argmax")
    candidate = evaluate_part1(validation, predict_policy_probabilities(bundle, validation), POLICIES[selected]["threshold"], empty_set_policy="argmax")
    results = {CONTROL: control}
    if selected != CONTROL:
        results[selected] = candidate
    for name, result in results.items():
        result.save(folder / "validation" / name)
    pd.DataFrame([{"policy": name, **POLICIES[name], **r.summary} for name, r in results.items()]).to_csv(folder / "validation_comparison.csv", index=False)
    pd.concat([reason_error_counts(r).assign(policy=name) for name, r in results.items()]).to_csv(folder / "validation_by_reason.csv", index=False)
    oof_gain = oof[selected].summary["f1_samples"] - oof[CONTROL].summary["f1_samples"]
    val_gain = candidate.summary["f1_samples"] - control.summary["f1_samples"]
    passes = bool(selected != CONTROL and oof_gain > 1e-12 and val_gain > 1e-12)
    gate = {"selected_policy": selected, "parameters": POLICIES[selected],
            "oof_f1_gain": oof_gain, "validation_f1_gain": val_gain,
            "passes_candidate_gate": passes, "current_model_max_probability_difference": current_error,
            "linear_svm_max_probability_difference": linear_error,
            "final_model_replaced": False, "public_or_private_score_known": False,
            "nonselected_policy_validation_scores_evaluated": False}
    write_json(folder / "candidate_gate.json", gate)
    write_json(folder / "paired_oof_diagnostics.json", paired_diagnostics(oof[CONTROL], oof[selected], part=1))
    write_json(folder / "paired_validation_diagnostics.json", paired_diagnostics(control, candidate, part=1))
    path = None
    if passes:
        test = pd.read_csv(dataset_path(root, "part1", "test"))
        scores = predict_policy_probabilities(bundle, test)
        decisions = pd.DataFrame(policy_decisions(bundle, scores), columns=REASONS)
        decisions.insert(0, "ID", test.ID.to_numpy())
        sample = pd.read_csv(dataset_path(root, "part1", "test").parent / "part1_sample_submission.csv")
        output = folder / "submissions"; output.mkdir(exist_ok=True)
        scores.to_csv(output / "part1_candidate_probabilities.csv", index=False)
        path = output / "part1_candidate.csv"
        build_submission(sample, decisions).to_csv(path, index=False)
        validate_submission(pd.read_csv(path), sample, 1)
        path = path.relative_to(root).as_posix()
    return gate, path


def report(folder, summary):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), constrained_layout=True)
    for ax, split, title in zip(axes, ["oof", "validation"], ["Training OOF: four fixed policies", "Validation: current and locked OOF choice"]):
        table = pd.read_csv(folder / f"{split}_comparison.csv")
        colors = ["#007f8b" if p == summary["selected_policy"] else "#a4b9c7" for p in table.policy]
        bars = ax.barh(table.policy.str.replace("_", " "), table.f1_samples, color=colors)
        ax.bar_label(bars, fmt="%.4f", padding=4, fontsize=9)
        ax.set_xlim(0, 1.08); ax.invert_yaxis(); ax.set_title(title, fontsize=11)
        ax.spines[["top", "right"]].set_visible(False)
    fig.savefig(folder / "comparison.png", dpi=180); plt.close(fig)
    blocks = []
    for name in ["oof_comparison", "oof_by_reason", "validation_comparison", "validation_by_reason"]:
        blocks.append("<h2>" + html.escape(name.replace("_", " ").title()) + "</h2>" +
                      pd.read_csv(folder / f"{name}.csv").to_html(index=False, float_format=lambda v: f"{v:.6f}"))
    decision = ("The locked OOF choice improves both OOF and validation sample F1. A separate candidate CSV is available."
                if summary["passes_candidate_gate"] else
                "The locked choice does not improve both OOF and validation. Retain the existing final model; no candidate CSV is generated.")
    page = """<!doctype html><html lang="en"><meta charset="utf-8"><title>Part 1 Threshold and Blend Study</title>
<style>body{font:16px system-ui;margin:36px;max-width:1450px;color:#203047}table{border-collapse:collapse;font-size:13px}td,th{padding:7px;border:1px solid #cbd5df}th{background:#edf2f6}pre{background:#f3f5f7;padding:16px;white-space:pre-wrap}img{max-width:100%}</style>
<h1>Part 1: Four Frozen Threshold and Blend Policies</h1>
<p>The three single-model policies use the retained K-Means + LR probabilities with common thresholds 0.35, 0.30, and 0.40. The fixed blend uses 75% retained LR and 25% calibrated gold-label linear SVM, threshold 0.35. Five independent reason probabilities are averaged by reason, without normalizing across reasons. All policies apply the argmax fallback.</p>
<p>No model, scaler, clusterer, or calibrator is refitted. Component OOF probabilities use identical four-fold gold holdouts and their original fit boundaries. One policy is selected by pooled OOF sample F1; the declared order preserves the current policy on a tie. Only this choice and the current control are evaluated on official validation. An unselected policy is not substituted after seeing validation.</p>
<p>Training OOF and validation have been inspected in previous work. Selection scores and paired intervals are exploratory, not independent evidence of generalization. Gold labels are nonrandom, Funding support is small, and prior calibration does not guarantee probabilities remain calibrated in later cohorts. The fixed blend is a hypothesis, not an assumed improvement.</p>"""
    page += "<h2>Decision</h2><p>" + decision + "</p><pre>" + html.escape(json.dumps(summary, indent=2)) + "</pre>"
    page += '<img src="comparison.png" alt="Four-policy OOF and validation comparison">' + "".join(blocks)
    page += "<h2>Paired validation difference</h2><pre>" + html.escape(json.dumps(read_json(folder / "paired_validation_diagnostics.json"), indent=2)) + "</pre>"
    page += "<p>Existing final artifacts, Part 2, previous notebooks and studies, report PDF, and delivery ZIP are unchanged. No submission is uploaded.</p></html>"
    (folder / "study_report.html").write_text(page)


def run_study(root):
    root = Path(root).resolve(); folder = root / STUDY
    if cache_complete(folder):
        verify_hashes(root, read_json(folder / "protected_artifacts.json"))
        verify_hashes(root, read_json(folder / "input_and_source_hashes.json"))
        return read_json(folder / "summary.json")
    for previous in ["refinement/oof_weight_char", "refinement/rbf_svm"]:
        if not cache_complete(root / previous):
            raise RuntimeError("Restore the frozen component study before comparing decision policies.")
    folder.mkdir(parents=True, exist_ok=True)
    plan = {"study": STUDY, "policies": POLICIES, "tie_order": list(POLICIES),
            "selection_metric": "sample F1 across the original 157 gold OOF rows",
            "validation_policy": "Evaluate retained current control and one locked OOF winner only",
            "candidate_gate": "New selected policy must strictly improve both OOF and validation F1",
            "empty_set_policy": "argmax", "component_refitting": False, "calibration_refitting": False,
            "per_reason_threshold_search": False, "blend_weight_search": False,
            "additional_policy_substitution_after_validation": False,
            "earlier_validation_and_public_feedback_seen": True,
            "new_public_feedback_used_for_selection": False,
            "python": platform.python_version(), "sklearn": sklearn.__version__}
    path = folder / "experiment_plan.json"
    if path.exists() and read_json(path) != plan:
        raise ValueError("The four-policy experiment plan changed.")
    write_json(path, plan)
    files = [root / p for p in ["midterm_threshold_blend.py", "midterm_next_round.py", "midterm_evaluation.py",
             "midterm_part1_finalization.py", "midterm_augmentation.py", "midterm_rbf_svm.py",
             "part1_final/selected_model.joblib", "refinement/rbf_svm/gold_linear_svm_c1.joblib",
             "refinement/oof_weight_char/part1/fold_assignments.csv",
             "refinement/oof_weight_char/part1/oof_weight_010.csv", "refinement/rbf_svm/oof_gold_linear_svm_c1.csv"]]
    files += [dataset_path(root, "part1", split) for split in ["train", "val", "test"]]
    hashes = {p.relative_to(root).as_posix(): file_sha256(p) for p in files}
    if (folder / "input_and_source_hashes.json").exists():
        if read_json(folder / "input_and_source_hashes.json") != hashes:
            raise ValueError("A frozen input or source changed during the comparison.")
        verify_hashes(root, read_json(folder / "protected_artifacts.json"))
    else:
        write_json(folder / "input_and_source_hashes.json", hashes)
        write_json(folder / "protected_artifacts.json", protected_hashes(root))
    lock, oof = run_oof(root, folder)
    gate, candidate_path = validation_and_candidate(root, folder, lock, oof)
    summary = {**gate, "candidate_submission": candidate_path, "gold_oof_rows": 157,
               "policies_evaluated_on_OOF": 4, "new_fits_performed": 0,
               "all_original_artifacts_unchanged": True, "uploads_performed": False,
               "report_pdf_and_delivery_zip_updated": False}
    report(folder, summary)
    verify_hashes(root, read_json(folder / "protected_artifacts.json")); verify_hashes(root, hashes)
    write_json(folder / "summary.json", summary); finish_cache(folder)
    return summary


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--project-root", type=Path, default=Path(__file__).resolve().parent)
    args = parser.parse_args()
    print(json.dumps(run_study(args.project_root), indent=2))


if __name__ == "__main__":
    main()

```


### Original midterm/reproduce_part1_diagnosis.py

Original archive member: `archive/midterm_before_cleanup_2026-11-02.zip ! midterm/reproduce_part1_diagnosis.py`

SHA-256: `486cf3c65700bcba035efde8ad49d4442550cb22db4c24fa72f329fc220f00c4`

```python
"""Execute notebook 09 while blocking all model and preprocessing fitting."""
from contextlib import ExitStack
from pathlib import Path
from unittest.mock import patch
import os
from reproduce import worker


def main():
    os.environ.setdefault('LOKY_MAX_CPU_COUNT', '1')
    from sklearn.cluster import KMeans
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.linear_model import LogisticRegression
    from sklearn.neighbors import KNeighborsClassifier
    from sklearn.preprocessing import StandardScaler
    from sklearn.semi_supervised import LabelSpreading
    methods = [(cls, 'fit') for cls in [KMeans, LogisticRegression, KNeighborsClassifier, StandardScaler, LabelSpreading]]
    methods += [(TfidfVectorizer, 'fit'), (TfidfVectorizer, 'fit_transform')]
    with ExitStack() as stack:
        for cls, method in methods:
            stack.enter_context(patch.object(cls, method, side_effect=RuntimeError('Fitting is disabled during prediction diagnosis.')))
        worker(Path(__file__).resolve().parent, '09_part1_prediction_diagnosis.ipynb', allow_new_training=True)
    print('Diagnosis completed with fitting blocked and historical evidence verified.')


if __name__ == '__main__':
    main()

```


### Original midterm/reproduce_refinement.py

Original archive member: `archive/midterm_before_cleanup_2026-11-02.zip ! midterm/reproduce_refinement.py`

SHA-256: `8373b7ead3d618826e8478abe0763b1093012e35e63d06d68e61c1d71db5bf27`

```python
"""Execute notebook 08, reusing verified saved studies by default."""
from contextlib import ExitStack
from pathlib import Path
from unittest.mock import patch
import argparse
import os
from reproduce import worker


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument('--allow-training', action='store_true', help='Allow fitting if a saved study is absent.')
    args = parser.parse_args()
    os.environ.setdefault('LOKY_MAX_CPU_COUNT', '1')
    root = Path(__file__).resolve().parent
    with ExitStack() as stack:
        if not args.allow_training:
            from sklearn.cluster import KMeans
            from sklearn.feature_extraction.text import TfidfVectorizer
            from sklearn.linear_model import LogisticRegression
            from sklearn.neighbors import KNeighborsClassifier
            from sklearn.preprocessing import StandardScaler
            from sklearn.semi_supervised import LabelSpreading
            methods = [(c, 'fit') for c in [KMeans, LogisticRegression, KNeighborsClassifier, StandardScaler, LabelSpreading]]
            methods += [(TfidfVectorizer, 'fit'), (TfidfVectorizer, 'fit_transform')]
            for cls, method in methods:
                stack.enter_context(patch.object(cls, method, side_effect=RuntimeError(
                    'Fitting is disabled. Restore the saved refinement studies or use --allow-training.')))
        worker(root, '08_targeted_refinement.ipynb', allow_new_training=True)
    print('Refinement notebook completed; historical snapshots verified.')


if __name__ == '__main__':
    main()

```


## Core models and report source appendix

Full Python source for the core experiments, saved-model reproduction, and figure/table generation follows. These review copies are checked against the executable files; model training remains opt-in.


In [6]:
import runpy
source_check = runpy.run_path(str(ROOT/'tools/release/sync_source_appendix.py'))
display(source_check['check'](ROOT))


{'status': 'PASS',
 'core_and_report_source_files': 15,
 'historical_source': 'Included separately above, with migration hashes.',
 'training_performed': False}

### report/scripts/generate_tables.py

SHA-256: `68e1cc96dbec1ddeea957089dcd3b75943a1c42b5b3dc6283f03795a57e5df14`

```python
"""Render report tables from saved, audited metrics; no fitting or inference."""
from pathlib import Path
import json

ROOT = Path(__file__).resolve().parents[1]
METRICS = ROOT / "analysis/report_metrics.json"

def main():
    data = json.loads(METRICS.read_text())
    out = ROOT / "tables"
    out.mkdir(exist_ok=True)
    names = {"supervised_lr": "Supervised LR", "initial_kmeans_lr": "Initial K-Means + LR",
             "retained_kmeans_lr": "Retained K-Means + LR", "fixed_ridge_157_gold": "Fixed ridge recipe (157 gold)"}
    reasons = ["Safety", "Funding", "Efficacy", "Recruitment", "Other"]
    row_end = r" \\" + "\n"
    tab = r"\begin{tabular}{lrrrrrrr}" + "\n" + r"\toprule" + "\n"
    tab += "Model & Sample F1 & Macro F1 & Safety & Funding & Efficacy & Recruit. & Other" + row_end
    tab += r"\midrule" + "\n"
    for row in data["part1_validation"]:
        per = {r["reason"]: r["f1"] for r in data["part1_validation_by_reason"] if r["model"] == row["model"]}
        values = [row["f1_samples"], row["f1_macro"], *[per[r] for r in reasons]]
        tab += names[row["model"]] + " & " + " & ".join(f"{v:.4f}" for v in values) + row_end
    tab += r"\bottomrule" + "\n" + r"\end{tabular}" + "\n"
    (out / "part1_validation.tex").write_text(tab)

    tab = r"\begin{tabular}{llrrrr}" + "\n" + r"\toprule" + "\n"
    tab += "Model & Phase & F1 & ROC-AUC & AP & Log loss" + row_end + r"\midrule" + "\n"
    for row in data["part2_validation_by_phase"]:
        name = "Baseline" if row["model"] == "baseline" else "Current phase-shared blend"
        values = [row[k] for k in ("f1", "roc_auc", "average_precision", "log_loss")]
        tab += name + " & " + row["phase"] + " & " + " & ".join(f"{v:.4f}" for v in values) + row_end
    tab += r"\bottomrule" + "\n" + r"\end{tabular}" + "\n"
    (out / "part2_phase.tex").write_text(tab)

    tab = r"\begin{tabular}{llrrll}" + "\n" + r"\toprule" + "\n"
    tab += "Phase & Reason group & $n$ & Obs./Pred. & Baseline errors & Current blend errors" + row_end + r"\midrule" + "\n"
    rows = [r for r in data["part2_failure_reason_comparison"] if r["reason_group"] != "matched_no_predicted_reason"]
    previous = None
    for row in rows:
        if previous is not None and row["phase"] != previous: tab += r"\midrule" + "\n"
        previous = row["phase"]
        n = row["n_trials"]
        reason = "No recorded reason" if row["reason_group"] == "no_recorded_stopping_reason" else row["reason_group"]
        origin = "--" if row["n_unmatched"] else f"{row['n_observed_reasons']}/{row['n_predicted_reasons']}"
        rates = []
        for model in ("baseline", "best"):
            count = row[f"n_predicted_success_{model}"]
            rates.append("--" if n == 0 else f"{count}/{n} ({100*count/n:.1f}" + r"\%)")
        tab += f"{row['phase']} & {reason} & {n} & {origin} & " + " & ".join(rates) + row_end
    tab += r"\bottomrule" + "\n" + r"\end{tabular}" + "\n"
    (out / "reason_errors.tex").write_text(tab)
    p = data["part2_reason_provenance"]
    (out / "reason_provenance_text.tex").write_text(
        f"The {p['failed_trial_phase_rows']} failed trial-phase rows represent {p['unique_failed_trials']} unique trials. "
        f"Of {p['matched_trial_phase_rows']} matched rows, {p['observed_reason_trial_phase_rows']} have observed reasons and "
        f"{p['predicted_reason_trial_phase_rows']} have predicted reasons; {p['unmatched_trial_phase_rows']} rows have no Part 1 match. "
        f"Among predicted rows, {p['predicted_training_resubstitution_rows']} are training resubstitution estimates. "
        "These are not out-of-fold or confirmed labels. Matched empty-reason groups have zero support "
        "after fallback and are omitted.\n")
    byphase = {(r["model"], r["phase"]): r for r in data["part2_validation_by_phase"]}
    fragments = []
    for phase in ("I", "II", "III"):
        a,b = byphase[("baseline",phase)],byphase[("phase_shared_best",phase)]
        fragments.append(f"Phase {phase} false-success counts change from {a['fp']}/{a['n_negative']} to {b['fp']}/{b['n_negative']}, "
                         f"while true-success false negatives change from {a['fn']} to {b['fn']}.")
    fragments.append("Phase I Recruitment errors fall from 10/22 to 7/22, while Safety errors rise from 4/10 to 5/10. "
                     "Phase II Other improves from 5/15 to 4/15, but Efficacy worsens from 4/18 to 5/18. "
                     "All five Phase I and all ten Phase III Efficacy failures remain false successes. "
                     "These groups include estimated reasons.")
    (out / "reason_interpretation_text.tex").write_text("\n\n".join(fragments) + "\n")

if __name__ == "__main__":
    main()

```


### report/scripts/report_analysis.py

SHA-256: `5d19b04101069df67c35123a1300b626be735129a9eda2e849bc3f0bc29e128c`

```python
"""Recompute report tables/figures from retained predictions; never fits a model."""
from __future__ import annotations
import argparse
import hashlib
import json
import sys
from pathlib import Path

PROJECT_ROOT = Path(__file__).resolve().parents[2]
SOURCE_ROOT = PROJECT_ROOT / "tools" / "src"
if str(SOURCE_ROOT) not in sys.path:
    sys.path.insert(0, str(SOURCE_ROOT))

import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score, f1_score, roc_auc_score

from midterm_evaluation import REASONS, evaluate_part1, evaluate_part2
from midterm_part2_baseline import analyze_failure_reasons

SOURCES = {
    'p1_supervised': 'models/part1/01_supervised_lr/evidence/original_run/validation_predictions.csv',
    'p1_initial_kmeans': 'models/part1/02_kmeans_lr/evidence/initial_run/validation_predictions.csv',
    'p1_retained_kmeans': 'models/part1/02_kmeans_lr/evidence/selected/selected_validation/validation_predictions.csv',
    'p1_fixed_ridge': 'models/part1/exploration/local_followup/historical_validation_predictions.csv',
    'p2_baseline': 'models/part2/01_tfidf_lr/evidence/analysis/validation_predictions.csv',
    'p2_phase_shared': 'models/part2/03_phase_shared/evidence/selection/validation_candidate/validation_predictions.csv',
    'historical_reason_lookup': 'models/part1/02_kmeans_lr/evidence/selected/reason_lookup.csv',
}


def sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


def clean_json(value):
    if isinstance(value, dict):
        return {str(k): clean_json(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [clean_json(v) for v in value]
    if isinstance(value, np.generic):
        value = value.item()
    if isinstance(value, float) and not np.isfinite(value):
        return None
    return value


def part1_tables(root):
    summary, by_reason, predictions = [], [], []
    truth_reference = None
    specifications = [('supervised_lr', 'p1_supervised', .5, 'prob_'),
                      ('initial_kmeans_lr', 'p1_initial_kmeans', .5, 'prob_'),
                      ('retained_kmeans_lr', 'p1_retained_kmeans', .35, 'prob_'),
                      ('fixed_ridge_157_gold', 'p1_fixed_ridge', .5, 'baseline_prob_')]
    for model, source, threshold, prefix in specifications:
        frame = pd.read_csv(root / SOURCES[source])
        reference = frame[['ID', *['true_' + r for r in REASONS]]].rename(columns={'true_' + r: r for r in REASONS})
        probabilities = frame[['ID', *[prefix + r for r in REASONS]]].rename(columns={prefix + r: r for r in REASONS})
        if truth_reference is None:
            truth_reference = reference.set_index('ID').sort_index()
        else:
            pd.testing.assert_frame_equal(reference.set_index('ID').sort_index(), truth_reference)
        result = evaluate_part1(reference, probabilities, threshold, empty_set_policy='argmax')
        y = result.predictions[['true_' + r for r in REASONS]].to_numpy()
        pred = result.predictions[['pred_' + r for r in REASONS]].to_numpy()
        summary.append(dict(model=model, threshold=threshold, gold_fit_rows=157,
                            decision_policy='threshold_then_argmax_if_empty',
                            **result.summary, exact_sets=int((pred == y).all(axis=1).sum()),
                            subset_accuracy=float((pred == y).all(axis=1).mean())))
        by_reason.append(result.by_group.assign(model=model))
        predictions.append(result.predictions.assign(model=model))
    return pd.DataFrame(summary), pd.concat(by_reason, ignore_index=True), pd.concat(predictions, ignore_index=True)


def part2_tables(root):
    lookup = pd.read_csv(root / SOURCES['historical_reason_lookup'])
    summaries, phases, analyses, predictions = [], [], {}, []
    reference_truth = None
    for model, source in [('baseline', 'p2_baseline'), ('phase_shared_best', 'p2_phase_shared')]:
        frame = pd.read_csv(root / SOURCES[source])
        reference = frame[['ID', 'trial_id', 'phase', 'label']]
        if reference_truth is None:
            reference_truth = reference.set_index('ID').sort_index()
        else:
            pd.testing.assert_frame_equal(reference.set_index('ID').sort_index(), reference_truth)
        result = evaluate_part2(reference, frame[['ID', 'p_success']], .5)
        y, p, pred = result.predictions.label, result.predictions.p_success, result.predictions.prediction
        summaries.append(dict(model=model, **result.summary,
            f1_pooled=float(f1_score(y, pred)), roc_auc_pooled=float(roc_auc_score(y, p)),
            average_precision_pooled=float(average_precision_score(y, p)),
            f1_equal_phase=float(result.by_group.f1.mean()),
            roc_auc_equal_phase=float(result.by_group.roc_auc.mean()),
            average_precision_equal_phase=float(result.by_group.average_precision.mean())))
        phases.append(result.by_group.assign(model=model))
        predictions.append(result.predictions.assign(model=model))
        analyses[model] = analyze_failure_reasons(result.predictions, lookup)
    before = analyses['baseline']['reason_rates']
    after = analyses['phase_shared_best']['reason_rates']
    keys = ['phase', 'reason_group', 'source_type']
    comparison = before.merge(after, on=keys, suffixes=('_baseline', '_best'), validate='one_to_one')
    for column in ['n_trials', 'n_observed_reasons', 'n_predicted_reasons', 'n_unmatched', 'n_training_resubstitution_reasons']:
        np.testing.assert_array_equal(comparison[column + '_baseline'], comparison[column + '_best'])
        comparison[column] = comparison[column + '_baseline']
    comparison['false_success_rate_delta'] = comparison.false_success_rate_best - comparison.false_success_rate_baseline
    failed = analyses['baseline']['failed_rows']
    provenance = {'lookup_file': SOURCES['historical_reason_lookup'],
        'lookup_sha256': sha256(root / SOURCES['historical_reason_lookup']), 'lookup_rows': len(lookup),
        'failed_trial_phase_rows': len(failed), 'unique_failed_trials': int(failed.trial_id.nunique()),
        'matched_trial_phase_rows': int(failed.matched_part1.sum()),
        'observed_reason_trial_phase_rows': int(failed.source_type.eq('observed').sum()),
        'predicted_reason_trial_phase_rows': int(failed.source_type.eq('predicted').sum()),
        'predicted_training_resubstitution_rows': int(failed.prediction_is_training_resubstitution.eq(True).sum()),
        'unmatched_trial_phase_rows': int(failed.source_type.eq('unmatched').sum()),
        'reasons_used_as_predictors': False,
        'reason_model': 'historical KMeans+LR lookup, unchanged for before/after comparison',
        'warning': 'Predicted reasons are model estimates; training resubstitution is not out-of-fold evidence. Multilabel groups overlap. Rates are descriptive, not causal.'}
    return pd.DataFrame(summaries), pd.concat(phases, ignore_index=True), analyses, comparison, provenance, pd.concat(predictions, ignore_index=True)


def make_figures(root):
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    out = root / 'report/analysis'
    figures = root / 'report/figures'
    figures.mkdir(parents=True, exist_ok=True)
    p1 = pd.read_csv(out/'part1_validation_by_reason.csv')
    shown = ['supervised_lr', 'retained_kmeans_lr', 'fixed_ridge_157_gold']
    matrix = p1.pivot(index='reason', columns='model', values='f1').reindex(REASONS)[shown]
    ax = matrix.rename(columns={'supervised_lr':'Supervised LR', 'retained_kmeans_lr':'K-Means + LR', 'fixed_ridge_157_gold':'Fixed RBF ridge'}).plot.bar(figsize=(9,4.5), ylim=(0,1), rot=0)
    ax.set_ylabel('Validation F1 per reason'); ax.set_xlabel(''); ax.set_title('Part 1: same nonempty output policy, 78 validation trials',pad=35)
    ax.legend(loc='lower center',bbox_to_anchor=(.5,1.01),ncol=3,frameon=False); ax.figure.tight_layout(); ax.figure.savefig(figures/'part1_reason_f1.png', dpi=160); plt.close(ax.figure)
    p2 = pd.read_csv(out/'part2_validation_by_phase.csv')
    matrix = p2.pivot(index='phase', columns='model', values='log_loss').reindex(['I','II','III'])
    ax = matrix.rename(columns={'baseline':'TF-IDF baseline','phase_shared_best':'Phase-shared blend'}).plot.bar(figsize=(7.5,4.3), rot=0)
    ax.set_ylabel('Validation log loss (lower is better)'); ax.set_xlabel('Phase'); ax.set_title('Part 2: saved validation probabilities',pad=35)
    ax.legend(loc='lower center',bbox_to_anchor=(.5,1.01),ncol=2,frameon=False)
    ax.figure.tight_layout(); ax.figure.savefig(figures/'part2_phase_logloss.png', dpi=160); plt.close(ax.figure)
    comparison = pd.read_csv(out/'part2_failure_reason_comparison.csv')
    reason_order = [*REASONS, 'no_recorded_stopping_reason']
    labels = [*REASONS, 'No Part 1 match']
    fig, axes = plt.subplots(1,2,figsize=(11.8,4.5),sharey=True)
    for ax, suffix, title in zip(axes, ['baseline','best'], ['TF-IDF baseline','Phase-shared blend']):
        selected = comparison[comparison.reason_group.isin(reason_order)]
        rates = selected.pivot(index='reason_group',columns='phase',values='false_success_rate_'+suffix).reindex(reason_order)[['I','II','III']]
        counts = selected.pivot(index='reason_group',columns='phase',values='n_trials').reindex(reason_order)[['I','II','III']]
        image = ax.imshow(rates.to_numpy(), vmin=0, vmax=1, cmap='Blues', aspect='auto')
        for i in range(len(reason_order)):
            for j in range(3):
                value, count = rates.iloc[i,j], int(counts.iloc[i,j])
                text = 'NA (n=0)' if pd.isna(value) else f'{value:.2f}\n(n={count})'
                ax.text(j,i,text,ha='center',va='center',fontsize=9,color='white' if value>.6 else 'black')
        ax.set_xticks(range(3), ['I','II','III']); ax.set_yticks(range(6), labels); ax.set_title(title); ax.set_xlabel('Phase')
    fig.suptitle('False-success rate among failed validation trials; historical reason groups')
    fig.subplots_adjust(left=.18,right=.88,top=.84,bottom=.12,wspace=.12)
    fig.colorbar(image,ax=axes,location='right',fraction=.035,pad=.035,label='False-success rate')
    fig.savefig(figures/'part2_failure_reason_comparison.png',dpi=160); plt.close(fig)
    return sorted(str(p.relative_to(root)) for p in figures.glob('*.png'))


def build_report(root=None, figures=True):
    root = Path(root) if root is not None else PROJECT_ROOT
    out = root / 'report/analysis'; out.mkdir(parents=True, exist_ok=True)
    selection_path = root / 'models/selection.json'
    selection = json.loads(selection_path.read_text())
    public_models = {part: selection[part]['best_public'] for part in ('part1', 'part2')}
    for item in public_models.values():
        submission = root / item['submission']
        if sha256(submission) != item['sha256']:
            raise ValueError(f'Selection submission checksum mismatch: {submission}')
    p1, p1_reasons, p1_predictions = part1_tables(root)
    p2, p2_phases, analyses, comparison, provenance, p2_predictions = part2_tables(root)
    tables = {'part1_validation_summary':p1, 'part1_validation_by_reason':p1_reasons,
              'part1_validation_predictions':p1_predictions, 'part2_validation_summary':p2,
              'part2_validation_by_phase':p2_phases, 'part2_validation_predictions':p2_predictions,
              'part2_failure_reason_comparison':comparison}
    for key in ['reason_rates','source_rates','source_detail_rates','overall_failed_rates','failed_rows']:
        tables['part2_failure_' + key] = pd.concat([value[key].assign(model=model) for model,value in analyses.items()],ignore_index=True)
    for name, table in tables.items():
        table.to_csv(out/(name+'.csv'),index=False)
    report = {'selection': {'path': 'models/selection.json', 'sha256': sha256(selection_path),
                             'public_models': public_models},
        'part1_validation':p1.to_dict('records'), 'part1_validation_by_reason':p1_reasons.to_dict('records'),
        'part2_validation':p2.to_dict('records'), 'part2_validation_by_phase':p2_phases.to_dict('records'),
        'part2_failure_reason_comparison':comparison.to_dict('records'), 'part2_reason_provenance':provenance,
        'sources':{name:{'path':path,'sha256':sha256(root/path)} for name,path in SOURCES.items()},
        'limitations':['All reported validation labels were previously inspected during development; held-out fitting scope does not imply an untouched test.',
          'Part1 fixed ridge validation is the train157 fit saved as baseline in local_followup, not the final235-label submission model.',
          'Part1 original supervised raw score .8654 used empty sets; fair argmax score is recomputed here.',
          'Part2 primary validation log loss gives equal weight to each phase; pooled metrics are reported separately.',
          'Reason analysis freezes the historical Part1 lookup for both Part2 models and never feeds reasons into outcome predictors.'],
        'no_model_training_performed':True}
    report['figures'] = make_figures(root) if figures else []
    (out/'report_metrics.json').write_text(json.dumps(clean_json(report),indent=2,allow_nan=False)+'\n')
    return report


if __name__ == '__main__':
    parser=argparse.ArgumentParser(); parser.add_argument('--root',type=Path); parser.add_argument('--no-figures',action='store_true')
    args=parser.parse_args(); result=build_report(args.root,figures=not args.no_figures)
    print(json.dumps({'part1_validation':result['part1_validation'],'part2_validation':result['part2_validation'],
                      'reason_provenance':result['part2_reason_provenance']},indent=2))

```


### reproduce.py

SHA-256: `377a04451637bde9097fd28b71ef543ea1930c505c4a88e4427be2517cbe0c03`

```python
"""Verify both frozen submissions and the best public Part 1 and Part 2 files.

This runner uses saved fitted models and the supplied test features. It never
fits a model, uses hidden labels, or changes the recorded submission files.
After verification, --output-dir exports byte-identical best submissions as
part1.csv and part2.csv into a new or empty directory.
"""
from __future__ import annotations

import argparse
import hashlib
import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

import sys

sys.path.insert(0, str(Path(__file__).resolve().parent / 'tools/src'))

from midterm_evaluation import REASONS, dataset_path, part1_decisions
from midterm_finalization import predict_final_outcomes
from midterm_part1_finalization import predict_bundle
from midterm_part2_improvement import validate_submission
from midterm_rank_part2 import blend, predict_phase_shared


def digest(path: Path) -> str:
    value = hashlib.sha256()
    with path.open("rb") as source:
        for block in iter(lambda: source.read(1024 * 1024), b""):
            value.update(block)
    return value.hexdigest()


def check_part1(root: Path) -> dict:
    source = dataset_path(root, "part1", "test")
    test = pd.read_csv(source)
    bundle = joblib.load(root / 'models/part1/02_kmeans_lr/model.joblib')
    probabilities = predict_bundle(bundle, test)
    selected = part1_decisions(probabilities, bundle["thresholds"], empty_set_policy="argmax")
    predicted = [" ".join(reason for reason, active in zip(REASONS, row) if active)
                 for row in selected]
    generated = pd.DataFrame({"ID": test.ID, "Predicted": predicted})
    recorded = pd.read_csv(root / 'models/part1/02_kmeans_lr/submission.csv')
    sample = pd.read_csv(source.parent / "part1_sample_submission.csv")
    validate_submission(generated, sample, 1)
    if not generated.equals(recorded):
        raise AssertionError("Part 1 selected model differs from recorded submission.")
    return {"rows": len(generated), "model": 'models/part1/02_kmeans_lr/model.joblib',
            "submission": 'models/part1/02_kmeans_lr/submission.csv',
            "submission_sha256": digest(root / 'models/part1/02_kmeans_lr/submission.csv')}


def check_part1_best_public(root: Path) -> dict:
    """Recreate the current local-kernel submission from its saved fitted bundle."""
    selection = json.loads((root / 'models/selection.json').read_text())
    record = selection["part1"]["best_public"]
    source = dataset_path(root, "part1", "test")
    test = pd.read_csv(source)
    fitted = joblib.load(root / record["model"])
    probabilities = fitted.predict(test[[f"emb_{j}" for j in range(150)]].to_numpy())
    if probabilities.shape != (len(test), len(REASONS)) or not np.isfinite(probabilities).all():
        raise AssertionError("Best public Part 1 model returned invalid probabilities.")
    if ((probabilities < 0) | (probabilities > 1)).any():
        raise AssertionError("Best public Part 1 probabilities are outside [0, 1].")
    selected = part1_decisions(probabilities, fitted.recipes["final"]["threshold"],
                               empty_set_policy="argmax")
    generated = pd.DataFrame({
        "ID": test.ID,
        "Predicted": [" ".join(reason for reason, active in zip(REASONS, row) if active)
                      for row in selected],
    })
    sample = pd.read_csv(source.parent / "part1_sample_submission.csv")
    validate_submission(generated, sample, 1)
    recorded = pd.read_csv(root / record["submission"])
    if not generated.equals(recorded):
        raise AssertionError("Best public Part 1 model differs from its recorded submission.")
    return {"rows": len(generated), "model": record["model"],
            "submission": record["submission"],
            "submission_sha256": digest(root / record["submission"])}


def check_part2(root: Path) -> dict:
    source = dataset_path(root, "part2", "test")
    test = pd.read_csv(source)
    original = joblib.load(root / 'models/part2/02_text_blend/model.joblib')
    shared = joblib.load(root / 'models/part2/03_phase_shared/component.joblib')
    original_p = predict_final_outcomes(original, test)
    shared_p = predict_phase_shared(shared, test)
    blended = blend(test, original_p, shared_p, 0.5)
    sample = pd.read_csv(source.parent / "part2_sample_submission.csv")
    results = {}
    for label, computed, relative in (
        ("submitted_original", original_p, 'models/part2/02_text_blend/submission.csv'),
        ("best_public", blended, 'models/part2/03_phase_shared/submission.csv'),
    ):
        generated = computed.rename(columns={"p_success": "label"})
        recorded = pd.read_csv(root / relative)
        validate_submission(generated, sample, 2)
        if generated.ID.tolist() != recorded.ID.tolist():
            raise AssertionError(f"Part 2 {label} ID order differs.")
        gap = float(np.max(np.abs(generated.label.to_numpy() - recorded.label.to_numpy())))
        if gap > 1e-12:
            raise AssertionError(f"Part 2 {label} probabilities differ: max gap {gap}.")
        results[label] = {"rows": len(generated), "submission": relative,
                          "submission_sha256": digest(root / relative),
                          "max_absolute_probability_gap": gap}
    return results


def submission_records(value):
    """Visit every registered submission, including prior public selections."""
    if isinstance(value, dict):
        if "submission" in value:
            if "sha256" not in value:
                raise AssertionError(f"Registered submission has no checksum: {value['submission']}")
            yield value
        for child in value.values():
            yield from submission_records(child)
    elif isinstance(value, list):
        for child in value:
            yield from submission_records(child)


def check_registry(root: Path) -> dict:
    selection = json.loads((root / 'models/selection.json').read_text())
    # Keep the original registry checks mandatory when registering a new best.
    for task, label in (("part1", "best_public"), ("part1", "previous_best_public"),
                        ("part1", "rejected_uploaded_graph"), ("part2", "best_public"),
                        ("part2", "frozen_november_1_delivery"),
                        ("part2", "earlier_word_field_candidate")):
        if not isinstance(selection.get(task, {}).get(label), dict):
            raise AssertionError(f"Missing required submission record: {task}.{label}")
    records = list(submission_records(selection))
    if not records:
        raise AssertionError("Submission registry is empty.")
    for record in records:
        if digest(root / record["submission"]) != record["sha256"]:
            raise AssertionError(f"Recorded submission bytes changed: {record['submission']}")
        for model, expected in record.get("model_sha256", {}).items():
            if digest(root / model) != expected:
                raise AssertionError(f"Recorded model bytes changed: {model}")
    source = root / 'models/part2/03_phase_shared/submission.csv'
    if digest(source) != selection["part2"]["best_public"]["sha256"]:
        raise AssertionError("Canonical best public Part 2 file does not match its registered hash.")
    return {"status": "PASS", "recorded_submissions": len(records)}


def validate_output_directory(path: Path) -> None:
    """Reject files and nonempty directories before running inference or writing."""
    if path.exists():
        if not path.is_dir():
            raise NotADirectoryError(f"Output path is not a directory: {path}")
        if any(path.iterdir()):
            raise FileExistsError(f"Output directory must be new or empty: {path}")


def export_best_submissions(root: Path, output_dir: Path) -> dict:
    """Export canonical bytes after main has verified their saved-model predictions.

    Copying the checked canonical CSV bytes preserves the exact submitted float
    serialization and registered hashes. No model or recorded CSV is rewritten.
    """
    validate_output_directory(output_dir)
    selection = json.loads((root / "models/selection.json").read_text())
    payloads = {}
    for task in ("part1", "part2"):
        record = selection[task]["best_public"]
        contents = (root / record["submission"]).read_bytes()
        if hashlib.sha256(contents).hexdigest() != record["sha256"]:
            raise AssertionError(f"Recorded submission bytes changed: {record['submission']}")
        payloads[task] = contents
    output_dir.mkdir(parents=True, exist_ok=True)
    validate_output_directory(output_dir)
    exported = {}
    for task, contents in payloads.items():
        path = output_dir / f"{task}.csv"
        with path.open("xb") as destination:
            destination.write(contents)
        exported[task] = {"filename": path.name, "sha256": hashlib.sha256(contents).hexdigest()}
    return exported


def main(argv=None) -> None:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--project-root", type=Path, default=Path(__file__).resolve().parent)
    parser.add_argument("--output-dir", type=Path,
                        help="Optional new or empty directory for verified part1.csv and part2.csv.")
    args = parser.parse_args(argv)
    root = args.project_root.resolve()
    output_dir = args.output_dir.resolve() if args.output_dir is not None else None
    if output_dir is not None:
        validate_output_directory(output_dir)
    registry = check_registry(root)
    report = {"part1": check_part1(root), "part1_best_public": check_part1_best_public(root),
              "part2": check_part2(root),
              "registry": registry, "status": "PASS"}
    if output_dir is not None:
        report["exported_best"] = export_best_submissions(root, output_dir)
    print(json.dumps(report, indent=2))


if __name__ == "__main__":
    main()

```


### tools/src/midterm_augmentation.py

SHA-256: `f7baeffe67ea5115a24b6476dbce40a2ae68f066afaf0750ad23fc430f72fdca`

```python
"""Training-only cluster label recovery and weighted multilabel logistic regression."""

from __future__ import annotations

import inspect
import warnings

import numpy as np
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression


def observed_label_mask(labels: np.ndarray) -> np.ndarray:
    """Require either five observed binary labels or five unknown labels per row."""
    labels = np.asarray(labels, dtype=float)
    if labels.ndim != 2 or labels.shape[0] == 0 or labels.shape[1] != 5:
        raise ValueError("Labels must be a nonempty n-by-5 array.")
    observed = np.isin(labels, [0, 1]).all(axis=1)
    masked = (labels == -100).all(axis=1)
    if not (observed | masked).all() or not observed.any():
        raise ValueError("Each row must be fully observed (0/1) or fully masked (-100), with observed seeds present.")
    return observed


def membership_matrix(values: np.ndarray) -> np.ndarray:
    """Validate hard one-hot assignments or soft responsibilities."""
    values = np.asarray(values, dtype=float)
    if values.ndim != 2 or min(values.shape) < 1 or not np.isfinite(values).all():
        raise ValueError("Memberships must be a finite, nonempty matrix.")
    if ((values < 0) | (values > 1)).any() or not np.allclose(values.sum(axis=1), 1, atol=1e-8, rtol=0):
        raise ValueError("Memberships must be nonnegative and sum to one in every row.")
    return values


def cluster_memberships(clusterer, features: np.ndarray, method: str) -> np.ndarray:
    """K-Means uses hard assignments; GMM uses posterior component responsibilities."""
    if method == "kmeans":
        return membership_matrix(np.eye(clusterer.n_clusters)[clusterer.predict(features)])
    if method == "gmm":
        return membership_matrix(clusterer.predict_proba(features))
    raise ValueError("Method must be kmeans or gmm.")


def estimate_cluster_reason_rates(memberships, labels, smoothing_strength: float = 2.0):
    """Smooth visible-seed reason rates toward the visible-seed global prior.

    For cluster k and reason r:
      q[k,r] = (sum_i w[i,k]*y[i,r] + alpha*prior[r]) / (sum_i w[i,k] + alpha)
    Every sum over i uses only observed training seeds. A cluster with no observed
    membership mass falls back to the prior. These estimates are not calibrated
    population prevalences because the visible seeds were not sampled at random.
    """
    weights = membership_matrix(memberships)
    labels = np.asarray(labels, dtype=float)
    observed = observed_label_mask(labels)
    if len(weights) != len(labels):
        raise ValueError("Membership and label rows must align.")
    if not np.isfinite(smoothing_strength) or smoothing_strength <= 0:
        raise ValueError("Smoothing strength must be finite and positive.")
    prior = labels[observed].mean(axis=0)
    mass = weights[observed].sum(axis=0)
    positive_mass = weights[observed].T @ labels[observed]
    rates = (positive_mass + smoothing_strength * prior) / (mass[:, None] + smoothing_strength)
    return {"rates": rates, "global_prior": prior, "observed_membership_mass": mass,
            "positive_membership_mass": positive_mass, "smoothing_strength": float(smoothing_strength)}


def recover_labels(labels, memberships, mapping, *, threshold: float = 0.5, pseudo_weight: float = 0.1):
    """Fill every masked row, preserving observed labels exactly.

    Decision margins describe distance from the fixed decision boundary. They do
    not estimate label accuracy. Cluster assignment certainty has a separate meaning.
    """
    labels = np.asarray(labels, dtype=float)
    observed = observed_label_mask(labels)
    weights = membership_matrix(memberships)
    rates = np.asarray(mapping["rates"], dtype=float)
    if len(weights) != len(labels) or rates.shape != (weights.shape[1], 5):
        raise ValueError("Labels, memberships, and cluster rates have incompatible shapes.")
    if not np.isfinite(rates).all() or ((rates < 0) | (rates > 1)).any():
        raise ValueError("Reason rates must be finite values in [0, 1].")
    if not np.isfinite(threshold) or not 0 < threshold < 1:
        raise ValueError("The decision threshold must lie strictly between zero and one.")
    if not np.isfinite(pseudo_weight) or not 0 < pseudo_weight <= 1:
        raise ValueError("The pseudo-label weight must be in (0, 1].")
    scores = weights @ rates
    recovered = labels.copy().astype(int)
    recovered[~observed] = (scores[~observed] >= threshold).astype(int)
    # At threshold 0.5, the normalized margin equals 2*abs(score-0.5).
    margins = np.where(scores >= threshold, (scores-threshold)/(1-threshold), (threshold-scores)/threshold)
    entropy = -(weights * np.log(np.clip(weights, 1e-300, 1))).sum(axis=1)
    normalized_entropy = entropy / np.log(weights.shape[1]) if weights.shape[1] > 1 else np.zeros(len(weights))
    return {"labels": recovered, "observed": observed, "scores": scores,
            "sample_weight": np.where(observed, 1.0, pseudo_weight),
            "decision_margins": margins, "dominant_cluster": weights.argmax(axis=1),
            "max_membership": weights.max(axis=1), "normalized_membership_entropy": normalized_entropy}


def logistic_parameters(seed: int = 42) -> dict:
    """Use the baseline's effective L2 settings across sklearn versions."""
    parameters = {"C": 1.0, "solver": "liblinear", "max_iter": 500, "tol": 1e-4,
                  "class_weight": None, "fit_intercept": True, "random_state": seed}
    # sklearn 1.8+ expresses L2 through l1_ratio; earlier releases use penalty.
    if inspect.signature(LogisticRegression).parameters["penalty"].default == "deprecated":
        parameters["l1_ratio"] = 0.0
    else:
        parameters["penalty"] = "l2"
    return parameters


def fit_reason_classifiers(features, labels, sample_weight, *, seed: int = 42):
    """Fit five independent binary models with explicit per-row sample weights."""
    labels = np.asarray(labels)
    if not observed_label_mask(labels).all():
        raise ValueError("Classifier targets must be binary; recover masked labels first.")
    sample_weight = np.asarray(sample_weight, dtype=float)
    if sample_weight.shape != (len(labels),) or not np.isfinite(sample_weight).all() or (sample_weight <= 0).any():
        raise ValueError("Provide a finite positive weight for every training row.")
    estimators, warning_messages = [], []
    with warnings.catch_warnings(record=True) as recorded:
        warnings.simplefilter("always")
        for column in range(labels.shape[1]):
            if len(np.unique(labels[:, column])) != 2:
                raise ValueError("Every reason requires both observed classes.")
            estimator = LogisticRegression(**logistic_parameters(seed))
            estimator.fit(features, labels[:, column], sample_weight=sample_weight)
            estimators.append(estimator)
    for item in recorded:
        if issubclass(item.category, ConvergenceWarning):
            raise RuntimeError(f"Logistic regression did not converge: {item.message}")
        warning_messages.append(str(item.message))
    if any(int(estimator.n_iter_.max()) >= estimator.max_iter for estimator in estimators):
        raise RuntimeError("A classifier reached the fixed iteration limit.")
    return estimators, warning_messages


def predict_reason_probabilities(estimators, features) -> np.ndarray:
    """Return independent reason scores in the estimator list's column order."""
    if len(estimators) != 5:
        raise ValueError("Exactly five reason classifiers are required.")
    return np.column_stack([model.predict_proba(features)[:, list(model.classes_).index(1)] for model in estimators])


def predict_bundle(bundle: dict, frame):
    """Predict with a saved plain-data bundle, without refitting any component."""
    scaled = bundle["scaler"].transform(frame[bundle["feature_columns"]])
    return predict_reason_probabilities(bundle["classifiers"], scaled)

```


### tools/src/midterm_evaluation.py

SHA-256: `7909b7f7470a9866934a5a2b879ff1419e0680d0ef730fff5a3846ec31d5e0ef`

```python
"""Consistent validation metrics, grouped folds, and experiment records for the midterm."""

from __future__ import annotations

from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Mapping, Sequence
import hashlib
import json
import random
import re
import shutil
import tempfile
import uuid

import numpy as np
import pandas as pd
from sklearn.metrics import (
    average_precision_score, confusion_matrix, f1_score, log_loss,
    precision_score, recall_score, roc_auc_score,
)
from sklearn.model_selection import StratifiedGroupKFold


REASONS = ("Safety", "Funding", "Efficacy", "Recruitment", "Other")
PHASES = ("I", "II", "III")
PART1_DIRECTORY = 'data/part1'
PART2_DIRECTORY = 'data/part2'
LOG_LOSS_EPSILON = 1e-15
REGISTRY_COLUMNS = (
    "run_id", "created_at_utc", "task", "model_name", "seed", "hypothesis", "notes",
    "feature_spec_json", "hyperparameters_json", "thresholds_json", "validation_rows",
    "primary_metric", "primary_value", "higher_is_better", "run_directory",
)


def _validate_seed(seed: int) -> None:
    if not isinstance(seed, int) or isinstance(seed, bool) or not 0 <= seed < 2**32:
        raise ValueError("seed must be an integer in [0, 2**32).")


def set_random_seed(seed: int = 42) -> None:
    """Seed Python and NumPy; also pass this seed to each estimator's random_state."""
    _validate_seed(seed)
    random.seed(seed)
    np.random.seed(seed)


def _json_ready(value: Any) -> Any:
    if isinstance(value, Mapping):
        return {str(key): _json_ready(item) for key, item in value.items()}
    if isinstance(value, (list, tuple, np.ndarray)):
        return [_json_ready(item) for item in value]
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (float, np.floating)):
        return float(value) if np.isfinite(value) else None
    if isinstance(value, (np.bool_,)):
        return bool(value)
    return value


def _write_json(path: Path, value: Any) -> None:
    path.write_text(json.dumps(_json_ready(value), indent=2, allow_nan=False) + "\n", encoding="utf-8")


def file_sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def _unique_ids(frame: pd.DataFrame, name: str) -> None:
    if not isinstance(frame, pd.DataFrame) or "ID" not in frame:
        raise ValueError(f"{name} must be a DataFrame containing ID.")
    if frame.columns.duplicated().any():
        raise ValueError(f"{name} has duplicate column names.")
    if frame.empty or frame["ID"].isna().any() or not frame["ID"].is_unique:
        raise ValueError(f"{name} must have nonempty, unique, nonmissing IDs.")
    if not frame["ID"].map(lambda value: isinstance(value, str) and bool(value.strip())).all():
        raise ValueError(f"{name} IDs must be nonempty strings.")


def _align(reference: pd.DataFrame, predictions: pd.DataFrame, columns: Sequence[str]) -> pd.DataFrame:
    _unique_ids(reference, "reference")
    _unique_ids(predictions, "predictions")
    if set(predictions.columns) != {"ID", *columns}:
        raise ValueError(f"Prediction columns must be exactly ID and {list(columns)}.")
    reference_ids, prediction_ids = set(reference["ID"]), set(predictions["ID"])
    if reference_ids != prediction_ids:
        raise ValueError(
            f"Prediction IDs do not match reference IDs: "
            f"{len(reference_ids-prediction_ids)} missing, {len(prediction_ids-reference_ids)} extra."
        )
    return predictions.set_index("ID").loc[reference["ID"], list(columns)].reset_index()


def _numeric(frame: pd.DataFrame, columns: Sequence[str], *, probabilities: bool) -> np.ndarray:
    try:
        values = frame[list(columns)].to_numpy(dtype=np.float64)
    except (KeyError, TypeError, ValueError) as error:
        raise ValueError(f"Expected numeric columns: {list(columns)}.") from error
    if not np.isfinite(values).all():
        raise ValueError("Targets and probabilities must be finite and nonmissing.")
    valid = ((values >= 0) & (values <= 1)) if probabilities else np.isin(values, [0, 1])
    if not valid.all():
        label = "Probabilities must be in [0, 1]." if probabilities else "Targets must be 0/1; masked -100 is not a negative label."
        raise ValueError(label)
    return values


def _thresholds(value: float | Mapping[str, float], groups: Sequence[str]) -> dict[str, float]:
    if isinstance(value, Mapping):
        if set(value) != set(groups):
            raise ValueError(f"Threshold mappings must contain exactly {list(groups)}.")
        result = {group: float(value[group]) for group in groups}
    else:
        result = {group: float(value) for group in groups}
    if any(not np.isfinite(t) or not 0 <= t <= 1 for t in result.values()):
        raise ValueError("Every threshold must be finite and in [0, 1].")
    return result


@dataclass
class EvaluationResult:
    task: str
    summary: dict[str, float | int]
    by_group: pd.DataFrame
    predictions: pd.DataFrame
    thresholds: dict[str, float]

    def save(self, directory: Path) -> None:
        """Save metrics, support counts, and ID-aligned probabilities and decisions."""
        directory = Path(directory)
        directory.mkdir(parents=True, exist_ok=True)
        _write_json(directory / "metrics.json", {
            "task": self.task, "summary": self.summary, "thresholds": self.thresholds,
            "by_group": self.by_group.to_dict(orient="records"),
        })
        self.by_group.to_csv(directory / "metrics_by_group.csv", index=False)
        self.predictions.to_csv(directory / "validation_predictions.csv", index=False)


def part1_decisions(scores, thresholds=0.5, *, empty_set_policy="allow"):
    """Threshold independently, optionally assigning the highest score to an empty row.

    Ties use REASONS order. The default retains historical experiment semantics;
    the final submission explicitly requests the argmax policy.
    """
    values = np.asarray(scores, dtype=float)
    limits = _thresholds(thresholds, REASONS)
    if values.ndim != 2 or values.shape[1] != len(REASONS) or not np.isfinite(values).all() or ((values < 0) | (values > 1)).any():
        raise ValueError("Provide five finite reason probabilities in [0, 1].")
    if empty_set_policy not in {"allow", "argmax"}:
        raise ValueError("empty_set_policy must be allow or argmax.")
    decisions = (values >= np.array([limits[r] for r in REASONS])).astype(int)
    if empty_set_policy == "argmax":
        empty = np.flatnonzero(decisions.sum(axis=1) == 0)
        decisions[empty, values[empty].argmax(axis=1)] = 1
    return decisions


def evaluate_part1(
    reference: pd.DataFrame, probabilities: pd.DataFrame,
    thresholds: float | Mapping[str, float] = 0.5, *, empty_set_policy: str = "allow",
) -> EvaluationResult:
    """Evaluate the same reason-decision policy used at inference."""
    aligned = _align(reference, probabilities, REASONS)
    truth = _numeric(reference, REASONS, probabilities=False).astype(int)
    scores = _numeric(aligned, REASONS, probabilities=True)
    limits = _thresholds(thresholds, REASONS)
    decisions = part1_decisions(scores, limits, empty_set_policy=empty_set_policy)
    rows = []
    saved = reference[["ID"]].reset_index(drop=True).copy()
    for j, reason in enumerate(REASONS):
        y, pred = truth[:, j], decisions[:, j]
        rows.append({"reason": reason, "n_positive": int(y.sum()), "n_negative": int((1-y).sum()),
                     "f1": f1_score(y, pred, zero_division=0),
                     "precision": precision_score(y, pred, zero_division=0),
                     "recall": recall_score(y, pred, zero_division=0)})
        saved[f"true_{reason}"] = y
        saved[f"prob_{reason}"] = scores[:, j]
        saved[f"pred_{reason}"] = pred
    summary = {"n_rows": len(reference),
               "f1_samples": f1_score(truth, decisions, average="samples", zero_division=0),
               "f1_macro": f1_score(truth, decisions, average="macro", zero_division=0)}
    if empty_set_policy == "argmax":
        fallback = (scores >= np.array([limits[r] for r in REASONS])).sum(axis=1) == 0
        saved["empty_set_fallback_applied"] = fallback
        summary["n_empty_set_fallbacks"] = int(fallback.sum())
    return EvaluationResult("part1", summary, pd.DataFrame(rows), saved, limits)


def _outcome_reference(reference: pd.DataFrame) -> None:
    _unique_ids(reference, "outcome reference")
    required = {"trial_id", "phase", "label"}
    if not required.issubset(reference.columns):
        raise ValueError(f"Outcome references require {sorted(required)}.")
    _numeric(reference, ["label"], probabilities=False)
    if reference["trial_id"].isna().any() or not reference["phase"].isin(PHASES).all():
        raise ValueError("Trial IDs must be present and phases must be I, II, or III.")
    if reference.duplicated(["trial_id", "phase"]).any():
        raise ValueError("Each trial-phase pair must appear only once.")
    if reference.groupby("trial_id")["label"].nunique().gt(1).any():
        raise ValueError("A trial must have the same outcome label across phases.")


def evaluate_part2(
    reference: pd.DataFrame, probabilities: pd.DataFrame,
    thresholds: float | Mapping[str, float] = 0.5,
) -> EvaluationResult:
    """Evaluate success (label=1) per phase; compute AP and ROC-AUC from probabilities."""
    _outcome_reference(reference)
    aligned = _align(reference, probabilities, ["p_success"])
    scores = _numeric(aligned, ["p_success"], probabilities=True).ravel()
    limits = _thresholds(thresholds, PHASES)
    saved = reference[["ID", "trial_id", "phase", "label"]].reset_index(drop=True).copy()
    saved["p_success"] = scores
    saved["prediction"] = (scores >= saved["phase"].map(limits).to_numpy()).astype(int)
    rows = []
    for phase in PHASES:
        sub = saved.loc[saved["phase"].eq(phase)]
        y = sub["label"].to_numpy(dtype=int)
        p = sub["p_success"].to_numpy()
        pred = sub["prediction"].to_numpy()
        row = {"phase": phase, "n_rows": len(sub), "n_unique_trials": sub["trial_id"].nunique(),
               "n_positive": int(y.sum()), "n_negative": int((1-y).sum())}
        if not len(sub):
            row.update({k: np.nan for k in ("f1", "roc_auc", "average_precision", "log_loss", "tn", "fp", "fn", "tp")})
            row.update(roc_auc_status="undefined_no_rows", average_precision_status="undefined_no_rows")
        else:
            both_classes = len(np.unique(y)) == 2
            row.update(
                f1=f1_score(y, pred, zero_division=0),
                roc_auc=roc_auc_score(y, p) if both_classes else np.nan,
                average_precision=average_precision_score(y, p) if y.sum() else np.nan,
                log_loss=log_loss(y, np.clip(p, LOG_LOSS_EPSILON, 1-LOG_LOSS_EPSILON), labels=[0, 1]),
                roc_auc_status="ok" if both_classes else "undefined_single_class",
                average_precision_status="ok" if y.sum() else "undefined_no_positives",
            )
            tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
            row.update(tn=int(tn), fp=int(fp), fn=int(fn), tp=int(tp))
        rows.append(row)
    groups = pd.DataFrame(rows)
    summary = {"n_rows": len(saved), "n_unique_trials": saved["trial_id"].nunique(),
               "log_loss_pooled": log_loss(saved["label"], np.clip(scores, LOG_LOSS_EPSILON, 1-LOG_LOSS_EPSILON), labels=[0, 1]),
               "log_loss_equal_phase": float(groups["log_loss"].mean()) if groups["n_rows"].gt(0).all() else np.nan}
    return EvaluationResult("part2", summary, groups, saved, limits)


def validate_fit_scope(fit_trial_ids: Sequence[str], allowed_training_ids: Sequence[str]) -> dict:
    """Validate declared fit IDs against this task's training trial IDs."""
    if isinstance(fit_trial_ids, (str, bytes)):
        raise ValueError("Pass a sequence of trial IDs, not a string.")
    ids = list(fit_trial_ids)
    if not ids or any(not isinstance(item, str) or not item.strip() for item in ids):
        raise ValueError("Fit IDs must be nonempty strings and contain at least one row.")
    unexpected = set(ids) - set(allowed_training_ids)
    if unexpected:
        raise ValueError(f"Fit scope contains {len(unexpected)} nontraining trial IDs: {sorted(unexpected)[:5]}.")
    digest = hashlib.sha256("\n".join(sorted(set(ids))).encode()).hexdigest()
    return {"declared_fit_rows": len(ids), "unique_training_trials": len(set(ids)), "unique_trial_ids_sha256": digest}


def make_part2_grouped_folds(training: pd.DataFrame, n_splits: int = 5, seed: int = 42) -> pd.DataFrame:
    """Assign training rows to optional CV folds, grouping all phases of each trial."""
    _outcome_reference(training)
    set_random_seed(seed)
    if not isinstance(n_splits, int) or n_splits < 2 or n_splits > training["trial_id"].nunique():
        raise ValueError("n_splits must be between 2 and the number of training trials.")
    strata = training["phase"].astype(str) + "_" + training["label"].astype(str)
    splitter = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    assignment = np.full(len(training), -1, dtype=int)
    for fold, (_, held_out) in enumerate(splitter.split(np.zeros(len(training)), strata, groups=training["trial_id"])):
        assignment[held_out] = fold
    result = training[["ID", "trial_id", "phase", "label"]].reset_index(drop=True).copy()
    result["fold"] = assignment
    if (assignment < 0).any() or result.groupby("trial_id")["fold"].nunique().gt(1).any():
        raise RuntimeError("Invalid grouped fold assignment.")
    return result


def dataset_path(project_root: Path, task: str, split: str) -> Path:
    if task not in ("part1", "part2") or split not in ("train", "val", "test"):
        raise ValueError("Expected task part1/part2 and split train/val/test.")
    directory, prefix = (PART1_DIRECTORY, "part1") if task == "part1" else (PART2_DIRECTORY, "outcome")
    return Path(project_root) / directory / f"{prefix}_{split}.csv"


def initialize_protocol(project_root: Path, seed: int = 42, n_splits: int = 5) -> dict:
    """Create reproducible split records and optional grouped folds without fitting models."""
    project_root = Path(project_root).resolve()
    set_random_seed(seed)
    protocol = {
        "version": "1.0", "seed": seed, "selection_split": "provided_validation",
        "part1": {"primary_metric": "f1_samples", "higher_is_better": True,
                  "report": ["f1_samples", "f1_macro", "f1_per_reason", "positive_support_per_reason"]},
        "part2": {"primary_metric": "log_loss_equal_phase", "higher_is_better": False,
                  "report_per_phase": ["f1", "roc_auc", "average_precision", "log_loss", "positive_support", "negative_support"],
                  "positive_class": 1, "PR_AUC_definition": "sklearn average_precision_score; no trapezoidal PR integration"},
        "decision_rule": "probability >= threshold", "initial_threshold": 0.5,
        "f1_zero_division": 0, "log_loss_clipping_epsilon": LOG_LOSS_EPSILON,
        "undefined_auc_policy": "ROC-AUC: missing for single-class/no-row phases; AP: missing for no-positive/no-row phases; JSON uses null.",
        "preprocessing": "Fit each model, scaler, vocabulary, clusterer, and imputer using training data only; transform validation/test.",
        "part2_forbidden_inputs": ["ID", "trial_id", "label", *REASONS, "emb_0 through emb_149"],
        "hyperparameter_search": "Start with a small prespecified set. Record a hypothesis and every tried setting, including thresholds and unsuccessful runs. No automatic search is performed here.",
        "part2_optional_cv": {"n_splits": n_splits, "group": "trial_id", "strata": "phase + outcome label",
                              "method": "StratifiedGroupKFold", "fit_preprocessing_inside_each_training_fold": True},
    }
    split_manifest, task_data = {}, {}
    for task in ("part1", "part2"):
        task_data[task] = {}
        ids_per_split = {}
        for split in ("train", "val", "test"):
            path = dataset_path(project_root, task, split)
            df = pd.read_csv(path)
            _unique_ids(df, f"{task}_{split}")
            task_data[task][split] = df
            group_col = "ID" if task == "part1" else "trial_id"
            ids_per_split[split] = set(df[group_col])
            split_manifest[f"{task}_{split}"] = {"relative_path": str(path.relative_to(project_root)),
                "sha256": file_sha256(path), "rows": len(df), "unique_trials": df[group_col].nunique()}
        for a, b in (("train", "val"), ("train", "test"), ("val", "test")):
            if ids_per_split[a] & ids_per_split[b]:
                raise ValueError(f"{task}: trial overlap between {a} and {b}.")
    folder = project_root / 'data/audit/protocol'
    folder.mkdir(parents=True, exist_ok=True)
    for filename, contents in (("protocol.json", protocol), ("data_split_manifest.json", split_manifest)):
        path = folder / filename
        if path.exists() and json.loads(path.read_text()) != _json_ready(contents):
            raise ValueError(f"Existing {filename} differs. Review and version the protocol before replacing it.")
        _write_json(path, contents)
    fold_path = folder / "part2_train_folds.csv"
    if fold_path.exists():
        # Retain the assignments used by saved OOF predictions across library versions.
        folds = pd.read_csv(fold_path)
        reference = task_data["part2"]["train"][["ID", "trial_id", "phase", "label"]].reset_index(drop=True)
        if (list(folds.columns) != [*reference.columns, "fold"] or
                not folds[list(reference.columns)].equals(reference)):
            raise ValueError("Saved folds do not match the current training rows and labels.")
        if (folds["fold"].isna().any() or set(folds["fold"]) != set(range(n_splits)) or
                folds.groupby("trial_id")["fold"].nunique().gt(1).any()):
            raise ValueError("Saved folds must contain every fold and keep each trial together.")
    else:
        folds = make_part2_grouped_folds(task_data["part2"]["train"], n_splits, seed)
        folds.to_csv(fold_path, index=False)
    fold_summary = folds.groupby(["fold", "phase"]).agg(
        rows=("ID", "size"), unique_trials=("trial_id", "nunique"), positives=("label", "sum"), success_rate=("label", "mean")
    ).reset_index()
    fold_summary.to_csv(folder / "part2_fold_summary.csv", index=False)
    log_dir = project_root / 'models/new_experiments'
    (log_dir / "runs").mkdir(parents=True, exist_ok=True)
    if not (log_dir / "experiment_log.csv").exists():
        pd.DataFrame(columns=REGISTRY_COLUMNS).to_csv(log_dir / "experiment_log.csv", index=False)
    return {"protocol": protocol, "split_manifest": split_manifest, "fold_summary": fold_summary}


class ExperimentLogger:
    """Save new runs separately from the curated historical experiment index."""

    def __init__(self, project_root: Path):
        self.root = Path(project_root).resolve()
        protocol_path = self.root / 'data/audit/protocol' / "protocol.json"
        if not protocol_path.exists():
            initialize_protocol(self.root)
        self.protocol = json.loads(protocol_path.read_text())
        self.manifest = json.loads((protocol_path.parent / "data_split_manifest.json").read_text())
        self.directory = self.root / 'models/new_experiments'
        (self.directory / "runs").mkdir(parents=True, exist_ok=True)

    def log_validation_run(
        self, *, task: str, probabilities: pd.DataFrame, model_name: str,
        feature_spec: Mapping[str, Any], hyperparameters: Mapping[str, Any],
        fit_trial_ids: Mapping[str, Sequence[str]], hypothesis: str, notes: str,
        seed: int = 42, thresholds: float | Mapping[str, float] = 0.5, run_id: str | None = None,
    ) -> Path:
        if task not in ("part1", "part2"):
            raise ValueError("task must be part1 or part2.")
        if not model_name.strip() or not hypothesis.strip() or not notes.strip():
            raise ValueError("Supply model_name, hypothesis, and notes for each completed run.")
        _validate_seed(seed)
        data = {}
        for split in ("train", "val"):
            path = dataset_path(self.root, task, split)
            if file_sha256(path) != self.manifest[f"{task}_{split}"]["sha256"]:
                raise ValueError(f"{task}_{split} changed after protocol initialization.")
            data[split] = pd.read_csv(path)
        input_columns = feature_spec.get("input_columns", [])
        if not isinstance(input_columns, (list, tuple)) or not input_columns:
            raise ValueError("feature_spec must declare a nonempty input_columns list of raw feature names.")
        forbidden = {"ID", "trial_id", "label", *REASONS}
        if task == "part2":
            forbidden |= {f"emb_{i}" for i in range(150)}
        bad_columns = set(input_columns) & forbidden
        if bad_columns or not set(input_columns).issubset(data["train"].columns):
            raise ValueError(f"Invalid or forbidden input columns: {sorted(bad_columns | (set(input_columns)-set(data['train'].columns)))}.")
        group_column = "ID" if task == "part1" else "trial_id"
        if not fit_trial_ids:
            raise ValueError("Declare fit_trial_ids separately for the model and every learned preprocessor.")
        scope = {component: validate_fit_scope(ids, data["train"][group_column]) for component, ids in fit_trial_ids.items()}
        evaluator = evaluate_part1 if task == "part1" else evaluate_part2
        result = evaluator(data["val"], probabilities, thresholds)
        primary_metric = self.protocol[task]["primary_metric"]
        primary_value = result.summary[primary_metric]
        if not np.isfinite(primary_value):
            raise ValueError("The primary validation metric is undefined; investigate before logging a run.")
        now = datetime.now(timezone.utc)
        run_id = run_id or f"{task}_{now.strftime('%Y%m%dT%H%M%SZ')}_{uuid.uuid4().hex[:8]}"
        if not re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9._-]{0,79}", run_id):
            raise ValueError("run_id must be a safe filename of 1-80 characters.")
        destination = self.directory / "runs" / run_id
        if destination.exists():
            raise FileExistsError(f"Run already exists: {run_id}")
        config = {"run_id": run_id, "created_at_utc": now.isoformat(), "task": task, "split": "validation",
                  "model_name": model_name, "feature_spec": feature_spec, "hyperparameters": hyperparameters,
                  "seed": seed, "thresholds": result.thresholds, "hypothesis": hypothesis, "notes": notes,
                  "declared_fit_scope": scope, "protocol": self.protocol,
                  "input_files": {f"{task}_{split}": self.manifest[f"{task}_{split}"] for split in ("train", "val")},
                  "evaluation_module_sha256": file_sha256(Path(__file__))}
        staged = Path(tempfile.mkdtemp(prefix=".pending_", dir=self.directory))
        try:
            _write_json(staged / "config.json", config)
            result.save(staged)
            staged.rename(destination)
        except Exception:
            shutil.rmtree(staged, ignore_errors=True)
            raise
        self.rebuild_registry()
        return destination

    def rebuild_registry(self) -> pd.DataFrame:
        """Rebuild the summary from saved runs so it can be recovered without losing history."""
        records = []
        for folder in sorted((self.directory / "runs").iterdir()):
            if not folder.is_dir():
                continue
            config = json.loads((folder / "config.json").read_text())
            metrics = json.loads((folder / "metrics.json").read_text())
            primary = config["protocol"][config["task"]]
            record = {key: config[key] for key in ("run_id", "created_at_utc", "task", "model_name", "seed", "hypothesis", "notes")}
            record.update({f"{key}_json": json.dumps(config[key], sort_keys=True) for key in ("feature_spec", "hyperparameters", "thresholds")})
            record.update(validation_rows=metrics["summary"]["n_rows"], primary_metric=primary["primary_metric"],
                          primary_value=metrics["summary"][primary["primary_metric"]], higher_is_better=primary["higher_is_better"],
                          run_directory=str(folder.relative_to(self.root)))
            records.append(record)
        registry = pd.DataFrame(records, columns=REGISTRY_COLUMNS)
        temporary = self.directory / f".experiment_log_{uuid.uuid4().hex}.tmp"
        registry.to_csv(temporary, index=False)
        temporary.replace(self.directory / "experiment_log.csv")
        return registry

```


### tools/src/midterm_finalization.py

SHA-256: `0c1a5d8be2f180dbb772d0ac31359f89978e7fb26a11e15b87a7839a989a7cf8`

```python
"""Predict historical/final outcome bundles and dispatch the final closeout."""
from pathlib import Path
import json
import numpy as np
import pandas as pd
from midterm_evaluation import file_sha256
from midterm_part2_baseline import predict_outcome_probabilities
from midterm_refinement import predict_field_outcomes

WORD_RUN = 'part2_equal_probability_blend_seed42_v1'
FINAL_RUN = 'part2_word_character_blend_seed42_v1'
FIELD_RUN = 'field_tfidf_lr_c1_v1'


def write_json(path,value):
    Path(path).write_text(json.dumps(value,indent=2,allow_nan=False)+'\n')


def check_run(directory, checksum_name):
    directory=Path(directory)
    for name,digest in json.loads((directory/checksum_name).read_text()).items():
        if Path(name).is_absolute() or '..' in Path(name).parts or file_sha256(directory/name)!=digest:
            raise AssertionError(f'Immutable experiment changed: {directory.name}/{name}')


def blend_probabilities(reference, components, weights):
    """Align components by ID before combining success probabilities."""
    w=np.asarray(weights,dtype=float)
    if len(components)!=len(w) or not np.isfinite(w).all() or (w<0).any() or not np.isclose(w.sum(),1):
        raise ValueError('Blend weights must be finite, nonnegative, and sum to one.')
    if reference.ID.isna().any() or not reference.ID.is_unique:raise ValueError('Reference IDs must be unique.')
    scores=[]
    for component in components:
        if component.ID.isna().any() or not component.ID.is_unique or set(component.ID)!=set(reference.ID):
            raise ValueError('Component ID sets must match the reference exactly.')
        p=component.set_index('ID').loc[reference.ID,'p_success'].to_numpy(dtype=float)
        if not np.isfinite(p).all() or ((p<0)|(p>1)).any():raise ValueError('Invalid component probabilities.')
        scores.append(p)
    return pd.DataFrame({'ID':reference.ID.to_numpy(),'p_success':np.sum(w[:,None]*np.array(scores),axis=0)})


def predict_final_outcomes(bundle,frame):
    """Preserve old word-only inference and support the promoted character blend."""
    if bundle.get('positive_class')!=1 or bundle.get('reason_features_used') is not False:
        raise ValueError('Unexpected final outcome bundle.')
    if bundle.get('weights')!=[.5,.5]:
        raise ValueError('The selected nested blend uses fixed equal weights.')
    kind=bundle.get('kind')
    if kind=='equal_probability_blend':
        parts=[predict_outcome_probabilities(bundle['baseline'],frame),predict_field_outcomes(bundle['field'],frame)]
    elif kind=='word_character_probability_blend':
        if bundle.get('flat_weights')!={'concatenated_word':.25,'separate_field_word':.25,'character':.5}:
            raise ValueError('The promoted blend has fixed 25/25/50 component weights.')
        parts=[predict_final_outcomes(bundle['word_blend'],frame),predict_outcome_probabilities(bundle['character'],frame)]
    else:
        raise ValueError('Unexpected final outcome bundle kind.')
    return blend_probabilities(frame,parts,bundle['weights'])


def integrate_final_models(root):
    raise RuntimeError(
        "The historical fifteen-notebook integration is stored in "
        "../archive/midterm_before_cleanup_2026-11-02.zip. "
        "Use python reproduce.py for this curated project."
    )

```


### tools/src/midterm_history.py

SHA-256: `f19bcecba725b246e88ed5af9551a2a6b0e38fd12b003417c2f08dc2573aa6dc`

```python
"""Resolve frozen evidence after an authorized update to current final outputs."""
from pathlib import Path
import json
from midterm_evaluation import file_sha256


def history_manifest(root):
    path=Path(root)/'history/manifest.json'
    return json.loads(path.read_text()) if path.exists() else {'archived':{},'regenerated':{}}


def historical_path(root, relative):
    root=Path(root);relative=str(relative)
    if Path(relative).is_absolute() or '..' in Path(relative).parts:
        raise ValueError('History paths must remain inside the project.')
    record=history_manifest(root)['archived'].get(relative)
    if record is None:return root/relative
    target=root/record['archive_path']
    if not str(target.resolve()).startswith(str((root/'history').resolve())+'/'):
        raise ValueError('Invalid archive location.')
    if file_sha256(target)!=record['sha256']:
        raise AssertionError(f'Archived evidence changed: {relative}')
    return target


def generated_archive_exception(root, relative, digest):
    """Only explicitly recorded prior delivery ZIPs may omit archived bytes."""
    if str(relative) != 'deliverables/midterm_code.zip':
        return False
    record=history_manifest(root).get('regenerated',{}).get(str(relative),{})
    return digest in [record.get('before_sha256'),*record.get('known_prior_sha256',[])]


def resolve_expected_path(root, relative, digest):
    """Resolve an exact historical digest, retaining multiple prior versions."""
    root=Path(root);relative=str(relative);path=Path(relative)
    if path.is_absolute() or '..' in path.parts:
        raise ValueError('History paths must remain inside the project.')
    current=root/path
    if current.is_file() and file_sha256(current)==digest:
        return current
    manifest=history_manifest(root)
    record=manifest.get('archived',{}).get(relative,{})
    archive_path=record.get('archive_path') if record.get('sha256')==digest else None
    archive_path=manifest.get('versions',{}).get(relative,{}).get(digest,archive_path)
    if archive_path:
        target=root/archive_path
        if not target.resolve().is_relative_to((root/'history').resolve()):
            raise ValueError('Invalid archive location.')
        if not target.is_file() or file_sha256(target)!=digest:
            raise AssertionError(f'Archived evidence changed: {relative}')
        return target
    if not current.is_file():
        raise FileNotFoundError(current)
    raise AssertionError(f'Original file changed without an exact archived version: {relative}')


def verify_snapshot(root, snapshot):
    """Verify original bytes in place or in the declared pre-integration archive."""
    root=Path(root);manifest=history_manifest(root)
    counts={'verified_current':0,'verified_archived':0,'excluded_regenerated_archive':0}
    for relative,digest in snapshot.items():
        if generated_archive_exception(root,relative,digest):
            counts['excluded_regenerated_archive']+=1
            continue
        path=resolve_expected_path(root,relative,digest)
        if file_sha256(path)!=digest:raise AssertionError(f'Original file changed: {relative}')
        counts['verified_current' if path==root/relative else 'verified_archived']+=1
    return {'status':'PASS','snapshot_entries':len(snapshot),**counts}


def baseline_model_path(root):
    return Path(root)/'models/part2/01_tfidf_lr/model.joblib'


def augmentation_metadata_compatible(saved,current):
    """Separate code provenance and CPU discovery warnings from fitted-result identity.

    Callers additionally require identical parameters, saved labels, predictions,
    and artifact checksums. Solver warnings and all other metadata remain strict.
    """
    import math
    left=dict(saved);right=dict(current)
    provenance=['training_code_sha256','augmentation_module_sha256']
    for key in provenance:
        left.pop(key,None);right.pop(key,None)
    for metadata in [left,right]:
        for key in ['cluster_warnings','classifier_warnings']:
            metadata[key]=[warning for warning in metadata.get(key,[])
                if not warning.startswith('Could not find the number of physical cores for the following reason:')]
    key='baseline_control_max_probability_difference'
    a=left.pop(key,None);b=right.pop(key,None)
    numeric=(isinstance(a,(float,int)) and isinstance(b,(float,int)) and
             math.isfinite(a) and math.isfinite(b) and max(abs(a),abs(b))<=1e-10 and abs(a-b)<=1e-12)
    return numeric and left==right

```


### tools/src/midterm_next_round.py

SHA-256: `aa278bf974a96c2ff1d14ef3707168d499c6213a75cf63511eded0e12ec9177e`

```python
"""Training-only OOF comparisons for pseudo-label weights and character text.

Run explicitly with --allow-training once. Later calls verify and reuse the
completed study. Current final models, submissions, and delivery files are
protected by checksums throughout this separate candidate study.
"""
from __future__ import annotations

import argparse
import html
import json
import os
from pathlib import Path
import platform
import warnings

import joblib
import numpy as np
import pandas as pd
import sklearn
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from threadpoolctl import threadpool_limits

from midterm_augmentation import logistic_parameters, observed_label_mask
from midterm_evaluation import (
    PHASES, REASONS, dataset_path, evaluate_part1, evaluate_part2,
    file_sha256, part1_decisions,
)
from midterm_finalization import blend_probabilities, predict_final_outcomes
from midterm_part1_finalization import build_submission
from midterm_part2_baseline import fit_phase_baseline, predict_outcome_probabilities
from midterm_part2_improvement import validate_submission
from midterm_refinement import (
    EMBEDDINGS, TFIDF, fit_field_outcomes, predict_field_outcomes,
    predict_reasons, prepare_reason_context,
)

STUDY = "refinement/oof_weight_char"
P1_WEIGHTS = {"weight_000": 0.0, "weight_003": 0.03, "weight_010": 0.1}
P1_CONTROL = "weight_010"
P2_CONTROL = "current_equal_blend"
P2_NEW = ("character", "current_plus_character")
CHAR_TFIDF = {
    "analyzer": "char_wb", "ngram_range": (3, 5), "min_df": 3,
    "max_features": 30000, "lowercase": True, "strip_accents": "unicode",
    "sublinear_tf": True, "smooth_idf": True, "use_idf": True, "norm": "l2",
}


def write_json(path, value):
    Path(path).write_text(json.dumps(value, indent=2, allow_nan=False) + "\n")


def read_json(path):
    return json.loads(Path(path).read_text())


def directory_hashes(directory):
    directory = Path(directory)
    return {p.relative_to(directory).as_posix(): file_sha256(p)
            for p in sorted(directory.rglob("*")) if p.is_file()
            and "__pycache__" not in p.parts and p.name != "completed_checksums.json"}


def verify_hashes(directory, hashes):
    directory = Path(directory)
    for relative, digest in hashes.items():
        path = Path(relative)
        if path.is_absolute() or ".." in path.parts:
            raise ValueError("Unsafe artifact path in checksum manifest.")
        if not (directory / path).is_file() or file_sha256(directory / path) != digest:
            from midterm_history import generated_archive_exception, resolve_expected_path
            if generated_archive_exception(directory,relative,digest):
                continue
            try:
                resolve_expected_path(directory,relative,digest)
            except (AssertionError,FileNotFoundError) as error:
                raise ValueError(f"Protected or completed artifact changed: {relative}") from error


def protected_hashes(root):
    """Cover current finals, registered runs, old studies, and hand-in files."""
    folders = ['models/part1/02_kmeans_lr/evidence/selected', 'models/part2/02_text_blend/evidence/selection', 'models', "submissions",
               "deliverables", "report", "refinement"]
    files = [root / 'tools/archive/records/final_selection_nov01.json']
    study = root / STUDY
    for folder in folders:
        files.extend(p for p in (root / folder).rglob("*")
                     if p.is_file() and not p.is_relative_to(study)
                     and "__pycache__" not in p.parts)
    return {p.relative_to(root).as_posix(): file_sha256(p) for p in sorted(set(files))}


def validate_fold_scope(fit, held, *, part):
    """Assert held-out identities are absent before any estimator is fitted."""
    if set(fit.ID) & set(held.ID):
        raise ValueError("Held-out row IDs overlap fitting data.")
    if part == 2 and set(fit.trial_id) & set(held.trial_id):
        raise ValueError("A trial crosses a training/held-out fold boundary.")
    return {
        "fit_row_ids": fit.ID.tolist(), "held_out_row_ids": held.ID.tolist(),
        "fit_trial_ids": sorted(fit.trial_id.unique()) if part == 2 else fit.ID.tolist(),
        "held_out_trial_ids": sorted(held.trial_id.unique()) if part == 2 else held.ID.tolist(),
        "row_overlap": 0, "trial_overlap": 0,
        "all_transforms_fitted_after_exclusion": True,
    }


def verify_oof(reference, predictions, assignment, *, part):
    """Require exact coverage, original labels, and one assigned fold per row."""
    if not predictions.ID.is_unique or set(predictions.ID) != set(reference.ID):
        raise ValueError("OOF predictions must cover every reference row exactly once.")
    if not assignment.ID.is_unique or set(assignment.ID) != set(reference.ID):
        raise ValueError("Fold assignments must cover exactly the reference rows.")
    ordered = predictions.set_index("ID").loc[reference.ID].reset_index()
    expected = assignment.set_index("ID").loc[reference.ID, "fold"].to_numpy()
    if not np.array_equal(ordered.fold.to_numpy(), expected):
        raise ValueError("OOF predictions have incorrect fold assignments.")
    if part == 1:
        truth = ordered[[f"true_{r}" for r in REASONS]].to_numpy()
        if not np.array_equal(truth, reference[list(REASONS)].to_numpy()):
            raise ValueError("OOF reason labels differ from the original gold labels.")
        scores = ordered[[f"prob_{r}" for r in REASONS]].to_numpy()
    else:
        if not np.array_equal(ordered.label.to_numpy(), reference.label.to_numpy()):
            raise ValueError("OOF outcome labels changed.")
        if not np.array_equal(ordered.trial_id, reference.trial_id) or not np.array_equal(ordered.phase, reference.phase):
            raise ValueError("OOF trial or phase identities changed.")
        if ordered.groupby("trial_id").fold.nunique().gt(1).any():
            raise ValueError("Combined-phase trial rows must stay in the same fold.")
        scores = ordered.p_success.to_numpy()
    if not np.isfinite(scores).all() or ((scores < 0) | (scores > 1)).any():
        raise ValueError("OOF probabilities must be finite and inside [0, 1].")
    return ordered


def validate_grouped_assignment(train, assignment):
    """Use saved folds as authoritative across sklearn splitter versions."""
    if not assignment.ID.is_unique or set(assignment.ID) != set(train.ID):
        raise ValueError("Saved grouped folds do not cover the training rows exactly.")
    assignment = assignment.set_index("ID").loc[train.ID].reset_index()
    pd.testing.assert_frame_equal(assignment[["ID", "trial_id", "phase", "label"]],
                                  train[["ID", "trial_id", "phase", "label"]].reset_index(drop=True))
    if set(assignment.fold) != set(range(5)) or assignment.groupby("trial_id").fold.nunique().gt(1).any():
        raise ValueError("Saved folds must keep each trial in exactly one of five folds.")
    for fold in range(5):
        for mask in [assignment.fold.eq(fold), ~assignment.fold.eq(fold)]:
            subset = assignment.loc[mask]
            if set(subset.phase) != set(PHASES) or not subset.groupby("phase").label.nunique().eq(2).all():
                raise ValueError("Every training and held-out phase requires both outcomes.")
    return assignment


def reason_error_counts(evaluation):
    rows = []
    for reason in REASONS:
        y = evaluation.predictions[f"true_{reason}"].to_numpy()
        p = evaluation.predictions[f"pred_{reason}"].to_numpy()
        rows.append({"reason": reason, "tp": int(((y == 1) & (p == 1)).sum()),
                     "fp": int(((y == 0) & (p == 1)).sum()),
                     "fn": int(((y == 1) & (p == 0)).sum())})
    return evaluation.by_group.merge(pd.DataFrame(rows), on="reason", validate="one_to_one")


def fit_reason_weight(frame, context, weight):
    """Use identical hard K-Means pseudo targets; vary only their row weight."""
    if not np.isfinite(weight) or weight < 0:
        raise ValueError("Pseudo-label weight must be finite and nonnegative.")
    observed = context["observed"]
    x = context["x"]
    gold = frame.loc[observed, list(REASONS)].to_numpy(dtype=int)
    pseudo = (context["pseudo_q"] >= .5).astype(int)
    if weight == 0:
        train_x, train_y = x[observed], gold
        weights = np.ones(len(gold))
    else:
        train_x = x
        train_y = frame[list(REASONS)].to_numpy(dtype=int).copy()
        train_y[~observed] = pseudo
        weights = np.where(observed, 1.0, weight)
    models = []
    for index in range(len(REASONS)):
        model = LogisticRegression(**{**logistic_parameters(42), "C": .1})
        with warnings.catch_warnings(record=True) as caught, threadpool_limits(limits=1):
            warnings.simplefilter("always")
            model.fit(train_x, train_y[:, index], sample_weight=weights)
        if any(issubclass(w.category, ConvergenceWarning) for w in caught) or model.n_iter_.max() >= 500:
            raise RuntimeError("A reason classifier did not converge.")
        models.append(model)
    return {"kind": "kmeans_weight_ablation", "pseudo_weight": weight,
            "scaler": context["scaler"], "clusterer": context["clusterer"],
            "mapping": context["mapping"], "models": models, "threshold": .35,
            "empty_set_policy": "argmax", "fit_ids": frame.ID.tolist(),
            "seed_ids": frame.loc[observed, "ID"].tolist()}


def p1_assignment(train):
    observed = observed_label_mask(train[list(REASONS)].to_numpy())
    gold = train.loc[observed].reset_index(drop=True)
    strata = gold[list(REASONS)].to_numpy().argmax(axis=1)
    if np.bincount(strata).min() < 4:
        raise ValueError("Four-fold first-positive stratification lacks support.")
    folds = np.full(len(gold), -1)
    for fold, (_, held) in enumerate(StratifiedKFold(n_splits=4, shuffle=True, random_state=42).split(gold, strata)):
        folds[held] = fold
    return gold, gold[["ID"]].assign(fold=folds)


def finish_cache(directory):
    write_json(directory / "completed_checksums.json", directory_hashes(directory))


def cache_complete(directory):
    manifest = directory / "completed_checksums.json"
    if not manifest.exists():
        return False
    verify_hashes(directory, read_json(manifest))
    return True


def run_p1_oof(root, folder, train):
    gold, assignment = p1_assignment(train)
    assignment.to_csv(folder / "fold_assignments.csv", index=False)
    fold_metrics, frames = [], {key: [] for key in P1_WEIGHTS}
    for fold in range(4):
        cache = folder / "fold_cache" / f"fold_{fold}"
        if not cache_complete(cache):
            cache.mkdir(parents=True, exist_ok=True)
            held_ids = set(assignment.loc[assignment.fold.eq(fold), "ID"])
            fit = train.loc[~train.ID.isin(held_ids)].reset_index(drop=True)
            held = gold.loc[gold.ID.isin(held_ids)].reset_index(drop=True)
            scope = validate_fold_scope(fit, held, part=1)
            scope["label_seed_ids"] = fit.loc[observed_label_mask(fit[list(REASONS)].to_numpy()), "ID"].tolist()
            context = prepare_reason_context(fit, 10)
            if set(context["seed_ids"]) & held_ids or set(context["fit_ids"]) & held_ids:
                raise AssertionError("Held-out rows entered scaling, clustering, or mapping.")
            write_json(cache / "fit_scope.json", scope)
            for setting, weight in P1_WEIGHTS.items():
                bundle = fit_reason_weight(fit, context, weight)
                result = evaluate_part1(held, predict_reasons(bundle, held), .35, empty_set_policy="argmax")
                result.predictions.assign(fold=fold).to_csv(cache / f"{setting}.csv", index=False)
                write_json(cache / f"{setting}_metrics.json", result.summary)
            finish_cache(cache)
        for setting in P1_WEIGHTS:
            frames[setting].append(pd.read_csv(cache / f"{setting}.csv"))
            fold_metrics.append({"setting": setting, "fold": fold, **read_json(cache / f"{setting}_metrics.json")})
        print(f"Part 1 OOF fold {fold + 1}/4 complete", flush=True)
    fold_table = pd.DataFrame(fold_metrics)
    fold_table.to_csv(folder / "fold_metrics.csv", index=False)
    comparisons, by_reason, evaluations = [], [], {}
    for setting, weight in P1_WEIGHTS.items():
        oof = verify_oof(gold, pd.concat(frames[setting], ignore_index=True), assignment, part=1)
        scores = oof[["ID", *[f"prob_{r}" for r in REASONS]]].rename(columns={f"prob_{r}": r for r in REASONS})
        evaluation = evaluate_part1(gold, scores, .35, empty_set_policy="argmax")
        evaluations[setting] = evaluation
        oof.to_csv(folder / f"oof_{setting}.csv", index=False)
        selected_folds = fold_table.loc[fold_table.setting.eq(setting)]
        comparisons.append({"setting": setting, "pseudo_weight": weight, **evaluation.summary,
                            "mean_fold_f1": selected_folds.f1_samples.mean(),
                            "std_fold_f1": selected_folds.f1_samples.std()})
        by_reason.append(reason_error_counts(evaluation).assign(setting=setting))
    table = pd.DataFrame(comparisons)
    table["delta_f1_vs_control"] = table.f1_samples - evaluations[P1_CONTROL].summary["f1_samples"]
    table.to_csv(folder / "oof_comparison.csv", index=False)
    pd.concat(by_reason).to_csv(folder / "oof_by_reason.csv", index=False)
    # Ties preserve the incumbent, then prefer the smaller change in weight.
    winner = max(P1_WEIGHTS, key=lambda s: (evaluations[s].summary["f1_samples"], P1_WEIGHTS[s]))
    lock = {"setting": winner, "pseudo_weight": P1_WEIGHTS[winner], "metric": "pooled_OOF_sample_F1",
            "score": evaluations[winner].summary["f1_samples"],
            "validation_labels_used": False, "test_predictions_used": False,
            "folds": 4, "tie_policy": "Prefer the larger weight, including the incumbent 0.1."}
    write_json(folder / "selection_locked_before_validation.json", lock)
    return lock, evaluations


def fit_character(train):
    return fit_phase_baseline(train, tfidf_parameters=CHAR_TFIDF, classifier_parameters={"C": 1.0}, seed=42)


def p2_fold_predictions(fit, held):
    """Fit all three representations inside this grouped training fold."""
    concat, _, _, _ = fit_phase_baseline(
        fit, tfidf_parameters={**TFIDF, "max_features": 20000}, classifier_parameters={"C": 1.0}, seed=42)
    field, _ = fit_field_outcomes(fit)
    character, diagnostics, fit_ids, caught = fit_character(fit)
    parts = {"concatenated_word": predict_outcome_probabilities(concat, held),
             "separate_field_word": predict_field_outcomes(field, held),
             "character": predict_outcome_probabilities(character, held)}
    parts[P2_CONTROL] = blend_probabilities(held, [parts["concatenated_word"], parts["separate_field_word"]], [.5, .5])
    parts["current_plus_character"] = blend_probabilities(held, [parts[P2_CONTROL], parts["character"]], [.5, .5])
    return parts, diagnostics, {"character_fit_ids": fit_ids, "character_warnings": caught}


def run_p2_oof(root, folder, train):
    assignment = validate_grouped_assignment(train, pd.read_csv(root / 'data/audit/protocol/part2_train_folds.csv'))
    assignment.to_csv(folder / "fold_assignments.csv", index=False)
    settings = ("concatenated_word", "separate_field_word", P2_CONTROL, *P2_NEW)
    frames, metrics, phase_metrics = {s: [] for s in settings}, [], []
    for fold in range(5):
        cache = folder / "fold_cache" / f"fold_{fold}"
        if not cache_complete(cache):
            cache.mkdir(parents=True, exist_ok=True)
            held_ids = set(assignment.loc[assignment.fold.eq(fold), "ID"])
            fit = train.loc[~train.ID.isin(held_ids)].reset_index(drop=True)
            held = train.loc[train.ID.isin(held_ids)].reset_index(drop=True)
            scope = validate_fold_scope(fit, held, part=2)
            scope["feature_columns"] = ["criteria", "diseases_cleaned", "drugs"]
            scope["reason_features_used"] = False
            parts, diagnostics, fit_metadata = p2_fold_predictions(fit, held)
            write_json(cache / "fit_scope.json", {**scope, **fit_metadata})
            diagnostics.to_csv(cache / "character_fit_diagnostics.csv", index=False)
            for setting, probabilities in parts.items():
                result = evaluate_part2(held, probabilities, .5)
                result.predictions.assign(fold=fold).to_csv(cache / f"{setting}.csv", index=False)
                write_json(cache / f"{setting}_metrics.json", result.summary)
                result.by_group.to_csv(cache / f"{setting}_by_phase.csv", index=False)
            finish_cache(cache)
        for setting in settings:
            frames[setting].append(pd.read_csv(cache / f"{setting}.csv"))
            metrics.append({"setting": setting, "fold": fold, **read_json(cache / f"{setting}_metrics.json")})
            phase_metrics.append(pd.read_csv(cache / f"{setting}_by_phase.csv").assign(setting=setting, fold=fold))
        print(f"Part 2 grouped OOF fold {fold + 1}/5 complete", flush=True)
    fold_table = pd.DataFrame(metrics)
    fold_table.to_csv(folder / "fold_metrics.csv", index=False)
    pd.concat(phase_metrics).to_csv(folder / "fold_metrics_by_phase.csv", index=False)
    evaluations, comparisons, by_phase = {}, [], []
    for setting in settings:
        oof = verify_oof(train, pd.concat(frames[setting], ignore_index=True), assignment, part=2)
        result = evaluate_part2(train, oof[["ID", "p_success"]], .5)
        evaluations[setting] = result
        oof.to_csv(folder / f"oof_{setting}.csv", index=False)
        selected_folds = fold_table.loc[fold_table.setting.eq(setting)]
        comparisons.append({"setting": setting, **result.summary,
                            "mean_fold_log_loss": selected_folds.log_loss_equal_phase.mean(),
                            "std_fold_log_loss": selected_folds.log_loss_equal_phase.std()})
        by_phase.append(result.by_group.assign(setting=setting))
    table = pd.DataFrame(comparisons)
    table["delta_log_loss_vs_control"] = table.log_loss_equal_phase - evaluations[P2_CONTROL].summary["log_loss_equal_phase"]
    table.to_csv(folder / "oof_comparison.csv", index=False)
    pd.concat(by_phase).to_csv(folder / "oof_by_phase.csv", index=False)
    # Word-only components are diagnostics; new candidates are char or the fixed blend.
    eligible = (P2_CONTROL, *P2_NEW)
    winner = min(eligible, key=lambda s: (evaluations[s].summary["log_loss_equal_phase"], eligible.index(s)))
    lock = {"setting": winner, "metric": "equal_phase_log_loss_from_all_OOF_rows",
            "score": evaluations[winner].summary["log_loss_equal_phase"],
            "eligible_settings": list(eligible), "validation_labels_used": False,
            "test_predictions_used": False, "folds": 5,
            "character_weight_in_new_blend": .5, "blend_weight_search": False}
    write_json(folder / "selection_locked_before_validation.json", lock)
    return lock, evaluations


def predict_candidate_outcomes(bundle, frame):
    """Loadable prediction entry point for this study's outcome candidates."""
    if bundle.get("kind") == "character_only":
        return predict_outcome_probabilities(bundle["character"], frame)
    if bundle.get("kind") == "current_plus_character" and bundle.get("weights") == [.5, .5]:
        return blend_probabilities(frame, [predict_final_outcomes(bundle["current"], frame),
            predict_outcome_probabilities(bundle["character"], frame)], [.5, .5])
    raise ValueError("Unexpected candidate outcome bundle.")


def row_f1(result):
    y = result.predictions[[f"true_{r}" for r in REASONS]].to_numpy()
    d = result.predictions[[f"pred_{r}" for r in REASONS]].to_numpy()
    return 2 * (y * d).sum(axis=1) / (y.sum(axis=1) + d.sum(axis=1))


def paired_diagnostics(before, after, *, part):
    """Describe paired differences; intervals exclude selection uncertainty."""
    left, right = before.predictions, after.predictions
    if not np.array_equal(left.ID, right.ID):
        raise ValueError("Paired diagnostics require aligned IDs.")
    rng = np.random.default_rng(42)
    if part == 1:
        delta = row_f1(after) - row_f1(before)
        draws = delta[rng.integers(0, len(delta), size=(2000, len(delta)))].mean(axis=1)
        value = float(delta.mean())
        details = {"rows_improved": int((delta > 1e-12).sum()), "rows_worsened": int((delta < -1e-12).sum())}
    else:
        y = left.label.to_numpy()
        p0, p1 = [np.clip(frame.p_success.to_numpy(), 1e-15, 1 - 1e-15) for frame in [left, right]]
        loss0 = -(y * np.log(p0) + (1 - y) * np.log1p(-p0))
        loss1 = -(y * np.log(p1) + (1 - y) * np.log1p(-p1))
        delta = loss1 - loss0
        ids = sorted(left.trial_id.unique())
        sums = np.zeros((len(ids), 3)); counts = np.zeros_like(sums)
        trial_index = {trial: i for i, trial in enumerate(ids)}
        for i, phase in enumerate(PHASES):
            subset = np.flatnonzero(left.phase.to_numpy() == phase)
            positions = np.array([trial_index[t] for t in left.iloc[subset].trial_id])
            np.add.at(sums[:, i], positions, delta[subset])
            np.add.at(counts[:, i], positions, 1)
        draws = []
        for _ in range(2000):
            sampled = rng.integers(0, len(ids), size=len(ids))
            denominator = counts[sampled].sum(axis=0)
            if (denominator > 0).all():
                draws.append(float((sums[sampled].sum(axis=0) / denominator).mean()))
        value = after.summary["log_loss_equal_phase"] - before.summary["log_loss_equal_phase"]
        details = {"bootstrap_unit": "trial_id, preserving all phase rows"}
    return {"delta": value, "descriptive_95_percentile_interval": np.quantile(draws, [.025, .975]).tolist(),
            "bootstrap_draws": 2000, **details,
            "limitation": "Descriptive only; repeated validation use, model selection, and nonrandom label sampling are not accounted for."}


def full_p1(root, folder, train, lock, oof):
    validation = pd.read_csv(dataset_path(root, "part1", "val"))
    current = pd.read_csv(root / 'models/part1/02_kmeans_lr/evidence/selected/selected_validation/validation_predictions.csv')
    current_scores = current[["ID", *[f"prob_{r}" for r in REASONS]]].rename(columns={f"prob_{r}": r for r in REASONS})
    control = evaluate_part1(validation, current_scores, .35, empty_set_policy="argmax")
    context = prepare_reason_context(train, 10)
    pseudo = train.loc[~context["observed"], ["ID"]].copy()
    q = context["pseudo_q"]
    for i, reason in enumerate(REASONS):
        pseudo[f"cluster_rate_{reason}"] = q[:, i]
        pseudo[f"pseudo_{reason}"] = (q[:, i] >= .5).astype(int)
    pseudo["source"] = "training_only_kmeans_cluster_rate_threshold_0.5"
    pseudo.to_csv(folder / "all_masked_pseudo_labels.csv", index=False)
    pd.DataFrame({"reason": REASONS, "gold_positive_rows": train.loc[context["observed"], list(REASONS)].sum().to_numpy(),
                  "pseudo_positive_rows": (q >= .5).sum(axis=0), "masked_rows": len(q)}).to_csv(folder / "pseudo_label_support.csv", index=False)
    control_refit = fit_reason_weight(train, context, .1)
    check = predict_reasons(control_refit, validation)
    error = float(np.abs(check[list(REASONS)].to_numpy() - current_scores.set_index("ID").loc[validation.ID, list(REASONS)].to_numpy()).max())
    if error > 1e-6:
        raise ValueError(f"The weight-0.1 control does not reproduce the retained model: {error}")
    bundle = control_refit if lock["setting"] == P1_CONTROL else fit_reason_weight(train, context, lock["pseudo_weight"])
    candidate = evaluate_part1(validation, predict_reasons(bundle, validation), .35, empty_set_policy="argmax")
    joblib.dump(bundle, folder / "oof_selected_model.joblib", compress=3)
    control.save(folder / "validation" / P1_CONTROL)
    if lock["setting"] != P1_CONTROL:
        candidate.save(folder / "validation" / lock["setting"])
    rows = [{"setting": P1_CONTROL, **control.summary}]
    if lock["setting"] != P1_CONTROL:
        rows.append({"setting": lock["setting"], **candidate.summary})
    pd.DataFrame(rows).to_csv(folder / "validation_comparison.csv", index=False)
    pd.concat([reason_error_counts(control).assign(setting=P1_CONTROL),
               reason_error_counts(candidate).assign(setting=lock["setting"], role="OOF_selected")]).to_csv(folder / "validation_by_reason.csv", index=False)
    oof_gain = oof[lock["setting"]].summary["f1_samples"] - oof[P1_CONTROL].summary["f1_samples"]
    val_gain = candidate.summary["f1_samples"] - control.summary["f1_samples"]
    gate = {"oof_selected_setting": lock["setting"], "oof_f1_gain": oof_gain,
            "validation_f1_gain": val_gain, "control_refit_max_probability_difference": error,
            "passes_candidate_gate": bool(lock["setting"] != P1_CONTROL and oof_gain > 1e-12 and val_gain > 1e-12),
            "final_model_replaced": False, "public_score_known": False}
    write_json(folder / "paired_oof_diagnostics.json", paired_diagnostics(oof[P1_CONTROL], oof[lock["setting"]], part=1))
    write_json(folder / "paired_validation_diagnostics.json", paired_diagnostics(control, candidate, part=1))
    write_json(folder / "candidate_gate.json", gate)
    return gate, bundle


def full_p2(root, folder, train, lock, oof):
    validation = pd.read_csv(dataset_path(root, "part2", "val"))
    word_run = root / 'results/experiments/runs/part2_equal_probability_blend_seed42_v1'
    current_bundle = joblib.load(word_run / "model.joblib")
    current_prob = predict_final_outcomes(current_bundle, validation)
    saved = pd.read_csv(word_run / "validation_predictions.csv")
    error = float(np.abs(current_prob.p_success.to_numpy() - saved.set_index("ID").loc[validation.ID, "p_success"].to_numpy()).max())
    if error > 1e-12:
        raise ValueError("The current outcome model does not reproduce its saved validation predictions.")
    character, diagnostics, fit_ids, caught = fit_character(train)
    diagnostics.to_csv(folder / "full_character_fit_diagnostics.csv", index=False)
    write_json(folder / "full_character_fit_scope.json", {"fit_ids": fit_ids, "warnings": caught,
               "reason_features_used": False, "validation_rows_in_fit": 0})
    char_bundle = {"kind": "character_only", "character": character, "reason_features_used": False}
    blend_bundle = {"kind": "current_plus_character", "current": current_bundle,
                    "character": character, "weights": [.5, .5], "reason_features_used": False}
    bundles = {"character": char_bundle, "current_plus_character": blend_bundle}
    joblib.dump(char_bundle, folder / "character_model.joblib", compress=3)
    probabilities = {P2_CONTROL: current_prob, **{s: predict_candidate_outcomes(b, validation) for s, b in bundles.items()}}
    evaluations = {s: evaluate_part2(validation, p, .5) for s, p in probabilities.items()}
    for setting, result in evaluations.items():
        result.save(folder / "validation" / setting)
    pd.DataFrame([{"setting": s, **r.summary} for s, r in evaluations.items()]).to_csv(folder / "validation_comparison.csv", index=False)
    pd.concat([r.by_group.assign(setting=s) for s, r in evaluations.items()]).to_csv(folder / "validation_by_phase.csv", index=False)
    control, candidate = evaluations[P2_CONTROL], evaluations[lock["setting"]]
    oof_gain = oof[lock["setting"]].summary["log_loss_equal_phase"] - oof[P2_CONTROL].summary["log_loss_equal_phase"]
    val_gain = candidate.summary["log_loss_equal_phase"] - control.summary["log_loss_equal_phase"]
    gate = {"oof_selected_setting": lock["setting"], "oof_log_loss_delta": oof_gain,
            "validation_log_loss_delta": val_gain, "current_model_max_probability_difference": error,
            "passes_candidate_gate": bool(lock["setting"] != P2_CONTROL and oof_gain < -1e-12 and val_gain < -1e-12),
            "final_model_replaced": False, "public_score_known": False,
            "validation_policy": "All three predeclared settings are reported; only the training-OOF winner can pass the gate."}
    if lock["setting"] != P2_CONTROL:
        joblib.dump(bundles[lock["setting"]], folder / "oof_selected_model.joblib", compress=3)
    write_json(folder / "paired_oof_diagnostics.json", paired_diagnostics(oof[P2_CONTROL], oof[lock["setting"]], part=2))
    write_json(folder / "paired_validation_diagnostics.json", paired_diagnostics(control, candidate, part=2))
    # Reasons are joined after predicting, using the unchanged Part 1 lookup.
    from midterm_part2_baseline import analyze_failure_reasons
    lookup = pd.read_csv(root / 'models/part1/02_kmeans_lr/evidence/selected/reason_lookup.csv')
    before = analyze_failure_reasons(control.predictions, lookup)
    after = analyze_failure_reasons(candidate.predictions, lookup)
    for key in ["reason_rates", "source_rates", "overall_failed_rates"]:
        on = ["phase", "reason_group"] + (["source_type"] if key == "source_rates" else [])
        joined = before[key].merge(after[key], on=on, suffixes=("_current", "_candidate"), validate="one_to_one")
        if not np.array_equal(joined.n_trials_current, joined.n_trials_candidate):
            raise AssertionError("Failure-reason group support changed.")
        joined["delta_false_success_rate"] = joined.false_success_rate_candidate - joined.false_success_rate_current
        joined.to_csv(folder / f"validation_{key}_comparison.csv", index=False)
    write_json(folder / "candidate_gate.json", gate)
    return gate, bundles.get(lock["setting"])


def candidate_submissions(root, folder, gates, bundles):
    output = folder / "submissions"
    output.mkdir(exist_ok=True)
    paths = {}
    for task, gate in gates.items():
        if not gate["passes_candidate_gate"]:
            continue
        test = pd.read_csv(dataset_path(root, task, "test"))
        part = 1 if task == "part1" else 2
        sample = pd.read_csv(dataset_path(root, task, "test").parent / f"{task}_sample_submission.csv")
        if part == 1:
            scores = predict_reasons(bundles[task], test)
            decisions = pd.DataFrame(part1_decisions(scores[list(REASONS)], .35, empty_set_policy="argmax"), columns=REASONS)
            decisions.insert(0, "ID", test.ID.to_numpy())
            submission = build_submission(sample, decisions)
        else:
            scores = predict_candidate_outcomes(bundles[task], test)
            submission = sample[["ID"]].merge(scores, on="ID", validate="one_to_one", sort=False).rename(columns={"p_success": "label"})
        scores.to_csv(output / f"{task}_candidate_probabilities.csv", index=False)
        path = output / f"{task}_candidate.csv"
        submission.to_csv(path, index=False)
        validate_submission(pd.read_csv(path), sample, part)
        paths[task] = path.relative_to(root).as_posix()
    return paths


def make_report(folder, summary):
    """Create English tables and a standalone plot from persisted study results."""
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), constrained_layout=True)
    for ax, task, metric, title in zip(axes, ["part1", "part2"], ["f1_samples", "log_loss_equal_phase"],
                                     ["Part 1: training OOF sample F1 (higher is better)", "Part 2: grouped OOF log loss (lower is better)"]):
        table = pd.read_csv(folder / task / "oof_comparison.csv")
        colors = ["#007f8b" if s == summary[task]["oof_selected_setting"] else "#a4b9c7" for s in table.setting]
        bars = ax.barh(table.setting.str.replace("_", " "), table[metric], color=colors)
        ax.bar_label(bars, fmt="%.4f", padding=4, fontsize=9)
        ax.set_title(title, fontsize=11); ax.set_xlim(0, float(table[metric].max()) * 1.18)
        ax.invert_yaxis(); ax.spines[["top", "right"]].set_visible(False)
    fig.savefig(folder / "oof_comparison.png", dpi=180)
    plt.close(fig)
    sections = []
    for task in ["part1", "part2"]:
        sections.append(f"<h2>{html.escape(task.title())}</h2>")
        for name in ["oof_comparison", "validation_comparison", "validation_by_reason" if task == "part1" else "validation_by_phase"]:
            sections.append(f"<h3>{html.escape(name.replace('_', ' ').title())}</h3>" + pd.read_csv(folder / task / f"{name}.csv").to_html(index=False, float_format=lambda v: f"{v:.6f}"))
        sections.append("<pre>" + html.escape(json.dumps(summary[task], indent=2)) + "</pre>")
    page = """<!doctype html><html lang="en"><meta charset="utf-8"><title>OOF Weight and Character Study</title>
<style>body{font:16px system-ui;margin:36px;max-width:1350px;color:#203047}table{border-collapse:collapse;font-size:13px}td,th{padding:7px;border:1px solid #cbd5df}th{background:#edf2f6}pre{background:#f3f5f7;padding:16px;white-space:pre-wrap}img{max-width:100%}</style>
<h1>OOF Weight and Character Study</h1>
<p>All settings were fixed before this study's validation evaluation. Selection uses training-only out-of-fold predictions. Current final artifacts remain unchanged.</p>
<p>Part 1 uses 157 observed rows in four folds; held-out rows are excluded from scaling, K-Means, and label mapping. All 1,014 originally masked rows remain unlabeled inputs. Part 2 uses the original five folds grouped by trial_id, with vocabularies fitted inside each training fold. Reasons are used only after prediction for error analysis.</p>
<p>The Part 1 observed labels are nonrandom, and official validation and public scores have been consulted in earlier work. OOF selection and descriptive intervals do not remove these limitations. These candidates have no known public or private score.</p>
<img src="oof_comparison.png" alt="Training-only OOF comparisons">""" + "".join(sections)
    page += "<h2>Candidate submissions</h2><pre>" + html.escape(json.dumps(summary["candidate_submissions"], indent=2)) + "</pre>"
    page += "<p>A candidate CSV is generated only when the training-OOF winner also improves the predeclared validation metric. No model is automatically promoted and no submission is uploaded. The existing report and delivery ZIP remain the prior completed snapshot.</p></html>"
    (folder / "study_report.html").write_text(page)


def run_study(root, *, allow_training=False):
    root = Path(root).resolve()
    folder = root / STUDY
    if cache_complete(folder):
        verify_hashes(root, read_json(folder / "protected_artifacts.json"))
        verify_hashes(root, read_json(folder / "input_and_source_hashes.json"))
        return read_json(folder / "summary.json")
    if not allow_training:
        raise RuntimeError("The study is missing or incomplete. Run midterm_next_round.py --allow-training to fit the predeclared experiments.")
    os.environ.setdefault("LOKY_MAX_CPU_COUNT", "1")
    folder.mkdir(parents=True, exist_ok=True)
    for name in ["part1", "part2"]:
        (folder / name).mkdir(exist_ok=True)
    plan = {
        "study": STUDY, "seed": 42, "part1_weights": P1_WEIGHTS,
        "part1_C": .1, "part1_threshold": .35, "empty_set_policy": "argmax",
        "part1_kmeans_clusters": 10, "part1_smoothing": 2.0, "part1_folds": 4,
        "part1_selection": "pooled OOF sample F1; ties prefer the incumbent",
        "part2_folds": 5, "part2_group": "trial_id", "part2_C": 1.0,
        "character_tfidf": CHAR_TFIDF, "new_blend_weights": [.5, .5],
        "part2_selection": "equal-phase log loss on all OOF rows; ties preserve incumbent",
        "validation_policy": "P1 control and OOF winner; P2 three fixed settings reported, gate only the OOF winner",
        "candidate_gate": "Strict improvement in both OOF and official validation; no automatic promotion",
        "reason_features_used_in_part2": False, "public_feedback_used_for_new_selection": False,
        "existing_work_has_reused_validation_and_public_feedback": True,
        "python": platform.python_version(), "sklearn": sklearn.__version__,
    }
    plan = json.loads(json.dumps(plan))
    plan_path = folder / "experiment_plan.json"
    if plan_path.exists() and read_json(plan_path) != plan:
        raise ValueError("The declared plan changed; use a separate study directory.")
    write_json(plan_path, plan)
    sources = ["midterm_next_round.py", "midterm_refinement.py", "midterm_augmentation.py",
               "midterm_evaluation.py", "midterm_finalization.py", "midterm_part2_baseline.py",
               "midterm_part1_finalization.py", "midterm_part2_improvement.py",
               'data/audit/protocol/part2_train_folds.csv']
    files = [root / 'tools/src' / s for s in sources] + [dataset_path(root, t, split) for t in ["part1", "part2"] for split in ["train", "val", "test"]]
    inputs = {p.relative_to(root).as_posix(): file_sha256(p) for p in files}
    if (folder / "input_and_source_hashes.json").exists():
        if inputs != read_json(folder / "input_and_source_hashes.json"):
            raise ValueError("Inputs or source changed during an incomplete study.")
        verify_hashes(root, read_json(folder / "protected_artifacts.json"))
    else:
        write_json(folder / "input_and_source_hashes.json", inputs)
        write_json(folder / "protected_artifacts.json", protected_hashes(root))
    train1 = pd.read_csv(dataset_path(root, "part1", "train"))
    train2 = pd.read_csv(dataset_path(root, "part2", "train"))
    lock1, oof1 = run_p1_oof(root, folder / "part1", train1)
    lock2, oof2 = run_p2_oof(root, folder / "part2", train2)
    # Both selections are persisted before official validation labels are read.
    gate1, bundle1 = full_p1(root, folder / "part1", train1, lock1, oof1)
    gate2, bundle2 = full_p2(root, folder / "part2", train2, lock2, oof2)
    gates = {"part1": gate1, "part2": gate2}
    paths = candidate_submissions(root, folder, gates, {"part1": bundle1, "part2": bundle2})
    summary = {**gates, "candidate_submissions": paths,
               "protected_final_artifacts_unchanged": True,
               "no_validation_labels_used_in_fitting": True,
               "all_OOF_rows_predicted_once": True, "reason_features_used_in_part2": False,
               "public_or_private_scores_available_for_new_candidates": False,
               "uploads_performed": False, "report_and_delivery_zip_updated": False}
    make_report(folder, summary)
    verify_hashes(root, read_json(folder / "protected_artifacts.json"))
    verify_hashes(root, inputs)
    write_json(folder / "summary.json", summary)
    finish_cache(folder)
    return summary


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--project-root", type=Path, default=Path(__file__).resolve().parents[2])
    parser.add_argument("--allow-training", action="store_true")
    args = parser.parse_args()
    print(json.dumps(run_study(args.project_root, allow_training=args.allow_training), indent=2))


if __name__ == "__main__":
    raise SystemExit("Historical study runner archived. Use python tools/src/train_core.py --help for the six fixed recipes.")

```


### tools/src/midterm_part1_finalization.py

SHA-256: `86c39636b704ec2555ea624409185fcee3d18868e1e2f60279175355e06fddf4`

```python
"""Controlled Part 1 feature experiments, submission formatting, and reason provenance."""

from __future__ import annotations

from collections import Counter
from pathlib import Path
import json
import warnings

import joblib
import numpy as np
import pandas as pd
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression

from midterm_augmentation import logistic_parameters, observed_label_mask, predict_reason_probabilities
from midterm_evaluation import REASONS, evaluate_part1, file_sha256, part1_decisions


def validate_saved_configuration(config, *, parameters, input_columns, input_files, seed, thresholds):
    """Check model identity independently of output paths and source-code fingerprints.

    Historical source hashes and package versions remain in the original record.
    Reusing its frozen model does not rerun the historical training code. Data,
    features, parameters, seed, and decisions must still match the requested run.
    """
    expected = {"hyperparameters": parameters, "input_files": input_files, "seed": seed,
                "thresholds": dict(thresholds)}
    for field, value in expected.items():
        if config.get(field) != value:
            raise ValueError(f"Existing run has different {field}; choose a new run ID for a different experiment.")
    if config.get("feature_spec", {}).get("input_columns") != list(input_columns):
        raise ValueError("Existing run uses different input columns; choose a new run ID.")
    if config.get("task") != "part1" or config.get("split") != "validation":
        raise ValueError("Expected an existing Part 1 validation run.")


def load_saved_step5_run(directory, *, parameters, input_columns, input_files, seed,
                         thresholds, required_lineage, validation):
    """Verify and load an immutable run without calling any estimator's fit method."""
    directory = Path(directory)
    config = json.loads((directory / "config.json").read_text())
    validate_saved_configuration(config, parameters=parameters, input_columns=input_columns,
                                 input_files=input_files, seed=seed, thresholds=thresholds)
    manifest = json.loads((directory / "artifact_checksums.json").read_text())
    for name, digest in manifest.items():
        relative = Path(name)
        if relative.is_absolute() or ".." in relative.parts:
            raise ValueError("Invalid relative path in the artifact checksum manifest.")
        path = directory / relative
        if not path.is_file() or file_sha256(path) != digest:
            raise ValueError(f"An immutable artifact changed: {path}")
    lineage = json.loads((directory / "lineage.json").read_text())
    for key, value in required_lineage.items():
        if lineage.get(key) != value:
            raise ValueError(f"Existing run has different lineage field {key}.")
    bundle = joblib.load(directory / "model.joblib")
    if bundle.get("step5_parameters") != parameters or bundle["thresholds"] != dict(thresholds):
        raise ValueError("Saved model configuration does not match the requested run.")
    saved = pd.read_csv(directory / "validation_predictions.csv")
    if not saved["ID"].is_unique or set(saved["ID"]) != set(validation["ID"]):
        raise ValueError("Saved validation IDs do not match the fixed validation split.")
    saved = saved.set_index("ID").loc[validation["ID"]]
    probabilities = pd.DataFrame(saved[[f"prob_{r}" for r in REASONS]].to_numpy(), columns=REASONS)
    probabilities.insert(0, "ID", validation["ID"].to_numpy())
    actual = predict_bundle(bundle, validation)
    if not np.allclose(actual, probabilities[list(REASONS)], atol=1e-12, rtol=0):
        raise ValueError("Saved model does not reproduce its recorded validation probabilities.")
    result = evaluate_part1(validation, probabilities, thresholds)
    if not np.array_equal(result.predictions[[f"pred_{r}" for r in REASONS]],
                          saved[[f"pred_{r}" for r in REASONS]]):
        raise ValueError("Saved validation decisions differ from the requested thresholds.")
    metrics = json.loads((directory / "metrics.json").read_text())
    for key, value in result.summary.items():
        if not np.isclose(value, metrics["summary"][key], atol=1e-14, rtol=0):
            raise ValueError(f"Saved metric {key} does not reproduce.")
    return {"bundle": bundle, "probabilities": probabilities, "evaluation": result,
            "lineage": lineage, "config": config}


def keep_or_create_selection(path, proposed):
    """Preserve training provenance while checking all material selection fields."""
    path = Path(path)
    if not path.exists():
        path.write_text(json.dumps(proposed, indent=2, allow_nan=False) + "\n", encoding="utf-8")
        return proposed
    saved = json.loads(path.read_text())
    descriptive_fields = {"training_code_sha256", "finalization_module_sha256", "runtime_versions",
                          "selected_model", "output_directory", "report_dir"}
    identity = lambda record: {key: value for key, value in record.items() if key not in descriptive_fields}
    if identity(saved) != identity(proposed):
        raise ValueError("The selected model, settings, data, or candidate results changed; use a new final-output location.")
    return saved


def require_same_inference_model(left, right):
    """Allow provenance-only differences while requiring exactly equal inference state."""
    for key in ["feature_columns", "reason_columns", "thresholds", "step5_parameters"]:
        if left.get(key) != right.get(key):
            raise ValueError(f"Inference model differs in {key}.")
    for name in ["with_mean", "with_std", "n_features_in_"]:
        if getattr(left["scaler"], name) != getattr(right["scaler"], name):
            raise ValueError(f"Scaler setting differs: {name}.")
    for name in ["mean_", "var_", "scale_", "n_samples_seen_", "feature_names_in_"]:
        if not np.array_equal(getattr(left["scaler"], name, None), getattr(right["scaler"], name, None)):
            raise ValueError(f"Scaler state differs: {name}.")
    encoders = [left.get("pathway_encoder"), right.get("pathway_encoder")]
    if (encoders[0] is None) != (encoders[1] is None):
        raise ValueError("Pathway feature blocks differ.")
    if encoders[0] is not None:
        for name in ["vocabulary_", "feature_names_", "min_df", "fit_ids_"]:
            if getattr(encoders[0], name) != getattr(encoders[1], name):
                raise ValueError(f"Pathway encoding differs: {name}.")
    if len(left["classifiers"]) != 5 or len(right["classifiers"]) != 5:
        raise ValueError("Five reason classifiers are required.")
    for a, b in zip(left["classifiers"], right["classifiers"]):
        if type(a) is not type(b) or a.get_params() != b.get_params():
            raise ValueError("Classifier settings differ.")
        for name in ["coef_", "intercept_", "classes_", "n_features_in_"]:
            if not np.array_equal(getattr(a, name), getattr(b, name)):
                raise ValueError(f"Classifier inference state differs: {name}.")
    return True


def semicolon_tokens(value) -> set[str]:
    """Parse pooled categorical values without treating missing values as tokens."""
    if pd.isna(value):
        return set()
    if not isinstance(value, str):
        raise ValueError("Pooled drug fields must be strings or missing values.")
    return {part.strip() for part in value.split(";") if part.strip()}


class BroadPathwayEncoder:
    """Unscaled binary level-0 pathways plus explicit availability indicators.

    Fit the vocabulary using training rows only. The supplied catalog defines the
    external taxonomy; catalog-wide drug/target counts are not predictive features.
    Unknown or infrequent pathways do not create new feature columns at transform.
    """

    def __init__(self, min_df: int = 5):
        if not isinstance(min_df, int) or min_df < 1:
            raise ValueError("min_df must be a positive integer.")
        self.min_df = min_df

    def fit(self, frame: pd.DataFrame, catalog: pd.DataFrame):
        required = {"reactome_id", "pathway_name", "pathway_level"}
        if not required.issubset(catalog) or not catalog["reactome_id"].is_unique:
            raise ValueError("The pathway catalog must contain unique IDs, names, and levels.")
        allowed = catalog.loc[catalog["pathway_level"].eq(0)].set_index("reactome_id")
        counts = Counter(token for value in frame["drug_pathways"]
                         for token in semicolon_tokens(value) if token in allowed.index)
        self.vocabulary_ = sorted(token for token, count in counts.items() if count >= self.min_df)
        if not self.vocabulary_:
            raise ValueError("No broad pathways meet the fixed training-frequency rule.")
        self.training_document_frequency_ = {token: counts[token] for token in self.vocabulary_}
        self.pathway_names_ = allowed.loc[self.vocabulary_, "pathway_name"].to_dict()
        self.feature_names_ = [f"pathway__{token}" for token in self.vocabulary_] + ["has_matched_drug", "has_any_pathway"]
        self.fit_ids_ = frame["ID"].tolist()
        return self

    def transform(self, frame: pd.DataFrame) -> np.ndarray:
        if not hasattr(self, "vocabulary_"):
            raise ValueError("Fit the training vocabulary before transforming.")
        tokens = frame["drug_pathways"].map(semicolon_tokens)
        matrix = np.zeros((len(frame), len(self.feature_names_)), dtype=float)
        for j, token in enumerate(self.vocabulary_):
            matrix[:, j] = tokens.map(lambda values: token in values).to_numpy(dtype=float)
        matrix[:, -2] = frame["matched_drugs"].map(semicolon_tokens).map(bool).to_numpy(dtype=float)
        matrix[:, -1] = tokens.map(bool).to_numpy(dtype=float)
        return matrix

    def feature_table(self) -> pd.DataFrame:
        return pd.DataFrame([{"feature": f"pathway__{token}", "reactome_id": token,
                              "pathway_name": self.pathway_names_[token], "pathway_level": 0,
                              "training_document_frequency": self.training_document_frequency_[token]}
                             for token in self.vocabulary_])


def make_features(frame, scaler, embedding_columns, pathway_encoder=None):
    """Keep the inherited embedding scale and append the optional binary block."""
    features = scaler.transform(frame[list(embedding_columns)])
    if pathway_encoder is not None:
        features = np.column_stack([features, pathway_encoder.transform(frame)])
    return features


def fit_classifiers(features, labels, weights, *, C: float, seed: int = 42):
    """Hold all effective baseline settings fixed except the declared C value."""
    labels = np.asarray(labels)
    weights = np.asarray(weights, dtype=float)
    if not observed_label_mask(labels).all():
        raise ValueError("Use observed or recovered binary targets, never -100.")
    if not np.isfinite(C) or C <= 0 or weights.shape != (len(labels),) or not np.isfinite(weights).all() or (weights <= 0).any():
        raise ValueError("C and all training weights must be finite and positive.")
    estimators = []
    with warnings.catch_warnings(record=True) as recorded:
        warnings.simplefilter("always")
        for j in range(5):
            parameters = {**logistic_parameters(seed), "C": C}
            model = LogisticRegression(**parameters)
            model.fit(features, labels[:, j], sample_weight=weights)
            estimators.append(model)
    if any(issubclass(w.category, ConvergenceWarning) for w in recorded) or any(model.n_iter_.max() >= 500 for model in estimators):
        raise RuntimeError("A classifier failed to converge under the prespecified iteration limit.")
    return estimators, [str(w.message) for w in recorded]


def predict_bundle(bundle, frame):
    features = make_features(frame, bundle["scaler"], bundle["feature_columns"], bundle.get("pathway_encoder"))
    return predict_reason_probabilities(bundle["classifiers"], features)


def select_primary(candidates):
    """Maximize sample-wise F1, retaining the first prespecified candidate on ties."""
    if not candidates:
        raise ValueError("Supply at least one candidate.")
    scores = [row["evaluation"].summary["f1_samples"] for row in candidates]
    if not np.isfinite(scores).all():
        raise ValueError("Selection scores must be finite.")
    return candidates[int(np.argmax(scores))]


def binary_decisions(scores, thresholds, *, empty_set_policy="allow"):
    scores = np.asarray(scores, dtype=float)
    limits = np.array([thresholds[reason] for reason in REASONS], dtype=float)
    if scores.ndim != 2 or scores.shape[1] != 5 or not np.isfinite(scores).all() or ((scores < 0) | (scores > 1)).any():
        raise ValueError("Provide five finite reason probabilities in [0, 1].")
    if not np.isfinite(limits).all() or ((limits <= 0) | (limits >= 1)).any():
        raise ValueError("Every threshold must lie strictly between zero and one.")
    return part1_decisions(scores, thresholds, empty_set_policy=empty_set_policy)


def reason_strings(decisions):
    decisions = np.asarray(decisions)
    if decisions.ndim != 2 or decisions.shape[1] != 5 or not np.isin(decisions, [0, 1]).all():
        raise ValueError("Reason decisions must be a binary n-by-5 array.")
    return [" ".join(reason for reason, positive in zip(REASONS, row) if positive) for row in decisions]


def build_submission(sample, predictions):
    """Align sample IDs and reject empty sets that Kaggle parses as null."""
    if list(sample.columns) != ["ID", "Predicted"] or set(predictions.columns) != {"ID", *REASONS}:
        raise ValueError("Unexpected sample or prediction columns.")
    for frame in [sample, predictions]:
        if frame["ID"].isna().any() or not frame["ID"].is_unique:
            raise ValueError("Submission IDs must be unique and nonmissing.")
    if set(sample["ID"]) != set(predictions["ID"]):
        raise ValueError("Test and sample IDs must match exactly.")
    aligned = predictions.set_index("ID").loc[sample["ID"], list(REASONS)]
    strings = reason_strings(aligned.to_numpy())
    if any(not value.strip() for value in strings):
        raise ValueError("Empty reason sets cannot be submitted; apply the declared argmax fallback first.")
    return pd.DataFrame({"ID": sample["ID"].to_numpy(), "Predicted": strings})


def build_reason_lookup(frames, scores_by_split, thresholds, model_run_id, *, empty_set_policy="allow"):
    """Prefer observed truth; use model estimates only for masked train and test.

    Training-row predictions are explicitly marked as resubstitution estimates
    from a model trained on their recovered targets. They are not held-out labels.
    """
    tables = []
    for split in ["train", "val", "test"]:
        frame = frames[split]
        scores = np.asarray(scores_by_split[split])
        decisions = binary_decisions(scores, thresholds, empty_set_policy=empty_set_policy)
        raw_empty = binary_decisions(scores, thresholds).sum(axis=1) == 0
        if len(scores) != len(frame):
            raise ValueError("Scores must align with split rows.")
        observed = observed_label_mask(frame[list(REASONS)].to_numpy()) if split != "test" else np.zeros(len(frame), dtype=bool)
        used = decisions.copy()
        if observed.any():
            used[observed] = frame.loc[observed, list(REASONS)].to_numpy(dtype=int)
        source = np.full(len(frame), f"model_prediction_{split}", dtype=object)
        source[observed] = f"observed_{split}"
        scope = np.full(len(frame), "test_inference" if split == "test" else "training_resubstitution_with_pseudo_targets", dtype=object)
        scope[observed] = "observed_label"
        table = pd.DataFrame({"ID": frame["ID"].to_numpy(), "part1_split": split,
            "reason_source": source, "is_observed": observed, "prediction_scope": scope,
            "model_run_id": np.where(observed, None, model_run_id),
            "prediction_is_training_resubstitution": (~observed) & (split == "train")})
        for j, reason in enumerate(REASONS):
            table[reason] = used[:, j]
            table[f"model_score_{reason}"] = np.where(observed, np.nan, scores[:, j])
        if empty_set_policy == "argmax":
            table["empty_set_fallback_applied"] = raw_empty & (~observed)
            table["decision_policy"] = np.where(observed, "observed_label", "threshold_then_argmax_if_empty")
        table["reason_count"] = used.sum(axis=1)
        table["has_assigned_reason"] = used.sum(axis=1) > 0
        table["reasons"] = reason_strings(used)
        tables.append(table)
    lookup = pd.concat(tables, ignore_index=True)
    if lookup["ID"].isna().any() or not lookup["ID"].is_unique:
        raise ValueError("Part 1 split IDs must be unique and disjoint.")
    return lookup


def link_unsuccessful_validation(outcomes, reason_lookup):
    """Keep every failed trial-phase row and distinguish unmatched from empty predictions."""
    failed = outcomes.loc[outcomes["label"].eq(0), ["ID", "trial_id", "phase", "label"]].copy()
    if failed.duplicated(["trial_id", "phase"]).any():
        raise ValueError("A trial must occur at most once within a phase.")
    renamed = reason_lookup.rename(columns={"ID": "part1_id"})
    joined = failed.merge(renamed, left_on="trial_id", right_on="part1_id", how="left", validate="many_to_one", indicator=True)
    joined["matched_part1"] = joined.pop("_merge").eq("both")
    joined["reason_group"] = np.where(~joined["matched_part1"], "no_recorded_stopping_reason",
        np.where(joined["reason_count"].fillna(0).eq(0), "matched_no_predicted_reason", "assigned_reasons"))
    for reason in REASONS:
        joined[reason] = joined[reason].astype("Int64")
    if len(joined) != len(failed):
        raise RuntimeError("A linkage operation changed the failed-row count.")
    return joined

```


### tools/src/midterm_part1_local_consensus.py

SHA-256: `d125c149c48ba96bc389e37500ea13b82b1daa920dde5e78aa657c2767df8cbe`

```python
"""Optional replay of the historical nested Part 1 local-model / consensus study.

Run: python tools/src/midterm_part1_local_consensus.py --out NEW_DIRECTORY
This reruns the recorded model search, so an explicit new output directory is
required. For fixed selected recipes, use python tools/src/train_core.py --help.
No recorded public submission is replaced or uploaded.
"""
from __future__ import annotations

import argparse
from dataclasses import dataclass
import hashlib
import importlib.metadata
import json
from pathlib import Path
import time
import warnings

import joblib
import numpy as np
import pandas as pd
from scipy.linalg import solve
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import connected_components
from scipy.spatial.distance import cdist, pdist
from sklearn.cluster import KMeans
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from threadpoolctl import threadpool_limits

from midterm_augmentation import (cluster_memberships, estimate_cluster_reason_rates,
    logistic_parameters, predict_reason_probabilities, recover_labels)
from midterm_evaluation import REASONS, dataset_path, part1_decisions
from midterm_part1_finalization import fit_classifiers

EMB = [f"emb_{j}" for j in range(150)]
SEEDS = (42, 137, 2026)
THRESHOLDS = (.35, .50, .25)  # This order resolves ties, not a tuned ordering.
TOL = 1e-4
EPS = 1e-12
OUTNAME = "part1_local_consensus_2026_11_02"
CONFIGS = [dict(geometry=g, family=f, bandwidth=b, strength=s)
           for g in ("column", "global", "unit")
           for b in (.5, 2.)
           for f, strengths in (("svm", (.3, 3.)), ("ridge", (.1, 1.)))
           for s in strengths]
LINEAR = dict(geometry="column", family="linear", bandwidth=1., strength=1.)
PLAN = {
    "version": 1, "gold_rows": 235, "unlabeled_rows": 1014,
    "outer_folds": 4, "repeat_seeds": list(SEEDS), "inner_folds": 3,
    "calibration": "three grouped cross-fits within each fitting partition; regularized sigmoid",
    "grouping": "connected components at raw Euclidean distance <= 1e-4 across gold and unlabeled",
    "fit_scope": "remove entire held group, including unlabeled counterparts; no test features in any fit",
    "bandwidth": "gamma = multiplier / median nonzero squared distance among training gold vectors",
    "local_configs": CONFIGS, "shared_thresholds": list(THRESHOLDS),
    "controls": ["current KMeans+LR threshold .35", "same model with threshold selected by inner OOF"],
    "inner_complementarity_gate": {
        "minimum_better_groups": 3, "minimum_oracle_gain": .01,
        "maximum_local_F1_deficit": .04,
    },
    "consensus": {
        "teachers": "gold-only linear and selected local model, three calibrated committee members each",
        "positive_minimum_across_all_six": .8, "negative_maximum_across_all_six": .1,
        "otherwise": "abstain independently for each reason",
        "target": "mean committee probability", "base_pseudo_weight": .1,
        "positive_weight_cap": ".5 * gold positive count",
        "negative_weight_cap": "min(.25 * gold negative count, 2 * gold positive count)",
        "student": "five LR C=1 with exact soft cross-entropy via weighted positive/negative copies",
        "rounds": 1,
    },
    "blends": "current model + best eligible local/consensus source; source weights .25/.5/.75, inner only",
    "candidate_gate": {
        "mean_OOF_gain_vs_retuned_control": .01, "positive_repeats_at_least": 2,
        "worst_repeat_gain_minimum": -.01, "each_source_gain_minimum": -.01,
        "late_56_gain_minimum": -.02,
    },
    "caveat": "All 235 labels were previously inspected. Nested resampling is selection evidence, not a fresh external test.",
}


def digest(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


def write_json(path, value):
    Path(path).write_text(json.dumps(value, indent=2, allow_nan=False) + "\n", encoding="utf-8")


def row_f1(y, pred):
    denom = y.sum(1) + pred.sum(1)
    return np.divide(2 * (y * pred).sum(1), denom, out=np.zeros(len(y), float), where=denom > 0)


def decide(p, threshold):
    return part1_decisions(np.clip(p, 0, 1), threshold, empty_set_policy="argmax")


def f1(y, p, threshold):
    return float(row_f1(y, decide(p, threshold)).mean())


def best_threshold(y, p):
    values = [f1(y, p, t) for t in THRESHOLDS]
    k = int(np.argmax(values))
    return THRESHOLDS[k], values[k]


def near_groups(x, tolerance=TOL):
    distances = cdist(x, x)
    _, groups = connected_components(csr_matrix(distances <= tolerance), directed=False)
    return groups


@dataclass
class Data:
    frame: pd.DataFrame
    x: np.ndarray
    y: np.ndarray
    groups: np.ndarray
    gold: np.ndarray
    unlabeled: np.ndarray

    def scope(self, gold, eligible):
        gold, eligible = np.asarray(gold, int), np.asarray(eligible, int)
        if not set(gold) <= set(eligible):
            raise ValueError("Gold rows must belong to the feature fitting scope")
        if not np.isin(self.y[gold], [0, 1]).all():
            raise ValueError("Only observed binary labels may train supervised fits")
        return Scope(self, gold, eligible)


@dataclass
class Scope:
    data: Data
    gold: np.ndarray
    eligible: np.ndarray

    def without(self, held):
        held_groups = np.unique(self.data.groups[held])
        keep = ~np.isin(self.data.groups[self.eligible], held_groups)
        gold = self.gold[~np.isin(self.data.groups[self.gold], held_groups)]
        return self.data.scope(gold, self.eligible[keep])

    def folds(self, seed, n=3):
        y = self.data.y[self.gold]
        splitter = StratifiedGroupKFold(n_splits=n, shuffle=True, random_state=seed)
        for _, held in splitter.split(self.gold, y.argmax(1), self.data.groups[self.gold]):
            yield held, self.without(self.gold[held])


@dataclass
class Geometry:
    kind: str
    mean: np.ndarray
    scale: np.ndarray

    @classmethod
    def fit(cls, x, kind):
        if kind == "unit":
            return cls(kind, np.zeros(x.shape[1]), np.ones(x.shape[1]))
        mean = x.mean(0)
        scale = x.std(0) if kind == "column" else np.full(x.shape[1], np.sqrt(np.mean((x - mean) ** 2)))
        return cls(kind, mean, np.where(scale > 1e-12, scale, 1.))

    def transform(self, x):
        if self.kind == "unit":
            norms = np.linalg.norm(x, axis=1, keepdims=True)
            return x / np.maximum(norms, 1e-12)
        return (x - self.mean) / self.scale


@dataclass
class BaseModel:
    geometry: Geometry
    family: str
    train_x: np.ndarray
    gamma: float
    estimators: list
    coefficients: np.ndarray | None

    def raw(self, x):
        z = self.geometry.transform(x)
        matrix = z if self.family == "linear" else np.exp(-self.gamma * cdist(z, self.train_x, "sqeuclidean"))
        if self.family == "ridge":
            return matrix @ self.coefficients
        return np.column_stack([np.full(len(z), m) if isinstance(m, float)
                                else m.decision_function(matrix) for m in self.estimators])


def fit_base(scope, config):
    d = scope.data
    geom = Geometry.fit(d.x[scope.eligible], config["geometry"])
    z = geom.transform(d.x[scope.gold])
    y = d.y[scope.gold]
    family = config["family"]
    distances = pdist(z, "sqeuclidean")
    median = float(np.median(distances[distances > 1e-12])) if np.any(distances > 1e-12) else 1.
    gamma = config["bandwidth"] / median
    matrix = z if family == "linear" else np.exp(-gamma * cdist(z, z, "sqeuclidean"))
    if family == "ridge":
        coefficients = solve(matrix + config["strength"] * np.eye(len(z)), y, assume_a="pos")
        return BaseModel(geom, family, z, gamma, [], coefficients)
    estimators = []
    for j in range(5):
        if len(np.unique(y[:, j])) == 1:
            rate = (y[:, j].sum() + 1) / (len(y) + 2)
            estimators.append(float(np.log(rate / (1 - rate))))
            continue
        if family == "svm":
            model = SVC(C=config["strength"], kernel="precomputed", max_iter=20000, tol=1e-4)
        else:
            model = LogisticRegression(**{**logistic_parameters(42), "C": config["strength"]})
        model.fit(matrix, y[:, j])
        estimators.append(model)
    return BaseModel(geom, family, z, gamma, estimators, None)


@dataclass
class Sigmoid:
    scalers: list
    estimators: list

    @classmethod
    def fit(cls, raw, y):
        scalers, models = [], []
        for j in range(5):
            s = StandardScaler().fit(raw[:, [j]])
            if len(np.unique(y[:, j])) == 1:
                m = float((y[:, j].sum() + 1) / (len(y) + 2))
            else:
                m = LogisticRegression(C=1., max_iter=500, solver="lbfgs")
                m.fit(s.transform(raw[:, [j]]), y[:, j])
            scalers.append(s)
            models.append(m)
        return cls(scalers, models)

    def transform(self, raw):
        return np.column_stack([np.full(len(raw), m) if isinstance(m, float)
                                else m.predict_proba(s.transform(raw[:, [j]]))[:, 1]
                                for j, (s, m) in enumerate(zip(self.scalers, self.estimators))])


@dataclass
class Calibrated:
    base: BaseModel
    sigmoid: Sigmoid
    members: list

    def predict(self, x):
        return self.sigmoid.transform(self.base.raw(x))

    def committee(self, x):
        return np.stack([self.sigmoid.transform(m.raw(x)) for m in self.members])


def fit_calibrated(scope, config, seed):
    raw = np.full((len(scope.gold), 5), np.nan)
    members = []
    for held, inner in scope.folds(seed):
        m = fit_base(inner, config)
        raw[held] = m.raw(scope.data.x[scope.gold[held]])
        members.append(m)
    if not np.isfinite(raw).all():
        raise RuntimeError("Incomplete calibration cross-fit")
    return Calibrated(fit_base(scope, config), Sigmoid.fit(raw, scope.data.y[scope.gold]), members)


@dataclass
class LRModel:
    geometry: Geometry
    estimators: list

    def predict(self, x):
        return predict_reason_probabilities(self.estimators, self.geometry.transform(x))


def fit_current(scope):
    d = scope.data
    # K-Means seeding is order-sensitive: retain original CSV train order.
    eligible = scope.eligible[np.argsort(d.frame.iloc[scope.eligible]["fit_order"].to_numpy(), kind="stable")]
    geom = Geometry.fit(d.x[eligible], "column")
    z = geom.transform(d.x[eligible])
    labels = np.full((len(z), 5), -100, int)
    position = {idx: k for k, idx in enumerate(eligible)}
    for idx in scope.gold:
        labels[position[idx]] = d.y[idx]
    km = KMeans(n_clusters=10, n_init=20, max_iter=500, tol=1e-4, random_state=42, algorithm="lloyd").fit(z)
    memberships = cluster_memberships(km, z, "kmeans")
    mapping = estimate_cluster_reason_rates(memberships, labels, smoothing_strength=2.)
    augmented = recover_labels(labels, memberships, mapping, threshold=.5, pseudo_weight=.1)
    models, _ = fit_classifiers(z, augmented["labels"], augmented["sample_weight"], C=.1, seed=42)
    return LRModel(geom, models)


def pseudo_targets(local_committee, linear_committee, gold_y):
    """Abstentions stay NaN; caps apply independently by reason and predicted sign."""
    all_scores = np.concatenate([local_committee, linear_committee], axis=0)
    positive = all_scores.min(0) >= .8
    negative = all_scores.max(0) <= .1
    accepted = positive | negative
    targets = np.where(accepted, all_scores.mean(0), np.nan)
    weights = accepted.astype(float) * .1
    report = []
    for j, reason in enumerate(REASONS):
        npos, nneg = int(gold_y[:, j].sum()), int((1 - gold_y[:, j]).sum())
        caps = (.5 * npos, min(.25 * nneg, 2. * npos))
        for sign, mask, cap in zip(("positive", "negative"), (positive[:, j], negative[:, j]), caps):
            mass = float(weights[mask, j].sum())
            if mass > cap:
                weights[mask, j] *= cap / mass
            report.append(dict(reason=reason, sign=sign, accepted=int(mask.sum()),
                               uncapped_mass=mass, effective_mass=float(weights[mask, j].sum()), cap=float(cap)))
    return targets, weights, report


def fit_consensus(scope, local, linear):
    d = scope.data
    unlab = np.intersect1d(scope.eligible, d.unlabeled)
    q, weights, report = pseudo_targets(local.committee(d.x[unlab]), linear.committee(d.x[unlab]), d.y[scope.gold])
    geom = Geometry.fit(d.x[scope.eligible], "column")
    gold_x, unknown_x = geom.transform(d.x[scope.gold]), geom.transform(d.x[unlab])
    models = []
    for j in range(5):
        accepted = np.isfinite(q[:, j]) & (weights[:, j] > 0)
        x = np.concatenate([gold_x, unknown_x[accepted], unknown_x[accepted]])
        y = np.concatenate([d.y[scope.gold, j], np.ones(accepted.sum()), np.zeros(accepted.sum())])
        w = np.concatenate([np.ones(len(scope.gold)), weights[accepted, j] * q[accepted, j],
                            weights[accepted, j] * (1 - q[accepted, j])])
        model = LogisticRegression(**logistic_parameters(42)).fit(x, y, sample_weight=w)
        models.append(model)
    accepted_frame = pd.DataFrame({"ID": d.frame.iloc[unlab]["ID"].to_numpy()})
    for j, reason in enumerate(REASONS):
        accepted_frame[f"target_{reason}"] = q[:, j]
        accepted_frame[f"weight_{reason}"] = weights[:, j]
    return LRModel(geom, models), report, accepted_frame


def complementary(y, local_p, local_t, current_p, current_t, groups):
    a, b = row_f1(y, decide(local_p, local_t)), row_f1(y, decide(current_p, current_t))
    group_deltas = pd.DataFrame({"group": groups, "delta": a - b}).groupby("group").delta.mean()
    stats = {"better_groups": int((group_deltas > EPS).sum()),
             "oracle_gain": float(np.maximum(a, b).mean() - b.mean()),
             "local_gap": float(a.mean() - b.mean())}
    passed = stats["better_groups"] >= 3 and stats["oracle_gain"] >= .01 and stats["local_gap"] >= -.04
    return bool(passed), stats


@dataclass
class FittedProcedure:
    current: LRModel
    local: Calibrated
    consensus: LRModel | None
    recipes: dict
    selection: dict

    def predict_all(self, x):
        p = {"current": self.current.predict(x), "local": self.local.predict(x)}
        if self.consensus is not None:
            p["consensus"] = self.consensus.predict(x)
        result = {}
        for name, recipe in self.recipes.items():
            w = recipe["source_weight"]
            result[name] = (1 - w) * p["current"] + w * p[recipe["source"]]
        return result

    def predict(self, x):
        return self.predict_all(x)["final"]


def fit_procedure(scope, seed, progress=None):
    """Every trigger, configuration, threshold and blend weight uses inner data only."""
    d, y = scope.data, scope.data.y[scope.gold]
    folds = list(scope.folds(seed))
    control_p = np.full((len(y), 5), np.nan)
    for held, inner in folds:
        control_p[held] = fit_current(inner).predict(d.x[scope.gold[held]])
    control_t, control_f = best_threshold(y, control_p)
    best, candidates = None, []
    for n, config in enumerate(CONFIGS):
        p, models = np.full((len(y), 5), np.nan), []
        for k, (held, inner) in enumerate(folds):
            m = fit_calibrated(inner, config, seed + 100 + k)
            p[held] = m.predict(d.x[scope.gold[held]])
            models.append(m)
        threshold, value = best_threshold(y, p)
        candidates.append(dict(config=config, threshold=threshold, inner_f1=value))
        if best is None or value > best[0] + EPS:
            best = value, config, threshold, p, models
        if progress and (n + 1) % 8 == 0:
            progress(f"local configurations {n+1}/{len(CONFIGS)}")
    local_f, config, local_t, local_p, inner_locals = best
    enabled, complement = complementary(y, local_p, local_t, control_p, control_t, d.groups[scope.gold])
    inner_probs = {"current": control_p, "local": local_p}
    thresholds, values = {"current": control_t, "local": local_t}, {"current": control_f, "local": local_f}
    if enabled:
        student_p = np.full((len(y), 5), np.nan)
        for k, (held, inner) in enumerate(folds):
            linear = fit_calibrated(inner, LINEAR, seed + 100 + k)
            student, _, _ = fit_consensus(inner, inner_locals[k], linear)
            student_p[held] = student.predict(d.x[scope.gold[held]])
        student_t, student_f = best_threshold(y, student_p)
        inner_probs["consensus"], thresholds["consensus"], values["consensus"] = student_p, student_t, student_f
    # Ties prefer the current method, then the local model, then the student.
    stage2 = max(values, key=values.get)
    recipes = {
        "current_fixed": dict(source="current", source_weight=0., threshold=.35),
        "current_retuned": dict(source="current", source_weight=0., threshold=control_t),
        "local": dict(source="local", source_weight=1., threshold=local_t),
        "after_consensus": dict(source=stage2, source_weight=float(stage2 != "current"), threshold=thresholds[stage2]),
    }
    final_recipe, final_f = recipes["after_consensus"].copy(), values[stage2]
    blend_records = []
    # Only complementary sources enter the blend stage.
    sources = [s for s in ("local", "consensus") if s in inner_probs]
    for source in sources:
        permitted, _ = complementary(y, inner_probs[source], thresholds[source], control_p, control_t, d.groups[scope.gold])
        if not permitted:
            continue
        for weight in (.25, .5, .75):
            p = (1 - weight) * control_p + weight * inner_probs[source]
            threshold, value = best_threshold(y, p)
            blend_records.append(dict(source=source, weight=weight, threshold=threshold, inner_f1=value))
            if value > final_f + EPS:
                final_f, final_recipe = value, dict(source=source, source_weight=weight, threshold=threshold)
    recipes["final"] = final_recipe
    current = fit_current(scope)
    local = fit_calibrated(scope, config, seed + 900)
    student, pseudo_report, pseudo_rows = None, [], None
    if enabled:
        linear = fit_calibrated(scope, LINEAR, seed + 900)
        student, pseudo_report, pseudo_rows = fit_consensus(scope, local, linear)
    selection = dict(local_config=config, local_threshold=local_t, current_threshold=control_t,
                     consensus_enabled=enabled, complementarity=complement,
                     inner_f1=values, final_inner_f1=final_f, recipes=recipes,
                     candidates=candidates, blends=blend_records, pseudo_summary=pseudo_report,
                     fit_gold_ids=d.frame.iloc[scope.gold]["ID"].tolist(),
                     fit_feature_ids=d.frame.iloc[scope.eligible]["ID"].tolist())
    return FittedProcedure(current, local, student, recipes, selection), pseudo_rows


def load_data(root):
    train = pd.read_csv(dataset_path(root, "part1", "train")).copy()
    val = pd.read_csv(dataset_path(root, "part1", "val")).copy()
    train["fit_order"] = np.arange(len(train))
    val["fit_order"] = len(train) + np.arange(len(val))
    observed = train[list(REASONS)].isin([0, 1]).all(axis=1)
    labeled = train.loc[observed].assign(source="train")
    val = val.assign(source="validation")
    unknown = train.loc[~observed].assign(source="unlabeled")
    frame = pd.concat([labeled, val, unknown], ignore_index=True)
    x, y = frame[EMB].to_numpy(), frame[list(REASONS)].to_numpy(dtype=int)
    groups = near_groups(x)
    data = Data(frame, x, y, groups, np.arange(len(labeled) + len(val)), np.arange(len(labeled) + len(val), len(frame)))
    if len(data.gold) != 235 or len(data.unlabeled) != 1014:
        raise ValueError("Unexpected input counts")
    return data, pd.read_csv(dataset_path(root, "part1", "test"))


def prediction_frame(data, evaluated, fitted, tag, repeat=-1, fold=-1):
    frame = data.frame.iloc[evaluated][["ID", "source", "start_year"]].reset_index(drop=True).copy()
    frame["group"], frame["evaluation"], frame["repeat"], frame["fold"] = data.groups[evaluated], tag, repeat, fold
    y = data.y[evaluated]
    for j, reason in enumerate(REASONS):
        frame[f"true_{reason}"] = y[:, j]
    for name, probs in fitted.predict_all(data.x[evaluated]).items():
        t = fitted.recipes[name]["threshold"]
        pred = decide(probs, t)
        frame[f"{name}_f1"] = row_f1(y, pred)
        frame[f"{name}_exact"] = (y == pred).all(1)
        frame[f"{name}_set"] = [" ".join(r for r, v in zip(REASONS, row) if v) for row in pred]
        frame[f"{name}_threshold"] = t
        for j, reason in enumerate(REASONS):
            frame[f"{name}_prob_{reason}"] = probs[:, j]
            frame[f"{name}_pred_{reason}"] = pred[:, j]
    return frame


def summarize(frame):
    names = [c[:-3] for c in frame if c.endswith("_f1")]
    return {name: {"sample_f1": float(frame[f"{name}_f1"].mean()),
                   "exact_set_rate": float(frame[f"{name}_exact"].mean())} for name in names}


def group_bootstrap(frame, seed=42):
    # Keep all repeats and all members of a group together in each bootstrap draw.
    x = frame.assign(delta=frame.final_f1 - frame.current_retuned_f1).groupby("group").delta.agg(["sum", "count"])
    rng = np.random.default_rng(seed)
    sampled = rng.integers(len(x), size=(2000, len(x)))
    means = x["sum"].to_numpy()[sampled].sum(1) / x["count"].to_numpy()[sampled].sum(1)
    return [float(v) for v in np.quantile(means, [.025, .975])]


def protected_hashes(root):
    paths = [root / "models/selection.json"]
    paths += list((root / "models").rglob("*.joblib"))
    paths += list((root / "models").rglob("submission.csv"))
    return {str(p): digest(p) for p in paths if p.is_file()}


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--root", type=Path, default=Path(__file__).resolve().parents[2])
    parser.add_argument("--out", type=Path, required=True,
                        help="New replay directory; existing directories are rejected.")
    args = parser.parse_args()
    root = args.root.resolve()
    out = args.out.resolve()
    out.mkdir(parents=True, exist_ok=False)
    plan_path = out / "experiment_plan.json"
    write_json(plan_path, PLAN)
    before = protected_hashes(root)
    write_json(out / "protected_before.json", before)
    data, test = load_data(root)
    original_scope = data.scope(np.flatnonzero(data.frame.source.to_numpy() == "train"),
                                np.flatnonzero(data.frame.source.to_numpy() != "validation"))
    validation_indices = np.flatnonzero(data.frame.source.to_numpy() == "validation")
    original_probabilities = fit_current(original_scope).predict(data.x[validation_indices])
    recorded = pd.read_csv(root / 'models/part1/02_kmeans_lr/evidence/selected/selected_validation/validation_predictions.csv').set_index("ID")
    recorded = recorded.loc[data.frame.iloc[validation_indices].ID, [f"prob_{r}" for r in REASONS]].to_numpy()
    parity_gap = float(np.max(np.abs(original_probabilities - recorded)))
    if parity_gap > 1e-10:
        raise RuntimeError(f"Preflight baseline parity failed: {parity_gap}")
    metadata = {"data_hashes": {str(dataset_path(root, "part1", s)): digest(dataset_path(root, "part1", s))
                                for s in ("train", "val", "test")},
                "source_sha256": digest(__file__),
                "versions": {p: importlib.metadata.version(p) for p in ("numpy", "pandas", "scikit-learn", "scipy")},
                "gold_near_groups": int(len(np.unique(data.groups[data.gold]))),
                "all_near_groups": int(len(np.unique(data.groups))),
                "preflight_current_validation_probability_gap": parity_gap}
    write_json(out / "metadata.json", metadata)
    data.frame[["ID", "source", "start_year"]].assign(group=data.groups).to_csv(out / "groups.csv", index=False)
    full_scope = data.scope(data.gold, np.arange(len(data.frame)))
    pieces, selections, pseudo = [], [], []
    started = time.time()
    for repeat, seed in enumerate(SEEDS):
        for fold, (held, scope) in enumerate(full_scope.folds(seed, n=4)):
            label = f"repeat {repeat+1}/3 fold {fold+1}/4"
            print(f"{label}: fitting {len(scope.gold)} gold rows", flush=True)
            fitted, pseudo_rows = fit_procedure(scope, seed + fold * 17,
                                                lambda s: print(f"{label}: {s}", flush=True))
            result = prediction_frame(data, data.gold[held], fitted, "grouped_oof", repeat, fold)
            pieces.append(result)
            record = dict(repeat=repeat, fold=fold, **fitted.selection)
            selections.append(record)
            if pseudo_rows is not None:
                pseudo.append(pseudo_rows.assign(repeat=repeat, fold=fold))
            pd.concat(pieces, ignore_index=True).to_csv(out / "oof_predictions.csv", index=False)
            write_json(out / "fold_selections.json", selections)
            print(f"{label}: {json.dumps(summarize(result))}", flush=True)
    oof = pd.concat(pieces, ignore_index=True)
    if not oof.groupby(["repeat", "ID"]).size().eq(1).all() or len(oof) != 235 * len(SEEDS):
        raise RuntimeError("Incomplete or duplicated outer OOF rows")
    if pseudo:
        pd.concat(pseudo, ignore_index=True).to_csv(out / "oof_pseudo_targets.csv", index=False)

    # Historical validation is diagnostic: previous rounds already inspected it.
    source = data.frame.source.to_numpy()
    years = data.frame.start_year.to_numpy()
    diagnostics, diag_tables = {}, []
    scopes = {
        "historical_validation": (np.flatnonzero(source == "train"), np.flatnonzero(source != "validation"), np.flatnonzero(source == "validation")),
        "late_train_20": (np.flatnonzero((source == "train") & (years <= 2006)), np.flatnonzero((source != "validation") & (years <= 2006)), np.flatnonzero((source == "train") & (years >= 2007) & (years <= 2009))),
        "late_combined_56": (np.flatnonzero((source != "unlabeled") & (years <= 2006)), np.flatnonzero(years <= 2006), np.flatnonzero((source != "unlabeled") & (years >= 2007) & (years <= 2009))),
    }
    for name, (gold, eligible, held) in scopes.items():
        print(f"diagnostic {name}", flush=True)
        scope = data.scope(gold, eligible)
        fitted, _ = fit_procedure(scope, 42, lambda s: print(f"{name}: {s}", flush=True))
        result = prediction_frame(data, held, fitted, name)
        result["near_early_gold_match"] = np.isin(data.groups[held], data.groups[gold])
        result.to_csv(out / f"{name}_predictions.csv", index=False)
        diag_tables.append(result)
        diagnostics[name] = {"metrics": summarize(result), "rows": len(result), "selection": fitted.selection,
                             "novel_group_metrics": summarize(result.loc[~result.near_early_gold_match])}
        if name == "historical_validation":
            saved = pd.read_csv(root / 'models/part1/02_kmeans_lr/evidence/selected/selected_validation/validation_predictions.csv').set_index("ID").loc[result.ID]
            actual = result[[f"current_fixed_prob_{r}" for r in REASONS]].to_numpy()
            expected = saved[[f"prob_{r}" for r in REASONS]].to_numpy()
            gap = float(np.max(np.abs(actual - expected)))
            if gap > 1e-10:
                raise RuntimeError(f"Current baseline does not reproduce saved validation: {gap}")
            diagnostics[name]["current_saved_probability_max_gap"] = gap
        write_json(out / "diagnostics.json", diagnostics)

    repeats = {str(k): summarize(v) for k, v in oof.groupby("repeat")}
    by_source = {str(k): summarize(v) for k, v in oof.groupby("source")}
    delta = lambda x: x["final"]["sample_f1"] - x["current_retuned"]["sample_f1"]
    repeat_gains = [delta(x) for x in repeats.values()]
    average = summarize(oof)
    time_gain = delta(diagnostics["late_combined_56"]["metrics"])
    gates = {"mean_oof_gain": delta(average) >= .01 - EPS,
             "positive_at_least_two_repeats": sum(v > EPS for v in repeat_gains) >= 2,
             "worst_repeat": min(repeat_gains) >= -.01 - EPS,
             "both_sources": all(delta(x) >= -.01 - EPS for x in by_source.values()),
             "late_56_no_large_regression": time_gain >= -.02 - EPS}
    table = pd.concat([oof, *diag_tables], ignore_index=True)
    changed = table.loc[table.final_set != table.current_retuned_set].copy()
    changed["row_f1_delta"] = changed.final_f1 - changed.current_retuned_f1
    changed.to_csv(out / "changed_trials.csv", index=False)
    reason_rows = []
    for split, frame in table.groupby("evaluation"):
        for model in average:
            for reason in REASONS:
                y, pred = frame[f"true_{reason}"].to_numpy(), frame[f"{model}_pred_{reason}"].to_numpy()
                reason_rows.append(dict(evaluation=split, model=model, reason=reason, observations=len(y),
                    support=int(y.sum()), predicted=int(pred.sum()), f1=float(f1_score(y, pred, zero_division=0))))
    pd.DataFrame(reason_rows).to_csv(out / "metrics_by_reason.csv", index=False)
    year_rows = []
    for (split, year), frame in table.groupby(["evaluation", "start_year"]):
        year_rows.append(dict(evaluation=split, year=int(year), observations=len(frame),
                             **{m: v["sample_f1"] for m, v in summarize(frame).items()}))
    pd.DataFrame(year_rows).to_csv(out / "metrics_by_year.csv", index=False)
    group_delta = oof.assign(delta=oof.final_f1 - oof.current_retuned_f1).groupby("group").delta.mean()
    summary = dict(overall=average, by_repeat=repeats, by_source=by_source, repeat_gains=repeat_gains,
                   mean_gain=delta(average), group_bootstrap_95_percent=group_bootstrap(oof),
                   group_wins=int((group_delta > EPS).sum()), group_losses=int((group_delta < -EPS).sum()),
                   diagnostic_metrics={k: v["metrics"] for k, v in diagnostics.items()},
                   consensus_enabled_folds=sum(s["consensus_enabled"] for s in selections),
                   candidate_gate=gates, candidate_allowed=all(gates.values()),
                   elapsed_seconds=time.time() - started,
                   caveat=PLAN["caveat"], candidate=None)
    write_json(out / "summary.json", summary)
    if all(gates.values()):
        print("Candidate gate passed; fitting locked procedure on all 235 gold labels", flush=True)
        fitted, pseudo_rows = fit_procedure(full_scope, 42, lambda s: print(f"final refit: {s}", flush=True))
        probabilities = fitted.predict(test[EMB].to_numpy())
        predicted = decide(probabilities, fitted.recipes["final"]["threshold"])
        submission = pd.DataFrame({"ID": test.ID, "Predicted": [" ".join(r for r, v in zip(REASONS, row) if v) for row in predicted]})
        contents = submission.to_csv(index=False).encode()
        sha = hashlib.sha256(contents).hexdigest()
        filename = f"part1_local_consensus_{sha[:12]}.csv"
        (out / filename).write_bytes(contents)
        joblib.dump(fitted, out / "candidate_model.joblib")
        restored = joblib.load(out / "candidate_model.joblib")
        np.testing.assert_allclose(restored.predict(test[EMB].to_numpy()), probabilities, atol=1e-12, rtol=0)
        if len(submission) != 312 or not submission.ID.is_unique or submission.Predicted.str.len().eq(0).any():
            raise RuntimeError("Invalid submission format")
        if pseudo_rows is not None:
            pseudo_rows.to_csv(out / "final_pseudo_targets.csv", index=False)
        write_json(out / "final_selection.json", fitted.selection)
        saved = pd.read_csv(root / 'models/part1/02_kmeans_lr/submission.csv').set_index("ID").loc[test.ID]
        summary["candidate"] = dict(filename=filename, sha256=sha, rows=312,
                                    changed_vs_public_best=int((submission.Predicted.to_numpy() != saved.Predicted.to_numpy()).sum()),
                                    model_sha256=digest(out / "candidate_model.joblib"), kaggle_uploaded=False)
        write_json(out / "summary.json", summary)
    after = protected_hashes(root)
    if after != before:
        raise RuntimeError("A protected file changed during this experiment")
    write_json(out / "verification.json", dict(protected_hashes_unchanged=True, protected_hashes=after,
               oof_rows=len(oof), unique_gold_ids=oof.ID.nunique(), gold_groups=oof.group.nunique(),
               outer_fold_count=len(selections), output_hashes={p.name: digest(p) for p in out.glob("*") if p.is_file()}))
    print(json.dumps(summary, indent=2), flush=True)


if __name__ == "__main__":
    # Use an importable class namespace in serialized models, not __main__.
    from midterm_part1_local_consensus import main as run_main
    with threadpool_limits(limits=1), warnings.catch_warnings():
        warnings.simplefilter("error", category=ConvergenceWarning)
        run_main()

```


### tools/src/midterm_part2_baseline.py

SHA-256: `7f720851987d03c630a371127c998bd34ace0ad0ab1ba3276c5302919f743f90`

```python
"""Phase-specific text baselines and provenance-aware failure-reason analysis."""

from __future__ import annotations

from math import sqrt
from pathlib import Path
import json
import warnings

import joblib
import numpy as np
import pandas as pd
from sklearn.exceptions import ConvergenceWarning
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from threadpoolctl import threadpool_limits

from midterm_augmentation import logistic_parameters
from midterm_evaluation import PHASES, REASONS, evaluate_part2, file_sha256
from midterm_part1_finalization import link_unsuccessful_validation

TEXT_COLUMNS = ("criteria", "diseases_cleaned", "drugs")
REASON_GROUPS = (*REASONS, "no_recorded_stopping_reason", "matched_no_predicted_reason")


def build_trial_text(frame: pd.DataFrame) -> pd.Series:
    """Use exactly three text fields; fill missing fields with an empty string.

    Brackets and quotes in the supplied disease/drug list strings are retained
    here and discarded by the vectorizer's word tokenizer. Negations remain.
    Identifiers, outcomes, years, drug-knowledge fields, and reasons are excluded.
    """
    if not set(TEXT_COLUMNS).issubset(frame.columns):
        raise ValueError("Expected criteria, diseases_cleaned, and drugs text fields.")
    return frame[list(TEXT_COLUMNS)].astype("string").fillna("").agg("\n".join, axis=1)


def validate_outcome_rows(frame, *, labels_required=True):
    required = {"ID", "trial_id", "phase", *TEXT_COLUMNS}
    if labels_required:
        required.add("label")
    if not required.issubset(frame.columns) or frame.empty:
        raise ValueError("Expected nonempty trial-phase rows with the declared columns.")
    if frame["ID"].isna().any() or not frame["ID"].is_unique or frame["trial_id"].isna().any():
        raise ValueError("Row IDs must be unique, with nonmissing row and trial IDs.")
    if not frame["phase"].isin(PHASES).all() or frame.duplicated(["trial_id", "phase"]).any():
        raise ValueError("Each trial may occur at most once in each of phases I, II, III.")
    if labels_required:
        if not frame["label"].isin([0, 1]).all():
            raise ValueError("Outcome labels must be binary, with success=1.")
        if frame.groupby("trial_id")["label"].nunique().gt(1).any():
            raise ValueError("Combined-phase copies of one trial must have the same outcome.")


def normalized_json(value):
    """Normalize tuple/list representations before comparing saved parameters."""
    return json.loads(json.dumps(value, sort_keys=True, allow_nan=False))


def fit_phase_baseline(train, *, tfidf_parameters, classifier_parameters, seed=42):
    """Fit one vocabulary and one binary classifier per training phase."""
    validate_outcome_rows(train)
    pipelines, diagnostics, fit_ids, warning_rows = {}, [], {}, []
    for phase in PHASES:
        subset = train.loc[train["phase"].eq(phase)]
        if subset.empty or subset["label"].nunique() != 2:
            raise ValueError(f"Phase {phase} requires training examples of both classes.")
        parameters = {**logistic_parameters(seed), **classifier_parameters, "random_state": seed}
        pipeline = Pipeline([("tfidf", TfidfVectorizer(**tfidf_parameters)),
                             ("classifier", LogisticRegression(**parameters))])
        with threadpool_limits(limits=1), warnings.catch_warnings(record=True) as recorded:
            warnings.simplefilter("always")
            pipeline.fit(build_trial_text(subset), subset["label"].to_numpy(dtype=int))
        classifier = pipeline.named_steps["classifier"]
        if any(issubclass(item.category, ConvergenceWarning) for item in recorded) or classifier.n_iter_.max() >= classifier.max_iter:
            raise RuntimeError(f"Phase {phase} failed to converge under the fixed iteration limit.")
        if not np.array_equal(classifier.classes_, [0, 1]):
            raise RuntimeError("The positive class must be success=1.")
        vectorizer = pipeline.named_steps["tfidf"]
        matrix = vectorizer.transform(build_trial_text(subset))
        diagnostics.append({"phase": phase, "training_rows": len(subset),
            "unique_training_trials": subset["trial_id"].nunique(),
            "training_successes": int(subset["label"].sum()),
            "training_failures": int(subset["label"].eq(0).sum()),
            "missing_criteria_rows": int(subset["criteria"].isna().sum()),
            "vocabulary_size": len(vectorizer.vocabulary_), "training_nonzero_values": int(matrix.nnz),
            "zero_training_vectors": int((matrix.getnnz(axis=1) == 0).sum()),
            "classifier_iterations": int(classifier.n_iter_.max()), "converged": True})
        fit_ids[f"tfidf_phase_{phase}"] = subset["trial_id"].tolist()
        fit_ids[f"classifier_phase_{phase}"] = subset["trial_id"].tolist()
        warning_rows.extend({"phase": phase, "warning": str(item.message)} for item in recorded)
        pipelines[phase] = pipeline
    bundle = {"pipelines": pipelines, "text_columns": list(TEXT_COLUMNS),
        "thresholds": dict.fromkeys(PHASES, .5), "positive_class": 1,
        "seed": seed, "tfidf_parameters": normalized_json(tfidf_parameters),
        "classifier_parameters": normalized_json(classifier_parameters),
        "reason_features_used": False}
    return bundle, pd.DataFrame(diagnostics), fit_ids, warning_rows


def predict_outcome_probabilities(bundle, frame):
    """Preserve row order while routing each trial to its phase's fitted pipeline."""
    validate_outcome_rows(frame, labels_required=False)
    if list(bundle["text_columns"]) != list(TEXT_COLUMNS) or bundle["positive_class"] != 1:
        raise ValueError("Unexpected text inputs or positive-class convention.")
    probabilities = np.full(len(frame), np.nan)
    phases = frame["phase"].to_numpy()
    for phase in PHASES:
        positions = np.flatnonzero(phases == phase)
        if not len(positions):
            continue
        pipeline = bundle["pipelines"][phase]
        classes = list(pipeline.classes_)
        probabilities[positions] = pipeline.predict_proba(build_trial_text(frame.iloc[positions]))[:, classes.index(1)]
    if not np.isfinite(probabilities).all() or ((probabilities < 0) | (probabilities > 1)).any():
        raise RuntimeError("Predictions must be finite success probabilities in [0, 1].")
    return pd.DataFrame({"ID": frame["ID"].to_numpy(), "p_success": probabilities})


def load_saved_baseline(directory, *, hyperparameters, feature_spec, input_files, seed, validation):
    """Reuse immutable runs after validating real settings, inputs, and predictions."""
    directory = Path(directory)
    config = json.loads((directory / "config.json").read_text())
    for field, expected in {"task": "part2", "split": "validation", "hyperparameters": hyperparameters,
            "feature_spec": feature_spec, "input_files": input_files, "seed": seed,
            "thresholds": dict.fromkeys(PHASES, .5)}.items():
        if config.get(field) != normalized_json(expected):
            raise ValueError(f"Existing run has different {field}; use a new run ID for different settings.")
    manifest = json.loads((directory / "artifact_checksums.json").read_text())
    for name, digest in manifest.items():
        relative = Path(name)
        if relative.is_absolute() or ".." in relative.parts or file_sha256(directory / relative) != digest:
            raise ValueError(f"An immutable run artifact changed: {name}")
    bundle = joblib.load(directory / "model.joblib")
    probabilities = predict_outcome_probabilities(bundle, validation)
    result = evaluate_part2(validation, probabilities, .5)
    saved = pd.read_csv(directory / "validation_predictions.csv").set_index("ID").loc[validation["ID"]]
    if not np.allclose(saved["p_success"], result.predictions["p_success"], atol=1e-12, rtol=0):
        raise ValueError("Saved model does not reproduce its validation probabilities.")
    if not np.array_equal(saved["prediction"], result.predictions["prediction"]):
        raise ValueError("Saved validation decisions differ from the fixed threshold.")
    metrics = json.loads((directory / "metrics.json").read_text())
    for key, value in result.summary.items():
        if not np.isclose(value, metrics["summary"][key], atol=1e-12, rtol=0):
            raise ValueError(f"Saved primary metrics do not reproduce: {key}")
    return bundle, result, pd.read_csv(directory / "training_diagnostics.csv")


def wilson_interval(k, n, z=1.959963984540054):
    """Descriptive binomial interval; it excludes uncertainty in predicted reasons."""
    if n < 0 or k < 0 or k > n:
        raise ValueError("Require 0 <= k <= n.")
    if n == 0:
        return np.nan, np.nan
    rate = k/n
    denominator = 1 + z*z/n
    center = (rate + z*z/(2*n))/denominator
    half = z*sqrt(rate*(1-rate)/n + z*z/(4*n*n))/denominator
    return max(0., center-half), min(1., center+half)


def _rate_row(subset, phase, group, source_type="all"):
    n = len(subset)
    k = int(subset["prediction"].eq(1).sum())
    lower, upper = wilson_interval(k, n)
    observed = int(subset["source_type"].eq("observed").sum())
    predicted = int(subset["source_type"].eq("predicted").sum())
    unmatched = int(subset["source_type"].eq("unmatched").sum())
    source_count = sum(count > 0 for count in (observed, predicted, unmatched))
    evidence = ("no_trials" if n == 0 else "mixed" if source_count > 1 else
                "unmatched" if unmatched else "observed_only" if observed else "predicted_only")
    return {"phase": phase, "reason_group": group, "source_type": source_type,
        "n_trials": n, "n_predicted_success": k, "false_success_rate": k/n if n else np.nan,
        "wilson_95_lower": lower, "wilson_95_upper": upper,
        "n_observed_reasons": observed, "n_predicted_reasons": predicted, "n_unmatched": unmatched,
        "n_training_resubstitution_reasons": int(subset["prediction_is_training_resubstitution"].eq(True).sum()),
        "label_evidence": evidence, "small_support": n < 5,
        "rate_status": "ok" if n else "undefined_no_trials"}


def analyze_failure_reasons(validation_predictions, reason_lookup):
    """Compute conditional false-success rates, retaining multilabel and unmatched rows."""
    required = {"ID", "trial_id", "phase", "label", "prediction", "p_success"}
    if not required.issubset(validation_predictions):
        raise ValueError("Provide saved outcome labels, probabilities, and binary decisions.")
    if not validation_predictions["ID"].is_unique or validation_predictions.duplicated(["trial_id", "phase"]).any():
        raise ValueError("Outcome IDs and trial-phase pairs must be unique.")
    if not validation_predictions["label"].isin([0, 1]).all() or not validation_predictions["prediction"].isin([0, 1]).all():
        raise ValueError("Outcome labels and decisions must be binary.")
    scores = validation_predictions["p_success"].to_numpy(dtype=float)
    if not validation_predictions["phase"].isin(PHASES).all() or not np.isfinite(scores).all() or ((scores < 0) | (scores > 1)).any():
        raise ValueError("Use known phases and finite success probabilities in [0, 1].")
    lookup_columns = {"ID", *REASONS, "is_observed", "reason_source", "reason_count", "prediction_is_training_resubstitution"}
    if not lookup_columns.issubset(reason_lookup) or not reason_lookup["ID"].is_unique:
        raise ValueError("Use a unique Part 1 lookup with explicit reason provenance.")
    if not reason_lookup[list(REASONS)].isin([0, 1]).all().all() or not reason_lookup["is_observed"].isin([True, False]).all():
        raise ValueError("Lookup reasons must be binary with known observed/predicted status.")
    if not np.array_equal(reason_lookup[list(REASONS)].sum(axis=1), reason_lookup["reason_count"]):
        raise ValueError("Lookup reason counts do not match its label vectors.")
    if (reason_lookup["is_observed"] & reason_lookup["reason_count"].eq(0)).any():
        raise ValueError("Observed Part 1 reason vectors should contain a recorded reason.")
    joined = link_unsuccessful_validation(validation_predictions, reason_lookup)
    joined = joined.merge(validation_predictions[["ID", "p_success", "prediction"]], on="ID", how="left", validate="one_to_one")
    joined["source_type"] = np.where(~joined["matched_part1"], "unmatched",
        np.where(joined["is_observed"].eq(True), "observed", "predicted"))
    joined["incorrectly_predicted_success"] = joined["prediction"].eq(1)
    primary, by_source, source_detail, overall = [], [], [], []
    for phase in PHASES:
        phase_rows = joined.loc[joined["phase"].eq(phase)]
        overall.append(_rate_row(phase_rows, phase, "all_failed_trials"))
        for group in REASON_GROUPS:
            mask = phase_rows[group].eq(1).fillna(False) if group in REASONS else phase_rows["reason_group"].eq(group)
            subset = phase_rows.loc[mask]
            primary.append(_rate_row(subset, phase, group))
            sources = ("unmatched",) if group == "no_recorded_stopping_reason" else ("predicted",) if group == "matched_no_predicted_reason" else ("observed", "predicted")
            for source in sources:
                by_source.append(_rate_row(subset.loc[subset["source_type"].eq(source)], phase, group, source))
            for source, detail in subset.groupby("reason_source", dropna=False):
                row = _rate_row(detail, phase, group)
                row["reason_source"] = source if pd.notna(source) else "no_part1_match"
                source_detail.append(row)
    return {"failed_rows": joined, "reason_rates": pd.DataFrame(primary),
            "source_rates": pd.DataFrame(by_source), "source_detail_rates": pd.DataFrame(source_detail),
            "overall_failed_rates": pd.DataFrame(overall)}

```


### tools/src/midterm_part2_improvement.py

SHA-256: `a8c690d363c3be1ab43053e67007512ab20e220d874d53ea74137871055b41a4`

```python
"""Controlled outcome-model improvements, fixed selection, and submission validation."""
from __future__ import annotations

from collections import Counter
from pathlib import Path
import json
import re
import warnings

import joblib
import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from threadpoolctl import threadpool_limits

from midterm_augmentation import logistic_parameters
from midterm_evaluation import PHASES, evaluate_part2, file_sha256
from midterm_part1_finalization import BroadPathwayEncoder
from midterm_part2_baseline import TEXT_COLUMNS, build_trial_text, normalized_json, validate_outcome_rows

BASELINE_RUN = 'part2_tfidf_lr_by_phase_seed42_v1'
STRUCTURED_COLUMNS = ('start_year', 'icdcodes', 'drug_pathways', 'matched_drugs')


def icd_categories(value):
    """Extract unique three-character ICD categories from nested list strings."""
    if pd.isna(value):
        return set()
    codes = re.findall(r'\b[A-Z][0-9][0-9A-Z](?:\.[0-9A-Z]{1,4})?\b', str(value).upper())
    return {code[:3] for code in codes}


class OutcomeMetadataEncoder:
    """Fit ICD vocabulary, broad pathways, and bounded year values on one training phase."""
    def __init__(self, min_df=5):
        self.min_df = min_df

    def fit(self, frame, catalog):
        if not set(STRUCTURED_COLUMNS).issubset(frame):
            raise ValueError('Missing declared structured inputs.')
        counts = Counter(token for value in frame['icdcodes'] for token in icd_categories(value))
        self.icd_vocabulary_ = sorted(token for token, count in counts.items() if count >= self.min_df)
        self.icd_df_ = {token: counts[token] for token in self.icd_vocabulary_}
        self.pathways_ = BroadPathwayEncoder(min_df=self.min_df).fit(frame, catalog)
        years = pd.to_numeric(frame['start_year'], errors='raise').to_numpy(dtype=float)
        if np.isinf(years).any() or np.isnan(years).all():
            raise ValueError('Training years require finite observed values.')
        self.year_median_ = float(np.nanmedian(years))
        self.year_min_ = float(np.nanmin(years))
        self.year_max_ = float(np.nanmax(years))
        self.year_scaler_ = StandardScaler().fit(np.where(np.isnan(years), self.year_median_, years).reshape(-1, 1))
        self.feature_names_ = ([f'icd__{token}' for token in self.icd_vocabulary_] + ['has_any_icd'] +
            self.pathways_.feature_names_ + ['year_standardized', 'year_missing'])
        self.fit_trial_ids_ = frame['trial_id'].tolist()
        return self

    def transform(self, frame):
        if not hasattr(self, 'feature_names_'):
            raise ValueError('Fit metadata preprocessing before transform.')
        categories = frame['icdcodes'].map(icd_categories)
        columns = [categories.map(lambda tokens: token in tokens).to_numpy(dtype=float)
                   for token in self.icd_vocabulary_]
        columns.append(categories.map(bool).to_numpy(dtype=float))
        icd = np.column_stack(columns)
        years = pd.to_numeric(frame['start_year'], errors='raise').to_numpy(dtype=float)
        if np.isinf(years).any():
            raise ValueError('Years must be finite or missing.')
        missing = np.isnan(years)
        imputed = np.where(missing, self.year_median_, years)
        bounded = np.clip(imputed, self.year_min_, self.year_max_)
        year = self.year_scaler_.transform(bounded.reshape(-1, 1))
        return sparse.csr_matrix(np.column_stack([icd, self.pathways_.transform(frame), year, missing.astype(float)]))

    def audit_record(self):
        return {'icd_vocabulary': self.icd_vocabulary_, 'icd_training_document_frequencies': self.icd_df_,
                'pathway_vocabulary': self.pathways_.vocabulary_, 'feature_names': self.feature_names_,
                'year_median': self.year_median_, 'year_min': self.year_min_, 'year_max': self.year_max_,
                'year_scaler_mean': self.year_scaler_.mean_.tolist(),
                'year_scaler_scale': self.year_scaler_.scale_.tolist(), 'min_df': self.min_df}


def candidate_run_id(structured, C):
    tag = {0.1: '01', 1.0: '1', 10.0: '10'}[float(C)]
    return BASELINE_RUN if not structured and C == 1 else f'part2_s7_{"metadata" if structured else "text"}_lr_c{tag}_seed42_v1'


def candidate_settings(structured, C, catalog_sha):
    return {'baseline_vocabulary_run': BASELINE_RUN, 'C': float(C), 'structured_features': bool(structured),
        'metadata': {'columns': list(STRUCTURED_COLUMNS), 'icd_representation': 'binary_three_character_categories',
            'min_training_df': 5, 'pathway_level': 0, 'binary_features_unscaled': True,
            'year_imputation': 'training_phase_median', 'year_scaling': 'training_phase_StandardScaler',
            'year_clipping': 'training_phase_observed_min_max', 'missing_year_indicator': True,
            'block_weight': 1.0, 'catalog_sha256': catalog_sha} if structured else None,
        'classifier': {'solver': 'liblinear', 'max_iter': 500, 'tol': 1e-4, 'effective_regularization': 'L2',
                       'class_weight': None, 'fit_intercept': True, 'random_state': 42},
        'threshold': 0.5, 'fit_split': 'training_only', 'reason_features_used': False}


def fit_candidate(train, baseline, catalog, *, structured, C):
    """Reuse the training-fitted text vocabularies, and vary only declared factors."""
    validate_outcome_rows(train)
    models, diagnostics, fit_scope = {}, [], {}
    for phase in PHASES:
        rows = train.loc[train['phase'].eq(phase)]
        vectorizer = baseline['pipelines'][phase].named_steps['tfidf']
        text_matrix = vectorizer.transform(build_trial_text(rows))
        encoder = OutcomeMetadataEncoder().fit(rows, catalog) if structured else None
        matrix = sparse.hstack([text_matrix, encoder.transform(rows)], format='csr') if encoder else text_matrix
        classifier = LogisticRegression(**{**logistic_parameters(42), 'C': float(C)})
        with threadpool_limits(limits=1), warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter('always')
            classifier.fit(matrix, rows['label'].to_numpy(dtype=int))
        if any(issubclass(w.category, ConvergenceWarning) for w in caught) or classifier.n_iter_.max() >= 500:
            raise RuntimeError(f'Phase {phase} did not converge.')
        if not np.array_equal(classifier.classes_, [0, 1]):
            raise RuntimeError('Expected success=1 as the positive class.')
        models[phase] = {'vectorizer': vectorizer, 'metadata_encoder': encoder, 'classifier': classifier}
        for component in ['frozen_tfidf', 'classifier'] + (['metadata_encoder'] if structured else []):
            fit_scope[f'{component}_phase_{phase}'] = rows['trial_id'].tolist()
        diagnostics.append({'phase': phase, 'training_rows': len(rows), 'text_features': text_matrix.shape[1],
            'metadata_features': matrix.shape[1]-text_matrix.shape[1], 'total_features': matrix.shape[1],
            'classifier_iterations': int(classifier.n_iter_.max()), 'converged': True,
            'year_median': encoder.year_median_ if encoder else np.nan,
            'year_min': encoder.year_min_ if encoder else np.nan, 'year_max': encoder.year_max_ if encoder else np.nan})
    return {'models': models, 'text_columns': list(TEXT_COLUMNS), 'structured': bool(structured),
            'C': float(C), 'positive_class': 1, 'reason_features_used': False}, pd.DataFrame(diagnostics), fit_scope


def predict_candidate(bundle, frame):
    validate_outcome_rows(frame, labels_required=False)
    probabilities = np.full(len(frame), np.nan)
    for phase in PHASES:
        positions = np.flatnonzero(frame['phase'].to_numpy() == phase)
        if not len(positions):
            continue
        rows = frame.iloc[positions]
        model = bundle['models'][phase]
        matrix = model['vectorizer'].transform(build_trial_text(rows))
        if model['metadata_encoder'] is not None:
            matrix = sparse.hstack([matrix, model['metadata_encoder'].transform(rows)], format='csr')
        probabilities[positions] = model['classifier'].predict_proba(matrix)[:, 1]
    if not np.isfinite(probabilities).all() or ((probabilities < 0) | (probabilities > 1)).any():
        raise ValueError('Expected finite success probabilities in [0, 1].')
    return pd.DataFrame({'ID': frame['ID'].to_numpy(), 'p_success': probabilities})


def load_candidate(directory, settings, validation, input_files):
    """Check frozen run settings, inputs, artifacts, and reproduced predictions."""
    directory = Path(directory)
    config = json.loads((directory/'config.json').read_text())
    if config['hyperparameters'] != normalized_json(settings) or config['input_files'] != input_files:
        raise ValueError('Run settings or training/validation inputs changed; use a new run ID.')
    if config['task'] != 'part2' or config['thresholds'] != dict.fromkeys(PHASES, .5):
        raise ValueError('Unexpected evaluation convention.')
    manifest = json.loads((directory/'artifact_checksums.json').read_text())
    for name, digest in manifest.items():
        if Path(name).is_absolute() or '..' in Path(name).parts or file_sha256(directory/name) != digest:
            raise ValueError(f'Immutable run artifact changed: {name}')
    bundle = joblib.load(directory/'model.joblib')
    result = evaluate_part2(validation, predict_candidate(bundle, validation), .5)
    saved = pd.read_csv(directory/'validation_predictions.csv').set_index('ID').loc[validation['ID']]
    if not np.allclose(result.predictions['p_success'], saved['p_success'], atol=1e-12, rtol=0):
        raise ValueError('Saved model predictions do not reproduce.')
    if not np.array_equal(result.predictions['prediction'], saved['prediction']):
        raise ValueError('Saved binary predictions do not reproduce.')
    return bundle, result


def validate_submission(frame, sample, part):
    """Require exact sample IDs/order/schema and the competition-specific output domain."""
    expected = ['ID', 'Predicted'] if part == 1 else ['ID', 'label'] if part == 2 else None
    if expected is None or list(frame) != expected or list(sample) != expected:
        raise ValueError('Incorrect submission columns.')
    if not frame['ID'].is_unique or frame['ID'].isna().any() or frame['ID'].tolist() != sample['ID'].tolist():
        raise ValueError('Submission IDs/order must match the supplied sample exactly.')
    if part == 2:
        scores = pd.to_numeric(frame['label'], errors='raise').to_numpy(dtype=float)
        if not np.isfinite(scores).all() or ((scores < 0) | (scores > 1)).any():
            raise ValueError('Outcome submissions require probabilities in [0, 1].')
    else:
        allowed = {'Safety','Funding','Efficacy','Recruitment','Other'}
        if frame['Predicted'].isna().any():
            raise ValueError('Reason submissions cannot contain null values.')
        for value in frame['Predicted']:
            if not isinstance(value, str) or not value.strip():
                raise ValueError('Reason submissions require at least one reason per row.')
            tokens = value.split()
            if len(tokens) != len(set(tokens)) or not set(tokens).issubset(allowed):
                raise ValueError('Invalid or repeated reason labels.')
    return {'part': part, 'rows': len(frame), 'unique_ids': frame['ID'].nunique(), 'columns': expected, 'status': 'PASS'}

```


### tools/src/midterm_rank_part2.py

SHA-256: `e219d258888871f22cc82880f22f6a3f71a44ff626534bb659f6e22760050b3f`

```python
"""Phase-shared and recency-weighted Part 2 candidate experiment.

The temporal screen selects one text model with only outcome_train labels.
Grouped OOF chooses a frozen blend weight against the current final model;
the supplied validation set is checked once after those choices are locked.
"""
from __future__ import annotations

import argparse
import json
import math
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.exceptions import ConvergenceWarning
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from threadpoolctl import threadpool_limits

from midterm_augmentation import logistic_parameters
from midterm_evaluation import PHASES, dataset_path, evaluate_part2, file_sha256
from midterm_finalization import predict_final_outcomes
from midterm_next_round import CHAR_TFIDF, validate_grouped_assignment, verify_oof
from midterm_part2_baseline import build_trial_text, fit_phase_baseline, predict_outcome_probabilities, validate_outcome_rows
from midterm_part2_improvement import validate_submission
from midterm_refinement import TFIDF, fit_field_outcomes, predict_field_outcomes

STUDY = Path('models/part2/03_phase_shared/evidence/selection')
SETTINGS = (
    {"alpha": 0.0, "half_life": None},
    {"alpha": 0.35, "half_life": None},
    {"alpha": 1.0, "half_life": None},
    {"alpha": 0.35, "half_life": 12},
    {"alpha": 0.35, "half_life": 6},
)
BLEND_WEIGHTS = (0.0, 0.25, 0.5, 1.0)
WORD_PARAMS = {**TFIDF, "max_features": 30000, "dtype": np.float32}


def write_json(path: Path, value) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(value, indent=2, allow_nan=False) + "\n")


def recency_weights(frame: pd.DataFrame, half_life: int | None, *, reference_year: int) -> np.ndarray:
    if half_life is None:
        return np.ones(len(frame), dtype=float)
    years = pd.to_numeric(frame.start_year, errors="coerce").to_numpy(dtype=float)
    observed = years[np.isfinite(years)]
    if not len(observed) or np.any(observed > reference_year):
        raise ValueError("Recency weights require known training years not later than the reference year.")
    years = np.where(np.isfinite(years), years, np.median(observed))
    weights = np.exp(-math.log(2) * (reference_year - years) / half_life)
    return weights / weights.mean()  # Same total weight keeps LR regularization comparable.


def phase_design(text_matrix, phases: pd.Series, alpha: float):
    """Shared text, optional phase-specific text interactions, and phase intercepts."""
    phase = phases.to_numpy()
    if not np.isin(phase, PHASES).all():
        raise ValueError("Only phases I, II, and III are allowed.")
    indicators = sparse.csr_matrix(np.column_stack([phase == p for p in PHASES]).astype(np.float32))
    blocks = [text_matrix]
    if alpha:
        blocks.extend(text_matrix.multiply((phase == p).astype(np.float32)[:, None]) * alpha for p in PHASES)
    blocks.append(indicators)
    return sparse.hstack(blocks, format="csr", dtype=np.float32)


def fit_phase_shared(train: pd.DataFrame, setting: dict, *, reference_year: int):
    validate_outcome_rows(train)
    vectorizer = TfidfVectorizer(**WORD_PARAMS)
    x_text = vectorizer.fit_transform(build_trial_text(train))
    x = phase_design(x_text, train.phase, setting["alpha"])
    classifier = LogisticRegression(**{**logistic_parameters(42), "C": 1.0, "max_iter": 1000})
    weights = recency_weights(train, setting["half_life"], reference_year=reference_year)
    with threadpool_limits(limits=1), warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        classifier.fit(x, train.label.to_numpy(dtype=int), sample_weight=weights)
    if any(issubclass(w.category, ConvergenceWarning) for w in caught) or classifier.n_iter_.max() >= classifier.max_iter:
        raise RuntimeError("Phase-shared LR failed to converge.")
    return {"vectorizer": vectorizer, "classifier": classifier, "setting": dict(setting),
            "reason_features_used": False, "positive_class": 1,
            "fit_row_ids": train.ID.tolist(), "fit_trial_ids": sorted(train.trial_id.unique()),
            "reference_year": reference_year, "vocabulary_size": len(vectorizer.vocabulary_)}


def predict_phase_shared(bundle: dict, frame: pd.DataFrame) -> pd.DataFrame:
    validate_outcome_rows(frame, labels_required=False)
    if bundle["reason_features_used"] is not False or bundle["positive_class"] != 1:
        raise ValueError("Unexpected model input scope or positive class.")
    text = bundle["vectorizer"].transform(build_trial_text(frame))
    x = phase_design(text, frame.phase, bundle["setting"]["alpha"])
    p = bundle["classifier"].predict_proba(x)[:, list(bundle["classifier"].classes_).index(1)]
    if not np.isfinite(p).all() or ((p < 0) | (p > 1)).any():
        raise ValueError("Invalid success probabilities.")
    return pd.DataFrame({"ID": frame.ID.to_numpy(), "p_success": p})


def blend(reference: pd.DataFrame, current: pd.DataFrame, proposed: pd.DataFrame, weight: float) -> pd.DataFrame:
    if not 0 <= weight <= 1 or not current.ID.is_unique or not proposed.ID.is_unique:
        raise ValueError("Invalid weight or duplicate prediction IDs.")
    if set(reference.ID) != set(current.ID) or set(reference.ID) != set(proposed.ID):
        raise ValueError("Both components must cover exactly the reference IDs.")
    a = current.set_index("ID").loc[reference.ID, "p_success"].to_numpy(dtype=float)
    b = proposed.set_index("ID").loc[reference.ID, "p_success"].to_numpy(dtype=float)
    if not np.isfinite(a).all() or not np.isfinite(b).all() or ((a < 0) | (a > 1)).any() or ((b < 0) | (b > 1)).any():
        raise ValueError("Invalid component probabilities.")
    return pd.DataFrame({"ID": reference.ID.to_numpy(), "p_success": (1-weight)*a + weight*b})


def temporal_screen(train: pd.DataFrame, folder: Path):
    """Past <=2009 predicts 2010-2013, matching the direction of the hidden test."""
    year = pd.to_numeric(train.start_year, errors="coerce")
    fit = train.loc[year.le(2009)].reset_index(drop=True)
    held = train.loc[year.between(2010, 2013)].reset_index(drop=True)
    if len(fit) < 1000 or len(held) < 500 or set(fit.trial_id) & set(held.trial_id):
        raise ValueError("Temporal split is too small or leaks a trial.")
    # Retrain each component of the current 25/25/50 final blend on <=2009.
    # This is the direct temporal comparator for the new pooled model.
    baseline, _, _, _ = fit_phase_baseline(fit, tfidf_parameters={**TFIDF, "max_features": 20000},
                                          classifier_parameters={"C": 1.0})
    word_scores = predict_outcome_probabilities(baseline, held)
    base_result = evaluate_part2(held, word_scores, .5)
    base_result.save(folder / "temporal_phase_word_diagnostic")
    field, _ = fit_field_outcomes(fit)
    field_scores = predict_field_outcomes(field, held)
    character, _, _, _ = fit_phase_baseline(fit, tfidf_parameters=CHAR_TFIDF,
                                            classifier_parameters={"C": 1.0})
    character_scores = predict_outcome_probabilities(character, held)
    word_blend = blend(held, word_scores, field_scores, .5)
    final_scores = blend(held, word_blend, character_scores, .5)
    final_result = evaluate_part2(held, final_scores, .5)
    final_result.save(folder / "temporal_current_final_control")
    rows = [{"setting_id": "phase_word_control", "alpha": None, "half_life": None,
             **base_result.summary},
            {"setting_id": "current_final_control", "alpha": None, "half_life": None,
             **final_result.summary}]
    phase_rows = [base_result.by_group.assign(setting_id="phase_word_control"),
                  final_result.by_group.assign(setting_id="current_final_control")]
    for i, setting in enumerate(SETTINGS):
        model = fit_phase_shared(fit, setting, reference_year=2009)
        result = evaluate_part2(held, predict_phase_shared(model, held), .5)
        result.save(folder / f"temporal_setting_{i}")
        rows.append({"setting_id": i, **setting, **result.summary})
        phase_rows.append(result.by_group.assign(setting_id=i))
        print(f"Part 2 temporal setting {i+1}/{len(SETTINGS)} complete", flush=True)
    table = pd.DataFrame(rows)
    table.to_csv(folder / "temporal_comparison.csv", index=False)
    pd.concat(phase_rows).to_csv(folder / "temporal_by_phase.csv", index=False)
    winner = min(range(len(SETTINGS)), key=lambda i: (float(table.loc[table.setting_id.eq(i), "log_loss_equal_phase"].iloc[0]), i))
    return winner, float(final_result.summary["log_loss_equal_phase"]), float(table.loc[table.setting_id.eq(winner), "log_loss_equal_phase"].iloc[0]), {
        "fit_rows": len(fit), "held_rows": len(held), "fit_trials": fit.trial_id.nunique(),
        "held_trials": held.trial_id.nunique(), "cutoff": 2009,
        "missing_year_rows_excluded": int(year.isna().sum()), "trial_overlap": 0,
    }


def grouped_oof(train: pd.DataFrame, assignment: pd.DataFrame, setting: dict, folder: Path):
    pieces = []
    fold_rows = []
    for fold in sorted(assignment.fold.unique()):
        held_ids = set(assignment.loc[assignment.fold.eq(fold), "ID"])
        fit = train.loc[~train.ID.isin(held_ids)].reset_index(drop=True)
        held = train.loc[train.ID.isin(held_ids)].reset_index(drop=True)
        if set(fit.trial_id) & set(held.trial_id):
            raise ValueError("A trial crosses the grouped OOF split.")
        model = fit_phase_shared(fit, setting, reference_year=2013)
        result = evaluate_part2(held, predict_phase_shared(model, held), .5)
        pieces.append(result.predictions.assign(fold=int(fold)))
        fold_rows.append({"fold": int(fold), **result.summary})
        print(f"Part 2 grouped OOF fold {int(fold)+1}/5 complete", flush=True)
    oof = verify_oof(train, pd.concat(pieces, ignore_index=True), assignment, part=2)
    oof.to_csv(folder / "oof_phase_shared.csv", index=False)
    pd.DataFrame(fold_rows).to_csv(folder / "oof_by_fold.csv", index=False)
    return oof


def run_study(root: Path):
    folder = root / STUDY
    folder.mkdir(parents=True, exist_ok=True)
    frozen = {name: file_sha256(root / name) for name in (
        'tools/archive/records/final_selection_nov01.json', 'models/part2/02_text_blend/model.joblib',
        'models/part1/02_kmeans_lr/submission.csv', 'models/part2/02_text_blend/submission.csv')}
    train = pd.read_csv(dataset_path(root, "part2", "train"))
    write_json(folder / "experiment_plan.json", {
        "settings": SETTINGS, "blend_weights": BLEND_WEIGHTS, "word_tfidf": {k: (str(v) if k == "dtype" else v) for k, v in WORD_PARAMS.items()},
        "temporal_cutoff": 2009, "selection_order": "temporal setting, then grouped OOF blend weight, then one official validation check",
        "reason_features_used": False, "test_labels_used": False,
    })
    winner, temporal_control, temporal_winner, split = temporal_screen(train, folder)
    write_json(folder / "temporal_selection_locked.json", {"winner": winner, "setting": SETTINGS[winner],
        "current_final_control_loss": temporal_control, "winner_loss": temporal_winner, "split": split,
        "validation_labels_used": False, "test_labels_used": False})
    assignment = pd.read_csv(root / 'data/audit/protocol/part2_train_folds.csv')
    assignment = validate_grouped_assignment(train, assignment)
    oof = grouped_oof(train, assignment, SETTINGS[winner], folder)
    current_saved = pd.read_csv(root / "refinement/oof_weight_char/part2/oof_current_plus_character.csv")
    current_oof = verify_oof(train, current_saved, assignment, part=2)
    current_scores = current_oof[["ID", "p_success"]]
    proposed_scores = oof[["ID", "p_success"]]
    rows = []
    for weight in BLEND_WEIGHTS:
        result = evaluate_part2(train, blend(train, current_scores, proposed_scores, weight), .5)
        rows.append({"weight": weight, **result.summary})
        result.by_group.assign(weight=weight).to_csv(folder / f"oof_by_phase_weight_{str(weight).replace('.', 'p')}.csv", index=False)
    table = pd.DataFrame(rows)
    table.to_csv(folder / "oof_blend_comparison.csv", index=False)
    chosen_weight = min(BLEND_WEIGHTS, key=lambda w: (float(table.loc[table.weight.eq(w), "log_loss_equal_phase"].iloc[0]), BLEND_WEIGHTS.index(w)))
    write_json(folder / "selection_locked_before_validation.json", {
        "setting_id": winner, "setting": SETTINGS[winner], "blend_weight": chosen_weight,
        "selection_metric": "equal-phase grouped OOF log loss", "validation_labels_used": False,
        "test_labels_used": False, "fit_preprocessing_inside_each_fold": True})
    # Check the exact OOF-selected blend on the historical future holdout.
    temporal_reference = pd.read_csv(folder / "temporal_current_final_control/validation_predictions.csv")
    temporal_final = temporal_reference[["ID", "p_success"]]
    temporal_shared = pd.read_csv(folder / f"temporal_setting_{winner}/validation_predictions.csv")[["ID", "p_success"]]
    temporal_blend = evaluate_part2(temporal_reference,
        blend(temporal_reference, temporal_final, temporal_shared, chosen_weight), .5)
    temporal_blend.save(folder / "temporal_selected_blend")
    temporal_blend_loss = float(temporal_blend.summary["log_loss_equal_phase"])
    full = fit_phase_shared(train, SETTINGS[winner], reference_year=2013)
    joblib.dump(full, folder / "selected_phase_shared_model.joblib", compress=3)
    validation = pd.read_csv(dataset_path(root, "part2", "val"))
    final = joblib.load(root / 'models/part2/02_text_blend/model.joblib')
    current_val = predict_final_outcomes(final, validation)
    saved_val = pd.read_csv(root / 'models/part2/02_text_blend/evidence/selection/selected_validation/validation_predictions.csv')
    error = np.max(np.abs(current_val.set_index("ID").loc[validation.ID, "p_success"].to_numpy() -
                          saved_val.set_index("ID").loc[validation.ID, "p_success"].to_numpy()))
    if error > 1e-12:
        raise ValueError("Current final model does not reproduce saved validation probabilities.")
    shared_val = predict_phase_shared(full, validation)
    control = evaluate_part2(validation, current_val, .5)
    candidate = evaluate_part2(validation, blend(validation, current_val, shared_val, chosen_weight), .5)
    control.save(folder / "validation_control")
    candidate.save(folder / "validation_candidate")
    candidate.by_group.assign(setting="candidate").to_csv(folder / "validation_candidate_by_phase.csv", index=False)
    oof_control = float(table.loc[table.weight.eq(0), "log_loss_equal_phase"].iloc[0])
    oof_candidate = float(table.loc[table.weight.eq(chosen_weight), "log_loss_equal_phase"].iloc[0])
    val_control = float(control.summary["log_loss_equal_phase"])
    val_candidate = float(candidate.summary["log_loss_equal_phase"])
    passes = bool(chosen_weight > 0 and temporal_blend_loss < temporal_control - 1e-12 and
                  oof_candidate < oof_control - 1e-12 and val_candidate < val_control - 1e-12)
    summary = {"setting_id": winner, "setting": SETTINGS[winner], "blend_weight": chosen_weight,
               "temporal_current_final_control_loss": temporal_control, "temporal_phase_shared_loss": temporal_winner,
               "temporal_selected_blend_loss": temporal_blend_loss,
               "oof_current_final_loss": oof_control, "oof_candidate_loss": oof_candidate,
               "validation_current_final_loss": val_control, "validation_candidate_loss": val_candidate,
               "validation_reproduction_max_abs_error": float(error),
               "passes_local_gate": passes, "final_replaced": False, "kaggle_uploaded": False}
    if passes:
        test = pd.read_csv(dataset_path(root, "part2", "test"))
        current_test = predict_final_outcomes(final, test)
        shared_test = predict_phase_shared(full, test)
        probabilities = blend(test, current_test, shared_test, chosen_weight)
        sample = pd.read_csv(dataset_path(root, "part2", "test").parent / "part2_sample_submission.csv")
        submission = probabilities.rename(columns={"p_success": "label"})
        submission.to_csv(folder / "part2_phase_shared_candidate.csv", index=False)
        validate_submission(submission, sample, 2)
        summary["candidate_submission"] = str((folder / "part2_phase_shared_candidate.csv").relative_to(root))
    write_json(folder / "summary.json", summary)
    for name, digest in frozen.items():
        if file_sha256(root / name) != digest:
            raise RuntimeError(f"A locked final artifact changed: {name}")
    print(json.dumps(summary, indent=2), flush=True)
    return summary


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--project-root", type=Path, default=Path(__file__).resolve().parents[2])
    args = parser.parse_args()
    run_study(args.project_root.resolve())


if __name__ == "__main__":
    raise SystemExit("Historical study runner archived. Use python tools/src/train_core.py --help for the six fixed recipes.")

```


### tools/src/midterm_refinement.py

SHA-256: `b64e37af5e5e996cc5ea910976b5bcecea09d62a9a98f2544438905d9546efa0`

```python
"""Bounded follow-up experiments after the first public leaderboard submission."""
from __future__ import annotations

from pathlib import Path
import json
import warnings
import joblib
import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.cluster import KMeans
from sklearn.exceptions import ConvergenceWarning
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from threadpoolctl import threadpool_limits

from midterm_augmentation import estimate_cluster_reason_rates,logistic_parameters,observed_label_mask,predict_reason_probabilities
from midterm_evaluation import REASONS,PHASES,dataset_path,evaluate_part1,evaluate_part2,file_sha256
from midterm_part1_finalization import binary_decisions,build_reason_lookup,build_submission
from midterm_part2_baseline import TEXT_COLUMNS,analyze_failure_reasons,predict_outcome_probabilities
from midterm_part2_improvement import validate_submission

EMBEDDINGS=[f'emb_{i}' for i in range(150)]
P1_SETTINGS=[{'method':m,'clusters':k,'C':c} for m,k in
             [('supervised',0),('hard',10),('soft',10),('soft',30)] for c in [.1,1.]]
TFIDF={'ngram_range':(1,2),'min_df':3,'lowercase':True,'strip_accents':'unicode',
       'sublinear_tf':True,'smooth_idf':True,'use_idf':True,'norm':'l2','stop_words':None}


def write_json(path,value):
    Path(path).write_text(json.dumps(value,indent=2,allow_nan=False)+'\n')


def setting_id(setting):
    return f'p1ref_{setting["method"]}_k{setting["clusters"]}_c{str(setting["C"]).replace(".","")}_v1'


def fit_binary_targets(gold_x,gold_y,pseudo_x,pseudo_q,*,C,pseudo_weight=.1):
    """Soft targets use two weighted binary rows; their total weight stays fixed.

    For q in [0,1], weighted positive and negative rows contribute exactly
    -w*(q*log(p)+(1-q)*log(1-p)) to the logistic loss.
    """
    models=[]
    for j in range(len(REASONS)):
        if pseudo_q is None:
            x=gold_x;y=gold_y[:,j];weights=np.ones(len(y))
        else:
            q=pseudo_q[:,j]
            if not np.isfinite(q).all() or ((q<0)|(q>1)).any():raise ValueError('Invalid pseudo-target probabilities.')
            x=np.vstack([gold_x,pseudo_x,pseudo_x])
            y=np.concatenate([gold_y[:,j],np.ones(len(q)),np.zeros(len(q))])
            weights=np.concatenate([np.ones(len(gold_y)),pseudo_weight*q,pseudo_weight*(1-q)])
        model=LogisticRegression(**{**logistic_parameters(42),'C':C})
        with threadpool_limits(limits=1),warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter('always');model.fit(x,y,sample_weight=weights)
        if any(issubclass(w.category,ConvergenceWarning) for w in caught) or model.n_iter_.max()>=500:
            raise RuntimeError('A refinement classifier failed to converge.')
        models.append(model)
    return models


def prepare_reason_context(frame,clusters,seed=42):
    """Exclude held-out rows before fitting either scaling or clustering."""
    observed=observed_label_mask(frame[list(REASONS)].to_numpy())
    scaler=StandardScaler().fit(frame[EMBEDDINGS]);x=scaler.transform(frame[EMBEDDINGS])
    context={'scaler':scaler,'x':x,'observed':observed,'fit_ids':frame.ID.tolist(),
             'seed_ids':frame.loc[observed,'ID'].tolist()}
    if clusters:
        clusterer=KMeans(n_clusters=clusters,init='k-means++',n_init=20,max_iter=500,tol=1e-4,algorithm='lloyd',random_state=seed)
        with threadpool_limits(limits=1):clusterer.fit(x)
        memberships=np.eye(clusters)[clusterer.labels_]
        mapping=estimate_cluster_reason_rates(memberships,frame[list(REASONS)].to_numpy(),2.)
        context.update(clusterer=clusterer,mapping=mapping,pseudo_q=(memberships@mapping['rates'])[~observed])
    return context


def fit_reason_setting(frame,context,setting):
    observed=context['observed'];x=context['x']
    q=context.get('pseudo_q')
    if setting['method']=='hard':q=(q>=.5).astype(float)
    if setting['method']=='supervised':q=None
    models=fit_binary_targets(x[observed],frame.loc[observed,list(REASONS)].to_numpy(dtype=int),x[~observed],q,C=setting['C'])
    return {'setting':setting,'scaler':context['scaler'],'models':models,
            'threshold':.35,'empty_set_policy':'argmax','fit_ids':context['fit_ids'],'seed_ids':context['seed_ids'],
            'clusterer':context.get('clusterer'),'mapping':context.get('mapping'),'pseudo_q':q}


def predict_reasons(bundle,frame):
    scores=predict_reason_probabilities(bundle['models'],bundle['scaler'].transform(frame[EMBEDDINGS]))
    result=pd.DataFrame(scores,columns=REASONS);result.insert(0,'ID',frame.ID.to_numpy());return result


def save_run(directory,config,bundle,evaluation,extras=None):
    directory=Path(directory)
    if directory.exists():
        if json.loads((directory/'config.json').read_text())!=config:raise ValueError('Existing refinement run has different settings.')
        for name,digest in json.loads((directory/'checksums.json').read_text()).items():
            if file_sha256(directory/name)!=digest:raise ValueError(f'Changed immutable artifact: {name}')
        return
    directory.mkdir(parents=True)
    write_json(directory/'config.json',config);evaluation.save(directory)
    joblib.dump(bundle,directory/'model.joblib',compress=3)
    if extras is not None:write_json(directory/'fit_metadata.json',extras)
    write_json(directory/'checksums.json',{p.name:file_sha256(p) for p in directory.iterdir() if p.is_file()})


def fit_field_outcomes(train):
    """Give criteria, disease and drug text separate training-fitted L2 TF-IDF blocks."""
    models={};diagnostics=[]
    for phase in PHASES:
        rows=train.loc[train.phase.eq(phase)];vectors={};blocks=[]
        for field,limit in zip(TEXT_COLUMNS,[20000,5000,5000]):
            vector=TfidfVectorizer(**TFIDF,max_features=limit)
            matrix=vector.fit_transform(rows[field].fillna('').astype(str))
            vectors[field]=vector;blocks.append(matrix)
        x=sparse.hstack(blocks,format='csr')/np.sqrt(3.)
        classifier=LogisticRegression(**logistic_parameters(42))
        with threadpool_limits(limits=1),warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter('always');classifier.fit(x,rows.label.to_numpy(dtype=int))
        if any(issubclass(w.category,ConvergenceWarning) for w in caught) or classifier.n_iter_.max()>=500:
            raise RuntimeError('Field-text classifier did not converge.')
        models[phase]={'vectorizers':vectors,'classifier':classifier,'fit_trial_ids':rows.trial_id.tolist()}
        diagnostics.append({'phase':phase,'iterations':int(classifier.n_iter_.max()),'rows':len(rows),
                            **{f'{field}_features':len(v.vocabulary_) for field,v in vectors.items()}})
    return {'models':models,'reason_features_used':False,'scale':float(1/np.sqrt(3.))},pd.DataFrame(diagnostics)


def predict_field_outcomes(bundle,frame):
    scores=np.full(len(frame),np.nan)
    for phase in PHASES:
        positions=np.flatnonzero(frame.phase.to_numpy()==phase)
        if not len(positions):continue
        rows=frame.iloc[positions];model=bundle['models'][phase]
        blocks=[model['vectorizers'][field].transform(rows[field].fillna('').astype(str)) for field in TEXT_COLUMNS]
        x=sparse.hstack(blocks,format='csr')*bundle['scale']
        scores[positions]=model['classifier'].predict_proba(x)[:,1]
    if not np.isfinite(scores).all():raise ValueError('Missing phase-routed predictions.')
    return pd.DataFrame({'ID':frame.ID.to_numpy(),'p_success':scores})


def run_refinement(root):
    root=Path(root);folder=root/'refinement';folder.mkdir(exist_ok=True)
    inputs={f'{task}_{split}':file_sha256(dataset_path(root,task,split)) for task in ['part1','part2'] for split in ['train','val']}
    plan={'seed':42,'part1_settings':P1_SETTINGS,'part1_cv_folds':4,'part1_threshold':.35,
        'part1_empty_policy':'argmax','pseudo_weight':.1,'smoothing_strength':2.,
        'cv_strategy':'StratifiedKFold on first-positive reason; held-out rows excluded from all fitted transforms and seeds',
        'official_validation_policy':'Evaluate one supervised and one augmented winner selected by training CV',
        'part2_settings':['baseline','separate_field_tfidf_lr_c1','equal_probability_blend'],
        'part2_field_weights':[float(1/np.sqrt(3.))]*3,'part2_threshold':.5,
        'selection_uses_public_scores':False,'original_final_outputs_modified':False,'input_sha256':inputs}
    pp=folder/'experiment_plan.json'
    if pp.exists():
        if json.loads(pp.read_text())!=plan:raise ValueError('Refinement plan changed; use a new study folder.')
    else:write_json(pp,plan)
    if (folder/'completed_checksums.json').exists():
        for name,digest in json.loads((folder/'completed_checksums.json').read_text()).items():
            if file_sha256(folder/name)!=digest:raise ValueError(f'Changed completed study output: {name}')
        return json.loads((folder/'summary.json').read_text())
    p1=folder/'part1';p2=folder/'part2';submissions=folder/'submissions'
    for path in [p1,p2,submissions]:path.mkdir(exist_ok=True)
    train=pd.read_csv(dataset_path(root,'part1','train'));mask=observed_label_mask(train[list(REASONS)].to_numpy())
    observed_positions=np.flatnonzero(mask);strata=train.loc[mask,list(REASONS)].to_numpy().argmax(axis=1)
    if min(np.bincount(strata))<4:raise ValueError('First-positive reason stratification requires four examples per stratum.')
    cv=StratifiedKFold(n_splits=4,shuffle=True,random_state=42)
    fold_rows=[];fold_provenance=[];oof={setting_id(s):[] for s in P1_SETTINGS}
    for fold,(seed_local,hold_local) in enumerate(cv.split(observed_positions,strata)):
        held=observed_positions[hold_local];fit=train.drop(index=train.index[held]).reset_index(drop=True)
        reference=train.iloc[held].copy()
        assert set(fit.ID).isdisjoint(reference.ID)
        contexts={k:prepare_reason_context(fit,k) for k in [0,10,30]}
        fold_provenance.append({'fold':fold,'fit_ids':fit.ID.tolist(),'held_out_ids':reference.ID.tolist(),
            'label_seed_ids':fit.loc[observed_label_mask(fit[list(REASONS)].to_numpy()),'ID'].tolist()})
        for setting in P1_SETTINGS:
            bundle=fit_reason_setting(fit,contexts[setting['clusters']],setting)
            probabilities=predict_reasons(bundle,reference)
            evaluation=evaluate_part1(reference,probabilities,.35,empty_set_policy='argmax')
            fold_rows.append({'setting_id':setting_id(setting),'fold':fold,**setting,**evaluation.summary})
            oof[setting_id(setting)].append(evaluation.predictions.assign(fold=fold))
        print(f'Completed training-only reason CV fold {fold+1}/4',flush=True)
    folds=pd.DataFrame(fold_rows);folds.to_csv(p1/'cv_fold_metrics.csv',index=False)
    table=folds.groupby(['setting_id','method','clusters','C'],as_index=False).agg(
        mean_f1_samples=('f1_samples','mean'),std_f1_samples=('f1_samples','std'),
        mean_f1_macro=('f1_macro','mean'),cv_rows=('n_rows','sum'))
    table=table.sort_values(['mean_f1_samples','setting_id'],ascending=[False,True]).reset_index(drop=True)
    table.to_csv(p1/'cv_comparison.csv',index=False);write_json(p1/'cv_fit_scope.json',fold_provenance)
    for key,frames in oof.items():pd.concat(frames).to_csv(p1/f'oof_{key}.csv',index=False)
    supervised_id=table.loc[table.method.eq('supervised'),'setting_id'].iloc[0]
    augmented_id=table.loc[table.method.ne('supervised'),'setting_id'].iloc[0]
    write_json(p1/'cv_selection_locked.json',{'supervised_id':supervised_id,'augmented_id':augmented_id,
        'metric':'mean_fold_sample_f1','validation_labels_used':False,'test_used':False})
    validation=pd.read_csv(dataset_path(root,'part1','val'))
    current_probs=pd.read_csv(root/'models/part1/02_kmeans_lr/evidence/selected/selected_validation/validation_predictions.csv')
    current_scores=current_probs[['ID',*[f'prob_{reason}' for reason in REASONS]]].rename(columns={f'prob_{reason}':reason for reason in REASONS})
    current_eval=evaluate_part1(validation,current_scores,.35,empty_set_policy='argmax')
    models={};evaluations={'current_submission':current_eval};summary_rows=[{'setting_id':'current_submission',**current_eval.summary}]
    for key in [supervised_id,augmented_id]:
        setting=next(s for s in P1_SETTINGS if setting_id(s)==key)
        context=prepare_reason_context(train,setting['clusters']);bundle=fit_reason_setting(train,context,setting)
        probabilities=predict_reasons(bundle,validation);evaluation=evaluate_part1(validation,probabilities,.35,empty_set_policy='argmax')
        config={'task':'part1','setting':setting,'threshold':.35,'decision_policy':'argmax_if_empty','selection':'training_CV_winner',
                'input_sha256':{k:v for k,v in inputs.items() if k.startswith('part1_')}}
        save_run(p1/'runs'/key,config,bundle,evaluation,{'fit_ids':bundle['fit_ids'],'seed_ids':bundle['seed_ids'],
            'kmeans_iterations':int(context['clusterer'].n_iter_) if setting['clusters'] else None})
        models[key]=bundle;evaluations[key]=evaluation;summary_rows.append({'setting_id':key,**evaluation.summary})
        if setting['clusters']:
            q=context['pseudo_q'];rows=train.loc[~mask,['ID']].copy()
            for j,reason in enumerate(REASONS):rows[f'cluster_probability_{reason}']=q[:,j]
            rows.to_csv(p1/'augmented_cluster_probabilities.csv',index=False)
            target_summary=pd.DataFrame({'reason':REASONS,'masked_rows':len(q),'mean_cluster_probability':q.mean(axis=0),
                'hard_positive_rows_at_05':(q>=.5).sum(axis=0),'effective_positive_pseudo_weight':(.1*q).sum(axis=0)})
            target_summary.to_csv(p1/'pseudo_target_summary.csv',index=False)
    validation_comparison=pd.DataFrame(summary_rows);validation_comparison.to_csv(p1/'validation_comparison.csv',index=False)
    pd.concat([e.by_group.assign(setting_id=k) for k,e in evaluations.items()]).to_csv(p1/'validation_by_reason.csv',index=False)
    candidate_eval=evaluations[augmented_id]
    better=candidate_eval.summary['f1_samples']>current_eval.summary['f1_samples']+1e-12
    chosen_id=augmented_id if better else 'current_submission'
    p1_selection={'chosen_candidate':chosen_id,'augmented_cv_winner':augmented_id,'supervised_cv_winner':supervised_id,
        'improved_official_validation':better,'source_current_score':current_eval.summary['f1_samples'],
        'augmented_candidate_score':candidate_eval.summary['f1_samples'],'test_used_for_selection':False}
    write_json(p1/'submission_selection_locked.json',p1_selection)
    # Paired resampling describes validation uncertainty, without a significance claim.
    def row_f1(e):
        y=e.predictions[[f'true_{v}' for v in REASONS]].to_numpy();d=e.predictions[[f'pred_{v}' for v in REASONS]].to_numpy()
        return 2*(y*d).sum(axis=1)/(y.sum(axis=1)+d.sum(axis=1))
    delta=row_f1(candidate_eval)-row_f1(current_eval)
    rng=np.random.default_rng(42);boot=delta[rng.integers(0,len(delta),size=(5000,len(delta)))].mean(axis=1)
    write_json(p1/'paired_validation_uncertainty.json',{'delta_sample_f1':float(delta.mean()),
        'descriptive_95_percentile_interval':np.quantile(boot,[.025,.975]).tolist(),
        'warning':'Does not account for previous validation selection or nonrandom labeled seeds.'})
    if better:
        test=pd.read_csv(dataset_path(root,'part1','test'));bundle=models[augmented_id]
        probs=predict_reasons(bundle,test);decisions=binary_decisions(probs[list(REASONS)],dict.fromkeys(REASONS,.35),empty_set_policy='argmax')
        pred=pd.DataFrame(decisions,columns=REASONS);pred.insert(0,'ID',test.ID.to_numpy())
        sample=pd.read_csv(dataset_path(root,'part1','test').parent/'part1_sample_submission.csv')
        submission=build_submission(sample,pred);probs.to_csv(submissions/'part1_candidate_probabilities.csv',index=False)
        submission.to_csv(submissions/'part1_candidate.csv',index=False);validate_submission(pd.read_csv(submissions/'part1_candidate.csv'),sample,1)
        frames={'train':train,'val':validation,'test':test}
        lookup=build_reason_lookup(frames,{s:predict_reasons(bundle,f)[list(REASONS)].to_numpy() for s,f in frames.items()},
            dict.fromkeys(REASONS,.35),augmented_id,empty_set_policy='argmax')
        lookup.to_csv(p1/'reason_lookup_candidate.csv',index=False)
    # Field-balanced outcome model and an a priori equal-probability blend.
    outcome_train=pd.read_csv(dataset_path(root,'part2','train'));outcome_val=pd.read_csv(dataset_path(root,'part2','val'))
    base=joblib.load(root/'models/part2/01_tfidf_lr/model.joblib')
    base_probs=predict_outcome_probabilities(base,outcome_val)
    field,diagnostics=fit_field_outcomes(outcome_train);field_probs=predict_field_outcomes(field,outcome_val)
    blend_probs=base_probs.copy();blend_probs['p_success']=.5*base_probs.p_success+.5*field_probs.p_success
    p2_probabilities={'baseline':base_probs,'field_tfidf_lr_c1':field_probs,'blend_equal':blend_probs}
    p2_evaluations={k:evaluate_part2(outcome_val,v,.5) for k,v in p2_probabilities.items()}
    save_run(p2/'runs/field_tfidf_lr_c1_v1',{'task':'part2','C':1.,'fields':list(TEXT_COLUMNS),'field_scale':float(1/np.sqrt(3.)),
        'tfidf':{**TFIDF,'ngram_range':[1,2]},'max_features_by_field':[20000,5000,5000],
        'input_sha256':{k:v for k,v in inputs.items() if k.startswith('part2_')}},field,p2_evaluations['field_tfidf_lr_c1'])
    diagnostics.to_csv(p2/'field_diagnostics.csv',index=False)
    p2_comparison=pd.DataFrame([{'setting_id':k,**v.summary} for k,v in p2_evaluations.items()]).sort_values('log_loss_equal_phase')
    p2_comparison.to_csv(p2/'validation_comparison.csv',index=False)
    pd.concat([v.by_group.assign(setting_id=k) for k,v in p2_evaluations.items()]).to_csv(p2/'validation_by_phase.csv',index=False)
    for key,result in p2_evaluations.items():result.save(p2/key)
    best_p2=p2_comparison.iloc[0].setting_id;p2_better=best_p2!='baseline'
    write_json(p2/'submission_selection_locked.json',{'chosen_candidate':best_p2,'metric':'log_loss_equal_phase',
        'score':p2_evaluations[best_p2].summary['log_loss_equal_phase'],'test_used':False})
    if p2_better:
        test=pd.read_csv(dataset_path(root,'part2','test'));probs=predict_field_outcomes(field,test)
        if best_p2=='blend_equal':probs['p_success']=.5*probs.p_success+.5*predict_outcome_probabilities(base,test).p_success
        sample=pd.read_csv(dataset_path(root,'part2','test').parent/'part2_sample_submission.csv')
        submission=sample[['ID']].merge(probs,on='ID',validate='one_to_one',sort=False).rename(columns={'p_success':'label'})
        validate_submission(submission,sample,2);submission.to_csv(submissions/'part2_candidate.csv',index=False)
    frozen_lookup=pd.read_csv(root/'models/part1/02_kmeans_lr/evidence/selected/reason_lookup.csv')
    before=analyze_failure_reasons(p2_evaluations['baseline'].predictions,frozen_lookup)
    after=analyze_failure_reasons(p2_evaluations[best_p2].predictions,frozen_lookup)
    rates=before['reason_rates'].merge(after['reason_rates'],on=['phase','reason_group'],suffixes=('_baseline','_candidate'))
    rates.to_csv(p2/'reason_error_comparison.csv',index=False)
    summary={'part1':p1_selection,'part1_validation':validation_comparison.to_dict('records'),
        'part2':{'chosen_candidate':best_p2,'improved_official_validation':p2_better,
            'baseline_log_loss':p2_evaluations['baseline'].summary['log_loss_equal_phase'],
            'candidate_log_loss':p2_evaluations[best_p2].summary['log_loss_equal_phase']},
        'new_public_scores_known':False,'public_scores_used_to_select_settings':False,
        'original_final_outputs_modified':False,'model_source_sha256':file_sha256(Path(__file__))}
    write_json(folder/'summary.json',summary)
    checksums={str(p.relative_to(folder)):file_sha256(p) for p in folder.rglob('*') if p.is_file()}
    write_json(folder/'completed_checksums.json',checksums)
    return summary


if __name__ == "__main__":
    raise SystemExit("Historical study runner archived. Use python tools/src/train_core.py --help for the six fixed recipes.")

```


### tools/src/train_core.py

SHA-256: `b515735e06d3765b51d59d255f50d0e9c91fa63ddd76f05fdb3b012ca8fc1210`

```python
"""Train one of six fixed core recipes into a new output directory.

Examples: python tools/src/train_core.py part1_ridge --output models/retrained_ridge
          python tools/src/train_core.py part2_shared --output models/retrained_shared

This is an explicit training entry point, with no search or automatic promotion.
Part 1 LR/K-Means use the supplied training split; the selected ridge uses all
235 visible labels plus the 1,014 unlabeled training embeddings for geometry.
Part 2 uses outcome_train only. Use reproduce.py to verify the preserved models;
retraining can vary across library versions and is not a byte-for-byte restore.
"""
from __future__ import annotations

import argparse
import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from threadpoolctl import threadpool_limits

from midterm_augmentation import (cluster_memberships, estimate_cluster_reason_rates,
                                  observed_label_mask, recover_labels)
from midterm_evaluation import PHASES, REASONS, dataset_path, file_sha256
from midterm_finalization import predict_final_outcomes
from midterm_next_round import CHAR_TFIDF
from midterm_part1_finalization import fit_classifiers, predict_bundle
from midterm_part1_local_consensus import Data, EMB, fit_calibrated, near_groups
from midterm_part2_baseline import fit_phase_baseline, predict_outcome_probabilities
from midterm_rank_part2 import blend, fit_phase_shared, predict_phase_shared
from midterm_refinement import TFIDF, fit_field_outcomes

FAMILIES = ("part1_supervised", "part1_kmeans", "part1_ridge",
            "part2_baseline", "part2_text_blend", "part2_shared")
RIDGE_CONFIG = dict(geometry="global", family="ridge", bandwidth=.5, strength=.1)
SHARED_SETTING = dict(alpha=.35, half_life=6)
RECIPES = {
    "part1_supervised": dict(scaler_scope="all training embeddings", classifier_scope="observed training labels",
                            C=1., threshold=.5, seed=42),
    "part1_kmeans": dict(clusters=10, smoothing=2., pseudo_threshold=.5, pseudo_weight=.1,
                        C=.1, threshold=.35, seed=42),
    "part1_ridge": dict(**RIDGE_CONFIG, threshold=.5, calibration_C=1., calibration_folds=3,
                       calibration_seed=942, scope="train plus validation; test excluded"),
    "part2_baseline": dict(C=1., max_word_features=20000, seed=42, scope="outcome_train"),
    "part2_text_blend": dict(word=.25, field=.25, character=.5, seed=42, scope="outcome_train"),
    "part2_shared": dict(**SHARED_SETTING, reference_year=2013, text_blend_weight=.5,
                        shared_weight=.5, seed=42, scope="outcome_train"),
}


def fit_part1_supervised(train):
    """Train-fold scaler over all rows; C=1 classifiers over observed labels only."""
    labels = train[list(REASONS)].to_numpy(dtype=int)
    observed = observed_label_mask(labels)
    scaler = StandardScaler().fit(train[EMB])
    x = scaler.transform(train[EMB])
    classifiers, _ = fit_classifiers(x[observed], labels[observed],
                                     np.ones(observed.sum()), C=1., seed=42)
    return dict(scaler=scaler, feature_columns=EMB, reason_columns=list(REASONS),
                classifiers=classifiers, pathway_encoder=None,
                thresholds=dict.fromkeys(REASONS, .5))


def fit_part1_kmeans(train):
    """K=10, smoothed hard pseudo-labels at .5, weight .1, C=.1, output .35."""
    labels = train[list(REASONS)].to_numpy(dtype=int)
    observed_label_mask(labels)
    scaler = StandardScaler().fit(train[EMB])
    x = scaler.transform(train[EMB])
    clusterer = KMeans(n_clusters=10, n_init=20, max_iter=500, tol=1e-4,
                       random_state=42, algorithm="lloyd").fit(x)
    memberships = cluster_memberships(clusterer, x, "kmeans")
    mapping = estimate_cluster_reason_rates(memberships, labels, smoothing_strength=2.)
    recovered = recover_labels(labels, memberships, mapping, threshold=.5, pseudo_weight=.1)
    classifiers, _ = fit_classifiers(x, recovered["labels"], recovered["sample_weight"],
                                     C=.1, seed=42)
    return dict(scaler=scaler, feature_columns=EMB, reason_columns=list(REASONS),
                classifiers=classifiers, pathway_encoder=None, clusterer=clusterer,
                reason_mapping=mapping, method="kmeans", thresholds=dict.fromkeys(REASONS, .35))


def ridge_data(train, validation):
    """Build the original pooled scope without loading any test features."""
    train, validation = train.copy(), validation.copy()
    observed = observed_label_mask(train[list(REASONS)].to_numpy())
    if not validation[list(REASONS)].isin([0, 1]).all(axis=None):
        raise ValueError("Ridge final fitting requires observed validation labels.")
    train["fit_order"] = np.arange(len(train))
    validation["fit_order"] = len(train) + np.arange(len(validation))
    frame = pd.concat([train.loc[observed].assign(source="train"),
                       validation.assign(source="validation"),
                       train.loc[~observed].assign(source="unlabeled")], ignore_index=True)
    if not frame.ID.is_unique:
        raise ValueError("Part 1 train and validation IDs must be unique and disjoint.")
    x, y = frame[EMB].to_numpy(), frame[list(REASONS)].to_numpy(dtype=int)
    n_gold = int(observed.sum()) + len(validation)
    return Data(frame, x, y, near_groups(x), np.arange(n_gold), np.arange(n_gold, len(frame)))


def fit_part1_ridge(train, validation):
    """Fixed selected kernel with three group-separated calibration folds, seed 942."""
    data = ridge_data(train, validation)
    scope = data.scope(data.gold, np.arange(len(data.x)))
    return fit_calibrated(scope, dict(RIDGE_CONFIG), seed=942)


def fit_part2_baseline(train):
    return fit_phase_baseline(train, tfidf_parameters={**TFIDF, "max_features": 20000},
                              classifier_parameters={"C": 1.}, seed=42)[0]


def fit_part2_text_blend(train):
    """The retained 25/25/50 concatenated word / separate field / character recipe."""
    baseline = fit_part2_baseline(train)
    field, _ = fit_field_outcomes(train)
    character = fit_phase_baseline(train, tfidf_parameters=CHAR_TFIDF,
                                   classifier_parameters={"C": 1.}, seed=42)[0]
    common = dict(weights=[.5, .5], positive_class=1, reason_features_used=False,
                  thresholds=dict.fromkeys(PHASES, .5))
    words = dict(kind="equal_probability_blend", baseline=baseline, field=field, **common)
    return dict(kind="word_character_probability_blend", word_blend=words,
                character=character, flat_weights={"concatenated_word": .25,
                    "separate_field_word": .25, "character": .5}, **common)


def fit_part2_shared(train):
    """The retained 50/50 text-blend / phase-shared recency-weighted recipe."""
    return dict(text_blend=fit_part2_text_blend(train),
                shared=fit_phase_shared(train, dict(SHARED_SETTING), reference_year=2013),
                shared_weight=.5)


def train_family(root: Path, family: str):
    """Return a self-contained fitted bundle; never save or select a production model."""
    if family not in FAMILIES:
        raise ValueError(f"Unknown family: {family}")
    task = "part1" if family.startswith("part1_") else "part2"
    train_path = dataset_path(root, task, "train")
    train = pd.read_csv(train_path)
    inputs = {str(train_path.relative_to(root)): file_sha256(train_path)}
    with threadpool_limits(limits=1):
        if family == "part1_ridge":
            val_path = dataset_path(root, "part1", "val")
            validation = pd.read_csv(val_path)
            inputs[str(val_path.relative_to(root))] = file_sha256(val_path)
            model = fit_part1_ridge(train, validation)
            label_ids = train.loc[observed_label_mask(train[list(REASONS)].to_numpy()), "ID"].tolist() + validation.ID.tolist()
            feature_ids = train.ID.tolist() + validation.ID.tolist()
        else:
            fitters = dict(part1_supervised=fit_part1_supervised, part1_kmeans=fit_part1_kmeans,
                           part2_baseline=fit_part2_baseline, part2_text_blend=fit_part2_text_blend,
                           part2_shared=fit_part2_shared)
            model = fitters[family](train)
            label_ids = (train.loc[observed_label_mask(train[list(REASONS)].to_numpy()), "ID"].tolist()
                         if task == "part1" else train.ID.tolist())
            feature_ids = train.ID.tolist()
    return dict(family=family, model=model, decision_threshold=.35 if family == "part1_kmeans" else .5,
                empty_set_policy="argmax" if task == "part1" else None,
                metadata=dict(recipe=dict(RECIPES[family]), input_sha256=inputs, fit_feature_ids=feature_ids,
                              fit_observed_label_ids=label_ids, seed=42,
                              calibration_seed=942 if family == "part1_ridge" else None,
                              validation_labels_used=family == "part1_ridge", test_features_used=False,
                              hyperparameter_search=False, production_selection_changed=False))


def predict_core(bundle, frame):
    """Return reason probabilities (Part 1) or ID/p_success (Part 2)."""
    family, model = bundle["family"], bundle["model"]
    if family in ("part1_supervised", "part1_kmeans"):
        return predict_bundle(model, frame)
    if family == "part1_ridge":
        return model.predict(frame[EMB].to_numpy())
    if family == "part2_baseline":
        return predict_outcome_probabilities(model, frame)
    if family == "part2_text_blend":
        return predict_final_outcomes(model, frame)
    if family == "part2_shared":
        return blend(frame, predict_final_outcomes(model["text_blend"], frame),
                     predict_phase_shared(model["shared"], frame), model["shared_weight"])
    raise ValueError(f"Unknown family: {family}")


def main(argv=None):
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("family", choices=FAMILIES)
    parser.add_argument("--project-root", type=Path, default=Path(__file__).resolve().parents[2])
    parser.add_argument("--output", type=Path, required=True, help="New directory; existing directories are rejected.")
    args = parser.parse_args(argv)
    output = args.output.resolve()
    output.mkdir(parents=True, exist_ok=False)
    fitted = train_family(args.project_root.resolve(), args.family)
    joblib.dump(fitted, output / "model.joblib", compress=3)
    manifest = dict(family=args.family, decision_threshold=fitted["decision_threshold"],
                    empty_set_policy=fitted["empty_set_policy"], **fitted["metadata"],
                    model_sha256=file_sha256(output / "model.joblib"))
    (output / "training_manifest.json").write_text(json.dumps(manifest, indent=2) + "\n")
    print(json.dumps(dict(family=args.family, output=str(output), production_selection_changed=False)))


if __name__ == "__main__":
    main()

```
